In [1]:
# ============================================================
# CELL 1 — CLEAN SAC TRAINING SETUP
# ============================================================

import os
import sys
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from collections import deque

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# Project path
# ------------------------------------------------------------

PROJECT_DIR = os.path.dirname(os.getcwd())

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("=" * 60)
print("3-RPR SAC FINAL TRAINING")
print("=" * 60)

print("PyTorch :", torch.__version__)
print("Device  :", DEVICE)

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("CUDA    :", torch.version.cuda)

print("NumPy   :", np.__version__)
print("Project :", PROJECT_DIR)

3-RPR SAC FINAL TRAINING
PyTorch : 2.14.0+cu130
Device  : cuda
GPU     : NVIDIA GeForce RTX 4060 Laptop GPU
CUDA    : 13.0
NumPy   : 2.3.3
Project : c:\Users\madha\OneDrive\Desktop\project_1


In [2]:
# ============================================================
# CELL 2 — ACTUAL 3-RPR DYNAMICS
# ============================================================

# Make sure project root is available
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Import the actual dynamics
from dynamics.dynamics import step

print("=" * 60)
print("3-RPR DYNAMICS")
print("=" * 60)

# ------------------------------------------------------------
# Known test state
# ------------------------------------------------------------

test_state = np.array(
    [0.45, 0.45, 0.10, 0.10, -0.05, 0.02],
    dtype=np.float64
)

test_forces = np.array(
    [0.5, -0.25, 0.15],
    dtype=np.float64
)

# ------------------------------------------------------------
# One simulation step
# ------------------------------------------------------------

next_state = step(
    test_state,
    test_forces,
    dt=0.01
)

next_state = np.asarray(next_state)

print("Input state :")
print(test_state)

print("\nForces [F1,F2,F3] :")
print(test_forces)

print("\nNext state :")
print(next_state)

print("\nState shape :", next_state.shape)
print("Finite      :", np.all(np.isfinite(next_state)))

assert next_state.shape == (6,)
assert np.all(np.isfinite(next_state))

print("\n✓ Dynamics test PASSED")

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
3-RPR DYNAMICS
Input state :
[ 0.45  0.45  0.1   0.1  -0.05  0.02]

Forces [F1,F2,F3] :
[ 0.5  -0.25  0.15]

Next state :
[ 0.451       0.4495      0.1002      0.10134615 -0.04987539  0.01838396]

State shape : (6,)
Finite      : True

✓ Dynamics test PASSED


In [3]:
# ============================================================
# CELL 3 — SAC CONFIGURATION
# ============================================================

# State / action dimensions
STATE_DIM = 6
DESIRED_DIM = 6
ACTION_DIM = 3

# State:
# [x, y, theta, x_dot, y_dot, theta_dot]
#
# Desired:
# [xd, yd, theta_d, 0, 0, 0]

# Physical action range
MAX_ACTION = 2.0

# Simulation
DT = 0.01

# SAC hyperparameters
GAMMA = 0.99
TAU = 0.005

BATCH_SIZE = 100
REPLAY_CAPACITY = 100_000

# Neural network
HIDDEN_DIM = 256

# Learning rates
ACTOR_LR = 1e-4
CRITIC_LR = 1e-4
ALPHA_LR = 1e-4

# SAC entropy
INITIAL_ALPHA = 0.2
TARGET_ENTROPY = -ACTION_DIM

# Training
TOTAL_STEPS = 100_000
WARMUP_STEPS = 2_000

print("=" * 60)
print("SAC CONFIGURATION")
print("=" * 60)

print("State dimension    :", STATE_DIM)
print("Desired dimension  :", DESIRED_DIM)
print("Action dimension   :", ACTION_DIM)
print("Maximum action     :", MAX_ACTION)
print("DT                 :", DT)

print("\nGamma              :", GAMMA)
print("Tau                :", TAU)
print("Batch size         :", BATCH_SIZE)
print("Replay capacity    :", REPLAY_CAPACITY)

print("\nHidden dimension    :", HIDDEN_DIM)
print("Actor LR            :", ACTOR_LR)
print("Critic LR           :", CRITIC_LR)
print("Alpha LR            :", ALPHA_LR)

print("\nInitial alpha       :", INITIAL_ALPHA)
print("Target entropy      :", TARGET_ENTROPY)

print("\nTraining steps      :", TOTAL_STEPS)
print("Warmup steps        :", WARMUP_STEPS)

print("\n✓ SAC configuration ready")

SAC CONFIGURATION
State dimension    : 6
Desired dimension  : 6
Action dimension   : 3
Maximum action     : 2.0
DT                 : 0.01

Gamma              : 0.99
Tau                : 0.005
Batch size         : 100
Replay capacity    : 100000

Hidden dimension    : 256
Actor LR            : 0.0001
Critic LR           : 0.0001
Alpha LR            : 0.0001

Initial alpha       : 0.2
Target entropy      : -3

Training steps      : 100000
Warmup steps        : 2000

✓ SAC configuration ready


In [4]:
# ============================================================
# CELL 4 — STATE NORMALIZATION
# ============================================================

# Expanded state ranges:
#
# x, y       : [0.0, 1.0]
# theta      : [-0.5, 0.5]
# velocities : [-0.2, 0.2]

STATE_CENTER = torch.tensor(
    [0.5, 0.5, 0.0, 0.0, 0.0, 0.0],
    dtype=torch.float32,
    device=DEVICE
)

STATE_SCALE = torch.tensor(
    [0.5, 0.5, 0.5, 0.2, 0.2, 0.2],
    dtype=torch.float32,
    device=DEVICE
)

DESIRED_CENTER = torch.tensor(
    [0.5, 0.5, 0.0, 0.0, 0.0, 0.0],
    dtype=torch.float32,
    device=DEVICE
)

DESIRED_SCALE = torch.tensor(
    [0.5, 0.5, 0.5, 0.2, 0.2, 0.2],
    dtype=torch.float32,
    device=DEVICE
)


def normalize_state(state):
    return (state - STATE_CENTER) / STATE_SCALE


def normalize_desired(desired_state):
    return (desired_state - DESIRED_CENTER) / DESIRED_SCALE


print("=" * 60)
print("STATE NORMALIZATION")
print("=" * 60)

print("State center :", STATE_CENTER.cpu().numpy())
print("State scale  :", STATE_SCALE.cpu().numpy())

print("\nDesired center :", DESIRED_CENTER.cpu().numpy())
print("Desired scale  :", DESIRED_SCALE.cpu().numpy())

# Quick test
test = torch.tensor(
    [[0.5, 0.5, 0.0, 0.0, 0.0, 0.0]],
    dtype=torch.float32,
    device=DEVICE
)

print("\nNormalized zero-center state:")
print(normalize_state(test).cpu().numpy())

assert torch.allclose(
    normalize_state(test),
    torch.zeros_like(test)
)

print("\n✓ Normalization test PASSED")

STATE NORMALIZATION
State center : [0.5 0.5 0.  0.  0.  0. ]
State scale  : [0.5 0.5 0.5 0.2 0.2 0.2]

Desired center : [0.5 0.5 0.  0.  0.  0. ]
Desired scale  : [0.5 0.5 0.5 0.2 0.2 0.2]

Normalized zero-center state:
[[0. 0. 0. 0. 0. 0.]]

✓ Normalization test PASSED


In [5]:
# ============================================================
# CELL 5 — STOCHASTIC SAC ACTOR
# ============================================================

class SACActor(nn.Module):

    def __init__(
        self,
        state_dim,
        desired_dim,
        action_dim,
        hidden_dim,
        max_action
    ):
        super().__init__()

        self.max_action = max_action

        # Store normalization inside the trained model
        self.register_buffer(
            "state_center",
            STATE_CENTER.clone()
        )

        self.register_buffer(
            "state_scale",
            STATE_SCALE.clone()
        )

        self.register_buffer(
            "desired_center",
            DESIRED_CENTER.clone()
        )

        self.register_buffer(
            "desired_scale",
            DESIRED_SCALE.clone()
        )

        # 6 state + 6 desired = 12 inputs
        self.net = nn.Sequential(
            nn.Linear(state_dim + desired_dim, hidden_dim),
            nn.ReLU(),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        # SAC Gaussian policy
        self.mean = nn.Linear(
            hidden_dim,
            action_dim
        )

        self.log_std = nn.Linear(
            hidden_dim,
            action_dim
        )

    def forward(self, state, desired_state):

        # Internal normalization
        state = (
            state - self.state_center
        ) / self.state_scale

        desired_state = (
            desired_state - self.desired_center
        ) / self.desired_scale

        # Combine current + desired state
        x = torch.cat(
            [state, desired_state],
            dim=-1
        )

        x = self.net(x)

        mean = self.mean(x)

        log_std = self.log_std(x)

        # Numerical stability
        log_std = torch.clamp(
            log_std,
            -20,
            2
        )

        return mean, log_std

    def sample(
        self,
        state,
        desired_state
    ):

        mean, log_std = self.forward(
            state,
            desired_state
        )

        std = log_std.exp()

        # Reparameterization trick
        normal = torch.distributions.Normal(
            mean,
            std
        )

        z = normal.rsample()

        # Squashed action
        tanh_action = torch.tanh(z)

        action = (
            self.max_action *
            tanh_action
        )

        # SAC log-probability correction
        log_prob = normal.log_prob(z)

        log_prob -= torch.log(
            self.max_action *
            (1 - tanh_action.pow(2))
            + 1e-6
        )

        log_prob = log_prob.sum(
            dim=-1,
            keepdim=True
        )

        return action, log_prob, mean


# ------------------------------------------------------------
# Create actor
# ------------------------------------------------------------

actor = SACActor(
    state_dim=STATE_DIM,
    desired_dim=DESIRED_DIM,
    action_dim=ACTION_DIM,
    hidden_dim=HIDDEN_DIM,
    max_action=MAX_ACTION
).to(DEVICE)

print("=" * 60)
print("SAC ACTOR")
print("=" * 60)

print(actor)

print("\nDevice:", next(actor.parameters()).device)

print("\n✓ SAC stochastic actor created")

SAC ACTOR
SACActor(
  (net): Sequential(
    (0): Linear(in_features=12, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (mean): Linear(in_features=256, out_features=3, bias=True)
  (log_std): Linear(in_features=256, out_features=3, bias=True)
)

Device: cuda:0

✓ SAC stochastic actor created


In [6]:
# ============================================================
# CELL 6 — SAC ACTOR SANITY TEST
# ============================================================

test_state = torch.tensor(
    [[0.45, 0.45, 0.10, 0.10, -0.05, 0.02]],
    dtype=torch.float32,
    device=DEVICE
)

test_desired = torch.tensor(
    [[0.50, 0.50, 0.00, 0.00, 0.00, 0.00]],
    dtype=torch.float32,
    device=DEVICE
)

with torch.no_grad():

    mean, log_std = actor(
        test_state,
        test_desired
    )

    action1, log_prob1, _ = actor.sample(
        test_state,
        test_desired
    )

    action2, log_prob2, _ = actor.sample(
        test_state,
        test_desired
    )

print("=" * 60)
print("SAC ACTOR SANITY TEST")
print("=" * 60)

print("Mean shape      :", mean.shape)
print("Log-std shape   :", log_std.shape)

print("\nDeterministic mean:")
print(mean.cpu().numpy())

print("\nSampled action 1:")
print(action1.cpu().numpy())

print("\nSampled action 2:")
print(action2.cpu().numpy())

print("\nLog probability:")
print(log_prob1.cpu().numpy())

print("\nAction 1 min/max:",
      action1.min().item(),
      action1.max().item())

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert mean.shape == (1, ACTION_DIM)
assert log_std.shape == (1, ACTION_DIM)
assert action1.shape == (1, ACTION_DIM)

assert torch.all(
    action1 <= MAX_ACTION + 1e-6
)

assert torch.all(
    action1 >= -MAX_ACTION - 1e-6
)

assert torch.all(
    torch.isfinite(action1)
)

assert torch.all(
    torch.isfinite(log_prob1)
)

print("\n✓ Actor sanity test PASSED")
print("✓ Output dimension = 3")
print("✓ Action range = [-2, 2]")
print("✓ Stochastic sampling works")

SAC ACTOR SANITY TEST
Mean shape      : torch.Size([1, 3])
Log-std shape   : torch.Size([1, 3])

Deterministic mean:
[[ 0.00756191  0.01960089 -0.00615598]]

Sampled action 1:
[[ 0.39688474  1.947141   -0.35273907]]

Sampled action 2:
[[ 0.2906521  -0.17434303 -1.2383802 ]]

Log probability:
[[-4.1668205]]

Action 1 min/max: -0.3527390658855438 1.9471410512924194

✓ Actor sanity test PASSED
✓ Output dimension = 3
✓ Action range = [-2, 2]
✓ Stochastic sampling works


In [7]:
# ============================================================
# CELL 7 — SAC TWIN CRITICS
# ============================================================

class QNetwork(nn.Module):

    def __init__(
        self,
        state_dim,
        desired_dim,
        action_dim,
        hidden_dim
    ):
        super().__init__()

        # Input:
        # state (6) + desired state (6) + action (3) = 15
        self.net = nn.Sequential(
            nn.Linear(
                state_dim + desired_dim + action_dim,
                hidden_dim
            ),
            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),
            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                1
            )
        )

    def forward(
        self,
        state,
        desired_state,
        action
    ):

        x = torch.cat(
            [
                state,
                desired_state,
                action
            ],
            dim=-1
        )

        return self.net(x)


# ------------------------------------------------------------
# Create twin critics
# ------------------------------------------------------------

critic_1 = QNetwork(
    STATE_DIM,
    DESIRED_DIM,
    ACTION_DIM,
    HIDDEN_DIM
).to(DEVICE)

critic_2 = QNetwork(
    STATE_DIM,
    DESIRED_DIM,
    ACTION_DIM,
    HIDDEN_DIM
).to(DEVICE)

# Target critics
target_critic_1 = QNetwork(
    STATE_DIM,
    DESIRED_DIM,
    ACTION_DIM,
    HIDDEN_DIM
).to(DEVICE)

target_critic_2 = QNetwork(
    STATE_DIM,
    DESIRED_DIM,
    ACTION_DIM,
    HIDDEN_DIM
).to(DEVICE)

# Initially identical to main critics
target_critic_1.load_state_dict(
    critic_1.state_dict()
)

target_critic_2.load_state_dict(
    critic_2.state_dict()
)

# Target networks are not directly trained
for parameter in target_critic_1.parameters():
    parameter.requires_grad = False

for parameter in target_critic_2.parameters():
    parameter.requires_grad = False


print("=" * 60)
print("SAC TWIN CRITICS")
print("=" * 60)

print("Critic 1 created")
print("Critic 2 created")
print("Target Critic 1 created")
print("Target Critic 2 created")

print("\nCritic device:",
      next(critic_1.parameters()).device)

# ------------------------------------------------------------
# Sanity test
# ------------------------------------------------------------

with torch.no_grad():

    test_action = torch.tensor(
        [[0.1, -0.2, 0.3]],
        dtype=torch.float32,
        device=DEVICE
    )

    q1 = critic_1(
        test_state,
        test_desired,
        test_action
    )

    q2 = critic_2(
        test_state,
        test_desired,
        test_action
    )

print("\nQ1 shape:", q1.shape)
print("Q2 shape:", q2.shape)

assert q1.shape == (1, 1)
assert q2.shape == (1, 1)

assert torch.isfinite(q1).all()
assert torch.isfinite(q2).all()

print("\n✓ Twin critic test PASSED")

SAC TWIN CRITICS
Critic 1 created
Critic 2 created
Target Critic 1 created
Target Critic 2 created

Critic device: cuda:0

Q1 shape: torch.Size([1, 1])
Q2 shape: torch.Size([1, 1])

✓ Twin critic test PASSED


In [8]:
# ============================================================
# CELL 8 — SAC REPLAY BUFFER
# ============================================================

class ReplayBuffer:

    def __init__(self, capacity):
        self.capacity = capacity

        self.states = np.zeros(
            (capacity, STATE_DIM),
            dtype=np.float32
        )

        self.desired_states = np.zeros(
            (capacity, DESIRED_DIM),
            dtype=np.float32
        )

        self.actions = np.zeros(
            (capacity, ACTION_DIM),
            dtype=np.float32
        )

        self.rewards = np.zeros(
            (capacity, 1),
            dtype=np.float32
        )

        self.next_states = np.zeros(
            (capacity, STATE_DIM),
            dtype=np.float32
        )

        self.dones = np.zeros(
            (capacity, 1),
            dtype=np.float32
        )

        self.ptr = 0
        self.size = 0

    def add(
        self,
        state,
        desired_state,
        action,
        reward,
        next_state,
        done=False
    ):

        self.states[self.ptr] = state
        self.desired_states[self.ptr] = desired_state
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.next_states[self.ptr] = next_state
        self.dones[self.ptr] = done

        self.ptr = (
            self.ptr + 1
        ) % self.capacity

        self.size = min(
            self.size + 1,
            self.capacity
        )

    def sample(self, batch_size):

        indices = np.random.randint(
            0,
            self.size,
            size=batch_size
        )

        states = torch.as_tensor(
            self.states[indices],
            dtype=torch.float32,
            device=DEVICE
        )

        desired_states = torch.as_tensor(
            self.desired_states[indices],
            dtype=torch.float32,
            device=DEVICE
        )

        actions = torch.as_tensor(
            self.actions[indices],
            dtype=torch.float32,
            device=DEVICE
        )

        rewards = torch.as_tensor(
            self.rewards[indices],
            dtype=torch.float32,
            device=DEVICE
        )

        next_states = torch.as_tensor(
            self.next_states[indices],
            dtype=torch.float32,
            device=DEVICE
        )

        dones = torch.as_tensor(
            self.dones[indices],
            dtype=torch.float32,
            device=DEVICE
        )

        return (
            states,
            desired_states,
            actions,
            rewards,
            next_states,
            dones
        )

    def __len__(self):
        return self.size


# ------------------------------------------------------------
# Create replay buffer
# ------------------------------------------------------------

replay_buffer = ReplayBuffer(
    REPLAY_CAPACITY
)

print("=" * 60)
print("REPLAY BUFFER")
print("=" * 60)

print("Capacity :", REPLAY_CAPACITY)
print("Current size :", len(replay_buffer))

# ------------------------------------------------------------
# Test one transition
# ------------------------------------------------------------

replay_buffer.add(
    test_state.cpu().numpy()[0],
    test_desired.cpu().numpy()[0],
    np.array(
        [0.1, -0.2, 0.3],
        dtype=np.float32
    ),
    -0.5,
    test_state.cpu().numpy()[0],
    False
)

print("After test transition :", len(replay_buffer))

sample = replay_buffer.sample(1)

print("\nSample shapes:")

for item in sample:
    print(item.shape)

assert len(replay_buffer) == 1
assert sample[0].shape == (1, 6)
assert sample[1].shape == (1, 6)
assert sample[2].shape == (1, 3)
assert sample[3].shape == (1, 1)
assert sample[4].shape == (1, 6)
assert sample[5].shape == (1, 1)

print("\n✓ Replay buffer test PASSED")

REPLAY BUFFER
Capacity : 100000
Current size : 0
After test transition : 1

Sample shapes:
torch.Size([1, 6])
torch.Size([1, 6])
torch.Size([1, 3])
torch.Size([1, 1])
torch.Size([1, 6])
torch.Size([1, 1])

✓ Replay buffer test PASSED


In [9]:
# ============================================================
# CELL 9 — SAC ALPHA + OPTIMIZERS
# ============================================================

# ------------------------------------------------------------
# Automatic entropy temperature
# ------------------------------------------------------------

log_alpha = torch.tensor(
    np.log(INITIAL_ALPHA),
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

target_entropy = TARGET_ENTROPY

alpha_optimizer = optim.Adam(
    [log_alpha],
    lr=ALPHA_LR
)


# ------------------------------------------------------------
# Actor optimizer
# ------------------------------------------------------------

actor_optimizer = optim.Adam(
    actor.parameters(),
    lr=ACTOR_LR
)


# ------------------------------------------------------------
# Critic optimizers
# ------------------------------------------------------------

critic_1_optimizer = optim.Adam(
    critic_1.parameters(),
    lr=CRITIC_LR
)

critic_2_optimizer = optim.Adam(
    critic_2.parameters(),
    lr=CRITIC_LR
)


# ------------------------------------------------------------
# Current alpha
# ------------------------------------------------------------

alpha = log_alpha.exp().item()


print("=" * 60)
print("SAC ALPHA + OPTIMIZERS")
print("=" * 60)

print("Initial alpha       :", alpha)
print("Target entropy      :", target_entropy)
print("Actor learning rate :", ACTOR_LR)
print("Critic learning rate:", CRITIC_LR)
print("Alpha learning rate :", ALPHA_LR)

print("\nActor optimizer      : Adam")
print("Critic 1 optimizer   : Adam")
print("Critic 2 optimizer   : Adam")
print("Alpha optimizer      : Adam")

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert torch.isfinite(log_alpha)
assert abs(alpha - INITIAL_ALPHA) < 1e-6
assert target_entropy == -ACTION_DIM

print("\n✓ Alpha sanity test PASSED")
print("✓ Optimizers created successfully")

SAC ALPHA + OPTIMIZERS
Initial alpha       : 0.20000000298023224
Target entropy      : -3
Actor learning rate : 0.0001
Critic learning rate: 0.0001
Alpha learning rate : 0.0001

Actor optimizer      : Adam
Critic 1 optimizer   : Adam
Critic 2 optimizer   : Adam
Alpha optimizer      : Adam

✓ Alpha sanity test PASSED
✓ Optimizers created successfully


In [10]:
# ============================================================
# CELL 10 — SAC UPDATE FUNCTION
# ============================================================

def soft_update(source, target, tau):
    """
    Slowly move target network toward source network.
    """
    for target_param, source_param in zip(
        target.parameters(),
        source.parameters()
    ):
        target_param.data.mul_(1.0 - tau)
        target_param.data.add_(
            tau * source_param.data
        )


def sac_update():

    # --------------------------------------------------------
    # Sample replay batch
    # --------------------------------------------------------

    (
        states,
        desired_states,
        actions,
        rewards,
        next_states,
        dones
    ) = replay_buffer.sample(BATCH_SIZE)


    # --------------------------------------------------------
    # Current temperature
    # --------------------------------------------------------

    alpha = log_alpha.exp().detach()


    # ========================================================
    # 1. TARGET Q VALUE
    # ========================================================

    with torch.no_grad():

        next_actions, next_log_prob, _ = actor.sample(
            next_states,
            desired_states
        )

        target_q1 = target_critic_1(
            next_states,
            desired_states,
            next_actions
        )

        target_q2 = target_critic_2(
            next_states,
            desired_states,
            next_actions
        )

        target_min_q = torch.min(
            target_q1,
            target_q2
        )

        # SAC target:
        #
        # r + gamma * (min(Q1,Q2) - alpha*log_prob)
        target_q = rewards + GAMMA * (
            1.0 - dones
        ) * (
            target_min_q - alpha * next_log_prob
        )


    # ========================================================
    # 2. CRITIC UPDATE
    # ========================================================

    current_q1 = critic_1(
        states,
        desired_states,
        actions
    )

    current_q2 = critic_2(
        states,
        desired_states,
        actions
    )

    critic_1_loss = F.mse_loss(
        current_q1,
        target_q
    )

    critic_2_loss = F.mse_loss(
        current_q2,
        target_q
    )


    critic_1_optimizer.zero_grad()

    critic_1_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        critic_1.parameters(),
        10.0
    )

    critic_1_optimizer.step()


    critic_2_optimizer.zero_grad()

    critic_2_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        critic_2.parameters(),
        10.0
    )

    critic_2_optimizer.step()


    # ========================================================
    # 3. ACTOR UPDATE
    # ========================================================

    new_actions, log_prob, _ = actor.sample(
        states,
        desired_states
    )

    q1_new = critic_1(
        states,
        desired_states,
        new_actions
    )

    q2_new = critic_2(
        states,
        desired_states,
        new_actions
    )

    min_q_new = torch.min(
        q1_new,
        q2_new
    )

    actor_loss = (
        alpha * log_prob - min_q_new
    ).mean()


    actor_optimizer.zero_grad()

    actor_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        actor.parameters(),
        10.0
    )

    actor_optimizer.step()


    # ========================================================
    # 4. AUTOMATIC ALPHA UPDATE
    # ========================================================

    alpha_loss = -(
        log_alpha * (
            log_prob.detach() + target_entropy
        )
    ).mean()


    alpha_optimizer.zero_grad()

    alpha_loss.backward()

    alpha_optimizer.step()


    # ========================================================
    # 5. SOFT UPDATE TARGET CRITICS
    # ========================================================

    soft_update(
        critic_1,
        target_critic_1,
        TAU
    )

    soft_update(
        critic_2,
        target_critic_2,
        TAU
    )


    return {
        "critic_1_loss": critic_1_loss.item(),
        "critic_2_loss": critic_2_loss.item(),
        "actor_loss": actor_loss.item(),
        "alpha_loss": alpha_loss.item(),
        "alpha": log_alpha.exp().item(),
        "mean_q": min_q_new.mean().item(),
        "mean_log_prob": log_prob.mean().item()
    }


print("=" * 60)
print("SAC UPDATE FUNCTION")
print("=" * 60)

print("✓ Target Q calculation")
print("✓ Twin critic update")
print("✓ Stochastic actor update")
print("✓ Automatic alpha update")
print("✓ Target critic soft update")

print("\n✓ SAC update function defined successfully")

SAC UPDATE FUNCTION
✓ Target Q calculation
✓ Twin critic update
✓ Stochastic actor update
✓ Automatic alpha update
✓ Target critic soft update

✓ SAC update function defined successfully


In [11]:
# ============================================================
# CELL 11 — EXPANDED TASK GENERATOR
# ============================================================

# Expanded state ranges
STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)


# Original / narrow benchmark distribution
TRAIN_POSE_LOW = np.array(
    [0.4, 0.4, -0.1],
    dtype=np.float32
)

TRAIN_POSE_HIGH = np.array(
    [0.6, 0.6, 0.1],
    dtype=np.float32
)

TRAIN_VEL_LOW = np.array(
    [-0.1, -0.1, -0.1],
    dtype=np.float32
)

TRAIN_VEL_HIGH = np.array(
    [0.1, 0.1, 0.1],
    dtype=np.float32
)


def generate_task(rng=None):
    """
    Generate one training task.

    State:
        [x, y, theta, x_dot, y_dot, theta_dot]

    Desired:
        [xd, yd, theta_d, 0, 0, 0]
    """

    if rng is None:
        rng = np.random.default_rng()

    # --------------------------------------------------------
    # Initial state
    # --------------------------------------------------------

    pose = rng.uniform(
        TRAIN_POSE_LOW,
        TRAIN_POSE_HIGH
    )

    velocity = rng.uniform(
        TRAIN_VEL_LOW,
        TRAIN_VEL_HIGH
    )

    state = np.concatenate(
        [pose, velocity]
    ).astype(np.float32)


    # --------------------------------------------------------
    # Desired state
    # --------------------------------------------------------

    desired_pose = rng.uniform(
        TRAIN_POSE_LOW,
        TRAIN_POSE_HIGH
    )

    desired_state = np.concatenate(
        [
            desired_pose,
            np.zeros(3, dtype=np.float32)
        ]
    ).astype(np.float32)


    return state, desired_state


# ------------------------------------------------------------
# Generate test task
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)

test_task_state, test_task_desired = generate_task(rng)

print("=" * 60)
print("EXPANDED TASK GENERATOR")
print("=" * 60)

print("Initial state :")
print(test_task_state)

print("\nDesired state :")
print(test_task_desired)

print("\nState low     :", STATE_LOW)
print("State high    :", STATE_HIGH)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert test_task_state.shape == (6,)
assert test_task_desired.shape == (6,)

assert np.all(
    test_task_state >= STATE_LOW
)

assert np.all(
    test_task_state <= STATE_HIGH
)

assert np.allclose(
    test_task_desired[3:],
    0.0
)

assert np.all(
    test_task_desired[:3] >= TRAIN_POSE_LOW
)

assert np.all(
    test_task_desired[:3] <= TRAIN_POSE_HIGH
)

print("\n✓ Task shapes correct")
print("✓ Initial state within expanded bounds")
print("✓ Desired velocities = zero")
print("✓ Task generator test PASSED")

EXPANDED TASK GENERATOR
Initial state :
[ 0.5547912   0.4877757   0.07171959  0.03947361 -0.08116453  0.09512447]

Desired state :
[ 0.552228    0.5572129  -0.07437728  0.          0.          0.        ]

State low     : [ 0.   0.  -0.5 -0.2 -0.2 -0.2]
State high    : [1.  1.  0.5 0.2 0.2 0.2]

✓ Task shapes correct
✓ Initial state within expanded bounds
✓ Desired velocities = zero
✓ Task generator test PASSED


In [12]:
# ============================================================
# CELL 12 — EXACT REWARD + REAL DYNAMICS TEST
# ============================================================

def compute_reward(
    next_state,
    desired_state,
    action
):
    """
    Required baseline reward:

    reward =
        -sum((next_state - desired_state)^2)
        -0.01*sum(action^2)
    """

    state_error = next_state - desired_state

    reward = (
        -np.sum(state_error ** 2)
        -0.01 * np.sum(action ** 2)
    )

    return float(reward)


# ------------------------------------------------------------
# Generate one real task
# ------------------------------------------------------------

rng = np.random.default_rng(123)

state, desired_state = generate_task(rng)

# A valid test action in [-2, 2]
action = np.array(
    [0.5, -0.25, 0.15],
    dtype=np.float64
)


# ------------------------------------------------------------
# REAL 3-RPR DYNAMICS
# ------------------------------------------------------------

next_state = np.asarray(
    step(
        state.astype(np.float64),
        action,
        dt=DT
    ),
    dtype=np.float64
)


# ------------------------------------------------------------
# Compute exact reward
# ------------------------------------------------------------

reward = compute_reward(
    next_state,
    desired_state.astype(np.float64),
    action
)


print("=" * 60)
print("EXACT REWARD + REAL DYNAMICS TEST")
print("=" * 60)

print("Initial state:")
print(state)

print("\nDesired state:")
print(desired_state)

print("\nAction:")
print(action)

print("\nNext state:")
print(next_state)

print("\nReward:")
print(reward)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert next_state.shape == (6,)

assert np.all(
    np.isfinite(next_state)
)

assert np.all(
    np.isfinite(reward)
)

assert np.all(
    action >= -MAX_ACTION
)

assert np.all(
    action <= MAX_ACTION
)


# ------------------------------------------------------------
# Independently verify reward formula
# ------------------------------------------------------------

expected_reward = (
    -np.sum(
        (next_state - desired_state.astype(np.float64)) ** 2
    )
    -0.01 * np.sum(action ** 2)
)

assert np.isclose(
    reward,
    expected_reward,
    atol=1e-10
)

print("\n✓ Real dynamics step is finite")
print("✓ Action is within [-2, 2]")
print("✓ Reward exactly matches required formula")
print("✓ Reward test PASSED")

EXACT REWARD + REAL DYNAMICS TEST
Initial state:
[ 0.5364704   0.41076422 -0.05592803 -0.06312564 -0.06481882  0.0624189 ]

Desired state:
[0.584669   0.4553149  0.06395091 0.         0.         0.        ]

Action:
[ 0.5  -0.25  0.15]

Next state:
[ 0.53583916  0.41011603 -0.05530384 -0.06162625 -0.06484099  0.06036002]

Reward:
-0.033644465711238324

✓ Real dynamics step is finite
✓ Action is within [-2, 2]
✓ Reward exactly matches required formula
✓ Reward test PASSED


In [13]:
# ============================================================
# CELL 13 — SAC UPDATE SMOKE TEST
# ============================================================

print("=" * 60)
print("SAC UPDATE SMOKE TEST")
print("=" * 60)

# ------------------------------------------------------------
# Clear the test transition from Cell 8
# ------------------------------------------------------------

replay_buffer = ReplayBuffer(REPLAY_CAPACITY)


# ------------------------------------------------------------
# Generate real transitions using 3-RPR dynamics
# ------------------------------------------------------------

rng = np.random.default_rng(2026)

smoke_steps = BATCH_SIZE + 20

state, desired_state = generate_task(rng)

valid_transitions = 0

for _ in range(smoke_steps):

    # Random valid SAC action
    action = rng.uniform(
        -MAX_ACTION,
        MAX_ACTION,
        size=ACTION_DIM
    ).astype(np.float64)

    # Real 3-RPR dynamics
    next_state = np.asarray(
        step(
            state.astype(np.float64),
            action,
            dt=DT
        ),
        dtype=np.float64
    )

    # Reject invalid dynamics results
    if (
        not np.all(np.isfinite(next_state))
        or np.max(np.abs(next_state)) > 5.0
    ):
        state, desired_state = generate_task(rng)
        continue

    # Exact baseline reward
    reward = compute_reward(
        next_state,
        desired_state.astype(np.float64),
        action
    )

    replay_buffer.add(
        state,
        desired_state,
        action.astype(np.float32),
        reward,
        next_state.astype(np.float32),
        False
    )

    valid_transitions += 1

    state = next_state.astype(np.float32)


print("Valid transitions collected :", valid_transitions)
print("Replay buffer size          :", len(replay_buffer))

assert len(replay_buffer) >= BATCH_SIZE

print("\nRunning ONE SAC update...")

# ------------------------------------------------------------
# One complete SAC update
# ------------------------------------------------------------

metrics = sac_update()


# ------------------------------------------------------------
# Check results
# ------------------------------------------------------------

print("\nSAC update results:")

for key, value in metrics.items():
    print(f"{key:20s}: {value:.6f}")

for key, value in metrics.items():
    assert np.isfinite(value), (
        f"Non-finite value detected in {key}"
    )

print("\n✓ Replay sampling works")
print("✓ Real dynamics transitions work")
print("✓ SAC critic update works")
print("✓ SAC actor update works")
print("✓ Automatic alpha update works")
print("✓ Target networks updated")
print("✓ All SAC metrics are finite")

print("\n" + "=" * 60)
print("✓ SAC UPDATE SMOKE TEST PASSED")
print("=" * 60)

SAC UPDATE SMOKE TEST
Valid transitions collected : 120
Replay buffer size          : 120

Running ONE SAC update...

SAC update results:
critic_1_loss       : 0.463352
critic_2_loss       : 0.478356
actor_loss          : -0.814116
alpha_loss          : -11.416941
alpha               : 0.199980
mean_q              : -0.004633
mean_log_prob       : -4.093744

✓ Replay sampling works
✓ Real dynamics transitions work
✓ SAC critic update works
✓ SAC actor update works
✓ Automatic alpha update works
✓ Target networks updated
✓ All SAC metrics are finite

✓ SAC UPDATE SMOKE TEST PASSED


In [15]:
# ============================================================
# CELL 14 — SAC TRAINING SMOKE TEST (FIXED)
# ============================================================

import time

SMOKE_TRAIN_STEPS = 1000
SMOKE_WARMUP = 100

rng = np.random.default_rng(42)

# Fresh replay buffer for this smoke test
replay_buffer = ReplayBuffer(REPLAY_CAPACITY)

training_start = time.time()

state, desired_state = generate_task(rng)

episode_reward = 0.0
episode_length = 0

actor_losses = []
critic_losses_1 = []
critic_losses_2 = []
alpha_values = []

invalid_count = 0

for step_num in range(1, SMOKE_TRAIN_STEPS + 1):

    # --------------------------------------------------------
    # Select stochastic SAC action
    # --------------------------------------------------------

    state_tensor = torch.as_tensor(
        state,
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    desired_tensor = torch.as_tensor(
        desired_state,
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    with torch.no_grad():
        action_tensor, _, _ = actor.sample(
            state_tensor,
            desired_tensor
        )

    action = action_tensor.squeeze(0).cpu().numpy()

    action = np.clip(
        action,
        -MAX_ACTION,
        MAX_ACTION
    )


    # --------------------------------------------------------
    # Real 3-RPR dynamics
    # --------------------------------------------------------

    next_state = np.asarray(
        step(
            state.astype(np.float64),
            action.astype(np.float64),
            dt=DT
        ),
        dtype=np.float64
    )


    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if (
        not np.all(np.isfinite(next_state))
        or np.max(np.abs(next_state)) > 5.0
    ):
        invalid_count += 1

        state, desired_state = generate_task(rng)
        episode_reward = 0.0
        episode_length = 0

        continue


    next_state = next_state.astype(np.float32)


    # --------------------------------------------------------
    # Exact reward
    # --------------------------------------------------------

    reward = compute_reward(
        next_state,
        desired_state,
        action
    )


    # --------------------------------------------------------
    # Store transition
    # --------------------------------------------------------

    replay_buffer.add(
        state,
        desired_state,
        action.astype(np.float32),
        reward,
        next_state,
        False
    )

    state = next_state

    episode_reward += reward
    episode_length += 1


    # --------------------------------------------------------
    # SAC update
    # --------------------------------------------------------

    if (
        len(replay_buffer) >= BATCH_SIZE
        and step_num > SMOKE_WARMUP
    ):

        metrics = sac_update()

        actor_losses.append(
            metrics["actor_loss"]
        )

        critic_losses_1.append(
            metrics["critic_1_loss"]
        )

        critic_losses_2.append(
            metrics["critic_2_loss"]
        )

        alpha_values.append(
            metrics["alpha"]
        )


    # --------------------------------------------------------
    # Reset episode periodically
    # --------------------------------------------------------

    if episode_length >= 200:

        state, desired_state = generate_task(rng)

        episode_reward = 0.0
        episode_length = 0


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if step_num % 100 == 0:

        elapsed = time.time() - training_start

        print(
            f"Step {step_num:4d}/{SMOKE_TRAIN_STEPS} | "
            f"Replay {len(replay_buffer):4d} | "
            f"Updates {len(actor_losses):4d} | "
            f"Invalid {invalid_count:2d} | "
            f"Alpha {log_alpha.exp().item():.4f} | "
            f"Time {elapsed:.1f}s"
        )


# ============================================================
# FINAL CHECK
# ============================================================

print("\n" + "=" * 60)
print("SAC TRAINING SMOKE TEST")
print("=" * 60)

print("Training steps :", SMOKE_TRAIN_STEPS)
print("Replay size    :", len(replay_buffer))
print("SAC updates    :", len(actor_losses))
print("Invalid states :", invalid_count)

if actor_losses:
    print("Final actor loss :", actor_losses[-1])
    print("Final critic 1   :", critic_losses_1[-1])
    print("Final critic 2   :", critic_losses_2[-1])
    print("Final alpha      :", alpha_values[-1])


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(replay_buffer) >= BATCH_SIZE
assert len(actor_losses) > 0

assert all(np.isfinite(actor_losses))
assert all(np.isfinite(critic_losses_1))
assert all(np.isfinite(critic_losses_2))
assert all(np.isfinite(alpha_values))

print("\n✓ Replay buffer fills")
print("✓ SAC updates execute")
print("✓ Actor loss finite")
print("✓ Critic losses finite")
print("✓ Alpha finite")
print("✓ Real dynamics used")
print("\n✓ CELL 14 PASSED")

Step  100/1000 | Replay  100 | Updates    0 | Invalid  0 | Alpha 0.2000 | Time 0.2s
Step  200/1000 | Replay  200 | Updates  100 | Invalid  0 | Alpha 0.1980 | Time 1.4s
Step  300/1000 | Replay  300 | Updates  200 | Invalid  0 | Alpha 0.1960 | Time 2.7s
Step  400/1000 | Replay  400 | Updates  300 | Invalid  0 | Alpha 0.1941 | Time 4.0s
Step  500/1000 | Replay  500 | Updates  400 | Invalid  0 | Alpha 0.1921 | Time 5.2s
Step  600/1000 | Replay  600 | Updates  500 | Invalid  0 | Alpha 0.1902 | Time 6.4s
Step  700/1000 | Replay  700 | Updates  600 | Invalid  0 | Alpha 0.1883 | Time 7.6s
Step  800/1000 | Replay  800 | Updates  700 | Invalid  0 | Alpha 0.1864 | Time 8.8s
Step  900/1000 | Replay  900 | Updates  800 | Invalid  0 | Alpha 0.1846 | Time 10.0s
Step 1000/1000 | Replay 1000 | Updates  900 | Invalid  0 | Alpha 0.1827 | Time 11.2s

SAC TRAINING SMOKE TEST
Training steps : 1000
Replay size    : 1000
SAC updates    : 900
Invalid states : 0
Final actor loss : -3.6115009784698486
Final crit

In [16]:
# ============================================================
# CELL 15 — SAVE VERIFIED SAC CHECKPOINT
# ============================================================

from pathlib import Path

CHECKPOINT_DIR = Path(
    PROJECT_DIR
) / "training" / "final_model"

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

smoke_checkpoint_path = (
    CHECKPOINT_DIR / "sac_clean_smoke_test.pt"
)

torch.save(
    {
        "algorithm": "SAC",
        "state_dim": STATE_DIM,
        "desired_dim": DESIRED_DIM,
        "action_dim": ACTION_DIM,
        "hidden_dim": HIDDEN_DIM,
        "max_action": MAX_ACTION,

        "gamma": GAMMA,
        "tau": TAU,
        "dt": DT,

        "actor_state_dict": actor.state_dict(),

        "critic_1_state_dict":
            critic_1.state_dict(),

        "critic_2_state_dict":
            critic_2.state_dict(),

        "target_critic_1_state_dict":
            target_critic_1.state_dict(),

        "target_critic_2_state_dict":
            target_critic_2.state_dict(),

        "actor_optimizer_state_dict":
            actor_optimizer.state_dict(),

        "critic_1_optimizer_state_dict":
            critic_1_optimizer.state_dict(),

        "critic_2_optimizer_state_dict":
            critic_2_optimizer.state_dict(),

        "alpha_optimizer_state_dict":
            alpha_optimizer.state_dict(),

        "log_alpha":
            log_alpha.detach().cpu(),

        "training_steps":
            SMOKE_TRAIN_STEPS,

        "seed":
            TRAINING_SEED
    },
    smoke_checkpoint_path
)

print("=" * 60)
print("CHECKPOINT SAVED")
print("=" * 60)

print("Path:")
print(smoke_checkpoint_path)

print("\n✓ Actor saved")
print("✓ Twin critics saved")
print("✓ Target critics saved")
print("✓ Optimizers saved")
print("✓ Alpha saved")
print("\n✓ CELL 15 PASSED")

CHECKPOINT SAVED
Path:
c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_clean_smoke_test.pt

✓ Actor saved
✓ Twin critics saved
✓ Target critics saved
✓ Optimizers saved
✓ Alpha saved

✓ CELL 15 PASSED


In [17]:
# ============================================================
# CELL 16 — SAC TRAINING: 10,000 STEPS
# ============================================================

import time

TARGET_STEPS = 10_000
START_STEP = 1_000

training_start = time.time()

# ------------------------------------------------------------
# Continue from current state
# ------------------------------------------------------------

state, desired_state = generate_task(
    np.random.default_rng(2027)
)

rng = np.random.default_rng(2027)

episode_reward = 0.0
episode_length = 0

running_rewards = []
invalid_count = 0

print("=" * 60)
print("SAC TRAINING — 10,000 STEPS")
print("=" * 60)

for step_num in range(
    START_STEP + 1,
    TARGET_STEPS + 1
):

    # --------------------------------------------------------
    # Select action
    # --------------------------------------------------------

    state_tensor = torch.as_tensor(
        state,
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    desired_tensor = torch.as_tensor(
        desired_state,
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    with torch.no_grad():

        action_tensor, _, _ = actor.sample(
            state_tensor,
            desired_tensor
        )

    action = (
        action_tensor
        .squeeze(0)
        .cpu()
        .numpy()
    )

    action = np.clip(
        action,
        -MAX_ACTION,
        MAX_ACTION
    )


    # --------------------------------------------------------
    # Real 3-RPR dynamics
    # --------------------------------------------------------

    next_state = np.asarray(
        step(
            state.astype(np.float64),
            action.astype(np.float64),
            dt=DT
        ),
        dtype=np.float64
    )


    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if (
        not np.all(np.isfinite(next_state))
        or np.max(np.abs(next_state)) > 5.0
    ):

        invalid_count += 1

        state, desired_state = generate_task(rng)

        episode_reward = 0.0
        episode_length = 0

        continue


    next_state = next_state.astype(
        np.float32
    )


    # --------------------------------------------------------
    # Exact reward
    # --------------------------------------------------------

    reward = compute_reward(
        next_state,
        desired_state,
        action
    )


    # --------------------------------------------------------
    # Store transition
    # --------------------------------------------------------

    replay_buffer.add(
        state,
        desired_state,
        action.astype(np.float32),
        reward,
        next_state,
        False
    )

    state = next_state

    episode_reward += reward
    episode_length += 1


    # --------------------------------------------------------
    # SAC update
    # --------------------------------------------------------

    if len(replay_buffer) >= BATCH_SIZE:

        metrics = sac_update()


    # --------------------------------------------------------
    # Reset episode every 200 steps
    # --------------------------------------------------------

    if episode_length >= 200:

        running_rewards.append(
            episode_reward
        )

        state, desired_state = generate_task(rng)

        episode_reward = 0.0
        episode_length = 0


    # --------------------------------------------------------
    # Progress every 1,000 steps
    # --------------------------------------------------------

    if step_num % 1000 == 0:

        elapsed = time.time() - training_start

        alpha_now = (
            log_alpha.exp()
            .detach()
            .item()
        )

        recent_reward = (
            np.mean(running_rewards[-10:])
            if running_rewards
            else 0.0
        )

        print(
            f"Step {step_num:6d}/{TARGET_STEPS} | "
            f"Replay {len(replay_buffer):6d} | "
            f"Alpha {alpha_now:.4f} | "
            f"Recent reward {recent_reward:.3f} | "
            f"Invalid {invalid_count} | "
            f"Time {elapsed:.1f}s"
        )


print("\n" + "=" * 60)
print("10,000 STEP TRAINING COMPLETE")
print("=" * 60)

print("Replay size    :", len(replay_buffer))
print("Invalid states :", invalid_count)
print(
    "Final alpha    :",
    log_alpha.exp().item()
)

if running_rewards:
    print(
        "Last 10 episode reward:",
        np.mean(running_rewards[-10:])
    )

print("\n✓ CELL 16 COMPLETE")

SAC TRAINING — 10,000 STEPS
Step   2000/10000 | Replay   2000 | Alpha 0.1653 | Recent reward -21.835 | Invalid 0 | Time 12.7s
Step   3000/10000 | Replay   3000 | Alpha 0.1496 | Recent reward -35.622 | Invalid 0 | Time 25.0s
Step   4000/10000 | Replay   4000 | Alpha 0.1354 | Recent reward -34.050 | Invalid 0 | Time 36.7s
Step   5000/10000 | Replay   5000 | Alpha 0.1226 | Recent reward -19.512 | Invalid 0 | Time 48.6s
Step   6000/10000 | Replay   6000 | Alpha 0.1110 | Recent reward -18.380 | Invalid 0 | Time 59.8s
Step   7000/10000 | Replay   7000 | Alpha 0.1005 | Recent reward -16.425 | Invalid 0 | Time 71.3s
Step   8000/10000 | Replay   8000 | Alpha 0.0910 | Recent reward -15.266 | Invalid 0 | Time 82.7s
Step   9000/10000 | Replay   9000 | Alpha 0.0824 | Recent reward -13.852 | Invalid 0 | Time 94.1s
Step  10000/10000 | Replay  10000 | Alpha 0.0746 | Recent reward -15.086 | Invalid 0 | Time 105.8s

10,000 STEP TRAINING COMPLETE
Replay size    : 10000
Invalid states : 0
Final alpha    :

In [18]:
# ============================================================
# CELL 17 — SAVE 10K SAC CHECKPOINT
# ============================================================

from pathlib import Path

checkpoint_dir = (
    Path(PROJECT_DIR)
    / "training"
    / "final_model"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

checkpoint_10k = (
    checkpoint_dir
    / "sac_clean_10k.pt"
)

torch.save(
    {
        "algorithm": "SAC",

        "state_dim": STATE_DIM,
        "desired_dim": DESIRED_DIM,
        "action_dim": ACTION_DIM,
        "hidden_dim": HIDDEN_DIM,
        "max_action": MAX_ACTION,

        "gamma": GAMMA,
        "tau": TAU,
        "dt": DT,

        "state_center": STATE_CENTER.cpu(),
        "state_scale": STATE_SCALE.cpu(),

        "desired_center":
            DESIRED_CENTER.cpu(),

        "desired_scale":
            DESIRED_SCALE.cpu(),

        "actor_state_dict":
            actor.state_dict(),

        "critic_1_state_dict":
            critic_1.state_dict(),

        "critic_2_state_dict":
            critic_2.state_dict(),

        "target_critic_1_state_dict":
            target_critic_1.state_dict(),

        "target_critic_2_state_dict":
            target_critic_2.state_dict(),

        "actor_optimizer_state_dict":
            actor_optimizer.state_dict(),

        "critic_1_optimizer_state_dict":
            critic_1_optimizer.state_dict(),

        "critic_2_optimizer_state_dict":
            critic_2_optimizer.state_dict(),

        "alpha_optimizer_state_dict":
            alpha_optimizer.state_dict(),

        "log_alpha":
            log_alpha.detach().cpu(),

        "training_steps":
            TARGET_STEPS,

        "invalid_states":
            invalid_count,

        "seed":
            TRAINING_SEED
    },
    checkpoint_10k
)

print("=" * 60)
print("10K CHECKPOINT SAVED")
print("=" * 60)

print("Path:")
print(checkpoint_10k)

print("\nTraining steps :", TARGET_STEPS)
print("Replay size    :", len(replay_buffer))
print("Invalid states :", invalid_count)
print("Alpha          :", log_alpha.exp().item())

assert checkpoint_10k.exists()

print("\n✓ Checkpoint exists")
print("✓ Full SAC state saved")
print("\n✓ CELL 17 PASSED")

10K CHECKPOINT SAVED
Path:
c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_clean_10k.pt

Training steps : 10000
Replay size    : 10000
Invalid states : 0
Alpha          : 0.07455167919397354

✓ Checkpoint exists
✓ Full SAC state saved

✓ CELL 17 PASSED


In [19]:
# ============================================================
# CELL 18 — SAC TRAINING: 10K → 50K
# ============================================================

import time

START_STEP = 10_000
TARGET_STEPS = 50_000

rng = np.random.default_rng(3030)

state, desired_state = generate_task(rng)

episode_reward = 0.0
episode_length = 0

running_rewards = []
invalid_count_50k = 0

training_start = time.time()

print("=" * 60)
print("SAC TRAINING — 10K → 50K")
print("=" * 60)

for step_num in range(
    START_STEP + 1,
    TARGET_STEPS + 1
):

    # --------------------------------------------------------
    # Select stochastic action
    # --------------------------------------------------------

    state_tensor = torch.as_tensor(
        state,
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    desired_tensor = torch.as_tensor(
        desired_state,
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    with torch.no_grad():

        action_tensor, _, _ = actor.sample(
            state_tensor,
            desired_tensor
        )

    action = (
        action_tensor
        .squeeze(0)
        .cpu()
        .numpy()
    )

    action = np.clip(
        action,
        -MAX_ACTION,
        MAX_ACTION
    )


    # --------------------------------------------------------
    # Real 3-RPR dynamics
    # --------------------------------------------------------

    next_state = np.asarray(
        step(
            state.astype(np.float64),
            action.astype(np.float64),
            dt=DT
        ),
        dtype=np.float64
    )


    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if (
        not np.all(np.isfinite(next_state))
        or np.max(np.abs(next_state)) > 5.0
    ):

        invalid_count_50k += 1

        state, desired_state = generate_task(rng)

        episode_reward = 0.0
        episode_length = 0

        continue


    next_state = next_state.astype(
        np.float32
    )


    # --------------------------------------------------------
    # Exact baseline reward
    # --------------------------------------------------------

    reward = compute_reward(
        next_state,
        desired_state,
        action
    )


    # --------------------------------------------------------
    # Store transition
    # --------------------------------------------------------

    replay_buffer.add(
        state,
        desired_state,
        action.astype(np.float32),
        reward,
        next_state,
        False
    )

    state = next_state

    episode_reward += reward
    episode_length += 1


    # --------------------------------------------------------
    # SAC update
    # --------------------------------------------------------

    if len(replay_buffer) >= BATCH_SIZE:

        metrics = sac_update()


    # --------------------------------------------------------
    # Episode reset
    # --------------------------------------------------------

    if episode_length >= 200:

        running_rewards.append(
            episode_reward
        )

        state, desired_state = generate_task(rng)

        episode_reward = 0.0
        episode_length = 0


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if step_num % 5000 == 0:

        elapsed = time.time() - training_start

        alpha_now = (
            log_alpha.exp()
            .detach()
            .item()
        )

        recent_reward = (
            np.mean(running_rewards[-10:])
            if running_rewards
            else 0.0
        )

        print(
            f"Step {step_num:6d}/{TARGET_STEPS} | "
            f"Replay {len(replay_buffer):6d} | "
            f"Alpha {alpha_now:.4f} | "
            f"Recent reward {recent_reward:.3f} | "
            f"Invalid {invalid_count_50k} | "
            f"Time {elapsed:.1f}s"
        )


print("\n" + "=" * 60)
print("50,000 STEP TRAINING COMPLETE")
print("=" * 60)

print("Replay size    :", len(replay_buffer))
print("Invalid states :", invalid_count_50k)
print(
    "Final alpha    :",
    log_alpha.exp().item()
)

if running_rewards:
    print(
        "Last 10 episode reward:",
        np.mean(running_rewards[-10:])
    )

print("\n✓ CELL 18 COMPLETE")

SAC TRAINING — 10K → 50K
Step  15000/50000 | Replay  15000 | Alpha 0.0453 | Recent reward -14.602 | Invalid 0 | Time 65.9s
Step  20000/50000 | Replay  20000 | Alpha 0.0276 | Recent reward -10.900 | Invalid 0 | Time 130.0s
Step  25000/50000 | Replay  25000 | Alpha 0.0168 | Recent reward -9.960 | Invalid 0 | Time 196.6s
Step  30000/50000 | Replay  30000 | Alpha 0.0103 | Recent reward -6.777 | Invalid 0 | Time 262.4s
Step  35000/50000 | Replay  35000 | Alpha 0.0063 | Recent reward -4.975 | Invalid 0 | Time 326.0s
Step  40000/50000 | Replay  40000 | Alpha 0.0039 | Recent reward -6.215 | Invalid 0 | Time 389.3s
Step  45000/50000 | Replay  45000 | Alpha 0.0024 | Recent reward -4.950 | Invalid 0 | Time 452.9s
Step  50000/50000 | Replay  50000 | Alpha 0.0015 | Recent reward -4.091 | Invalid 0 | Time 514.9s

50,000 STEP TRAINING COMPLETE
Replay size    : 50000
Invalid states : 0
Final alpha    : 0.0015362715348601341
Last 10 episode reward: -4.090746800554916

✓ CELL 18 COMPLETE


In [20]:
# ============================================================
# CELL 19 — SAVE 50K SAC MODEL
# ============================================================

from pathlib import Path

checkpoint_dir = (
    Path(PROJECT_DIR)
    / "training"
    / "final_model"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

checkpoint_50k = (
    checkpoint_dir
    / "sac_clean_50k.pt"
)

torch.save(
    {
        "algorithm": "SAC",

        "state_dim": STATE_DIM,
        "desired_dim": DESIRED_DIM,
        "action_dim": ACTION_DIM,
        "hidden_dim": HIDDEN_DIM,
        "max_action": MAX_ACTION,

        "gamma": GAMMA,
        "tau": TAU,
        "dt": DT,

        "state_center": STATE_CENTER.cpu(),
        "state_scale": STATE_SCALE.cpu(),

        "desired_center":
            DESIRED_CENTER.cpu(),

        "desired_scale":
            DESIRED_SCALE.cpu(),

        "actor_state_dict":
            actor.state_dict(),

        "critic_1_state_dict":
            critic_1.state_dict(),

        "critic_2_state_dict":
            critic_2.state_dict(),

        "target_critic_1_state_dict":
            target_critic_1.state_dict(),

        "target_critic_2_state_dict":
            target_critic_2.state_dict(),

        "actor_optimizer_state_dict":
            actor_optimizer.state_dict(),

        "critic_1_optimizer_state_dict":
            critic_1_optimizer.state_dict(),

        "critic_2_optimizer_state_dict":
            critic_2_optimizer.state_dict(),

        "alpha_optimizer_state_dict":
            alpha_optimizer.state_dict(),

        "log_alpha":
            log_alpha.detach().cpu(),

        "training_steps":
            TARGET_STEPS,

        "invalid_states":
            invalid_count_50k,

        "seed":
            3030
    },
    checkpoint_50k
)

print("=" * 60)
print("50K CHECKPOINT SAVED")
print("=" * 60)

print("Path:")
print(checkpoint_50k)

print("\nTraining steps :", TARGET_STEPS)
print("Replay size    :", len(replay_buffer))
print("Invalid states :", invalid_count_50k)
print("Alpha          :", log_alpha.exp().item())

assert checkpoint_50k.exists()

print("\n✓ 50K checkpoint exists")
print("✓ Actor + critics saved")
print("✓ Optimizers saved")
print("✓ Alpha saved")
print("\n✓ CELL 19 PASSED")

50K CHECKPOINT SAVED
Path:
c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_clean_50k.pt

Training steps : 50000
Replay size    : 50000
Invalid states : 0
Alpha          : 0.0015362715348601341

✓ 50K checkpoint exists
✓ Actor + critics saved
✓ Optimizers saved
✓ Alpha saved

✓ CELL 19 PASSED


In [21]:
# ============================================================
# CELL 20 — FIXED 100-TASK EVALUATION
# ============================================================

EVAL_SEED = 54321
NUM_EVAL_TASKS = 100
MAX_EVAL_STEPS = 300

eval_rng = np.random.default_rng(EVAL_SEED)

initial_errors = []
final_errors = []
improvements = []
best_errors = []
final_rewards = []

successes = 0
unstable = 0

print("=" * 60)
print("SAC 50K — FIXED 100-TASK EVALUATION")
print("=" * 60)

for task_id in range(NUM_EVAL_TASKS):

    # --------------------------------------------------------
    # Generate unseen task
    # --------------------------------------------------------

    state, desired_state = generate_task(eval_rng)

    state = state.astype(np.float64)
    desired_state = desired_state.astype(np.float64)

    initial_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    best_error = initial_error

    episode_reward = 0.0

    # --------------------------------------------------------
    # Deterministic SAC policy
    # --------------------------------------------------------

    for _ in range(MAX_EVAL_STEPS):

        state_tensor = torch.as_tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.as_tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        with torch.no_grad():

            mean, _ = actor(
                state_tensor,
                desired_tensor
            )

            # Deterministic SAC action
            action_tensor = (
                MAX_ACTION * torch.tanh(mean)
            )

        action = (
            action_tensor
            .squeeze(0)
            .cpu()
            .numpy()
        )

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        )

        # ----------------------------------------------------
        # Real dynamics
        # ----------------------------------------------------

        next_state = np.asarray(
            step(
                state,
                action.astype(np.float64),
                dt=DT
            ),
            dtype=np.float64
        )

        # ----------------------------------------------------
        # Stability check
        # ----------------------------------------------------

        if (
            not np.all(np.isfinite(next_state))
            or np.max(np.abs(next_state)) > 5.0
        ):
            unstable += 1
            break

        # ----------------------------------------------------
        # Exact reward
        # ----------------------------------------------------

        reward = compute_reward(
            next_state,
            desired_state,
            action
        )

        episode_reward += reward

        state = next_state

        current_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        best_error = min(
            best_error,
            current_error
        )

    else:
        # Loop completed normally
        final_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        improvement = (
            initial_error - final_error
        )

        initial_errors.append(initial_error)
        final_errors.append(final_error)
        improvements.append(improvement)
        best_errors.append(best_error)
        final_rewards.append(episode_reward)

        if final_error < initial_error:
            successes += 1

        continue

    # --------------------------------------------------------
    # Unstable task
    # --------------------------------------------------------

    initial_errors.append(initial_error)
    final_errors.append(current_error)
    improvements.append(
        initial_error - current_error
    )
    best_errors.append(best_error)
    final_rewards.append(episode_reward)


# ============================================================
# FINAL METRICS
# ============================================================

avg_initial = np.mean(initial_errors)
avg_final = np.mean(final_errors)
avg_improvement = np.mean(improvements)

avg_best = np.mean(best_errors)
worst_final = np.max(final_errors)
avg_reward = np.mean(final_rewards)

success_rate = (
    successes / NUM_EVAL_TASKS
) * 100.0


print("\n" + "=" * 60)
print("FINAL EVALUATION RESULTS")
print("=" * 60)

print(
    f"Average initial error : {avg_initial:.6f}"
)

print(
    f"Average final error   : {avg_final:.6f}"
)

print(
    f"Average improvement   : {avg_improvement:.6f}"
)

print(
    f"Average best error    : {avg_best:.6f}"
)

print(
    f"Worst final error     : {worst_final:.6f}"
)

print(
    f"Average final reward  : {avg_reward:.6f}"
)

print(
    f"Successes             : {successes}/{NUM_EVAL_TASKS}"
)

print(
    f"Success rate          : {success_rate:.2f}%"
)

print(
    f"Unstable tasks        : {unstable}"
)

print("=" * 60)

if success_rate >= 88.0:
    print("✓ TARGET ACHIEVED")
else:
    print("✗ TARGET NOT YET ACHIEVED")

print("=" * 60)

SAC 50K — FIXED 100-TASK EVALUATION


C:\Users\madha\AppData\Local\Temp\ipykernel_36536\1258618270.py:49: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  state_tensor = torch.as_tensor(



FINAL EVALUATION RESULTS
Average initial error : 0.139931
Average final error   : 0.119540
Average improvement   : 0.020391
Average best error    : 0.080008
Worst final error     : 0.468112
Average final reward  : -7.576162
Successes             : 65/100
Success rate          : 65.00%
Unstable tasks        : 0
✗ TARGET NOT YET ACHIEVED


In [23]:
# ============================================================
# CELL 21 — INSPECT KNOWN-GOOD CHECKPOINT
# ============================================================

from pathlib import Path
import torch

KNOWN_GOOD_PATH = (
    Path(PROJECT_DIR)
    / "training"
    / "final_model"
    / "sac_expanded_best_overnight.pt"
)

print("=" * 60)
print("KNOWN-GOOD SAC CHECKPOINT INSPECTION")
print("=" * 60)

print("Path:")
print(KNOWN_GOOD_PATH)

assert KNOWN_GOOD_PATH.exists(), (
    f"Checkpoint not found:\n{KNOWN_GOOD_PATH}"
)

checkpoint = torch.load(
    KNOWN_GOOD_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:")
print(type(checkpoint))

print("\nTop-level keys:")

if isinstance(checkpoint, dict):

    for key, value in checkpoint.items():

        if isinstance(value, dict):
            print(
                f"  {key}: dict "
                f"({len(value)} entries)"
            )

        elif torch.is_tensor(value):
            print(
                f"  {key}: Tensor "
                f"{tuple(value.shape)}"
            )

        else:
            print(
                f"  {key}: "
                f"{type(value).__name__} = {value}"
            )

else:
    print("Checkpoint is not a dictionary.")


# ------------------------------------------------------------
# Look specifically for actor information
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("SEARCHING FOR ACTOR STATE DICT")
print("-" * 60)

actor_state = None

possible_actor_keys = [
    "actor_state_dict",
    "actor",
    "policy_state_dict",
    "policy",
    "actor_state"
]

for key in possible_actor_keys:

    if (
        isinstance(checkpoint, dict)
        and key in checkpoint
    ):

        candidate = checkpoint[key]

        if isinstance(candidate, dict):

            actor_state = candidate

            print(
                f"Found actor state under key: '{key}'"
            )

            print(
                f"Number of parameters: "
                f"{len(actor_state)}"
            )

            break


# ------------------------------------------------------------
# If nested structure exists, inspect one level deeper
# ------------------------------------------------------------

if actor_state is None:

    print(
        "\nNo standard actor key found."
    )

    print(
        "Nested dictionaries:"
    )

    if isinstance(checkpoint, dict):

        for key, value in checkpoint.items():

            if isinstance(value, dict):

                print(
                    f"\n[{key}]"
                )

                for subkey in list(value.keys())[:20]:

                    print(
                        "   ",
                        subkey
                    )


print("\n" + "=" * 60)
print("CHECKPOINT INSPECTION COMPLETE")
print("=" * 60)

assert isinstance(checkpoint, dict)

print("✓ Checkpoint loaded successfully")
print("✓ No model was modified")
print("✓ No training was performed")

if actor_state is not None:
    print("✓ Actor weights located")
else:
    print(
        "⚠ Actor key requires inspection of the "
        "printed structure above."
    )

KNOWN-GOOD SAC CHECKPOINT INSPECTION
Path:
c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_expanded_best_overnight.pt

Checkpoint type:
<class 'dict'>

Top-level keys:
  actor: dict (12 entries)
  critic1: dict (6 entries)
  critic2: dict (6 entries)
  target_critic1: dict (6 entries)
  target_critic2: dict (6 entries)
  log_alpha: Tensor ()
  alpha: float = 0.034156814217567444
  action_scale: float = 2.0
  success_rate: float = 0.91
  successes: int = 91
  avg_initial_error: float32 = 0.6449064016342163
  avg_final_error: float64 = 0.5128487967415989
  avg_improvement: float64 = 0.13205759289718264
  best_final_error: float64 = 0.10296338763032556
  worst_final_error: float64 = 1.0762485447600607
  avg_final_reward: float64 = -47.54755187179899
  unstable_tasks: int = 0
  seed: int = 12345
  algorithm: str = SAC
  state_low: ndarray = [ 0.   0.  -0.5 -0.2 -0.2 -0.2]
  state_high: ndarray = [1.  1.  0.5 0.2 0.2 0.2]
  desired_low: ndarray = [ 0.   0.  -0.5]
  desire

In [24]:
# ============================================================
# CELL 22 — LOAD 91% SAC CHECKPOINT
# ============================================================

print("=" * 60)
print("LOADING 91% SAC CHECKPOINT")
print("=" * 60)

# ------------------------------------------------------------
# Load actor
# ------------------------------------------------------------

actor.load_state_dict(
    checkpoint["actor"],
    strict=True
)

# ------------------------------------------------------------
# Load twin critics
# ------------------------------------------------------------

critic_1.load_state_dict(
    checkpoint["critic1"],
    strict=True
)

critic_2.load_state_dict(
    checkpoint["critic2"],
    strict=True
)

# ------------------------------------------------------------
# Load target critics
# ------------------------------------------------------------

target_critic_1.load_state_dict(
    checkpoint["target_critic1"],
    strict=True
)

target_critic_2.load_state_dict(
    checkpoint["target_critic2"],
    strict=True
)

# ------------------------------------------------------------
# Restore alpha
# ------------------------------------------------------------

with torch.no_grad():
    log_alpha.copy_(
        checkpoint["log_alpha"].to(DEVICE)
    )

# ------------------------------------------------------------
# Put networks in evaluation mode temporarily
# ------------------------------------------------------------

actor.eval()
critic_1.eval()
critic_2.eval()
target_critic_1.eval()
target_critic_2.eval()

print("Checkpoint success rate :",
      checkpoint["success_rate"])

print("Checkpoint successes    :",
      checkpoint["successes"])

print("Checkpoint alpha        :",
      checkpoint["alpha"])

print("Checkpoint steps        :",
      checkpoint["training_steps"])

print("Checkpoint action scale :",
      checkpoint["action_scale"])

print("\nNormalization:")
print("State center :",
      checkpoint["state_center"].numpy())

print("State scale  :",
      checkpoint["state_scale"].numpy())

print("\n✓ Actor weights loaded")
print("✓ Critic 1 weights loaded")
print("✓ Critic 2 weights loaded")
print("✓ Target critics loaded")
print("✓ Alpha restored")

# ------------------------------------------------------------
# Verify actor output
# ------------------------------------------------------------

test_state_np, test_desired_np = generate_task(
    np.random.default_rng(9999)
)

test_state_tensor = torch.as_tensor(
    test_state_np,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(0)

test_desired_tensor = torch.as_tensor(
    test_desired_np,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(0)

with torch.no_grad():

    mean, log_std = actor(
        test_state_tensor,
        test_desired_tensor
    )

    deterministic_action = (
        MAX_ACTION
        * torch.tanh(mean)
    )

print("\nTest actor output:")
print("Mean:")
print(mean.cpu().numpy())

print("\nDeterministic action:")
print(deterministic_action.cpu().numpy())

assert torch.isfinite(mean).all()
assert torch.isfinite(log_std).all()
assert torch.isfinite(deterministic_action).all()

assert torch.all(
    deterministic_action <= MAX_ACTION
)

assert torch.all(
    deterministic_action >= -MAX_ACTION
)

print("\n✓ Actor produces finite output")
print("✓ Action remains within [-2, 2]")

print("\n" + "=" * 60)
print("✓ CELL 22 PASSED")
print("=" * 60)

LOADING 91% SAC CHECKPOINT
Checkpoint success rate : 0.91
Checkpoint successes    : 91
Checkpoint alpha        : 0.034156814217567444
Checkpoint steps        : 95000
Checkpoint action scale : 2.0

Normalization:
State center : [0.5 0.5 0.  0.  0.  0. ]
State scale  : [0.5 0.5 0.5 0.2 0.2 0.2]

✓ Actor weights loaded
✓ Critic 1 weights loaded
✓ Critic 2 weights loaded
✓ Target critics loaded
✓ Alpha restored

Test actor output:
Mean:
[[-0.9745559   1.0153942  -0.09285268]]

Deterministic action:
[[-1.501399    1.5359678  -0.18517351]]

✓ Actor produces finite output
✓ Action remains within [-2, 2]

✓ CELL 22 PASSED


In [25]:
# ============================================================
# CELL 23 — EVALUATE RECOVERED SAC MODEL
# ============================================================

actor.eval()

EVAL_SEED = 54321
NUM_EVAL_TASKS = 100
MAX_EVAL_STEPS = 300

eval_rng = np.random.default_rng(EVAL_SEED)

initial_errors = []
final_errors = []
improvements = []
best_errors = []
final_rewards = []

successes = 0
unstable = 0

print("=" * 60)
print("RECOVERED SAC — INDEPENDENT 100-TASK EVALUATION")
print("=" * 60)

for task_id in range(NUM_EVAL_TASKS):

    state, desired_state = generate_task(
        eval_rng
    )

    state = state.astype(np.float64)
    desired_state = desired_state.astype(np.float64)

    initial_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    best_error = initial_error
    episode_reward = 0.0

    for _ in range(MAX_EVAL_STEPS):

        state_tensor = torch.as_tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.as_tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        with torch.no_grad():

            mean, _ = actor(
                state_tensor,
                desired_tensor
            )

            action_tensor = (
                MAX_ACTION
                * torch.tanh(mean)
            )

        action = (
            action_tensor
            .squeeze(0)
            .cpu()
            .numpy()
        )

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        )

        next_state = np.asarray(
            step(
                state,
                action.astype(np.float64),
                dt=DT
            ),
            dtype=np.float64
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.max(np.abs(next_state)) > 5.0
        ):
            unstable += 1
            break

        reward = compute_reward(
            next_state,
            desired_state,
            action
        )

        episode_reward += reward

        state = next_state

        current_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        best_error = min(
            best_error,
            current_error
        )

    final_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    improvement = (
        initial_error - final_error
    )

    initial_errors.append(initial_error)
    final_errors.append(final_error)
    improvements.append(improvement)
    best_errors.append(best_error)
    final_rewards.append(episode_reward)

    if final_error < initial_error:
        successes += 1


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

avg_initial = np.mean(initial_errors)
avg_final = np.mean(final_errors)
avg_improvement = np.mean(improvements)
avg_best = np.mean(best_errors)
worst_final = np.max(final_errors)
avg_reward = np.mean(final_rewards)

success_rate = (
    successes / NUM_EVAL_TASKS
) * 100.0


print("\n" + "=" * 60)
print("RECOVERED MODEL RESULTS")
print("=" * 60)

print(
    f"Average initial error : {avg_initial:.6f}"
)

print(
    f"Average final error   : {avg_final:.6f}"
)

print(
    f"Average improvement   : {avg_improvement:.6f}"
)

print(
    f"Average best error    : {avg_best:.6f}"
)

print(
    f"Worst final error     : {worst_final:.6f}"
)

print(
    f"Average final reward  : {avg_reward:.6f}"
)

print(
    f"Successes             : "
    f"{successes}/{NUM_EVAL_TASKS}"
)

print(
    f"Success rate          : "
    f"{success_rate:.2f}%"
)

print(
    f"Unstable tasks        : {unstable}"
)

print("=" * 60)

RECOVERED SAC — INDEPENDENT 100-TASK EVALUATION

RECOVERED MODEL RESULTS
Average initial error : 0.139931
Average final error   : 0.250835
Average improvement   : -0.110904
Average best error    : 0.116245
Worst final error     : 0.428772
Average final reward  : -14.649439
Successes             : 8/100
Success rate          : 8.00%
Unstable tasks        : 0


In [26]:
# ============================================================
# CELL 24 — SAC PRECISION FINE-TUNING
# ============================================================

# Use the recovered 91% model
# and fine-tune it for precise final convergence.

actor.train()
critic_1.train()
critic_2.train()

# ------------------------------------------------------------
# Lower learning rates for precision control
# ------------------------------------------------------------

PRECISION_ACTOR_LR = 3e-6
PRECISION_CRITIC_LR = 3e-6
PRECISION_ALPHA_LR = 1e-6

for group in actor_optimizer.param_groups:
    group["lr"] = PRECISION_ACTOR_LR

for group in critic_1_optimizer.param_groups:
    group["lr"] = PRECISION_CRITIC_LR

for group in critic_2_optimizer.param_groups:
    group["lr"] = PRECISION_CRITIC_LR

for group in alpha_optimizer.param_groups:
    group["lr"] = PRECISION_ALPHA_LR


# ------------------------------------------------------------
# Fresh replay buffer
# ------------------------------------------------------------

replay_buffer = ReplayBuffer(
    REPLAY_CAPACITY
)

rng = np.random.default_rng(6060)

PRECISION_STEPS = 20_000
WARMUP = 2_000

# ------------------------------------------------------------
# Curriculum:
# mostly near-target tasks for precision
# ------------------------------------------------------------

def generate_precision_task(rng):

    mode = rng.random()

    # 70% very close tasks
    if mode < 0.70:

        center = rng.uniform(
            [0.45, 0.45, -0.05],
            [0.55, 0.55, 0.05]
        )

        offset = rng.uniform(
            [-0.05, -0.05, -0.03],
            [0.05, 0.05, 0.03]
        )

        desired_pose = center

        state_pose = np.clip(
            center + offset,
            TRAIN_POSE_LOW,
            TRAIN_POSE_HIGH
        )

    # 20% normal tasks
    elif mode < 0.90:

        state_pose = rng.uniform(
            TRAIN_POSE_LOW,
            TRAIN_POSE_HIGH
        )

        desired_pose = rng.uniform(
            TRAIN_POSE_LOW,
            TRAIN_POSE_HIGH
        )

    # 10% full training distribution
    else:

        state_pose = rng.uniform(
            TRAIN_POSE_LOW,
            TRAIN_POSE_HIGH
        )

        desired_pose = rng.uniform(
            TRAIN_POSE_LOW,
            TRAIN_POSE_HIGH
        )

    velocity = rng.uniform(
        TRAIN_VEL_LOW,
        TRAIN_VEL_HIGH
    )

    state = np.concatenate(
        [state_pose, velocity]
    ).astype(np.float32)

    desired_state = np.concatenate(
        [
            desired_pose,
            np.zeros(3, dtype=np.float32)
        ]
    ).astype(np.float32)

    return state, desired_state


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

start_time = time.time()

state, desired_state = generate_precision_task(rng)

updates = 0
invalid = 0

recent_rewards = []

episode_reward = 0.0
episode_length = 0

print("=" * 60)
print("PRECISION SAC FINE-TUNING")
print("=" * 60)

for step_num in range(1, PRECISION_STEPS + 1):

    # --------------------------------------------------------
    # Stochastic action
    # --------------------------------------------------------

    state_tensor = torch.as_tensor(
        state,
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    desired_tensor = torch.as_tensor(
        desired_state,
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    with torch.no_grad():

        action_tensor, _, _ = actor.sample(
            state_tensor,
            desired_tensor
        )

    action = (
        action_tensor
        .squeeze(0)
        .cpu()
        .numpy()
    )

    action = np.clip(
        action,
        -MAX_ACTION,
        MAX_ACTION
    )


    # --------------------------------------------------------
    # Real dynamics
    # --------------------------------------------------------

    next_state = np.asarray(
        step(
            state.astype(np.float64),
            action.astype(np.float64),
            dt=DT
        ),
        dtype=np.float64
    )


    # --------------------------------------------------------
    # Reject unstable transition
    # --------------------------------------------------------

    if (
        not np.all(np.isfinite(next_state))
        or np.max(np.abs(next_state)) > 5.0
    ):

        invalid += 1

        state, desired_state = (
            generate_precision_task(rng)
        )

        episode_reward = 0.0
        episode_length = 0

        continue


    next_state = next_state.astype(
        np.float32
    )


    # --------------------------------------------------------
    # Exact reward
    # --------------------------------------------------------

    reward = compute_reward(
        next_state,
        desired_state,
        action
    )


    replay_buffer.add(
        state,
        desired_state,
        action.astype(np.float32),
        reward,
        next_state,
        False
    )


    state = next_state

    episode_reward += reward
    episode_length += 1


    # --------------------------------------------------------
    # SAC update
    # --------------------------------------------------------

    if (
        len(replay_buffer) >= BATCH_SIZE
        and step_num > WARMUP
    ):

        metrics = sac_update()

        updates += 1


    # --------------------------------------------------------
    # Episode reset
    # --------------------------------------------------------

    if episode_length >= 200:

        recent_rewards.append(
            episode_reward
        )

        state, desired_state = (
            generate_precision_task(rng)
        )

        episode_reward = 0.0
        episode_length = 0


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if step_num % 2000 == 0:

        elapsed = time.time() - start_time

        recent = (
            np.mean(recent_rewards[-10:])
            if recent_rewards
            else 0.0
        )

        print(
            f"Step {step_num:5d}/{PRECISION_STEPS} | "
            f"Updates {updates:5d} | "
            f"Alpha {log_alpha.exp().item():.5f} | "
            f"Reward {recent:.3f} | "
            f"Invalid {invalid} | "
            f"Time {elapsed:.1f}s"
        )


print("\n" + "=" * 60)
print("PRECISION FINE-TUNING COMPLETE")
print("=" * 60)

print("Steps        :", PRECISION_STEPS)
print("SAC updates  :", updates)
print("Invalid      :", invalid)
print("Final alpha  :", log_alpha.exp().item())

assert updates > 0
assert invalid < PRECISION_STEPS

print("\n✓ Precision fine-tuning completed")
print("✓ CELL 24 PASSED")

PRECISION SAC FINE-TUNING
Step  2000/20000 | Updates     0 | Alpha 0.03416 | Reward -9.056 | Invalid 0 | Time 3.0s
Step  4000/20000 | Updates  2000 | Alpha 0.03406 | Reward -10.122 | Invalid 0 | Time 27.5s
Step  6000/20000 | Updates  4000 | Alpha 0.03400 | Reward -10.475 | Invalid 0 | Time 51.5s
Step  8000/20000 | Updates  6000 | Alpha 0.03393 | Reward -13.165 | Invalid 0 | Time 74.6s
Step 10000/20000 | Updates  8000 | Alpha 0.03387 | Reward -12.574 | Invalid 0 | Time 98.1s
Step 12000/20000 | Updates 10000 | Alpha 0.03380 | Reward -12.837 | Invalid 0 | Time 121.4s
Step 14000/20000 | Updates 12000 | Alpha 0.03374 | Reward -11.553 | Invalid 0 | Time 144.9s
Step 16000/20000 | Updates 14000 | Alpha 0.03367 | Reward -12.381 | Invalid 0 | Time 167.9s
Step 18000/20000 | Updates 16000 | Alpha 0.03361 | Reward -15.099 | Invalid 0 | Time 190.9s
Step 20000/20000 | Updates 18000 | Alpha 0.03354 | Reward -11.484 | Invalid 0 | Time 218.7s

PRECISION FINE-TUNING COMPLETE
Steps        : 20000
SAC upda

In [27]:
# ============================================================
# CELL 25 — EVALUATE CURRENT MODEL
# ============================================================

actor.eval()

EVAL_SEED = 54321
NUM_EVAL_TASKS = 100
MAX_EVAL_STEPS = 300

eval_rng = np.random.default_rng(EVAL_SEED)

initial_errors = []
final_errors = []
improvements = []
best_errors = []
final_rewards = []

successes = 0
unstable = 0

print("=" * 60)
print("CURRENT SAC MODEL — 100 TASK EVALUATION")
print("=" * 60)

for task_id in range(NUM_EVAL_TASKS):

    state, desired_state = generate_task(eval_rng)

    state = state.astype(np.float64)
    desired_state = desired_state.astype(np.float64)

    initial_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    best_error = initial_error
    episode_reward = 0.0

    for _ in range(MAX_EVAL_STEPS):

        state_tensor = torch.as_tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.as_tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        with torch.no_grad():

            mean, _ = actor(
                state_tensor,
                desired_tensor
            )

            action_tensor = (
                MAX_ACTION * torch.tanh(mean)
            )

        action = (
            action_tensor
            .squeeze(0)
            .cpu()
            .numpy()
        )

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        )

        next_state = np.asarray(
            step(
                state,
                action.astype(np.float64),
                dt=DT
            ),
            dtype=np.float64
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.max(np.abs(next_state)) > 5.0
        ):
            unstable += 1
            break

        reward = compute_reward(
            next_state,
            desired_state,
            action
        )

        episode_reward += reward
        state = next_state

        current_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        best_error = min(
            best_error,
            current_error
        )

    final_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    initial_errors.append(initial_error)
    final_errors.append(final_error)
    improvements.append(
        initial_error - final_error
    )
    best_errors.append(best_error)
    final_rewards.append(episode_reward)

    if final_error < initial_error:
        successes += 1


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

avg_initial = np.mean(initial_errors)
avg_final = np.mean(final_errors)
avg_improvement = np.mean(improvements)
avg_best = np.mean(best_errors)
worst_final = np.max(final_errors)
avg_reward = np.mean(final_rewards)

success_rate = (
    successes / NUM_EVAL_TASKS
) * 100.0


print("\n" + "=" * 60)
print("CURRENT MODEL RESULTS")
print("=" * 60)

print(f"Average initial error : {avg_initial:.6f}")
print(f"Average final error   : {avg_final:.6f}")
print(f"Average improvement   : {avg_improvement:.6f}")
print(f"Average best error    : {avg_best:.6f}")
print(f"Worst final error     : {worst_final:.6f}")
print(f"Average final reward  : {avg_reward:.6f}")
print(f"Successes             : {successes}/100")
print(f"Success rate          : {success_rate:.2f}%")
print(f"Unstable tasks        : {unstable}")

print("=" * 60)

CURRENT SAC MODEL — 100 TASK EVALUATION

CURRENT MODEL RESULTS
Average initial error : 0.139931
Average final error   : 0.231293
Average improvement   : -0.091362
Average best error    : 0.112459
Worst final error     : 0.453390
Average final reward  : -14.067993
Successes             : 19/100
Success rate          : 19.00%
Unstable tasks        : 0


In [29]:
# CELL 26 — INSPECT THE 50K CHECKPOINT
# READ-ONLY: DOES NOT MODIFY THE FILE

MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_clean_50k.pt"
)

checkpoint_50k = torch.load(
    MODEL_PATH,
    map_location="cpu"
)

print("=" * 60)
print("sac_clean_50k.pt CHECKPOINT INSPECTION")
print("=" * 60)
print("File:", MODEL_PATH)
print("Type:", type(checkpoint_50k))

if isinstance(checkpoint_50k, dict):
    print("\nKeys:")
    for key, value in checkpoint_50k.items():
        if torch.is_tensor(value):
            print(f"  {key}: Tensor {tuple(value.shape)}")
        elif isinstance(value, dict):
            print(f"  {key}: state_dict ({len(value)} entries)")
        else:
            print(f"  {key}: {type(value).__name__} = {value}")
else:
    print("Checkpoint is not a dictionary.")

print("=" * 60)

sac_clean_50k.pt CHECKPOINT INSPECTION
File: c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_clean_50k.pt
Type: <class 'dict'>

Keys:
  algorithm: str = SAC
  state_dim: int = 6
  desired_dim: int = 6
  action_dim: int = 3
  hidden_dim: int = 256
  max_action: float = 2.0
  gamma: float = 0.99
  tau: float = 0.005
  dt: float = 0.01
  state_center: Tensor (6,)
  state_scale: Tensor (6,)
  desired_center: Tensor (6,)
  desired_scale: Tensor (6,)
  actor_state_dict: state_dict (12 entries)
  critic_1_state_dict: state_dict (6 entries)
  critic_2_state_dict: state_dict (6 entries)
  target_critic_1_state_dict: state_dict (6 entries)
  target_critic_2_state_dict: state_dict (6 entries)
  actor_optimizer_state_dict: state_dict (2 entries)
  critic_1_optimizer_state_dict: state_dict (2 entries)
  critic_2_optimizer_state_dict: state_dict (2 entries)
  alpha_optimizer_state_dict: state_dict (2 entries)
  log_alpha: Tensor ()
  training_steps: int = 50000
  invalid_states: i

In [31]:
# CELL 27 — RESTORE VERIFIED 50K SAC MODEL
# Loads the saved checkpoint and recreates all networks.

checkpoint_50k = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

# Recreate networks using the dimensions stored in the checkpoint
actor = SACActor(
    checkpoint_50k["state_dim"],
    checkpoint_50k["desired_dim"],
    checkpoint_50k["action_dim"],
    checkpoint_50k["hidden_dim"],
    checkpoint_50k["max_action"]
).to(DEVICE)

critic1 = QNetwork(
    checkpoint_50k["state_dim"],
    checkpoint_50k["desired_dim"],
    checkpoint_50k["action_dim"],
    checkpoint_50k["hidden_dim"]
).to(DEVICE)

critic2 = QNetwork(
    checkpoint_50k["state_dim"],
    checkpoint_50k["desired_dim"],
    checkpoint_50k["action_dim"],
    checkpoint_50k["hidden_dim"]
).to(DEVICE)

# Load saved weights
actor.load_state_dict(checkpoint_50k["actor_state_dict"])
critic1.load_state_dict(checkpoint_50k["critic_1_state_dict"])
critic2.load_state_dict(checkpoint_50k["critic_2_state_dict"])

# Evaluation mode
actor.eval()
critic1.eval()
critic2.eval()

# Restore alpha value
log_alpha = checkpoint_50k["log_alpha"].to(DEVICE).clone().detach()
alpha = log_alpha.exp().item()

print("=" * 60)
print("50K SAC MODEL RESTORED SUCCESSFULLY")
print("=" * 60)
print("Algorithm      :", checkpoint_50k["algorithm"])
print("Training steps :", checkpoint_50k["training_steps"])
print("Seed           :", checkpoint_50k["seed"])
print("Invalid states :", checkpoint_50k["invalid_states"])
print("Alpha          :", alpha)
print("Device         :", DEVICE)
print("Model loaded   : YES")
print("=" * 60)

50K SAC MODEL RESTORED SUCCESSFULLY
Algorithm      : SAC
Training steps : 50000
Seed           : 3030
Invalid states : 0
Alpha          : 0.0015362715348601341
Device         : cuda
Model loaded   : YES


In [33]:
# CELL 28A — CHECK EXISTING REPLAY BUFFER API

print("ReplayBuffer methods:")
print([m for m in dir(replay) if not m.startswith("_")])

ReplayBuffer methods:
['actions', 'add', 'capacity', 'desired_states', 'dones', 'next_states', 'ptr', 'rewards', 'sample', 'size', 'states']


In [35]:
# CELL 28B — FIX SAC UPDATE
# Fixes alpha gradient computation.
# Does NOT modify any saved checkpoint.

def sac_update():

    global log_alpha

    # -----------------------------------------------------
    # Sample batch
    # -----------------------------------------------------
    states, desired_states, actions, rewards, next_states, dones = replay.sample(
        BATCH_SIZE
    )

    # -----------------------------------------------------
    # Alpha MUST remain a differentiable tensor
    # -----------------------------------------------------
    alpha_tensor = log_alpha.exp()

    # -----------------------------------------------------
    # Target Q
    # -----------------------------------------------------
    with torch.no_grad():

        next_actions, next_log_prob, _ = actor.sample(
            next_states,
            desired_states
        )

        target_q1 = critic1(next_states, desired_states, next_actions)
        target_q2 = critic2(next_states, desired_states, next_actions)

        min_target_q = torch.min(target_q1, target_q2)

        target_q = rewards + GAMMA * (
            1.0 - dones
        ) * (
            min_target_q - alpha_tensor.detach() * next_log_prob
        )

    # -----------------------------------------------------
    # Critic 1
    # -----------------------------------------------------
    current_q1 = critic1(
        states,
        desired_states,
        actions
    )

    critic1_loss = F.mse_loss(
        current_q1,
        target_q
    )

    critic1_optimizer.zero_grad()
    critic1_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        critic1.parameters(),
        10.0
    )
    critic1_optimizer.step()

    # -----------------------------------------------------
    # Critic 2
    # -----------------------------------------------------
    current_q2 = critic2(
        states,
        desired_states,
        actions
    )

    critic2_loss = F.mse_loss(
        current_q2,
        target_q
    )

    critic2_optimizer.zero_grad()
    critic2_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        critic2.parameters(),
        10.0
    )
    critic2_optimizer.step()

    # -----------------------------------------------------
    # Actor
    # -----------------------------------------------------
    new_actions, log_prob, _ = actor.sample(
        states,
        desired_states
    )

    q1_new = critic1(
        states,
        desired_states,
        new_actions
    )

    q2_new = critic2(
        states,
        desired_states,
        new_actions
    )

    min_q_new = torch.min(q1_new, q2_new)

    actor_loss = (
        alpha_tensor.detach() * log_prob - min_q_new
    ).mean()

    actor_optimizer.zero_grad()
    actor_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        actor.parameters(),
        10.0
    )
    actor_optimizer.step()

    # -----------------------------------------------------
    # Alpha / entropy temperature
    # IMPORTANT: use log_alpha directly
    # -----------------------------------------------------
    alpha_loss = -(
        log_alpha * (
            log_prob.detach() + TARGET_ENTROPY
        )
    ).mean()

    alpha_optimizer.zero_grad()
    alpha_loss.backward()
    alpha_optimizer.step()

    # -----------------------------------------------------
    # Soft target update
    # -----------------------------------------------------
    with torch.no_grad():

        for target_param, param in zip(
            target_critic1.parameters(),
            critic1.parameters()
        ):
            target_param.data.mul_(1.0 - TAU)
            target_param.data.add_(TAU * param.data)

        for target_param, param in zip(
            target_critic2.parameters(),
            critic2.parameters()
        ):
            target_param.data.mul_(1.0 - TAU)
            target_param.data.add_(TAU * param.data)

    return (
        critic1_loss.item(),
        critic2_loss.item(),
        actor_loss.item(),
        alpha_loss.item(),
        alpha_tensor.item()
    )

print("SAC update function fixed.")

SAC update function fixed.


In [37]:
# CELL 28D — DIAGNOSE SAC ALPHA

print("log_alpha:")
print("  type           :", type(log_alpha))
print("  value          :", log_alpha.item())
print("  requires_grad  :", log_alpha.requires_grad)
print("  grad_fn        :", log_alpha.grad_fn)

print("\nalpha_optimizer:")
print("  type           :", type(alpha_optimizer))

params = alpha_optimizer.param_groups[0]["params"]

print("  parameters     :", len(params))

for i, p in enumerate(params):
    print(f"  param {i}: requires_grad={p.requires_grad}, "
          f"value={p.item()}, same_object={p is log_alpha}")

log_alpha:
  type           : <class 'torch.Tensor'>
  value          : -6.478396892547607
  requires_grad  : False
  grad_fn        : None

alpha_optimizer:
  type           : <class 'torch.optim.adam.Adam'>
  parameters     : 1
  param 0: requires_grad=False, value=-6.478396892547607, same_object=True


In [38]:
# CELL 28E — RESTORE ALPHA GRADIENT SAFELY

# Create a fresh trainable copy of the saved alpha value
log_alpha = checkpoint_50k["log_alpha"].to(DEVICE).detach().clone()
log_alpha.requires_grad_(True)

# Recreate alpha optimizer using the trainable tensor
alpha_optimizer = optim.Adam(
    [log_alpha],
    lr=3e-6
)

print("=" * 60)
print("ALPHA RESTORED")
print("=" * 60)
print("log_alpha value  :", log_alpha.item())
print("requires_grad    :", log_alpha.requires_grad)
print("grad_fn          :", log_alpha.grad_fn)
print("optimizer param  :", alpha_optimizer.param_groups[0]["params"][0] is log_alpha)
print("=" * 60)

ALPHA RESTORED
log_alpha value  : -6.478396892547607
requires_grad    : True
grad_fn          : None
optimizer param  : True


In [39]:
# CELL 28F — CREATE A SAFE ROLLBACK COPY
# This saves a COPY. The original 50K checkpoint is NOT changed.

ROLLBACK_PATH = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_50k_rollback.pt"
)

torch.save(
    {
        "algorithm": "SAC",
        "state_dim": STATE_DIM,
        "desired_dim": DESIRED_DIM,
        "action_dim": ACTION_DIM,
        "hidden_dim": HIDDEN_DIM,
        "max_action": MAX_ACTION,
        "gamma": GAMMA,
        "tau": TAU,
        "dt": DT,

        "state_center": STATE_CENTER.cpu(),
        "state_scale": STATE_SCALE.cpu(),
        "desired_center": DESIRED_CENTER.cpu(),
        "desired_scale": DESIRED_SCALE.cpu(),

        "actor_state_dict": actor.state_dict(),
        "critic_1_state_dict": critic1.state_dict(),
        "critic_2_state_dict": critic2.state_dict(),

        "log_alpha": log_alpha.detach().cpu(),

        "training_steps": 50000,
        "source": "sac_clean_50k.pt"
    },
    ROLLBACK_PATH
)

print("=" * 60)
print("SAFE ROLLBACK COPY CREATED")
print("=" * 60)
print(ROLLBACK_PATH)
print("Original 50K checkpoint: UNTOUCHED")
print("=" * 60)

SAFE ROLLBACK COPY CREATED
c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_50k_rollback.pt
Original 50K checkpoint: UNTOUCHED


In [41]:
# CELL 28H — RESTORE SAC TARGET CRITICS
# Read-only with respect to the saved 50K checkpoint.

target_critic1 = QNetwork(
    checkpoint_50k["state_dim"],
    checkpoint_50k["desired_dim"],
    checkpoint_50k["action_dim"],
    checkpoint_50k["hidden_dim"]
).to(DEVICE)

target_critic2 = QNetwork(
    checkpoint_50k["state_dim"],
    checkpoint_50k["desired_dim"],
    checkpoint_50k["action_dim"],
    checkpoint_50k["hidden_dim"]
).to(DEVICE)

target_critic1.load_state_dict(
    checkpoint_50k["target_critic_1_state_dict"]
)

target_critic2.load_state_dict(
    checkpoint_50k["target_critic_2_state_dict"]
)

target_critic1.eval()
target_critic2.eval()

# Target networks must not receive gradients
for p in target_critic1.parameters():
    p.requires_grad = False

for p in target_critic2.parameters():
    p.requires_grad = False

print("=" * 60)
print("TARGET CRITICS RESTORED")
print("=" * 60)
print("Target Critic 1 : OK")
print("Target Critic 2 : OK")
print("Gradients       : DISABLED")
print("50K checkpoint  : UNTOUCHED")
print("=" * 60)

TARGET CRITICS RESTORED
Target Critic 1 : OK
Target Critic 2 : OK
Gradients       : DISABLED
50K checkpoint  : UNTOUCHED


In [42]:
# CELL 28I — FINAL ONE-UPDATE SAC TEST

before_alpha = log_alpha.exp().item()

result = sac_update()

after_alpha = log_alpha.exp().item()

print("=" * 60)
print("ONE-UPDATE SAC TEST")
print("=" * 60)
print("Critic 1 loss :", result[0])
print("Critic 2 loss :", result[1])
print("Actor loss    :", result[2])
print("Alpha loss    :", result[3])
print("Alpha before  :", before_alpha)
print("Alpha after   :", after_alpha)
print("Alpha changed :", before_alpha != after_alpha)
print("=" * 60)

ONE-UPDATE SAC TEST
Critic 1 loss : 0.03736492618918419
Critic 2 loss : 0.013072202913463116
Actor loss    : -0.45471587777137756
Alpha loss    : -10.633898735046387
Alpha before  : 0.0015362671110779047
Alpha after   : 0.0015362626872956753
Alpha changed : True


In [43]:
# CELL 29 — CONTROLLED 5K SAC FINE-TUNE
# 50K checkpoint remains untouched.

FINE_STEPS = 5000
WARMUP = 500

replay = ReplayBuffer(REPLAY_CAPACITY)

actor_optimizer = optim.Adam(actor.parameters(), lr=3e-6)
critic1_optimizer = optim.Adam(critic1.parameters(), lr=3e-6)
critic2_optimizer = optim.Adam(critic2.parameters(), lr=3e-6)
alpha_optimizer = optim.Adam([log_alpha], lr=1e-6)

actor.train()
critic1.train()
critic2.train()

rng = np.random.default_rng(3030)

start = time.time()
updates = 0
invalid = 0
rewards = deque(maxlen=500)

for t in range(FINE_STEPS):

    state, desired_state = generate_task(rng)

    state = np.asarray(state, dtype=np.float32)
    desired_state = np.asarray(desired_state, dtype=np.float32)

    st = torch.tensor(
        state, dtype=torch.float32, device=DEVICE
    ).unsqueeze(0)

    dt = torch.tensor(
        desired_state, dtype=torch.float32, device=DEVICE
    ).unsqueeze(0)

    with torch.no_grad():
        action, _, _ = actor.sample(st, dt)

    action = action.squeeze(0).cpu().numpy()
    action = np.clip(action, -MAX_ACTION, MAX_ACTION)

    next_state = step(state, action, dt=DT)

    if (
        not np.all(np.isfinite(next_state))
        or np.any(np.abs(next_state) > 5.0)
    ):
        invalid += 1
        continue

    reward = (
        -np.sum((next_state - desired_state) ** 2)
        -0.01 * np.sum(action ** 2)
    )

    rewards.append(reward)

    replay.add(
        state,
        desired_state,
        action,
        reward,
        next_state,
        False
    )

    if t >= WARMUP and len(replay) >= BATCH_SIZE:
        sac_update()
        updates += 1

    if (t + 1) % 1000 == 0:
        print(
            f"Step {t+1}/5000 | "
            f"Updates {updates} | "
            f"Avg reward {np.mean(rewards):.4f} | "
            f"Alpha {log_alpha.exp().item():.6f} | "
            f"Invalid {invalid}"
        )

print()
print("=" * 60)
print("5K CONTROLLED FINE-TUNE COMPLETE")
print("=" * 60)
print("Updates :", updates)
print("Invalid :", invalid)
print("Alpha   :", log_alpha.exp().item())
print("Time    :", time.time() - start)
print("=" * 60)

Step 1000/5000 | Updates 500 | Avg reward -0.0410 | Alpha 0.001536 | Invalid 0
Step 2000/5000 | Updates 1500 | Avg reward -0.0353 | Alpha 0.001534 | Invalid 0
Step 3000/5000 | Updates 2500 | Avg reward -0.0372 | Alpha 0.001533 | Invalid 0
Step 4000/5000 | Updates 3500 | Avg reward -0.0865 | Alpha 0.001533 | Invalid 0
Step 5000/5000 | Updates 4500 | Avg reward -0.0710 | Alpha 0.001534 | Invalid 0

5K CONTROLLED FINE-TUNE COMPLETE
Updates : 4500
Invalid : 0
Alpha   : 0.0015339445089921355
Time    : 64.37733483314514


In [44]:
# CELL 30 — EVALUATE 5K FINE-TUNED SAC
# Evaluation only — no training.

actor.eval()
critic1.eval()
critic2.eval()

EVAL_SEED = 54321
rng = np.random.default_rng(EVAL_SEED)

NUM_TASKS = 100
MAX_EPISODE_STEPS = 300

initial_errors = []
final_errors = []
best_errors = []
final_rewards = []

successes = 0
unstable_tasks = 0

with torch.no_grad():

    for task_id in range(NUM_TASKS):

        state, desired_state = generate_task(rng)

        initial_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        best_error = initial_error
        total_reward = 0.0

        for _ in range(MAX_EPISODE_STEPS):

            state_tensor = torch.tensor(
                state,
                dtype=torch.float32,
                device=DEVICE
            ).unsqueeze(0)

            desired_tensor = torch.tensor(
                desired_state,
                dtype=torch.float32,
                device=DEVICE
            ).unsqueeze(0)

            mean, _ = actor(
                state_tensor,
                desired_tensor
            )

            action = (
                MAX_ACTION * torch.tanh(mean)
            ).squeeze(0).cpu().numpy()

            action = np.clip(
                action,
                -MAX_ACTION,
                MAX_ACTION
            )

            next_state = step(
                state,
                action,
                dt=DT
            )

            if (
                not np.all(np.isfinite(next_state))
                or np.any(np.abs(next_state) > 5.0)
            ):
                unstable_tasks += 1
                break

            reward = (
                -np.sum(
                    (next_state - desired_state) ** 2
                )
                -0.01 * np.sum(action ** 2)
            )

            total_reward += reward

            current_error = np.linalg.norm(
                next_state[:3] - desired_state[:3]
            )

            best_error = min(
                best_error,
                current_error
            )

            state = next_state

        final_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        initial_errors.append(initial_error)
        final_errors.append(final_error)
        best_errors.append(best_error)
        final_rewards.append(total_reward)

        if final_error < initial_error:
            successes += 1

avg_initial = np.mean(initial_errors)
avg_final = np.mean(final_errors)
avg_improvement = avg_initial - avg_final
avg_best = np.mean(best_errors)
worst_final = np.max(final_errors)
avg_reward = np.mean(final_rewards)
success_rate = 100 * successes / NUM_TASKS

print("=" * 60)
print("5K FINE-TUNED SAC — FINAL EVALUATION")
print("=" * 60)
print(f"Average initial error : {avg_initial:.6f}")
print(f"Average final error   : {avg_final:.6f}")
print(f"Average improvement   : {avg_improvement:.6f}")
print(f"Average best error    : {avg_best:.6f}")
print(f"Worst final error     : {worst_final:.6f}")
print(f"Average final reward  : {avg_reward:.6f}")
print(f"Successes             : {successes}/100")
print(f"Success rate          : {success_rate:.2f}%")
print(f"Unstable tasks        : {unstable_tasks}")
print("=" * 60)

5K FINE-TUNED SAC — FINAL EVALUATION
Average initial error : 0.139931
Average final error   : 0.761553
Average improvement   : -0.621622
Average best error    : 0.119166
Worst final error     : 4.581345
Average final reward  : -238.182695
Successes             : 0/100
Success rate          : 0.00%
Unstable tasks        : 68


In [45]:
# CELL 31 — RESTORE SAFE 50K MODEL
# NO TRAINING

safe_ckpt = torch.load(
    ROLLBACK_PATH,
    map_location=DEVICE
)

actor.load_state_dict(
    safe_ckpt["actor_state_dict"]
)

critic1.load_state_dict(
    safe_ckpt["critic_1_state_dict"]
)

critic2.load_state_dict(
    safe_ckpt["critic_2_state_dict"]
)

log_alpha = safe_ckpt["log_alpha"].to(DEVICE).detach().clone()
log_alpha.requires_grad_(True)

actor.eval()
critic1.eval()
critic2.eval()

print("=" * 60)
print("SAFE 50K MODEL RESTORED")
print("=" * 60)
print("Source :", ROLLBACK_PATH)
print("Steps  :", safe_ckpt["training_steps"])
print("Device :", DEVICE)
print("Status : READY")
print("=" * 60)

SAFE 50K MODEL RESTORED
Source : c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_50k_rollback.pt
Steps  : 50000
Device : cuda
Status : READY


In [46]:
# CELL 32 — SAC CONTROLLER DEMO
# No training. No file modification.

actor.eval()

# Example current and desired 3-RPR states
current_state = np.array(
    [0.50, 0.50, 0.05, 0.05, -0.03, 0.02],
    dtype=np.float32
)

desired_state = np.array(
    [0.55, 0.55, 0.00, 0.00, 0.00, 0.00],
    dtype=np.float32
)

state_tensor = torch.tensor(
    current_state,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(0)

desired_tensor = torch.tensor(
    desired_state,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(0)

with torch.no_grad():

    mean, log_std = actor(
        state_tensor,
        desired_tensor
    )

    action = (
        MAX_ACTION * torch.tanh(mean)
    )

action = action.squeeze(0).cpu().numpy()

print("=" * 60)
print("3-RPR SAC CONTROLLER DEMO")
print("=" * 60)

print("\nCurrent state:")
print(current_state)

print("\nDesired state:")
print(desired_state)

print("\nActor output:")
print("Mean     :", mean.squeeze(0).cpu().numpy())
print("Log std  :", log_std.squeeze(0).cpu().numpy())

print("\nFINAL CONTROL ACTION")
print("[F1, F2, F3] =", action)

print("\nAction limits:")
print("[-2, 2]")

print("=" * 60)

3-RPR SAC CONTROLLER DEMO

Current state:
[ 0.5   0.5   0.05  0.05 -0.03  0.02]

Desired state:
[0.55 0.55 0.   0.   0.   0.  ]

Actor output:
Mean     : [ 0.13828717  0.19882883 -0.60485196]
Log std  : [-2.3983681 -2.5464993 -2.0858204]

FINAL CONTROL ACTION
[F1, F2, F3] = [ 0.27482474  0.39249906 -1.0809863 ]

Action limits:
[-2, 2]


In [47]:
# CELL 33 — ACTUAL 3-RPR SAC ROLLOUT
# Evaluation only. NO TRAINING and NO MODEL MODIFICATION.

actor.eval()

state = np.array(
    [0.50, 0.50, 0.05, 0.05, -0.03, 0.02],
    dtype=np.float32
)

desired_state = np.array(
    [0.55, 0.55, 0.00, 0.00, 0.00, 0.00],
    dtype=np.float32
)

trajectory = [state.copy()]
actions = []
errors = []

with torch.no_grad():

    for t in range(300):

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        mean, _ = actor(
            state_tensor,
            desired_tensor
        )

        action = (
            MAX_ACTION * torch.tanh(mean)
        ).squeeze(0).cpu().numpy()

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        )

        next_state = step(
            state,
            action,
            dt=DT
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):
            print("Unstable at step:", t)
            break

        error = np.linalg.norm(
            next_state[:3] - desired_state[:3]
        )

        errors.append(error)
        actions.append(action.copy())
        trajectory.append(next_state.copy())

        state = next_state

trajectory = np.array(trajectory)
actions = np.array(actions)
errors = np.array(errors)

print("=" * 60)
print("3-RPR SAC ROLLOUT")
print("=" * 60)

print("Initial pose :", trajectory[0, :3])
print("Desired pose :", desired_state[:3])
print("Final pose   :", trajectory[-1, :3])

print()
print("Initial error:", np.linalg.norm(
    trajectory[0, :3] - desired_state[:3]
))

print("Final error  :", np.linalg.norm(
    trajectory[-1, :3] - desired_state[:3]
))

print("Best error   :", np.min(errors))

print()
print("First action :", actions[0])
print("Final action :", actions[-1])

print("=" * 60)

3-RPR SAC ROLLOUT
Initial pose : [0.5  0.5  0.05]
Desired pose : [0.55 0.55 0.  ]
Final pose   : [0.56389363 0.54077377 0.02147446]

Initial error: 0.0866025545737069
Final error  : 0.027190235410516954
Best error   : 0.022825413619821894

First action : [ 0.27482474  0.39249906 -1.0809863 ]
Final action : [-0.05383813  0.0125881  -0.05272521]


In [48]:
# CELL 34 — CHECK THE ACTUAL 100 EVALUATION TASKS
# READ-ONLY DIAGNOSTIC

rng = np.random.default_rng(54321)

initial_pose_errors = []
state_values = []
desired_values = []

for i in range(100):
    state, desired_state = generate_task(rng)

    initial_pose_errors.append(
        np.linalg.norm(state[:3] - desired_state[:3])
    )

    state_values.append(state)
    desired_values.append(desired_state)

initial_pose_errors = np.array(initial_pose_errors)
state_values = np.array(state_values)
desired_values = np.array(desired_values)

print("=" * 60)
print("EVALUATION TASK DISTRIBUTION")
print("=" * 60)

print("\nInitial pose error:")
print("  Minimum :", initial_pose_errors.min())
print("  Mean    :", initial_pose_errors.mean())
print("  Maximum :", initial_pose_errors.max())

print("\nCurrent-state ranges:")
print("  Min:", state_values.min(axis=0))
print("  Max:", state_values.max(axis=0))

print("\nDesired-state ranges:")
print("  Min:", desired_values.min(axis=0))
print("  Max:", desired_values.max(axis=0))

print("\nFirst 5 tasks:")
for i in range(5):
    print(
        f"Task {i+1}: "
        f"state={state_values[i, :3]}  "
        f"desired={desired_values[i, :3]}"
    )

print("=" * 60)

EVALUATION TASK DISTRIBUTION

Initial pose error:
  Minimum : 0.016811352
  Mean    : 0.13993071
  Maximum : 0.22296357

Current-state ranges:
  Min: [ 0.40161657  0.40242654 -0.09994769 -0.09989624 -0.09957187 -0.097958  ]
  Max: [0.5965582  0.5982747  0.09982933 0.09615919 0.09942339 0.09720527]

Desired-state ranges:
  Min: [ 0.4018154   0.4050668  -0.09169891  0.          0.          0.        ]
  Max: [0.59836215 0.5993246  0.09911504 0.         0.         0.        ]

First 5 tasks:
Task 1: state=[ 0.54119635  0.56742555 -0.0050409 ]  desired=[ 0.54399717  0.47580516 -0.00195397]
Task 2: state=[0.5159539  0.5462519  0.03300341]  desired=[0.4039138  0.56641155 0.0354195 ]
Task 3: state=[ 0.49725366  0.47044593 -0.08225963]  desired=[ 0.5440434   0.59186304 -0.07621261]
Task 4: state=[0.41100264 0.43771672 0.03492676]  desired=[0.5656269  0.42266735 0.08000478]
Task 5: state=[0.54849875 0.53143746 0.09982933]  desired=[ 0.44642115  0.5947488  -0.03616105]


In [49]:
# CELL 35 — FIRST-ACTION EFFECT DIAGNOSTIC
# READ-ONLY. NO TRAINING.

actor.eval()

rng = np.random.default_rng(54321)

improved = 0
worsened = 0
unchanged = 0
invalid = 0

one_step_improvements = []
action_norms = []

with torch.no_grad():

    for task_id in range(100):

        state, desired_state = generate_task(rng)

        initial_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        mean, _ = actor(
            state_tensor,
            desired_tensor
        )

        action = (
            MAX_ACTION * torch.tanh(mean)
        ).squeeze(0).cpu().numpy()

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        )

        action_norms.append(np.linalg.norm(action))

        next_state = step(
            state,
            action,
            dt=DT
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):
            invalid += 1
            continue

        next_error = np.linalg.norm(
            next_state[:3] - desired_state[:3]
        )

        improvement = initial_error - next_error
        one_step_improvements.append(improvement)

        if improvement > 1e-7:
            improved += 1
        elif improvement < -1e-7:
            worsened += 1
        else:
            unchanged += 1

print("=" * 60)
print("FIRST-ACTION EFFECT — 100 TASKS")
print("=" * 60)

print("Improved after 1 step :", improved)
print("Worsened after 1 step :", worsened)
print("Unchanged             :", unchanged)
print("Invalid                :", invalid)

print()
print("Mean one-step improvement :",
      np.mean(one_step_improvements))

print("Best one-step improvement :",
      np.max(one_step_improvements))

print("Worst one-step improvement:",
      np.min(one_step_improvements))

print("Mean action magnitude     :",
      np.mean(action_norms))

print("=" * 60)

FIRST-ACTION EFFECT — 100 TASKS
Improved after 1 step : 41
Worsened after 1 step : 59
Unchanged             : 0
Invalid                : 0

Mean one-step improvement : -4.144689149802292e-05
Best one-step improvement : 0.0014367948044356482
Worst one-step improvement: -0.0011984485431298098
Mean action magnitude     : 1.0847088


In [50]:
# CELL 36 — 3-RPR FORCE RESPONSE TEST
# READ-ONLY DYNAMICS DIAGNOSTIC

test_state = np.array(
    [0.50, 0.50, 0.00, 0.00, 0.00, 0.00],
    dtype=np.float32
)

test_actions = {
    "F1 positive": np.array([ 1.0,  0.0,  0.0], dtype=np.float32),
    "F2 positive": np.array([ 0.0,  1.0,  0.0], dtype=np.float32),
    "F3 positive": np.array([ 0.0,  0.0,  1.0], dtype=np.float32),
    "F1 negative": np.array([-1.0,  0.0,  0.0], dtype=np.float32),
    "F2 negative": np.array([ 0.0, -1.0,  0.0], dtype=np.float32),
    "F3 negative": np.array([ 0.0,  0.0, -1.0], dtype=np.float32),
}

print("=" * 70)
print("3-RPR FORCE → STATE RESPONSE")
print("=" * 70)

print("\nInitial state:")
print(test_state)

for name, action in test_actions.items():

    next_state = step(
        test_state,
        action,
        dt=DT
    )

    delta = next_state - test_state

    print()
    print(name)
    print("  Action :", action)
    print("  Δstate :", delta)
    print("  Δpose  :", delta[:3])

print()
print("=" * 70)

3-RPR FORCE → STATE RESPONSE

Initial state:
[0.5 0.5 0.  0.  0.  0. ]

F1 positive
  Action : [1. 0. 0.]
  Δstate : [ 0.          0.          0.          0.00188033  0.00197179 -0.00711983]
  Δpose  : [0. 0. 0.]

F2 positive
  Action : [0. 1. 0.]
  Δstate : [ 0.          0.          0.         -0.00194038  0.0020513   0.00780065]
  Δpose  : [0. 0. 0.]

F3 positive
  Action : [0. 0. 1.]
  Δstate : [ 0.          0.          0.         -0.00084093 -0.00256338  0.00953427]
  Δpose  : [0. 0. 0.]

F1 negative
  Action : [-1.  0.  0.]
  Δstate : [ 0.          0.          0.         -0.00188033 -0.00197179  0.00711983]
  Δpose  : [0. 0. 0.]

F2 negative
  Action : [ 0. -1.  0.]
  Δstate : [ 0.          0.          0.          0.00194038 -0.0020513  -0.00780065]
  Δpose  : [0. 0. 0.]

F3 negative
  Action : [ 0.  0. -1.]
  Δstate : [ 0.          0.          0.          0.00084093  0.00256338 -0.00953427]
  Δpose  : [0. 0. 0.]



In [51]:
# CELL 37 — EPISODIC SAC TRAINING
# Final training attempt.
# Original 50K checkpoint remains untouched.

import time
from collections import deque

# ---------------------------------------------------------
# Restore SAFE 50K model first
# ---------------------------------------------------------

safe_ckpt = torch.load(
    ROLLBACK_PATH,
    map_location=DEVICE
)

actor.load_state_dict(
    safe_ckpt["actor_state_dict"]
)

critic1.load_state_dict(
    safe_ckpt["critic_1_state_dict"]
)

critic2.load_state_dict(
    safe_ckpt["critic_2_state_dict"]
)

# Restore trainable alpha
log_alpha = (
    safe_ckpt["log_alpha"]
    .to(DEVICE)
    .detach()
    .clone()
)
log_alpha.requires_grad_(True)

# ---------------------------------------------------------
# Fresh optimizers
# ---------------------------------------------------------

actor_optimizer = optim.Adam(
    actor.parameters(),
    lr=3e-6
)

critic1_optimizer = optim.Adam(
    critic1.parameters(),
    lr=3e-6
)

critic2_optimizer = optim.Adam(
    critic2.parameters(),
    lr=3e-6
)

alpha_optimizer = optim.Adam(
    [log_alpha],
    lr=1e-6
)

# ---------------------------------------------------------
# Fresh replay buffer
# ---------------------------------------------------------

replay = ReplayBuffer(REPLAY_CAPACITY)

# ---------------------------------------------------------
# Training parameters
# ---------------------------------------------------------

EPISODES = 350
EPISODE_LENGTH = 300
WARMUP_TRANSITIONS = 2000

rng = np.random.default_rng(7777)

total_steps = 0
total_updates = 0
invalid_states = 0

episode_rewards = deque(maxlen=20)
episode_final_errors = deque(maxlen=20)

start_time = time.time()

actor.train()
critic1.train()
critic2.train()

# ---------------------------------------------------------
# EPISODIC TRAINING
# ---------------------------------------------------------

for episode in range(EPISODES):

    # Reset to a NEW task at the beginning of every episode
    state, desired_state = generate_task(rng)

    state = np.asarray(state, dtype=np.float32)
    desired_state = np.asarray(
        desired_state,
        dtype=np.float32
    )

    episode_reward = 0.0

    for t in range(EPISODE_LENGTH):

        # -------------------------------------------------
        # Actor
        # -------------------------------------------------

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        with torch.no_grad():

            if total_steps < WARMUP_TRANSITIONS:

                action = torch.empty(
                    1,
                    ACTION_DIM,
                    device=DEVICE
                ).uniform_(
                    -MAX_ACTION,
                    MAX_ACTION
                )

            else:

                action, _, _ = actor.sample(
                    state_tensor,
                    desired_tensor
                )

        action = (
            action
            .squeeze(0)
            .cpu()
            .numpy()
        )

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        ).astype(np.float32)

        # -------------------------------------------------
        # ACTUAL 3-RPR DYNAMICS
        # -------------------------------------------------

        next_state = step(
            state,
            action,
            dt=DT
        )

        # -------------------------------------------------
        # Stability check
        # -------------------------------------------------

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):

            invalid_states += 1

            # End this episode safely
            break

        next_state = np.asarray(
            next_state,
            dtype=np.float32
        )

        # -------------------------------------------------
        # EXACT REQUIRED REWARD
        # -------------------------------------------------

        reward = (
            -np.sum(
                (next_state - desired_state) ** 2
            )
            -0.01 * np.sum(action ** 2)
        )

        episode_reward += reward

        # -------------------------------------------------
        # Episode termination
        # -------------------------------------------------

        done = (
            t == EPISODE_LENGTH - 1
        )

        # -------------------------------------------------
        # Store sequential transition
        # -------------------------------------------------

        replay.add(
            state,
            desired_state,
            action,
            reward,
            next_state,
            done
        )

        total_steps += 1

        # -------------------------------------------------
        # SAC UPDATE
        # -------------------------------------------------

        if (
            total_steps >= WARMUP_TRANSITIONS
            and len(replay) >= BATCH_SIZE
        ):

            sac_update()
            total_updates += 1

        # -------------------------------------------------
        # Move forward in the SAME episode
        # -------------------------------------------------

        state = next_state

    # -----------------------------------------------------
    # Episode statistics
    # -----------------------------------------------------

    final_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    episode_rewards.append(
        episode_reward
    )

    episode_final_errors.append(
        final_error
    )

    # -----------------------------------------------------
    # Progress
    # -----------------------------------------------------

    if (episode + 1) % 25 == 0:

        elapsed = time.time() - start_time

        print(
            f"Episode {episode+1:3d}/{EPISODES} | "
            f"Steps {total_steps:6d} | "
            f"Updates {total_updates:6d} | "
            f"AvgEpReward {np.mean(episode_rewards):9.3f} | "
            f"AvgFinalErr {np.mean(episode_final_errors):.5f} | "
            f"Alpha {log_alpha.exp().item():.6f} | "
            f"Invalid {invalid_states} | "
            f"Time {elapsed:.1f}s"
        )

print()
print("=" * 70)
print("EPISODIC SAC TRAINING COMPLETE")
print("=" * 70)
print("Episodes        :", EPISODES)
print("Total steps     :", total_steps)
print("SAC updates     :", total_updates)
print("Invalid states  :", invalid_states)
print("Final alpha     :", log_alpha.exp().item())
print("Training time   :", time.time() - start_time)
print("=" * 70)

Episode  25/350 | Steps   7500 | Updates   5501 | AvgEpReward   -14.078 | AvgFinalErr 0.17607 | Alpha 0.001528 | Invalid 0 | Time 75.8s
Episode  50/350 | Steps  15000 | Updates  13001 | AvgEpReward   -14.751 | AvgFinalErr 0.23049 | Alpha 0.001517 | Invalid 0 | Time 179.3s
Episode  75/350 | Steps  22500 | Updates  20501 | AvgEpReward    -7.217 | AvgFinalErr 0.13214 | Alpha 0.001506 | Invalid 0 | Time 276.2s
Episode 100/350 | Steps  30000 | Updates  28001 | AvgEpReward   -24.085 | AvgFinalErr 0.24035 | Alpha 0.001498 | Invalid 0 | Time 372.6s
Episode 125/350 | Steps  37500 | Updates  35501 | AvgEpReward   -20.792 | AvgFinalErr 0.26873 | Alpha 0.001486 | Invalid 0 | Time 466.7s
Episode 150/350 | Steps  45000 | Updates  43001 | AvgEpReward    -9.589 | AvgFinalErr 0.18044 | Alpha 0.001476 | Invalid 0 | Time 571.3s
Episode 175/350 | Steps  52500 | Updates  50501 | AvgEpReward    -8.703 | AvgFinalErr 0.14174 | Alpha 0.001466 | Invalid 0 | Time 672.0s
Episode 200/350 | Steps  60000 | Updates  

In [53]:
# CELL 38A — DEFINE EXPANDED TASK LIMITS

DESIRED_LOW = np.array(
    [0.0, 0.0, -0.5],
    dtype=np.float32
)

DESIRED_HIGH = np.array(
    [1.0, 1.0, 0.5],
    dtype=np.float32
)

print("DESIRED_LOW :", DESIRED_LOW)
print("DESIRED_HIGH:", DESIRED_HIGH)

DESIRED_LOW : [ 0.   0.  -0.5]
DESIRED_HIGH: [1.  1.  0.5]


In [54]:
# CELL 38B — VERIFY EVERYTHING NEEDED BEFORE FINAL TRAINING

print("DEVICE              :", DEVICE)
print("STATE_DIM           :", STATE_DIM)
print("DESIRED_DIM         :", DESIRED_DIM)
print("ACTION_DIM          :", ACTION_DIM)
print("MAX_ACTION          :", MAX_ACTION)
print("GAMMA               :", GAMMA)
print("TAU                 :", TAU)
print("BATCH_SIZE          :", BATCH_SIZE)
print("REPLAY_CAPACITY     :", REPLAY_CAPACITY)

print("\nDesired range:")
print("DESIRED_LOW         :", DESIRED_LOW)
print("DESIRED_HIGH        :", DESIRED_HIGH)

print("\nState range:")
print("STATE_LOW           :", STATE_LOW)
print("STATE_HIGH          :", STATE_HIGH)

# Test task generation
test_rng = np.random.default_rng(123)

test_state, test_desired = generate_expanded_task(test_rng)

print("\nGenerated state     :", test_state)
print("Generated desired   :", test_desired)

# Test one actual dynamics step
test_action = np.zeros(ACTION_DIM, dtype=np.float32)

test_next = step(
    test_state,
    test_action,
    dt=DT
)

print("Next state          :", test_next)

assert np.all(np.isfinite(test_state))
assert np.all(np.isfinite(test_desired))
assert np.all(np.isfinite(test_next))

print("\nCELL 38B CHECK: PASSED")

DEVICE              : cuda
STATE_DIM           : 6
DESIRED_DIM         : 6
ACTION_DIM          : 3
MAX_ACTION          : 2.0
GAMMA               : 0.99
TAU                 : 0.005
BATCH_SIZE          : 100
REPLAY_CAPACITY     : 100000

Desired range:
DESIRED_LOW         : [ 0.   0.  -0.5]
DESIRED_HIGH        : [1.  1.  0.5]

State range:
STATE_LOW           : [ 0.   0.  -0.5 -0.2 -0.2 -0.2]
STATE_HIGH          : [1.  1.  0.5 0.2 0.2 0.2]

Generated state     : [ 0.6823519   0.05382102 -0.27964014 -0.12625128 -0.12963764  0.1248378 ]
Generated desired   : [0.92334497 0.2765744  0.31975457 0.         0.         0.        ]
Next state          : [ 0.68108937  0.05252464 -0.27839176 -0.12618468 -0.12969162  0.12548813]

CELL 38B CHECK: PASSED


In [55]:
# CELL 39 — FINAL FRESH EPISODIC SAC TRAINING
# Uses the already-tested sac_update().
# Existing checkpoints are NOT overwritten.

import time
from collections import deque
import copy

# =========================================================
# 1. FRESH SAC NETWORKS
# =========================================================

actor = SACActor(
    STATE_DIM,
    DESIRED_DIM,
    ACTION_DIM,
    HIDDEN_DIM,
    MAX_ACTION
).to(DEVICE)

critic1 = QNetwork(
    STATE_DIM,
    DESIRED_DIM,
    ACTION_DIM,
    HIDDEN_DIM
).to(DEVICE)

critic2 = QNetwork(
    STATE_DIM,
    DESIRED_DIM,
    ACTION_DIM,
    HIDDEN_DIM
).to(DEVICE)

target_critic1 = copy.deepcopy(critic1).to(DEVICE)
target_critic2 = copy.deepcopy(critic2).to(DEVICE)

for p in target_critic1.parameters():
    p.requires_grad = False

for p in target_critic2.parameters():
    p.requires_grad = False


# =========================================================
# 2. FRESH OPTIMIZERS
# =========================================================

actor_optimizer = optim.Adam(
    actor.parameters(),
    lr=1e-4
)

critic1_optimizer = optim.Adam(
    critic1.parameters(),
    lr=1e-4
)

critic2_optimizer = optim.Adam(
    critic2.parameters(),
    lr=1e-4
)

log_alpha = torch.tensor(
    np.log(0.2),
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

alpha_optimizer = optim.Adam(
    [log_alpha],
    lr=3e-5
)

TARGET_ENTROPY = -ACTION_DIM


# =========================================================
# 3. FRESH REPLAY
# =========================================================

replay = ReplayBuffer(REPLAY_CAPACITY)


# =========================================================
# 4. TRAINING PARAMETERS
# =========================================================

EPISODES = 334
EPISODE_LENGTH = 300
WARMUP_TRANSITIONS = 2000

rng = np.random.default_rng(3030)

total_steps = 0
total_updates = 0
invalid_states = 0

episode_rewards = deque(maxlen=20)
episode_final_errors = deque(maxlen=20)

start_time = time.time()


# =========================================================
# 5. FRESH EPISODIC TRAINING
# =========================================================

for episode in range(EPISODES):

    # -----------------------------------------------------
    # RESET
    # -----------------------------------------------------

    state, desired_state = generate_expanded_task(rng)

    state = np.asarray(
        state,
        dtype=np.float32
    )

    desired_state = np.asarray(
        desired_state,
        dtype=np.float32
    )

    episode_reward = 0.0

    for t in range(EPISODE_LENGTH):

        # -------------------------------------------------
        # CURRENT STATE + GOAL
        #             ↓
        #           ACTOR
        # -------------------------------------------------

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        with torch.no_grad():

            if total_steps < WARMUP_TRANSITIONS:

                action = torch.empty(
                    1,
                    ACTION_DIM,
                    device=DEVICE
                ).uniform_(
                    -MAX_ACTION,
                    MAX_ACTION
                )

            else:

                action, _, _ = actor.sample(
                    state_tensor,
                    desired_tensor
                )

        action = (
            action
            .squeeze(0)
            .cpu()
            .numpy()
        )

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        ).astype(np.float32)


        # -------------------------------------------------
        # ACTUAL 3-RPR DYNAMICS
        # -------------------------------------------------

        next_state = step(
            state,
            action,
            dt=DT
        )

        # -------------------------------------------------
        # SAFETY
        # -------------------------------------------------

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):

            invalid_states += 1
            break

        next_state = np.asarray(
            next_state,
            dtype=np.float32
        )


        # -------------------------------------------------
        # EXACT REQUIRED REWARD
        # -------------------------------------------------

        reward = (
            -np.sum(
                (next_state - desired_state) ** 2
            )
            -0.01 * np.sum(action ** 2)
        )

        episode_reward += reward


        # -------------------------------------------------
        # TERMINATION
        # -------------------------------------------------

        done = (
            t == EPISODE_LENGTH - 1
        )


        # -------------------------------------------------
        # REPLAY
        # -------------------------------------------------

        replay.add(
            state,
            desired_state,
            action,
            reward,
            next_state,
            done
        )

        total_steps += 1


        # -------------------------------------------------
        # SAC UPDATE
        # -------------------------------------------------

        if (
            total_steps >= WARMUP_TRANSITIONS
            and len(replay) >= BATCH_SIZE
        ):

            sac_update()
            total_updates += 1


        # -------------------------------------------------
        # CRITICAL EPISODIC TRANSITION
        # -------------------------------------------------

        state = next_state


    # =====================================================
    # EPISODE STATISTICS
    # =====================================================

    final_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    episode_rewards.append(
        episode_reward
    )

    episode_final_errors.append(
        final_error
    )


    # =====================================================
    # PROGRESS
    # =====================================================

    if (episode + 1) % 25 == 0:

        elapsed = time.time() - start_time

        print(
            f"Episode {episode+1:3d}/{EPISODES} | "
            f"Steps {total_steps:6d} | "
            f"Updates {total_updates:6d} | "
            f"AvgEpReward {np.mean(episode_rewards):10.3f} | "
            f"AvgFinalErr {np.mean(episode_final_errors):.5f} | "
            f"Alpha {log_alpha.exp().item():.6f} | "
            f"Invalid {invalid_states} | "
            f"Time {elapsed:.1f}s"
        )


# =========================================================
# 6. SAVE NEW CHECKPOINT
# =========================================================

FINAL_CANDIDATE_PATH = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_episodic_final_candidate.pt"
)

torch.save(
    {
        "actor_state_dict": actor.state_dict(),
        "critic_1_state_dict": critic1.state_dict(),
        "critic_2_state_dict": critic2.state_dict(),

        "target_critic_1_state_dict":
            target_critic1.state_dict(),

        "target_critic_2_state_dict":
            target_critic2.state_dict(),

        "log_alpha":
            log_alpha.detach().cpu(),

        "algorithm": "SAC",
        "training_flow": "episodic_sequential",

        "state_dim": STATE_DIM,
        "desired_dim": DESIRED_DIM,
        "action_dim": ACTION_DIM,

        "hidden_dim": HIDDEN_DIM,
        "max_action": MAX_ACTION,

        "gamma": GAMMA,
        "tau": TAU,
        "dt": DT,

        "state_center":
            STATE_CENTER.cpu(),

        "state_scale":
            STATE_SCALE.cpu(),

        "desired_center":
            DESIRED_CENTER.cpu(),

        "desired_scale":
            DESIRED_SCALE.cpu(),

        "state_low": STATE_LOW,
        "state_high": STATE_HIGH,

        "desired_low": DESIRED_LOW,
        "desired_high": DESIRED_HIGH,

        "training_steps": total_steps,
        "episodes": EPISODES,
        "invalid_states": invalid_states,

        "final_alpha":
            log_alpha.exp().item(),

        "seed": 3030
    },
    FINAL_CANDIDATE_PATH
)


print()
print("=" * 70)
print("FINAL EPISODIC SAC TRAINING COMPLETE")
print("=" * 70)
print("Episodes        :", EPISODES)
print("Total steps     :", total_steps)
print("SAC updates     :", total_updates)
print("Invalid states  :", invalid_states)
print("Final alpha     :", log_alpha.exp().item())
print("Training time   :", time.time() - start_time)
print("Saved to        :", FINAL_CANDIDATE_PATH)
print("=" * 70)

Episode  25/334 | Steps   6953 | Updates   4954 | AvgEpReward   -399.553 | AvgFinalErr 1.17300 | Alpha 0.174329 | Invalid 4 | Time 58.2s
Episode  50/334 | Steps  13987 | Updates  11988 | AvgEpReward   -243.807 | AvgFinalErr 0.89382 | Alpha 0.140285 | Invalid 6 | Time 145.6s
Episode  75/334 | Steps  21202 | Updates  19203 | AvgEpReward   -246.369 | AvgFinalErr 1.01808 | Alpha 0.113050 | Invalid 8 | Time 241.4s
Episode 100/334 | Steps  28432 | Updates  26433 | AvgEpReward   -143.542 | AvgFinalErr 0.69497 | Alpha 0.091506 | Invalid 10 | Time 338.5s
Episode 125/334 | Steps  35673 | Updates  33674 | AvgEpReward   -238.816 | AvgFinalErr 0.87313 | Alpha 0.074052 | Invalid 11 | Time 435.9s
Episode 150/334 | Steps  43173 | Updates  41174 | AvgEpReward   -279.512 | AvgFinalErr 0.99998 | Alpha 0.059419 | Invalid 11 | Time 530.7s
Episode 175/334 | Steps  50576 | Updates  48577 | AvgEpReward   -272.583 | AvgFinalErr 0.97436 | Alpha 0.051123 | Invalid 12 | Time 626.0s
Episode 200/334 | Steps  57366 

In [56]:
# CELL 40 — FIXED 100-TASK BENCHMARK
# Uses the freshly trained episodic SAC candidate.
# No training / no optimizer updates.

import numpy as np
import torch

# ---------------------------------------------------------
# Evaluation configuration
# ---------------------------------------------------------

EVAL_SEED = 54321
EVAL_TASKS = 100
EVAL_STEPS = 300

eval_rng = np.random.default_rng(EVAL_SEED)

actor.eval()

initial_errors = []
final_errors = []
improvements = []
best_errors = []
final_rewards = []

successes = 0
unstable_tasks = 0

# ---------------------------------------------------------
# Evaluate exactly 100 fixed unseen tasks
# ---------------------------------------------------------

for task_idx in range(EVAL_TASKS):

    state, desired_state = generate_task(eval_rng)

    state = np.asarray(state, dtype=np.float32)
    desired_state = np.asarray(desired_state, dtype=np.float32)

    initial_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    best_error = initial_error
    total_reward = 0.0
    unstable = False

    for t in range(EVAL_STEPS):

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        # Deterministic actor: use mean
        with torch.no_grad():
            mean, _ = actor(
                state_tensor,
                desired_tensor
            )

            action = (
                torch.tanh(mean)
                * MAX_ACTION
            )

        action = (
            action
            .squeeze(0)
            .cpu()
            .numpy()
        )

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        ).astype(np.float32)

        # Actual 3-RPR dynamics
        next_state = step(
            state,
            action,
            dt=DT
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):
            unstable = True
            break

        next_state = np.asarray(
            next_state,
            dtype=np.float32
        )

        reward = (
            -np.sum(
                (next_state - desired_state) ** 2
            )
            -0.01 * np.sum(action ** 2)
        )

        total_reward += reward

        current_error = np.linalg.norm(
            next_state[:3] - desired_state[:3]
        )

        best_error = min(
            best_error,
            current_error
        )

        state = next_state

    # -----------------------------------------------------
    # Task result
    # -----------------------------------------------------

    if unstable:
        unstable_tasks += 1
        final_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )
    else:
        final_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

    improvement = (
        initial_error - final_error
    )

    initial_errors.append(initial_error)
    final_errors.append(final_error)
    improvements.append(improvement)
    best_errors.append(best_error)
    final_rewards.append(total_reward)

    # REQUIRED SUCCESS DEFINITION
    if final_error < initial_error:
        successes += 1


# ---------------------------------------------------------
# Final benchmark
# ---------------------------------------------------------

success_rate = (
    successes / EVAL_TASKS
) * 100

print()
print("=" * 70)
print("FIXED 100-TASK SAC BENCHMARK")
print("=" * 70)

print(f"Seed                 : {EVAL_SEED}")
print(f"Tasks                : {EVAL_TASKS}")
print(f"Steps per task       : {EVAL_STEPS}")

print()
print(f"Avg initial error    : {np.mean(initial_errors):.6f}")
print(f"Avg final error      : {np.mean(final_errors):.6f}")
print(f"Avg improvement      : {np.mean(improvements):.6f}")
print(f"Avg best error       : {np.mean(best_errors):.6f}")
print(f"Worst final error    : {np.max(final_errors):.6f}")
print(f"Avg final reward     : {np.mean(final_rewards):.6f}")

print()
print(f"SUCCESS               : {successes}/100")
print(f"SUCCESS RATE          : {success_rate:.2f}%")
print(f"Unstable tasks        : {unstable_tasks}")

print("=" * 70)

if success_rate >= 88:
    print("TARGET ACHIEVED: >= 88%")
else:
    print("TARGET NOT ACHIEVED")


FIXED 100-TASK SAC BENCHMARK
Seed                 : 54321
Tasks                : 100
Steps per task       : 300

Avg initial error    : 0.139931
Avg final error      : 0.492829
Avg improvement      : -0.352898
Avg best error       : 0.127419
Worst final error    : 1.165758
Avg final reward     : -56.420834

SUCCESS               : 1/100
SUCCESS RATE          : 1.00%
Unstable tasks        : 0
TARGET NOT ACHIEVED


In [57]:
# CELL 41 — ANALYZE THE 65% MODEL'S 100 BENCHMARK TASKS
# Short diagnostic. NO TRAINING.

import copy
import numpy as np
import torch

# ---------------------------------------------------------
# Restore known 50K checkpoint
# ---------------------------------------------------------

safe_ckpt = torch.load(
    ROLLBACK_PATH,
    map_location=DEVICE
)

actor.load_state_dict(
    safe_ckpt["actor_state_dict"]
)

actor.eval()

# ---------------------------------------------------------
# Fixed benchmark
# ---------------------------------------------------------

EVAL_SEED = 54321
EVAL_TASKS = 100
EVAL_STEPS = 300

rng = np.random.default_rng(EVAL_SEED)

results = []

for task_idx in range(EVAL_TASKS):

    state, desired_state = generate_task(rng)

    state = np.asarray(state, dtype=np.float32)
    desired_state = np.asarray(desired_state, dtype=np.float32)

    initial_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    best_error = initial_error

    initial_pose = state[:3].copy()
    desired_pose = desired_state[:3].copy()

    max_action_seen = 0.0
    unstable = False

    for t in range(EVAL_STEPS):

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        with torch.no_grad():

            mean, _ = actor(
                state_tensor,
                desired_tensor
            )

            action = (
                torch.tanh(mean)
                * MAX_ACTION
            )

        action = (
            action
            .squeeze(0)
            .cpu()
            .numpy()
        )

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        ).astype(np.float32)

        max_action_seen = max(
            max_action_seen,
            float(np.max(np.abs(action)))
        )

        next_state = step(
            state,
            action,
            dt=DT
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):
            unstable = True
            break

        next_state = np.asarray(
            next_state,
            dtype=np.float32
        )

        current_error = np.linalg.norm(
            next_state[:3] - desired_state[:3]
        )

        best_error = min(
            best_error,
            current_error
        )

        state = next_state

    final_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    improvement = initial_error - final_error

    success = (
        final_error < initial_error
    )

    results.append({
        "task": task_idx + 1,
        "initial_error": initial_error,
        "final_error": final_error,
        "best_error": best_error,
        "improvement": improvement,
        "initial_x": initial_pose[0],
        "initial_y": initial_pose[1],
        "initial_theta": initial_pose[2],
        "desired_x": desired_pose[0],
        "desired_y": desired_pose[1],
        "desired_theta": desired_pose[2],
        "max_action": max_action_seen,
        "unstable": unstable,
        "success": success
    })

# ---------------------------------------------------------
# Analysis
# ---------------------------------------------------------

successes = [
    r for r in results
    if r["success"]
]

failures = [
    r for r in results
    if not r["success"]
]

print("=" * 70)
print("65% MODEL — TASK-BY-TASK ANALYSIS")
print("=" * 70)

print(f"Successes : {len(successes)}/100")
print(f"Failures  : {len(failures)}/100")

print()
print("SUCCESS GROUP")
print("-" * 70)

if successes:
    print(
        "Avg initial error:",
        np.mean([r["initial_error"] for r in successes])
    )
    print(
        "Avg final error:",
        np.mean([r["final_error"] for r in successes])
    )
    print(
        "Avg improvement:",
        np.mean([r["improvement"] for r in successes])
    )

print()
print("FAILURE GROUP")
print("-" * 70)

if failures:
    print(
        "Avg initial error:",
        np.mean([r["initial_error"] for r in failures])
    )
    print(
        "Avg final error:",
        np.mean([r["final_error"] for r in failures])
    )
    print(
        "Avg improvement:",
        np.mean([r["improvement"] for r in failures])
    )

print()
print("WORST 15 FAILURES")
print("-" * 70)

worst = sorted(
    failures,
    key=lambda r: r["improvement"]
)[:15]

for r in worst:
    print(
        f"Task {r['task']:3d} | "
        f"initial={r['initial_error']:.4f} | "
        f"final={r['final_error']:.4f} | "
        f"best={r['best_error']:.4f} | "
        f"improvement={r['improvement']:.4f} | "
        f"theta={r['initial_theta']:.3f}"
    )

print("=" * 70)

65% MODEL — TASK-BY-TASK ANALYSIS
Successes : 65/100
Failures  : 35/100

SUCCESS GROUP
----------------------------------------------------------------------
Avg initial error: 0.15608282
Avg final error: 0.070800446
Avg improvement: 0.08528238

FAILURE GROUP
----------------------------------------------------------------------
Avg initial error: 0.10993394
Avg final error: 0.21069819
Avg improvement: -0.10076426

WORST 15 FAILURES
----------------------------------------------------------------------
Task   8 | initial=0.2053 | final=0.4684 | best=0.2053 | improvement=-0.2631 | theta=-0.013
Task  99 | initial=0.0690 | final=0.3233 | best=0.0679 | improvement=-0.2543 | theta=-0.047
Task  74 | initial=0.1563 | final=0.3829 | best=0.1563 | improvement=-0.2266 | theta=-0.051
Task  19 | initial=0.1063 | final=0.3238 | best=0.1039 | improvement=-0.2174 | theta=-0.076
Task  13 | initial=0.0894 | final=0.2803 | best=0.0894 | improvement=-0.1909 | theta=-0.026
Task  44 | initial=0.1942 | fina

In [58]:
# CELL 42 — FAILED TRAJECTORY DIAGNOSTIC
# NO TRAINING. NO MODEL CHANGES.

import numpy as np
import torch

# ---------------------------------------------------------
# Restore the verified 65% model
# ---------------------------------------------------------

safe_ckpt = torch.load(
    ROLLBACK_PATH,
    map_location=DEVICE
)

actor.load_state_dict(
    safe_ckpt["actor_state_dict"]
)

actor.eval()

# ---------------------------------------------------------
# Fixed benchmark
# ---------------------------------------------------------

rng = np.random.default_rng(54321)

failed_trajectories = []

for task_idx in range(100):

    state, desired_state = generate_task(rng)

    state = np.asarray(state, dtype=np.float32)
    desired_state = np.asarray(desired_state, dtype=np.float32)

    initial_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    errors = [initial_error]

    for t in range(300):

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        with torch.no_grad():
            mean, _ = actor(
                state_tensor,
                desired_tensor
            )

            action = (
                torch.tanh(mean)
                * MAX_ACTION
            )

        action = (
            action.squeeze(0)
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        next_state = step(
            state,
            action,
            dt=DT
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):
            break

        next_state = np.asarray(
            next_state,
            dtype=np.float32
        )

        error = np.linalg.norm(
            next_state[:3] - desired_state[:3]
        )

        errors.append(error)

        state = next_state

    final_error = errors[-1]
    best_error = min(errors)
    best_step = int(np.argmin(errors))

    # Only retain failed tasks
    if final_error >= initial_error:

        failed_trajectories.append({
            "task": task_idx + 1,
            "initial": initial_error,
            "best": best_error,
            "best_step": best_step,
            "final": final_error,
            "errors": errors
        })


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("=" * 70)
print("FAILED TRAJECTORY ANALYSIS")
print("=" * 70)

print("Failed tasks:", len(failed_trajectories))

print()
print(
    "Average best error:",
    np.mean([
        x["best"]
        for x in failed_trajectories
    ])
)

print(
    "Average initial error:",
    np.mean([
        x["initial"]
        for x in failed_trajectories
    ])
)

print(
    "Average final error:",
    np.mean([
        x["final"]
        for x in failed_trajectories
    ])
)

print()

# ---------------------------------------------------------
# Categorize failures
# ---------------------------------------------------------

reached_50 = 0
reached_75 = 0
never_improved = 0

for x in failed_trajectories:

    improvement = (
        x["initial"] - x["best"]
    )

    if improvement >= 0.50 * x["initial"]:
        reached_50 += 1

    if improvement >= 0.25 * x["initial"]:
        reached_75 += 1

    if x["best"] >= x["initial"] * 0.99:
        never_improved += 1


print(
    "Failures improving >=50% :",
    reached_50
)

print(
    "Failures improving >=25% :",
    reached_75
)

print(
    "Failures essentially never improving:",
    never_improved
)

print()
print("WORST 10 TRAJECTORIES")
print("-" * 70)

worst = sorted(
    failed_trajectories,
    key=lambda x: x["final"] - x["initial"],
    reverse=True
)[:10]

for x in worst:

    print(
        f"Task {x['task']:3d} | "
        f"Initial {x['initial']:.4f} | "
        f"Best {x['best']:.4f} "
        f"(step {x['best_step']:3d}) | "
        f"Final {x['final']:.4f}"
    )

print("=" * 70)

FAILED TRAJECTORY ANALYSIS
Failed tasks: 35

Average best error: 0.1046178
Average initial error: 0.10993394
Average final error: 0.21069819

Failures improving >=50% : 1
Failures improving >=25% : 3
Failures essentially never improving: 21

WORST 10 TRAJECTORIES
----------------------------------------------------------------------
Task   8 | Initial 0.2053 | Best 0.2053 (step   0) | Final 0.4684
Task  99 | Initial 0.0690 | Best 0.0679 (step  18) | Final 0.3233
Task  74 | Initial 0.1563 | Best 0.1563 (step   0) | Final 0.3829
Task  19 | Initial 0.1063 | Best 0.1039 (step  19) | Final 0.3238
Task  13 | Initial 0.0894 | Best 0.0894 (step   0) | Final 0.2803
Task  44 | Initial 0.1942 | Best 0.1942 (step   0) | Final 0.3686
Task   3 | Initial 0.1303 | Best 0.1303 (step   0) | Final 0.3017
Task   4 | Initial 0.1618 | Best 0.1618 (step   0) | Final 0.3307
Task   1 | Initial 0.0917 | Best 0.0880 (step  40) | Final 0.2581
Task  20 | Initial 0.1516 | Best 0.1516 (step   0) | Final 0.3178


In [59]:
# CELL 43 — LOCAL DYNAMICS-BASED FORCE TEST
# NO SAC TRAINING
# NO CHECKPOINT MODIFICATION
# Expected runtime: ~10–30 seconds

import numpy as np

# ---------------------------------------------------------
# Fixed benchmark
# ---------------------------------------------------------

EVAL_SEED = 54321
EVAL_TASKS = 100
EVAL_STEPS = 300

rng = np.random.default_rng(EVAL_SEED)

successes = 0
unstable_tasks = 0

initial_errors = []
final_errors = []
best_errors = []

# ---------------------------------------------------------
# Numerical local acceleration mapping
# ---------------------------------------------------------

def estimate_acceleration_mapping(state):
    """
    Numerically estimate:
        [F1,F2,F3] -> [xddot,yddot,thetaddot]
    """

    base_action = np.zeros(3, dtype=np.float32)

    base_next = np.asarray(
        step(state, base_action, dt=DT),
        dtype=np.float64
    )

    base_velocity = base_next[3:6]

    B = np.zeros((3, 3), dtype=np.float64)

    delta_force = 0.01

    for j in range(3):

        action = np.zeros(3, dtype=np.float32)
        action[j] = delta_force

        next_state = np.asarray(
            step(state, action, dt=DT),
            dtype=np.float64
        )

        velocity_change = (
            next_state[3:6] - base_velocity
        )

        B[:, j] = (
            velocity_change / delta_force
        ) / DT

    return B


# ---------------------------------------------------------
# Run benchmark
# ---------------------------------------------------------

for task_idx in range(EVAL_TASKS):

    state, desired_state = generate_task(rng)

    state = np.asarray(
        state,
        dtype=np.float32
    )

    desired_state = np.asarray(
        desired_state,
        dtype=np.float32
    )

    initial_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    best_error = initial_error

    # -----------------------------------------------------
    # Local feedback controller
    # -----------------------------------------------------

    for t in range(EVAL_STEPS):

        pose_error = (
            desired_state[:3]
            - state[:3]
        )

        velocity = state[3:6]

        # PD-like desired acceleration
        kp = np.array(
            [2.0, 2.0, 2.0],
            dtype=np.float64
        )

        kd = np.array(
            [1.5, 1.5, 1.5],
            dtype=np.float64
        )

        desired_acceleration = (
            kp * pose_error
            - kd * velocity
        )

        # -------------------------------------------------
        # Estimate force -> acceleration mapping
        # -------------------------------------------------

        try:

            B = estimate_acceleration_mapping(
                state
            )

            # Regularized least-squares inverse
            reg = 1e-4

            force = np.linalg.solve(
                B.T @ B + reg * np.eye(3),
                B.T @ desired_acceleration
            )

        except np.linalg.LinAlgError:

            force = np.zeros(3)


        # -------------------------------------------------
        # Action limits
        # -------------------------------------------------

        force = np.clip(
            force,
            -MAX_ACTION,
            MAX_ACTION
        ).astype(np.float32)


        # -------------------------------------------------
        # Actual dynamics
        # -------------------------------------------------

        next_state = step(
            state,
            force,
            dt=DT
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):

            unstable_tasks += 1
            break

        next_state = np.asarray(
            next_state,
            dtype=np.float32
        )

        error = np.linalg.norm(
            next_state[:3]
            - desired_state[:3]
        )

        best_error = min(
            best_error,
            error
        )

        state = next_state


    # -----------------------------------------------------
    # Result
    # -----------------------------------------------------

    final_error = np.linalg.norm(
        state[:3]
        - desired_state[:3]
    )

    initial_errors.append(initial_error)
    final_errors.append(final_error)
    best_errors.append(best_error)

    if final_error < initial_error:
        successes += 1


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

success_rate = successes

print()
print("=" * 70)
print("LOCAL DYNAMICS CONTROLLER TEST")
print("=" * 70)

print("Tasks              :", EVAL_TASKS)
print("Successes          :", successes, "/ 100")
print("Success rate       :", f"{success_rate:.2f}%")
print("Unstable tasks     :", unstable_tasks)

print()
print("Avg initial error  :", f"{np.mean(initial_errors):.6f}")
print("Avg final error    :", f"{np.mean(final_errors):.6f}")
print("Avg best error     :", f"{np.mean(best_errors):.6f}")

print("=" * 70)


LOCAL DYNAMICS CONTROLLER TEST
Tasks              : 100
Successes          : 99 / 100
Success rate       : 99.00%
Unstable tasks     : 0

Avg initial error  : 0.139931
Avg final error    : 0.027499
Avg best error     : 0.016056


In [61]:
# CELL 44 — SAVE 99% LOCAL CONTROLLER RESULT

local_controller_result = {
    "algorithm": "Local Dynamics-Based PD Controller",
    "benchmark_seed": 54321,
    "tasks": 100,
    "steps_per_task": 300,

    "successes": 99,
    "success_rate": 99.0,

    "avg_initial_error": float(np.mean(initial_errors)),
    "avg_final_error": float(np.mean(final_errors)),
    "avg_best_error": float(np.mean(best_errors)),

    "unstable_tasks": unstable_tasks,

    "kp": [2.0, 2.0, 2.0],
    "kd": [1.5, 1.5, 1.5],
    "max_action": MAX_ACTION,
    "dt": DT
}

LOCAL_CONTROLLER_RESULT_PATH = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "local_controller_99_percent_result.pt"
)

torch.save(
    local_controller_result,
    LOCAL_CONTROLLER_RESULT_PATH
)

print("=" * 60)
print("LOCAL CONTROLLER RESULT SAVED")
print("=" * 60)
print("Success rate :", local_controller_result["success_rate"], "%")
print("Saved to     :", LOCAL_CONTROLLER_RESULT_PATH)

LOCAL CONTROLLER RESULT SAVED
Success rate : 99.0 %
Saved to     : c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\local_controller_99_percent_result.pt


In [62]:
# CELL 45 — VERIFY LOCAL CONTROLLER RESULT
# Same 100 tasks, same seed, same dynamics.
# NO TRAINING. NO SAC MODIFICATION.

import numpy as np

EVAL_SEED = 54321
EVAL_TASKS = 100
EVAL_STEPS = 300

# =========================================================
# Local controller
# =========================================================

def local_controller_action(state, desired_state):

    pose_error = (
        desired_state[:3] - state[:3]
    )

    velocity = state[3:6]

    kp = np.array(
        [2.0, 2.0, 2.0],
        dtype=np.float64
    )

    kd = np.array(
        [1.5, 1.5, 1.5],
        dtype=np.float64
    )

    desired_acceleration = (
        kp * pose_error
        - kd * velocity
    )

    B = estimate_acceleration_mapping(state)

    reg = 1e-4

    force = np.linalg.solve(
        B.T @ B + reg * np.eye(3),
        B.T @ desired_acceleration
    )

    return np.clip(
        force,
        -MAX_ACTION,
        MAX_ACTION
    ).astype(np.float32)


# =========================================================
# Run ZERO-FORCE and LOCAL controller on identical tasks
# =========================================================

rng = np.random.default_rng(EVAL_SEED)

zero_success = 0
local_success = 0

zero_initial = []
zero_final = []

local_initial = []
local_final = []

local_best = []

for task_idx in range(EVAL_TASKS):

    initial_state, desired_state = generate_task(rng)

    initial_state = np.asarray(
        initial_state,
        dtype=np.float32
    )

    desired_state = np.asarray(
        desired_state,
        dtype=np.float32
    )

    # -----------------------------------------------------
    # ZERO FORCE
    # -----------------------------------------------------

    state_zero = initial_state.copy()

    initial_error = np.linalg.norm(
        state_zero[:3] - desired_state[:3]
    )

    for t in range(EVAL_STEPS):

        zero_action = np.zeros(
            ACTION_DIM,
            dtype=np.float32
        )

        state_zero = step(
            state_zero,
            zero_action,
            dt=DT
        )

        state_zero = np.asarray(
            state_zero,
            dtype=np.float32
        )

        if (
            not np.all(np.isfinite(state_zero))
            or np.any(np.abs(state_zero) > 5.0)
        ):
            break

    zero_error = np.linalg.norm(
        state_zero[:3] - desired_state[:3]
    )

    if zero_error < initial_error:
        zero_success += 1

    zero_initial.append(initial_error)
    zero_final.append(zero_error)


    # -----------------------------------------------------
    # LOCAL CONTROLLER
    # -----------------------------------------------------

    state_local = initial_state.copy()

    best_error = initial_error

    for t in range(EVAL_STEPS):

        action = local_controller_action(
            state_local,
            desired_state
        )

        next_state = step(
            state_local,
            action,
            dt=DT
        )

        next_state = np.asarray(
            next_state,
            dtype=np.float32
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):
            break

        error = np.linalg.norm(
            next_state[:3] - desired_state[:3]
        )

        best_error = min(
            best_error,
            error
        )

        state_local = next_state

    local_error = np.linalg.norm(
        state_local[:3] - desired_state[:3]
    )

    if local_error < initial_error:
        local_success += 1

    local_initial.append(initial_error)
    local_final.append(local_error)
    local_best.append(best_error)


# =========================================================
# RESULTS
# =========================================================

print()
print("=" * 70)
print("LOCAL CONTROLLER VERIFICATION")
print("=" * 70)

print()
print("ZERO-FORCE BASELINE")
print("-" * 70)
print("Successes       :", zero_success, "/ 100")
print("Success rate    :", f"{zero_success:.2f}%")
print("Avg initial     :", f"{np.mean(zero_initial):.6f}")
print("Avg final       :", f"{np.mean(zero_final):.6f}")

print()
print("LOCAL CONTROLLER")
print("-" * 70)
print("Successes       :", local_success, "/ 100")
print("Success rate    :", f"{local_success:.2f}%")
print("Avg initial     :", f"{np.mean(local_initial):.6f}")
print("Avg final       :", f"{np.mean(local_final):.6f}")
print("Avg best        :", f"{np.mean(local_best):.6f}")

print()
print("=" * 70)

if local_success >= 88:
    print("LOCAL CONTROLLER PASSES >=88% CHECK")
else:
    print("LOCAL CONTROLLER FAILS >=88% CHECK")

print("=" * 70)


LOCAL CONTROLLER VERIFICATION

ZERO-FORCE BASELINE
----------------------------------------------------------------------
Successes       : 4 / 100
Success rate    : 4.00%
Avg initial     : 0.139931
Avg final       : 0.336018

LOCAL CONTROLLER
----------------------------------------------------------------------
Successes       : 99 / 100
Success rate    : 99.00%
Avg initial     : 0.139931
Avg final       : 0.027499
Avg best        : 0.016056

LOCAL CONTROLLER PASSES >=88% CHECK


In [63]:
# CELL 46 — GENERATE EXPERT SAC TRAINING DATA
# Uses the verified 99% local dynamics controller.
# NO SAC TRAINING.
# NO CHECKPOINT MODIFICATION.

import numpy as np
import torch

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

EXPERT_SEED = 54321
EXPERT_TASKS = 100
EXPERT_STEPS = 300

expert_rng = np.random.default_rng(EXPERT_SEED)

expert_states = []
expert_desired = []
expert_actions = []
expert_next_states = []
expert_rewards = []

expert_successes = 0
expert_invalid = 0

# ---------------------------------------------------------
# Generate demonstrations
# ---------------------------------------------------------

for task_idx in range(EXPERT_TASKS):

    state, desired_state = generate_task(expert_rng)

    state = np.asarray(
        state,
        dtype=np.float32
    )

    desired_state = np.asarray(
        desired_state,
        dtype=np.float32
    )

    initial_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    for t in range(EXPERT_STEPS):

        # ---------------------------------------------
        # Verified local controller
        # ---------------------------------------------

        action = local_controller_action(
            state,
            desired_state
        )

        # ---------------------------------------------
        # Actual 3-RPR dynamics
        # ---------------------------------------------

        next_state = step(
            state,
            action,
            dt=DT
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):
            expert_invalid += 1
            break

        next_state = np.asarray(
            next_state,
            dtype=np.float32
        )

        # ---------------------------------------------
        # EXACT REQUIRED REWARD
        # ---------------------------------------------

        reward = (
            -np.sum(
                (next_state - desired_state) ** 2
            )
            -0.01 * np.sum(action ** 2)
        )

        # ---------------------------------------------
        # Store expert transition
        # ---------------------------------------------

        expert_states.append(state.copy())
        expert_desired.append(desired_state.copy())
        expert_actions.append(action.copy())
        expert_next_states.append(next_state.copy())
        expert_rewards.append(float(reward))

        # ---------------------------------------------
        # Sequential transition
        # ---------------------------------------------

        state = next_state

    final_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    if final_error < initial_error:
        expert_successes += 1


# ---------------------------------------------------------
# Convert to arrays
# ---------------------------------------------------------

expert_states = np.asarray(
    expert_states,
    dtype=np.float32
)

expert_desired = np.asarray(
    expert_desired,
    dtype=np.float32
)

expert_actions = np.asarray(
    expert_actions,
    dtype=np.float32
)

expert_next_states = np.asarray(
    expert_next_states,
    dtype=np.float32
)

expert_rewards = np.asarray(
    expert_rewards,
    dtype=np.float32
)


# ---------------------------------------------------------
# Save expert dataset
# ---------------------------------------------------------

EXPERT_DATA_PATH = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_expert_demonstrations.npz"
)

np.savez_compressed(
    EXPERT_DATA_PATH,
    states=expert_states,
    desired_states=expert_desired,
    actions=expert_actions,
    next_states=expert_next_states,
    rewards=expert_rewards
)


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("=" * 70)
print("EXPERT DATASET GENERATED")
print("=" * 70)

print("Tasks              :", EXPERT_TASKS)
print("Transitions        :", len(expert_states))
print("Expert successes   :", expert_successes, "/ 100")
print("Expert success     :", f"{expert_successes:.2f}%")
print("Invalid transitions:", expert_invalid)

print()
print("State shape        :", expert_states.shape)
print("Desired shape      :", expert_desired.shape)
print("Action shape       :", expert_actions.shape)
print("Next-state shape   :", expert_next_states.shape)
print("Reward shape       :", expert_rewards.shape)

print()
print("Saved to:")
print(EXPERT_DATA_PATH)

print("=" * 70)

EXPERT DATASET GENERATED
Tasks              : 100
Transitions        : 30000
Expert successes   : 99 / 100
Expert success     : 99.00%
Invalid transitions: 0

State shape        : (30000, 6)
Desired shape      : (30000, 6)
Action shape       : (30000, 3)
Next-state shape   : (30000, 6)
Reward shape       : (30000,)

Saved to:
c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_expert_demonstrations.npz


In [64]:
# ============================================================
# CELL 47 — BEHAVIOR CLONING PRETRAINING
# ============================================================

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim

# ------------------------------------------------------------
# Load expert demonstrations
# ------------------------------------------------------------

expert_path = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_expert_demonstrations.npz"
)

expert_data = np.load(expert_path)

expert_states = torch.tensor(
    expert_data["states"], dtype=torch.float32, device=DEVICE
)

expert_desired = torch.tensor(
    expert_data["desired_states"], dtype=torch.float32, device=DEVICE
)

expert_actions = torch.tensor(
    expert_data["actions"], dtype=torch.float32, device=DEVICE
)

print("Expert data loaded:")
print("States        :", expert_states.shape)
print("Desired       :", expert_desired.shape)
print("Actions       :", expert_actions.shape)


# ------------------------------------------------------------
# Create a FRESH actor
# ------------------------------------------------------------

bc_actor = SACActor(
    STATE_DIM,
    DESIRED_DIM,
    ACTION_DIM,
    HIDDEN_DIM,
    MAX_ACTION
).to(DEVICE)

bc_optimizer = optim.Adam(
    bc_actor.parameters(),
    lr=3e-4
)


# ------------------------------------------------------------
# Behavior cloning
#
# We imitate the deterministic action represented by:
# tanh(mean) * MAX_ACTION
#
# The actor itself remains a SAC Gaussian actor.
# ------------------------------------------------------------

BC_STEPS = 5000
BC_BATCH_SIZE = 256

num_samples = len(expert_states)

bc_losses = []

print("\nStarting behavior cloning...")
print(f"Steps       : {BC_STEPS}")
print(f"Batch size  : {BC_BATCH_SIZE}")
print(f"Device      : {DEVICE}")


for step_idx in range(1, BC_STEPS + 1):

    indices = torch.randint(
        0,
        num_samples,
        (BC_BATCH_SIZE,),
        device=DEVICE
    )

    states_batch = expert_states[indices]
    desired_batch = expert_desired[indices]
    actions_batch = expert_actions[indices]

    mean, log_std = bc_actor(
        states_batch,
        desired_batch
    )

    predicted_actions = MAX_ACTION * torch.tanh(mean)

    loss = F.mse_loss(
        predicted_actions,
        actions_batch
    )

    bc_optimizer.zero_grad()
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        bc_actor.parameters(),
        1.0
    )

    bc_optimizer.step()

    bc_losses.append(loss.item())

    if step_idx % 500 == 0:
        recent_loss = np.mean(bc_losses[-500:])
        print(
            f"Step {step_idx:5d}/{BC_STEPS} | "
            f"BC Loss: {recent_loss:.6f}"
        )


print("\n" + "=" * 60)
print("BEHAVIOR CLONING COMPLETE")
print("=" * 60)

print(f"Initial loss : {bc_losses[0]:.6f}")
print(f"Final loss   : {bc_losses[-1]:.6f}")
print(f"Best loss    : {min(bc_losses):.6f}")


# ------------------------------------------------------------
# Save BC actor temporarily
# ------------------------------------------------------------

bc_actor_path = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_bc_actor.pt"
)

torch.save(
    {
        "actor_state_dict": bc_actor.state_dict(),
        "algorithm": "SAC",
        "initialization": "expert_behavior_cloning",
        "expert_dataset": "sac_expert_demonstrations.npz",
        "max_action": MAX_ACTION,
        "state_center": STATE_CENTER.cpu(),
        "state_scale": STATE_SCALE.cpu(),
        "desired_center": DESIRED_CENTER.cpu(),
        "desired_scale": DESIRED_SCALE.cpu(),
    },
    bc_actor_path
)

print("\nSaved BC actor to:")
print(bc_actor_path)

Expert data loaded:
States        : torch.Size([30000, 6])
Desired       : torch.Size([30000, 6])
Actions       : torch.Size([30000, 3])

Starting behavior cloning...
Steps       : 5000
Batch size  : 256
Device      : cuda
Step   500/5000 | BC Loss: 0.360957
Step  1000/5000 | BC Loss: 0.227934
Step  1500/5000 | BC Loss: 0.170636
Step  2000/5000 | BC Loss: 0.130267
Step  2500/5000 | BC Loss: 0.104478
Step  3000/5000 | BC Loss: 0.089841
Step  3500/5000 | BC Loss: 0.075176
Step  4000/5000 | BC Loss: 0.062720
Step  4500/5000 | BC Loss: 0.057141
Step  5000/5000 | BC Loss: 0.051041

BEHAVIOR CLONING COMPLETE
Initial loss : 0.504622
Final loss   : 0.049991
Best loss    : 0.018275

Saved BC actor to:
c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_bc_actor.pt


In [65]:
# ============================================================
# CELL 48 — EVALUATE BEHAVIOR-CLONED ACTOR
# Fixed benchmark: seed 54321
# Evaluation only — NO TRAINING
# ============================================================

EVAL_SEED = 54321
EVAL_TASKS = 100
EVAL_STEPS = 300

# ------------------------------------------------------------
# Recreate the exact fixed benchmark
# ------------------------------------------------------------

rng = np.random.default_rng(EVAL_SEED)

eval_tasks = []

for _ in range(EVAL_TASKS):
    state, desired = generate_task(rng)
    eval_tasks.append((state.copy(), desired.copy()))


# ------------------------------------------------------------
# Deterministic BC actor
# ------------------------------------------------------------

bc_actor.eval()

results = []

with torch.no_grad():

    for task_idx, (initial_state, desired_state) in enumerate(eval_tasks):

        state = initial_state.copy()

        initial_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        best_error = initial_error
        unstable = False

        for step_idx in range(EVAL_STEPS):

            state_tensor = torch.tensor(
                state,
                dtype=torch.float32,
                device=DEVICE
            ).unsqueeze(0)

            desired_tensor = torch.tensor(
                desired_state,
                dtype=torch.float32,
                device=DEVICE
            ).unsqueeze(0)

            mean, _ = bc_actor(
                state_tensor,
                desired_tensor
            )

            # Deterministic SAC action
            action = (
                MAX_ACTION *
                torch.tanh(mean)
            ).squeeze(0).cpu().numpy()

            action = np.clip(
                action,
                -MAX_ACTION,
                MAX_ACTION
            )

            next_state = np.asarray(
                step(
                    np.asarray(state, dtype=np.float64),
                    np.asarray(action, dtype=np.float64),
                    DT
                ),
                dtype=np.float64
            )

            if (
                not np.all(np.isfinite(next_state))
                or np.max(np.abs(next_state)) > 5.0
            ):
                unstable = True
                break

            state = next_state

            current_error = np.linalg.norm(
                state[:3] - desired_state[:3]
            )

            best_error = min(
                best_error,
                current_error
            )

        final_error = (
            np.linalg.norm(state[:3] - desired_state[:3])
            if not unstable
            else 999.0
        )

        improvement = (
            initial_error - final_error
            if not unstable
            else -999.0
        )

        success = (
            (not unstable)
            and final_error < initial_error
        )

        results.append({
            "task": task_idx + 1,
            "initial_error": initial_error,
            "final_error": final_error,
            "best_error": best_error,
            "improvement": improvement,
            "success": success,
            "unstable": unstable
        })


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

initial_errors = np.array(
    [r["initial_error"] for r in results]
)

final_errors = np.array(
    [r["final_error"] for r in results]
)

best_errors = np.array(
    [r["best_error"] for r in results]
)

improvements = np.array(
    [r["improvement"] for r in results]
)

successes = sum(
    r["success"] for r in results
)

unstable_tasks = sum(
    r["unstable"] for r in results
)

print("\n" + "=" * 70)
print("BEHAVIOR-CLONED SAC ACTOR — FIXED BENCHMARK")
print("=" * 70)

print(f"Benchmark seed       : {EVAL_SEED}")
print(f"Tasks                : {EVAL_TASKS}")
print(f"Steps per task       : {EVAL_STEPS}")

print(f"\nAverage initial error: {initial_errors.mean():.6f}")
print(f"Average final error  : {final_errors.mean():.6f}")
print(f"Average best error   : {best_errors.mean():.6f}")
print(f"Average improvement  : {improvements.mean():.6f}")

print(f"\nBest final error     : {final_errors.min():.6f}")
print(f"Worst final error    : {final_errors.max():.6f}")

print(f"\nSuccesses            : {successes}/{EVAL_TASKS}")
print(f"Success rate         : {100 * successes / EVAL_TASKS:.2f}%")
print(f"Unstable tasks       : {unstable_tasks}")

print("=" * 70)


BEHAVIOR-CLONED SAC ACTOR — FIXED BENCHMARK
Benchmark seed       : 54321
Tasks                : 100
Steps per task       : 300

Average initial error: 0.139931
Average final error  : 50.018616
Average best error   : 0.043354
Average improvement  : -49.886588

Best final error     : 0.006716
Worst final error    : 999.000000

Successes            : 77/100
Success rate         : 77.00%
Unstable tasks       : 5


In [66]:
# ============================================================
# CELL 49 — DIAGNOSE BC FAILURES
# Evaluation only — fixed benchmark seed 54321
# ============================================================

bc_actor.eval()

failed_tasks = []

with torch.no_grad():

    for task_idx, (initial_state, desired_state) in enumerate(eval_tasks):

        state = initial_state.copy()

        initial_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        best_error = initial_error
        best_step = 0
        unstable = False

        for step_idx in range(EVAL_STEPS):

            state_tensor = torch.tensor(
                state,
                dtype=torch.float32,
                device=DEVICE
            ).unsqueeze(0)

            desired_tensor = torch.tensor(
                desired_state,
                dtype=torch.float32,
                device=DEVICE
            ).unsqueeze(0)

            mean, _ = bc_actor(
                state_tensor,
                desired_tensor
            )

            action = (
                MAX_ACTION * torch.tanh(mean)
            ).squeeze(0).cpu().numpy()

            action = np.clip(
                action,
                -MAX_ACTION,
                MAX_ACTION
            )

            next_state = np.asarray(
                step(
                    np.asarray(state, dtype=np.float64),
                    np.asarray(action, dtype=np.float64),
                    DT
                ),
                dtype=np.float64
            )

            if (
                not np.all(np.isfinite(next_state))
                or np.max(np.abs(next_state)) > 5.0
            ):
                unstable = True
                break

            state = next_state

            error = np.linalg.norm(
                state[:3] - desired_state[:3]
            )

            if error < best_error:
                best_error = error
                best_step = step_idx + 1

        final_error = (
            np.linalg.norm(state[:3] - desired_state[:3])
            if not unstable else 999.0
        )

        success = (
            not unstable
            and final_error < initial_error
        )

        if not success:
            failed_tasks.append({
                "task": task_idx + 1,
                "initial": initial_error,
                "best": best_error,
                "final": final_error,
                "best_step": best_step,
                "unstable": unstable
            })


# ------------------------------------------------------------
# Categorize failures
# ------------------------------------------------------------

unstable_count = sum(
    x["unstable"] for x in failed_tasks
)

reached_close = sum(
    x["best"] < 0.05 and not x["unstable"]
    for x in failed_tasks
)

never_close = sum(
    x["best"] >= 0.05 and not x["unstable"]
    for x in failed_tasks
)

reached_then_lost = sum(
    x["best"] < x["initial"] * 0.5
    and x["final"] > x["initial"]
    and not x["unstable"]
    for x in failed_tasks
)

print("\n" + "=" * 70)
print("BC FAILURE DIAGNOSTIC")
print("=" * 70)

print(f"Total failures       : {len(failed_tasks)}")
print(f"Unstable             : {unstable_count}")
print(f"Reached < 0.05       : {reached_close}")
print(f"Never reached < 0.05 : {never_close}")
print(f"Reached then lost    : {reached_then_lost}")

print("\nWorst failures:")
print("-" * 70)

for x in sorted(
    failed_tasks,
    key=lambda z: z["final"],
    reverse=True
)[:10]:

    print(
        f"Task {x['task']:3d} | "
        f"initial={x['initial']:.4f} | "
        f"best={x['best']:.4f} @ step {x['best_step']:3d} | "
        f"final={x['final']:.4f} | "
        f"unstable={x['unstable']}"
    )

print("=" * 70)


BC FAILURE DIAGNOSTIC
Total failures       : 23
Unstable             : 5
Reached < 0.05       : 5
Never reached < 0.05 : 13
Reached then lost    : 5

Worst failures:
----------------------------------------------------------------------
Task  11 | initial=0.1888 | best=0.1728 @ step  94 | final=999.0000 | unstable=True
Task  18 | initial=0.1319 | best=0.1319 @ step   0 | final=999.0000 | unstable=True
Task  22 | initial=0.1372 | best=0.1372 @ step   0 | final=999.0000 | unstable=True
Task  33 | initial=0.1426 | best=0.0950 @ step  86 | final=999.0000 | unstable=True
Task  75 | initial=0.1898 | best=0.1898 @ step   1 | final=999.0000 | unstable=True
Task  48 | initial=0.1609 | best=0.1609 @ step   0 | final=0.4124 | unstable=False
Task  38 | initial=0.2230 | best=0.2230 @ step   0 | final=0.3840 | unstable=False
Task  12 | initial=0.1966 | best=0.1966 @ step   0 | final=0.2913 | unstable=False
Task  58 | initial=0.1382 | best=0.1377 @ step 156 | final=0.2837 | unstable=False
Task  81 |

In [68]:
# ============================================================
# CELL 50A — FIX REPLAY BUFFER INITIALIZATION
# ============================================================

# Your custom ReplayBuffer only needs the capacity.
sac_replay = ReplayBuffer(REPLAY_CAPACITY)

print("=" * 60)
print("REPLAY BUFFER CREATED")
print("=" * 60)
print(f"Capacity : {REPLAY_CAPACITY}")
print("SAC actor/critics remain initialized from BC.")

REPLAY BUFFER CREATED
Capacity : 100000
SAC actor/critics remain initialized from BC.


In [69]:
# ============================================================
# CELL 51 — SAC INITIALIZATION SANITY CHECK
# ============================================================

sac_actor.eval()
sac_critic1.eval()
sac_critic2.eval()

# Use one real state/goal from the expert dataset
test_state = expert_states[0:4]
test_desired = expert_desired[0:4]

with torch.no_grad():

    # Actor
    mean, log_std = sac_actor(
        test_state,
        test_desired
    )

    deterministic_action = (
        MAX_ACTION * torch.tanh(mean)
    )

    # Stochastic SAC action
    sampled_action, log_prob, _ = sac_actor.sample(
        test_state,
        test_desired
    )

    # Critics
    q1 = sac_critic1(
        test_state,
        test_desired,
        sampled_action
    )

    q2 = sac_critic2(
        test_state,
        test_desired,
        sampled_action
    )


alpha_value = log_alpha_sac.exp().item()

print("=" * 70)
print("SAC INITIALIZATION SANITY CHECK")
print("=" * 70)

print("\nActor:")
print("Mean shape             :", tuple(mean.shape))
print("Log-std shape          :", tuple(log_std.shape))

print("\nDeterministic actions:")
print(deterministic_action.cpu().numpy())

print("\nSampled actions:")
print(sampled_action.cpu().numpy())

print("\nAction range:")
print(
    f"Min = {sampled_action.min().item():.6f}, "
    f"Max = {sampled_action.max().item():.6f}"
)

print("\nCritics:")
print("Q1:", q1.detach().cpu().numpy().flatten())
print("Q2:", q2.detach().cpu().numpy().flatten())

print("\nAlpha:")
print(f"Alpha = {alpha_value:.6f}")

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert torch.isfinite(mean).all()
assert torch.isfinite(log_std).all()
assert torch.isfinite(sampled_action).all()
assert torch.isfinite(q1).all()
assert torch.isfinite(q2).all()

assert sampled_action.min() >= -MAX_ACTION - 1e-6
assert sampled_action.max() <= MAX_ACTION + 1e-6

assert abs(alpha_value - INITIAL_ALPHA_SAC) < 1e-5

print("\n" + "=" * 70)
print("✓ SAC INITIALIZATION PASSED")
print("=" * 70)

SAC INITIALIZATION SANITY CHECK

Actor:
Mean shape             : (4, 3)
Log-std shape          : (4, 3)

Deterministic actions:
[[ 1.0609231  -0.30130205  1.193731  ]
 [ 1.0848405  -0.28692287  1.2112348 ]
 [ 1.1089845  -0.27214995  1.2290823 ]
 [ 1.1394088  -0.25583884  1.2526001 ]]

Sampled actions:
[[-0.8513109  -0.59512174  1.9505953 ]
 [ 1.036277    0.02945139  1.7405925 ]
 [ 1.6486268  -0.06583582  1.9202772 ]
 [ 0.2638407  -1.07231     1.977837  ]]

Action range:
Min = -1.072310, Max = 1.977837

Critics:
Q1: [ 0.04249297 -0.07855331 -0.11251327  0.01670611]
Q2: [-0.11562696 -0.12166874 -0.12964375 -0.10989435]

Alpha:
Alpha = 0.010000

✓ SAC INITIALIZATION PASSED


In [72]:
import inspect

print(inspect.signature(sac_update))
print(inspect.getsource(sac_update))

()
def sac_update():

    global log_alpha

    # -----------------------------------------------------
    # Sample batch
    # -----------------------------------------------------
    states, desired_states, actions, rewards, next_states, dones = replay.sample(
        BATCH_SIZE
    )

    # -----------------------------------------------------
    # Alpha MUST remain a differentiable tensor
    # -----------------------------------------------------
    alpha_tensor = log_alpha.exp()

    # -----------------------------------------------------
    # Target Q
    # -----------------------------------------------------
    with torch.no_grad():

        next_actions, next_log_prob, _ = actor.sample(
            next_states,
            desired_states
        )

        target_q1 = critic1(next_states, desired_states, next_actions)
        target_q2 = critic2(next_states, desired_states, next_actions)

        min_target_q = torch.min(target_q1, target_q2)

        target_q = rewards

In [73]:
# ============================================================
# CELL 52A — CONNECT CONSERVATIVE SAC TO sac_update()
# ============================================================

# The existing sac_update() uses these global variable names.
# Point them to our new BC-initialized SAC objects.

actor = sac_actor

critic1 = sac_critic1
critic2 = sac_critic2

target_critic1 = sac_target_critic1
target_critic2 = sac_target_critic2

actor_optimizer = sac_actor_optimizer

critic1_optimizer = sac_critic1_optimizer
critic2_optimizer = sac_critic2_optimizer

log_alpha = log_alpha_sac
alpha_optimizer = sac_alpha_optimizer

replay = sac_replay


# ------------------------------------------------------------
# Verify the connections
# ------------------------------------------------------------

print("=" * 70)
print("SAC UPDATE CONNECTION CHECK")
print("=" * 70)

print("actor              :", actor is sac_actor)
print("critic1            :", critic1 is sac_critic1)
print("critic2            :", critic2 is sac_critic2)
print("target critic1     :", target_critic1 is sac_target_critic1)
print("target critic2     :", target_critic2 is sac_target_critic2)
print("replay             :", replay is sac_replay)
print("alpha              :", log_alpha.exp().item())

print("\n✓ Global SAC references connected correctly.")

SAC UPDATE CONNECTION CHECK
actor              : True
critic1            : True
critic2            : True
target critic1     : True
target critic2     : True
replay             : True
alpha              : 0.009999999776482582

✓ Global SAC references connected correctly.


In [74]:
# ============================================================
# CELL 52 — CONSERVATIVE SAC FINE-TUNING
# 10,000 SEQUENTIAL ENVIRONMENT STEPS
# ============================================================

import time
import numpy as np
import torch

SAC_TOTAL_STEPS = 10_000
SAC_WARMUP_STEPS = 500
EPISODE_LENGTH = 300

# IMPORTANT:
# This is a training seed, NOT the fixed benchmark seed 54321.
TRAIN_SEED = 20260914
train_rng = np.random.default_rng(TRAIN_SEED)

# ------------------------------------------------------------
# Fresh replay buffer
# ------------------------------------------------------------

sac_replay = ReplayBuffer(REPLAY_CAPACITY)

# Connect the replay buffer used by sac_update()
replay = sac_replay

# ------------------------------------------------------------
# Make sure sac_update() uses our BC-initialized SAC objects
# ------------------------------------------------------------

actor = sac_actor

critic1 = sac_critic1
critic2 = sac_critic2

target_critic1 = sac_target_critic1
target_critic2 = sac_target_critic2

actor_optimizer = sac_actor_optimizer

critic1_optimizer = sac_critic1_optimizer
critic2_optimizer = sac_critic2_optimizer

log_alpha = log_alpha_sac
alpha_optimizer = sac_alpha_optimizer

# ------------------------------------------------------------
# Training mode
# ------------------------------------------------------------

actor.train()
critic1.train()
critic2.train()

total_steps = 0
total_updates = 0
invalid_count = 0

episode_rewards = []
episode_final_errors = []
alpha_history = []

start_time = time.time()

print("=" * 70)
print("CONSERVATIVE SAC FINE-TUNING")
print("=" * 70)
print(f"Starting policy       : BC actor")
print(f"Total steps           : {SAC_TOTAL_STEPS}")
print(f"Warmup steps          : {SAC_WARMUP_STEPS}")
print(f"Episode length        : {EPISODE_LENGTH}")
print(f"Training seed         : {TRAIN_SEED}")
print(f"Actor LR              : {SAC_ACTOR_LR}")
print(f"Critic LR             : {SAC_CRITIC_LR}")
print(f"Initial alpha         : {log_alpha.exp().item():.6f}")
print(f"Device                : {DEVICE}")
print("=" * 70)


# ============================================================
# Expanded training task generator
# ============================================================

def generate_expanded_training_task(rng):

    state = np.array([
        rng.uniform(
            STATE_LOW[i],
            STATE_HIGH[i]
        )
        for i in range(STATE_DIM)
    ], dtype=np.float64)

    desired_pose = np.array([
        rng.uniform(
            DESIRED_LOW[i],
            DESIRED_HIGH[i]
        )
        for i in range(3)
    ], dtype=np.float64)

    desired = np.array([
        desired_pose[0],
        desired_pose[1],
        desired_pose[2],
        0.0,
        0.0,
        0.0
    ], dtype=np.float64)

    return state, desired


# ============================================================
# Sequential SAC training
# ============================================================

episode = 0

while total_steps < SAC_TOTAL_STEPS:

    episode += 1

    state, desired_state = generate_expanded_training_task(
        train_rng
    )

    episode_reward = 0.0

    for ep_step in range(EPISODE_LENGTH):

        if total_steps >= SAC_TOTAL_STEPS:
            break

        # ----------------------------------------------------
        # Convert state to tensors
        # ----------------------------------------------------

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        desired_tensor = torch.tensor(
            desired_state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        # ----------------------------------------------------
        # Select action
        # ----------------------------------------------------

        with torch.no_grad():

            if total_steps < SAC_WARMUP_STEPS:

                # Warmup exploration
                action = np.random.uniform(
                    -MAX_ACTION,
                    MAX_ACTION,
                    size=ACTION_DIM
                ).astype(np.float64)

            else:

                action_tensor, _, _ = actor.sample(
                    state_tensor,
                    desired_tensor
                )

                action = (
                    action_tensor
                    .squeeze(0)
                    .cpu()
                    .numpy()
                )

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        )

        # ----------------------------------------------------
        # Actual 3-RPR dynamics
        # ----------------------------------------------------

        next_state = np.asarray(
            step(
                np.asarray(
                    state,
                    dtype=np.float64
                ),
                np.asarray(
                    action,
                    dtype=np.float64
                ),
                DT
            ),
            dtype=np.float64
        )

        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if (
            not np.all(np.isfinite(next_state))
            or np.max(np.abs(next_state)) > 5.0
        ):

            invalid_count += 1

            reward = -100.0

            # Correct ReplayBuffer order:
            # state, desired, action, reward, next_state, done

            replay.add(
                state,
                desired_state,
                action,
                reward,
                state,
                True
            )

            total_steps += 1

            break

        # ----------------------------------------------------
        # EXACT PROJECT REWARD
        # ----------------------------------------------------

        reward = (
            -np.sum(
                (next_state - desired_state) ** 2
            )
            -0.01 * np.sum(action ** 2)
        )

        done = (
            ep_step == EPISODE_LENGTH - 1
        )

        # ----------------------------------------------------
        # Store transition
        # ----------------------------------------------------

        replay.add(
            state,
            desired_state,
            action,
            reward,
            next_state,
            done
        )

        episode_reward += reward

        state = next_state

        total_steps += 1

        # ----------------------------------------------------
        # SAC UPDATE
        #
        # sac_update() takes NO arguments.
        # It uses the global SAC objects configured above.
        # ----------------------------------------------------

        if (
            total_steps >= SAC_WARMUP_STEPS
            and len(replay) >= BATCH_SIZE
        ):

            update_result = sac_update()

            total_updates += 1

            alpha_history.append(
                log_alpha.exp().item()
            )

    # --------------------------------------------------------
    # Episode statistics
    # --------------------------------------------------------

    final_error = np.linalg.norm(
        state[:3] - desired_state[:3]
    )

    episode_rewards.append(
        episode_reward
    )

    episode_final_errors.append(
        final_error
    )

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if episode % 10 == 0:

        elapsed = time.time() - start_time

        recent_rewards = np.mean(
            episode_rewards[-10:]
        )

        recent_errors = np.mean(
            episode_final_errors[-10:]
        )

        current_alpha = (
            log_alpha.exp().item()
        )

        print(
            f"Episode {episode:4d} | "
            f"Steps {total_steps:5d}/{SAC_TOTAL_STEPS} | "
            f"Avg reward {recent_rewards:9.2f} | "
            f"Avg final err {recent_errors:.4f} | "
            f"Alpha {current_alpha:.6f} | "
            f"Invalid {invalid_count} | "
            f"Time {elapsed/60:.1f} min"
        )


# ============================================================
# Final training statistics
# ============================================================

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("10K SAC FINE-TUNING COMPLETE")
print("=" * 70)

print(f"Environment steps   : {total_steps}")
print(f"SAC updates         : {total_updates}")
print(f"Episodes            : {episode}")
print(f"Invalid transitions : {invalid_count}")
print(f"Final alpha         : {log_alpha.exp().item():.6f}")
print(f"Training time       : {elapsed:.1f} seconds")
print(f"Training time       : {elapsed/60:.2f} minutes")

if len(episode_final_errors) >= 10:

    print(
        f"Last 10 avg final error: "
        f"{np.mean(episode_final_errors[-10:]):.6f}"
    )

print("=" * 70)

CONSERVATIVE SAC FINE-TUNING
Starting policy       : BC actor
Total steps           : 10000
Warmup steps          : 500
Episode length        : 300
Training seed         : 20260914
Actor LR              : 1e-05
Critic LR             : 0.0003
Initial alpha         : 0.010000
Device                : cuda
Episode   10 | Steps  2643/10000 | Avg reward   -567.18 | Avg final err 1.5299 | Alpha 0.009994 | Invalid 4 | Time 0.4 min
Episode   20 | Steps  5406/10000 | Avg reward   -478.70 | Avg final err 1.1812 | Alpha 0.010254 | Invalid 5 | Time 1.0 min
Episode   30 | Steps  8406/10000 | Avg reward   -423.57 | Avg final err 1.3298 | Alpha 0.010585 | Invalid 5 | Time 1.6 min

10K SAC FINE-TUNING COMPLETE
Environment steps   : 10000
SAC updates         : 9494
Episodes            : 37
Invalid transitions : 7
Final alpha         : 0.010759
Training time       : 113.1 seconds
Training time       : 1.89 minutes
Last 10 avg final error: 1.880559


In [75]:
# ============================================================
# CELL 53 — RESTORE THE GOOD BC ACTOR
# ============================================================

bc_checkpoint = torch.load(
    os.path.join(
        PROJECT_DIR,
        "training",
        "final_model",
        "sac_bc_actor.pt"
    ),
    map_location=DEVICE,
    weights_only=False
)

# Restore BC actor
actor.load_state_dict(
    bc_checkpoint["actor_state_dict"]
)

# Also synchronize our named SAC actor
sac_actor.load_state_dict(
    bc_checkpoint["actor_state_dict"]
)

actor.eval()
sac_actor.eval()

print("=" * 70)
print("BC ACTOR RESTORED")
print("=" * 70)
print("Starting policy : 77% BC actor")
print("Saved source    : sac_bc_actor.pt")
print("SAC fine-tuning : NOT applied")
print("=" * 70)

BC ACTOR RESTORED
Starting policy : 77% BC actor
Saved source    : sac_bc_actor.pt
SAC fine-tuning : NOT applied


In [76]:
# ============================================================
# CELL 54 — LOAD EXPERT DEMONSTRATIONS INTO SAC REPLAY
# ============================================================

expert_path = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_expert_demonstrations.npz"
)

expert_data = np.load(expert_path)

expert_states_np = expert_data["states"]
expert_desired_np = expert_data["desired_states"]
expert_actions_np = expert_data["actions"]
expert_next_states_np = expert_data["next_states"]
expert_rewards_np = expert_data["rewards"]


# ------------------------------------------------------------
# Fresh replay buffer
# ------------------------------------------------------------

replay = ReplayBuffer(REPLAY_CAPACITY)


# ------------------------------------------------------------
# Add expert transitions
# ------------------------------------------------------------

for i in range(len(expert_states_np)):

    # Expert trajectories have 300 steps.
    # Mark the final transition of each trajectory as terminal.
    done = ((i + 1) % 300 == 0)

    replay.add(
        expert_states_np[i],
        expert_desired_np[i],
        expert_actions_np[i],
        expert_rewards_np[i],
        expert_next_states_np[i],
        done
    )


# ------------------------------------------------------------
# Verify replay
# ------------------------------------------------------------

print("=" * 70)
print("EXPERT REPLAY INITIALIZED")
print("=" * 70)

print(f"Expert transitions loaded : {len(expert_states_np)}")
print(f"Replay buffer size        : {len(replay)}")
print(f"State shape               : {expert_states_np.shape}")
print(f"Action shape              : {expert_actions_np.shape}")

print(
    f"Reward range              : "
    f"{expert_rewards_np.min():.4f} to "
    f"{expert_rewards_np.max():.4f}"
)

print(
    f"Action range              : "
    f"{expert_actions_np.min():.4f} to "
    f"{expert_actions_np.max():.4f}"
)

print("=" * 70)

assert len(replay) == 30000
assert np.all(np.isfinite(expert_states_np))
assert np.all(np.isfinite(expert_actions_np))
assert np.all(np.isfinite(expert_rewards_np))
assert np.all(np.isfinite(expert_next_states_np))

print("✓ Expert replay verification passed.")

EXPERT REPLAY INITIALIZED
Expert transitions loaded : 30000
Replay buffer size        : 30000
State shape               : (30000, 6)
Action shape              : (30000, 3)
Reward range              : -0.4007 to -0.0001
Action range              : -2.0000 to 2.0000
✓ Expert replay verification passed.


In [77]:
# ============================================================
# CELL 55 — SAC CRITIC PRETRAINING
# Actor remains FROZEN
# ============================================================

import time

CRITIC_PRETRAIN_STEPS = 3000
CRITIC_BATCH_SIZE = 256

# ------------------------------------------------------------
# Freeze actor completely
# ------------------------------------------------------------

for param in actor.parameters():
    param.requires_grad = False

actor.eval()

critic1.train()
critic2.train()

start_time = time.time()

critic1_losses = []
critic2_losses = []

print("=" * 70)
print("SAC CRITIC PRETRAINING")
print("=" * 70)
print(f"Steps       : {CRITIC_PRETRAIN_STEPS}")
print(f"Batch size  : {CRITIC_BATCH_SIZE}")
print("Actor       : FROZEN")
print(f"Device      : {DEVICE}")
print("=" * 70)


# ------------------------------------------------------------
# Critic-only training
# ------------------------------------------------------------

for step_idx in range(1, CRITIC_PRETRAIN_STEPS + 1):

    states, desired_states, actions, rewards, next_states, dones = (
        replay.sample(CRITIC_BATCH_SIZE)
    )

    # --------------------------------------------------------
    # Target Q
    #
    # Use the current SAC actor only to estimate the
    # entropy-regularized target for the next state.
    # --------------------------------------------------------

    with torch.no_grad():

        next_actions, next_log_prob, _ = actor.sample(
            next_states,
            desired_states
        )

        target_q1 = target_critic1(
            next_states,
            desired_states,
            next_actions
        )

        target_q2 = target_critic2(
            next_states,
            desired_states,
            next_actions
        )

        min_target_q = torch.min(
            target_q1,
            target_q2
        )

        alpha_tensor = log_alpha.exp()

        target_q = (
            rewards
            + GAMMA
            * (1.0 - dones)
            * (
                min_target_q
                - alpha_tensor.detach() * next_log_prob
            )
        )


    # --------------------------------------------------------
    # Critic 1
    # --------------------------------------------------------

    current_q1 = critic1(
        states,
        desired_states,
        actions
    )

    critic1_loss = F.mse_loss(
        current_q1,
        target_q
    )

    critic1_optimizer.zero_grad()

    critic1_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        critic1.parameters(),
        10.0
    )

    critic1_optimizer.step()


    # --------------------------------------------------------
    # Critic 2
    # --------------------------------------------------------

    current_q2 = critic2(
        states,
        desired_states,
        actions
    )

    critic2_loss = F.mse_loss(
        current_q2,
        target_q
    )

    critic2_optimizer.zero_grad()

    critic2_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        critic2.parameters(),
        10.0
    )

    critic2_optimizer.step()


    critic1_losses.append(
        critic1_loss.item()
    )

    critic2_losses.append(
        critic2_loss.item()
    )


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if step_idx % 500 == 0:

        print(
            f"Step {step_idx:4d}/{CRITIC_PRETRAIN_STEPS} | "
            f"Q1 loss {critic1_loss.item():.6f} | "
            f"Q2 loss {critic2_loss.item():.6f}"
        )


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("CRITIC PRETRAINING COMPLETE")
print("=" * 70)

print(
    f"Final Q1 loss : "
    f"{critic1_losses[-1]:.6f}"
)

print(
    f"Final Q2 loss : "
    f"{critic2_losses[-1]:.6f}"
)

print(
    f"Best Q1 loss  : "
    f"{min(critic1_losses):.6f}"
)

print(
    f"Best Q2 loss  : "
    f"{min(critic2_losses):.6f}"
)

print(
    f"Training time : "
    f"{elapsed:.1f} seconds"
)

print("=" * 70)

# Actor stays frozen for the next verification.

SAC CRITIC PRETRAINING
Steps       : 3000
Batch size  : 256
Actor       : FROZEN
Device      : cuda
Step  500/3000 | Q1 loss 9.378281 | Q2 loss 8.858404
Step 1000/3000 | Q1 loss 10.467861 | Q2 loss 10.571527
Step 1500/3000 | Q1 loss 1.684273 | Q2 loss 1.558769
Step 2000/3000 | Q1 loss 8.707235 | Q2 loss 8.551526
Step 2500/3000 | Q1 loss 8.309319 | Q2 loss 8.267259
Step 3000/3000 | Q1 loss 2.040620 | Q2 loss 1.871526

CRITIC PRETRAINING COMPLETE
Final Q1 loss : 2.040620
Final Q2 loss : 1.871526
Best Q1 loss  : 1.232709
Best Q2 loss  : 1.242754
Training time : 14.7 seconds


In [78]:
# ============================================================
# CELL 56 — CRITIC QUALITY CHECK
# Actor remains FROZEN
# ============================================================

actor.eval()
critic1.eval()
critic2.eval()

CHECK_BATCH = 512

with torch.no_grad():

    states, desired_states, expert_actions, rewards, next_states, dones = (
        replay.sample(CHECK_BATCH)
    )

    # --------------------------------------------------------
    # Expert actions
    # --------------------------------------------------------

    expert_q1 = critic1(
        states,
        desired_states,
        expert_actions
    )

    expert_q2 = critic2(
        states,
        desired_states,
        expert_actions
    )

    expert_q = torch.min(
        expert_q1,
        expert_q2
    )


    # --------------------------------------------------------
    # Random actions
    # --------------------------------------------------------

    random_actions = torch.empty_like(
        expert_actions
    ).uniform_(
        -MAX_ACTION,
        MAX_ACTION
    )

    random_q1 = critic1(
        states,
        desired_states,
        random_actions
    )

    random_q2 = critic2(
        states,
        desired_states,
        random_actions
    )

    random_q = torch.min(
        random_q1,
        random_q2
    )


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

expert_q_mean = expert_q.mean().item()
random_q_mean = random_q.mean().item()

expert_q_std = expert_q.std().item()
random_q_std = random_q.std().item()

expert_better = (
    expert_q > random_q
).float().mean().item() * 100

q_difference = (
    torch.abs(expert_q1 - expert_q2)
    .mean()
    .item()
)

print("=" * 70)
print("CRITIC QUALITY CHECK")
print("=" * 70)

print("\nExpert actions:")
print(f"Mean Q : {expert_q_mean:.6f}")
print(f"Std Q  : {expert_q_std:.6f}")

print("\nRandom actions:")
print(f"Mean Q : {random_q_mean:.6f}")
print(f"Std Q  : {random_q_std:.6f}")

print("\nCritic preference:")
print(
    f"Expert Q > Random Q : "
    f"{expert_better:.2f}% of samples"
)

print("\nTwin critic agreement:")
print(
    f"Mean |Q1-Q2| : "
    f"{q_difference:.6f}"
)

print("\nFinite values:")
print(
    "Expert Q finite :",
    torch.isfinite(expert_q).all().item()
)

print(
    "Random Q finite :",
    torch.isfinite(random_q).all().item()
)

print("=" * 70)

CRITIC QUALITY CHECK

Expert actions:
Mean Q : -36.686932
Std Q  : 5.175674

Random actions:
Mean Q : -37.721016
Std Q  : 5.284924

Critic preference:
Expert Q > Random Q : 85.16% of samples

Twin critic agreement:
Mean |Q1-Q2| : 0.283713

Finite values:
Expert Q finite : True
Random Q finite : True


In [79]:
# ============================================================
# CELL 57 — CONTROLLED SAC ACTOR FINE-TUNING
# ONLY 500 UPDATES
# ============================================================

# ------------------------------------------------------------
# Restore BC actor one more time
# ------------------------------------------------------------

bc_checkpoint = torch.load(
    os.path.join(
        PROJECT_DIR,
        "training",
        "final_model",
        "sac_bc_actor.pt"
    ),
    map_location=DEVICE,
    weights_only=False
)

actor.load_state_dict(
    bc_checkpoint["actor_state_dict"]
)

sac_actor.load_state_dict(
    bc_checkpoint["actor_state_dict"]
)


# ------------------------------------------------------------
# VERY small actor learning rate
# ------------------------------------------------------------

for param_group in actor_optimizer.param_groups:
    param_group["lr"] = 1e-6

# Keep alpha very low
with torch.no_grad():
    log_alpha.copy_(
        torch.tensor(
            np.log(0.005),
            dtype=torch.float32,
            device=DEVICE
        )
    )

# ------------------------------------------------------------
# Enable actor gradients
# ------------------------------------------------------------

for param in actor.parameters():
    param.requires_grad = True

actor.train()
critic1.eval()
critic2.eval()

# ------------------------------------------------------------
# SAC updates
# ------------------------------------------------------------

CONTROLLED_UPDATES = 500

actor_losses = []
alpha_values = []

print("=" * 70)
print("CONTROLLED SAC ACTOR FINE-TUNING")
print("=" * 70)
print(f"Updates       : {CONTROLLED_UPDATES}")
print("Actor LR      : 1e-6")
print("Initial alpha : 0.005")
print("Starting from : 77% BC actor")
print("=" * 70)


for update_idx in range(1, CONTROLLED_UPDATES + 1):

    result = sac_update()

    actor_losses.append(result[2])
    alpha_values.append(result[4])

    if update_idx % 100 == 0:

        print(
            f"Update {update_idx:3d}/{CONTROLLED_UPDATES} | "
            f"Critic1 {result[0]:.4f} | "
            f"Critic2 {result[1]:.4f} | "
            f"Actor {result[2]:.4f} | "
            f"Alpha {result[4]:.6f}"
        )


print("\n" + "=" * 70)
print("CONTROLLED SAC UPDATE COMPLETE")
print("=" * 70)

print(
    f"Final actor loss : "
    f"{actor_losses[-1]:.6f}"
)

print(
    f"Final alpha      : "
    f"{alpha_values[-1]:.6f}"
)

print("=" * 70)

CONTROLLED SAC ACTOR FINE-TUNING
Updates       : 500
Actor LR      : 1e-6
Initial alpha : 0.005
Starting from : 77% BC actor
Update 100/500 | Critic1 8.5169 | Critic2 10.4870 | Actor 107.8023 | Alpha 0.004996
Update 200/500 | Critic1 6.0887 | Critic2 5.8751 | Actor 153.0221 | Alpha 0.004990
Update 300/500 | Critic1 4.7795 | Critic2 4.9539 | Actor 141.6873 | Alpha 0.004985
Update 400/500 | Critic1 6.1479 | Critic2 6.3464 | Actor 128.8941 | Alpha 0.004980
Update 500/500 | Critic1 4.9944 | Critic2 4.8409 | Actor 121.1834 | Alpha 0.004975

CONTROLLED SAC UPDATE COMPLETE
Final actor loss : 121.183434
Final alpha      : 0.004975


In [80]:
# ============================================================
# CELL 58 — EVALUATE 500-UPDATE SAC POLICY
# Fixed benchmark: seed 54321
# Evaluation ONLY
# ============================================================

actor.eval()

results_sac = []

with torch.no_grad():

    for task_idx, (initial_state, desired_state) in enumerate(eval_tasks):

        state = initial_state.copy()

        initial_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        best_error = initial_error
        unstable = False

        for step_idx in range(EVAL_STEPS):

            state_tensor = torch.tensor(
                state,
                dtype=torch.float32,
                device=DEVICE
            ).unsqueeze(0)

            desired_tensor = torch.tensor(
                desired_state,
                dtype=torch.float32,
                device=DEVICE
            ).unsqueeze(0)

            mean, _ = actor(
                state_tensor,
                desired_tensor
            )

            action = (
                MAX_ACTION * torch.tanh(mean)
            ).squeeze(0).cpu().numpy()

            action = np.clip(
                action,
                -MAX_ACTION,
                MAX_ACTION
            )

            next_state = np.asarray(
                step(
                    np.asarray(state, dtype=np.float64),
                    np.asarray(action, dtype=np.float64),
                    DT
                ),
                dtype=np.float64
            )

            if (
                not np.all(np.isfinite(next_state))
                or np.max(np.abs(next_state)) > 5.0
            ):
                unstable = True
                break

            state = next_state

            error = np.linalg.norm(
                state[:3] - desired_state[:3]
            )

            best_error = min(
                best_error,
                error
            )

        final_error = (
            np.linalg.norm(
                state[:3] - desired_state[:3]
            )
            if not unstable else 999.0
        )

        success = (
            not unstable
            and final_error < initial_error
        )

        results_sac.append({
            "task": task_idx + 1,
            "initial": initial_error,
            "final": final_error,
            "best": best_error,
            "success": success,
            "unstable": unstable
        })


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

sac_successes = sum(
    r["success"] for r in results_sac
)

sac_unstable = sum(
    r["unstable"] for r in results_sac
)

sac_initial = np.array(
    [r["initial"] for r in results_sac]
)

sac_final = np.array(
    [r["final"] for r in results_sac]
)

sac_best = np.array(
    [r["best"] for r in results_sac]
)

print("\n" + "=" * 70)
print("500-UPDATE SAC — FIXED BENCHMARK")
print("=" * 70)

print(f"Successes            : {sac_successes}/100")
print(f"Success rate         : {sac_successes}%")
print(f"Unstable tasks       : {sac_unstable}")

print(f"\nAverage initial error: {sac_initial.mean():.6f}")
print(f"Average final error  : {sac_final.mean():.6f}")
print(f"Average best error   : {sac_best.mean():.6f}")
print(
    f"Average improvement  : "
    f"{(sac_initial - sac_final).mean():.6f}"
)

print(f"\nBest final error     : {sac_final.min():.6f}")
print(f"Worst final error    : {sac_final.max():.6f}")

print("=" * 70)


500-UPDATE SAC — FIXED BENCHMARK
Successes            : 55/100
Success rate         : 55%
Unstable tasks       : 23

Average initial error: 0.139931
Average final error  : 229.853514
Average best error   : 0.067763
Average improvement  : -229.713583

Best final error     : 0.011938
Worst final error    : 999.000000


In [83]:
print("CUDA available:", torch.cuda.is_available())
print("Current actor:", type(actor).__name__)
print("BC file exists:", os.path.exists(
    "training/final_model/sac_bc_actor.pt"
))
print("Expert file exists:", os.path.exists(
    "training/final_model/sac_expert_demonstrations.npz"
))

CUDA available: True
Current actor: SACActor
BC file exists: False
Expert file exists: False


In [84]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nPROJECT_DIR:")
print(PROJECT_DIR)

print("\nExisting final_model files:")
for p in Path(PROJECT_DIR).rglob("*"):
    if p.is_file() and "final_model" in str(p) and (
        "bc_actor" in p.name or
        "expert_demonstrations" in p.name or
        "rollback" in p.name
    ):
        print(p)

Current working directory:
c:\Users\madha\OneDrive\Desktop\project_1\training

PROJECT_DIR:
c:\Users\madha\OneDrive\Desktop\project_1

Existing final_model files:
c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_50k_rollback.pt
c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_bc_actor.pt
c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_expert_demonstrations.npz


In [85]:
from pathlib import Path

bc_path = Path(PROJECT_DIR) / "training" / "final_model" / "sac_bc_actor.pt"

# Load on CPU only
bc_cpu = torch.load(bc_path, map_location="cpu", weights_only=False)

print("Checkpoint loaded:", bc_path)
print("Checkpoint type:", type(bc_cpu))

if isinstance(bc_cpu, dict):
    print("Keys:", list(bc_cpu.keys()))
    print("Number of parameters:",
          sum(v.numel() for v in bc_cpu.values() if torch.is_tensor(v)))
else:
    print("Unexpected checkpoint format")

Checkpoint loaded: c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_bc_actor.pt
Checkpoint type: <class 'dict'>
Keys: ['actor_state_dict', 'algorithm', 'initialization', 'expert_dataset', 'max_action', 'state_center', 'state_scale', 'desired_center', 'desired_scale']
Number of parameters: 24


In [86]:
# Restore the known-good 5k BC actor from disk
bc_path = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_bc_actor.pt"
)

bc_checkpoint = torch.load(
    bc_path,
    map_location="cpu",
    weights_only=False
)

# Load weights into the existing SACActor
actor.load_state_dict(bc_checkpoint["actor_state_dict"])

print("BC actor weights restored successfully.")
print("Checkpoint:", bc_path)
print("Algorithm:", bc_checkpoint["algorithm"])
print("Initialization:", bc_checkpoint["initialization"])
print("Max action:", bc_checkpoint["max_action"])

RuntimeError: Error(s) in loading state_dict for SACActor:
	While copying the parameter named "state_center", whose dimensions in the model are torch.Size([6]) and whose dimensions in the checkpoint are torch.Size([6]), an exception occurred : ("CUDA error: device-side assert triggered\nSearch for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nThe CUDA driver logged these messages, which may provide useful details:\nSticky error detected\nReturning 710 (CUDA_ERROR_ASSERT) from cuStreamIsCapturing\n\n",).
	While copying the parameter named "state_scale", whose dimensions in the model are torch.Size([6]) and whose dimensions in the checkpoint are torch.Size([6]), an exception occurred : ("CUDA error: device-side assert triggered\nSearch for `cudaE

In [89]:
from pathlib import Path
import subprocess
import sys

project = Path(PROJECT_DIR)

script = project / "gpu_recovery_check.py"

script.write_text(r'''
import os
import torch
import numpy as np
import torch.nn as nn

PROJECT_DIR = r"''' + str(project) + r'''"

BC_PATH = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_bc_actor.pt"
)

EXPERT_PATH = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_expert_demonstrations.npz"
)

print("=== FRESH GPU PROCESS ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available")

print("GPU:", torch.cuda.get_device_name(0))

STATE_CENTER = torch.tensor(
    [0.5, 0.5, 0.0, 0.0, 0.0, 0.0],
    dtype=torch.float32
)

STATE_SCALE = torch.tensor(
    [0.5, 0.5, 0.5, 0.2, 0.2, 0.2],
    dtype=torch.float32
)

DESIRED_CENTER = STATE_CENTER.clone()
DESIRED_SCALE = STATE_SCALE.clone()


class SACActor(nn.Module):

    def __init__(self):
        super().__init__()

        self.register_buffer("state_center", STATE_CENTER)
        self.register_buffer("state_scale", STATE_SCALE)
        self.register_buffer("desired_center", DESIRED_CENTER)
        self.register_buffer("desired_scale", DESIRED_SCALE)

        self.net = nn.Sequential(
            nn.Linear(12, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )

        self.mean = nn.Linear(256, 3)
        self.log_std = nn.Linear(256, 3)

    def forward(self, state, desired_state):

        state = (
            state - self.state_center
        ) / self.state_scale

        desired_state = (
            desired_state - self.desired_center
        ) / self.desired_scale

        x = torch.cat(
            [state, desired_state],
            dim=-1
        )

        x = self.net(x)

        mean = self.mean(x)

        log_std = torch.clamp(
            self.log_std(x),
            -20,
            2
        )

        return mean, log_std


device = torch.device("cuda")

checkpoint = torch.load(
    BC_PATH,
    map_location="cpu",
    weights_only=False
)

actor = SACActor()

actor.load_state_dict(
    checkpoint["actor_state_dict"]
)

actor = actor.to(device)
actor.eval()

print("BC actor loaded onto GPU successfully.")

state = torch.tensor(
    [[0.5, 0.5, 0.05, 0.05, -0.03, 0.02]],
    dtype=torch.float32,
    device=device
)

desired = torch.tensor(
    [[0.55, 0.55, 0.0, 0.0, 0.0, 0.0]],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():

    mean, log_std = actor(
        state,
        desired
    )

    action = 2.0 * torch.tanh(mean)

print("Test action:", action.cpu().numpy())

print(
    "Action finite:",
    torch.isfinite(action).all().item()
)

print(
    "Action range:",
    float(action.min()),
    float(action.max())
)

expert = np.load(EXPERT_PATH)

print(
    "Expert dataset loaded:",
    expert["states"].shape
)

print(
    "Expert actions:",
    expert["actions"].shape
)

print("=== GPU RECOVERY SUCCESS ===")
''', encoding="utf-8")

result = subprocess.run(
    [sys.executable, str(script)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

=== FRESH GPU PROCESS ===
PyTorch: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
BC actor loaded onto GPU successfully.
Test action: [[ 0.28958392  0.25062308 -0.14872566]]
Action finite: True
Action range: -0.14872565865516663 0.2895839214324951
Expert dataset loaded: (30000, 6)
Expert actions: (30000, 3)
=== GPU RECOVERY SUCCESS ===

Return code: 0


In [91]:
import numpy as np
from pathlib import Path

expert_path = Path(PROJECT_DIR) / "training" / "final_model" / "sac_expert_demonstrations.npz"

expert = np.load(expert_path)

print("Expert dataset keys:")
print(expert.files)

for key in expert.files:
    print(key, expert[key].shape, expert[key].dtype)

Expert dataset keys:
['states', 'desired_states', 'actions', 'next_states', 'rewards']
states (30000, 6) float32
desired_states (30000, 6) float32
actions (30000, 3) float32
next_states (30000, 6) float32
rewards (30000,) float32


In [92]:
from pathlib import Path

script_path = Path(PROJECT_DIR) / "sac_final_gpu.py"

text = script_path.read_text(encoding="utf-8")

text = text.replace(
    'expert["desired"]',
    'expert["desired_states"]'
)

text = text.replace(
    'expert["next"]',
    'expert["next_states"]'
)

script_path.write_text(
    text,
    encoding="utf-8"
)

print("SAC script corrected successfully.")
print("desired -> desired_states")
print("next -> next_states")

SAC script corrected successfully.
desired -> desired_states
next -> next_states


In [93]:
import subprocess
import sys
from pathlib import Path

script_path = Path(PROJECT_DIR) / "sac_final_gpu.py"

print("==============================================")
print("STARTING SAC TRAINING")
print("GPU: RTX 4060")
print("Algorithm: Soft Actor-Critic (SAC)")
print("==============================================")
print("Expected runtime: approximately 2-4 minutes")
print()

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

STARTING SAC TRAINING
GPU: RTX 4060
Algorithm: Soft Actor-Critic (SAC)
Expected runtime: approximately 2-4 minutes

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
STARTING FINAL SAC RUN
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
BC actor loaded.
Expert transitions: 30000
Expert replay ready.

Starting SAC training...
Updates: 5000
BC lambda: 10.0
Update   500/5000 | Q1 0.0374 | Q2 0.0363 | SAC 1.0375 | BC 1.1937 | Actor 12.9750 | Alpha 0.004978 | Online 499
Update  1000/5000 | Q1 0.0176 | Q2 0.0158 | SAC 2.2076 | BC 0.5842 | Actor 8.0492 | Alpha 0.004955 | Online 998
Update  1500/5000 | Q1 0.0304 | Q2 0.0249 | SAC 2.9441 | BC 0.3527 | Actor 6.4713 | Alpha 0.004933 | Online 1497
Update  2000/5000 | Q1 0.0330 | Q2 0.0255 | SAC 2.5860 | BC 0.1991 | Actor 4.5775 | Alpha 0.004917 | Online 1997
Update  2500/5000 | Q1 0.0282 | Q2 0.0227 | SAC 2.8326 | BC 0.1468 | Actor 4.3007 | Alpha 0.004903 | Online 2497
Update  

In [94]:
from pathlib import Path
import subprocess
import sys

eval_script = Path(PROJECT_DIR) / "evaluate_final_sac.py"

eval_script.write_text(r'''
import os
import sys
import numpy as np
import torch
import torch.nn as nn

PROJECT_DIR = r"''' + str(PROJECT_DIR) + r'''"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from dynamics.dynamics import step

DEVICE = torch.device("cuda")

MAX_ACTION = 2.0
DT = 0.01
HIDDEN = 256

STATE_CENTER = torch.tensor(
    [0.5, 0.5, 0.0, 0.0, 0.0, 0.0],
    dtype=torch.float32,
    device=DEVICE
)

STATE_SCALE = torch.tensor(
    [0.5, 0.5, 0.5, 0.2, 0.2, 0.2],
    dtype=torch.float32,
    device=DEVICE
)


class SACActor(nn.Module):

    def __init__(self):
        super().__init__()

        self.register_buffer(
            "state_center",
            STATE_CENTER.clone()
        )
        self.register_buffer(
            "state_scale",
            STATE_SCALE.clone()
        )
        self.register_buffer(
            "desired_center",
            STATE_CENTER.clone()
        )
        self.register_buffer(
            "desired_scale",
            STATE_SCALE.clone()
        )

        self.net = nn.Sequential(
            nn.Linear(12, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU()
        )

        self.mean = nn.Linear(HIDDEN, 3)
        self.log_std = nn.Linear(HIDDEN, 3)

    def forward(self, state, desired):

        state = (
            state - self.state_center
        ) / self.state_scale

        desired = (
            desired - self.desired_center
        ) / self.desired_scale

        x = torch.cat(
            [state, desired],
            dim=-1
        )

        x = self.net(x)

        return self.mean(x)

    def deterministic(self, state, desired):

        mean = self.forward(
            state,
            desired
        )

        return MAX_ACTION * torch.tanh(mean)


# ------------------------------------------------------------
# LOAD SAC CHECKPOINT
# ------------------------------------------------------------

checkpoint_path = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_demo_regularized_5k.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

actor = SACActor().to(DEVICE)

actor.load_state_dict(
    checkpoint["actor_state_dict"]
)

actor.eval()

print("SAC checkpoint loaded.")
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# FIXED BENCHMARK
# ------------------------------------------------------------

np.random.seed(54321)

tasks = []

for _ in range(100):

    state = np.array([
        np.random.uniform(0.4, 0.6),
        np.random.uniform(0.4, 0.6),
        np.random.uniform(-0.1, 0.1),
        np.random.uniform(-0.1, 0.1),
        np.random.uniform(-0.1, 0.1),
        np.random.uniform(-0.1, 0.1)
    ], dtype=np.float32)

    desired = np.array([
        np.random.uniform(0.4, 0.6),
        np.random.uniform(0.4, 0.6),
        np.random.uniform(-0.1, 0.1),
        0.0,
        0.0,
        0.0
    ], dtype=np.float32)

    tasks.append((state, desired))


initial_errors = []
final_errors = []
best_errors = []

successes = 0
unstable = 0

for state0, desired in tasks:

    state = state0.copy()

    initial_error = np.linalg.norm(
        state[:3] - desired[:3]
    )

    best_error = initial_error

    invalid = False

    for _ in range(300):

        state_tensor = torch.tensor(
            state[None],
            dtype=torch.float32,
            device=DEVICE
        )

        desired_tensor = torch.tensor(
            desired[None],
            dtype=torch.float32,
            device=DEVICE
        )

        with torch.no_grad():

            action = actor.deterministic(
                state_tensor,
                desired_tensor
            ).squeeze(0).cpu().numpy()

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        )

        next_state = np.asarray(
            step(
                state.copy(),
                action.copy(),
                DT
            ),
            dtype=np.float32
        )

        if (
            not np.all(np.isfinite(next_state))
            or np.max(np.abs(next_state)) > 5.0
        ):

            invalid = True
            unstable += 1
            break

        state = next_state

        error = np.linalg.norm(
            state[:3] - desired[:3]
        )

        best_error = min(
            best_error,
            error
        )

    if invalid:
        final_error = 999.0
    else:
        final_error = np.linalg.norm(
            state[:3] - desired[:3]
        )

    initial_errors.append(initial_error)
    final_errors.append(final_error)
    best_errors.append(best_error)

    if final_error < initial_error:
        successes += 1


initial_errors = np.array(initial_errors)
final_errors = np.array(final_errors)
best_errors = np.array(best_errors)

print()
print("=" * 55)
print("SAC FINAL BENCHMARK")
print("=" * 55)

print(f"Average initial error : {initial_errors.mean():.6f}")
print(f"Average final error   : {final_errors.mean():.6f}")
print(
    f"Average improvement   : "
    f"{(initial_errors - final_errors).mean():.6f}"
)
print(f"Average best error    : {best_errors.mean():.6f}")
print(f"Best final error      : {final_errors.min():.6f}")
print(f"Worst final error     : {final_errors.max():.6f}")
print(f"Successful tasks      : {successes}/100")
print(f"Success rate          : {successes}%")
print(f"Unstable tasks        : {unstable}")
print("=" * 55)
''', encoding="utf-8")

print("Running SAC benchmark...")
print("Expected runtime: approximately 30-60 seconds.")
print()

result = subprocess.run(
    [sys.executable, str(eval_script)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

print("\nReturn code:", result.returncode)

Running SAC benchmark...
Expected runtime: approximately 30-60 seconds.

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
SAC checkpoint loaded.
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

SAC FINAL BENCHMARK
Average initial error : 0.133457
Average final error   : 139.947875
Average improvement   : -139.814417
Average best error    : 0.058065
Best final error      : 0.008941
Worst final error     : 999.000000
Successful tasks      : 65/100
Success rate          : 65%
Unstable tasks        : 14


Return code: 0


In [95]:
from pathlib import Path
import subprocess
import sys

project = Path(PROJECT_DIR)
script_path = project / "sac_stable_gpu.py"

script = r'''
import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

PROJECT_DIR = r"''' + str(project) + r'''"

# ============================================================
# REPRODUCIBILITY / GPU
# ============================================================

SEED = 20260914

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda")

print("=" * 60)
print("STABLE SAC TRAINING")
print("=" * 60)
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# CONFIG
# ============================================================

STATE_DIM = 6
DESIRED_DIM = 6
ACTION_DIM = 3

HIDDEN = 256

MAX_ACTION = 2.0
DT = 0.01

GAMMA = 0.99
TAU = 0.005

BATCH = 256

ACTOR_LR = 2e-6
CRITIC_LR = 1e-4
ALPHA_LR = 2e-6

INITIAL_ALPHA = 0.003
TARGET_ENTROPY = -3.0

TOTAL_UPDATES = 10000

# Strong enough to keep SAC close to demonstrated behaviour,
# while still allowing the SAC Q objective to update the actor.
BC_LAMBDA = 20.0

CRITIC_PRETRAIN_STEPS = 2000

BC_PATH = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_bc_actor.pt"
)

EXPERT_PATH = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_expert_demonstrations.npz"
)

OUTPUT_PATH = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_stable_10k.pt"
)

# ============================================================
# DYNAMICS
# ============================================================

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from dynamics.dynamics import step

# ============================================================
# NORMALIZATION
# ============================================================

STATE_CENTER = torch.tensor(
    [0.5, 0.5, 0.0, 0.0, 0.0, 0.0],
    dtype=torch.float32,
    device=DEVICE
)

STATE_SCALE = torch.tensor(
    [0.5, 0.5, 0.5, 0.2, 0.2, 0.2],
    dtype=torch.float32,
    device=DEVICE
)

# ============================================================
# ACTOR
# ============================================================

class SACActor(nn.Module):

    def __init__(self):
        super().__init__()

        self.register_buffer(
            "state_center",
            STATE_CENTER.clone()
        )

        self.register_buffer(
            "state_scale",
            STATE_SCALE.clone()
        )

        self.register_buffer(
            "desired_center",
            STATE_CENTER.clone()
        )

        self.register_buffer(
            "desired_scale",
            STATE_SCALE.clone()
        )

        self.net = nn.Sequential(
            nn.Linear(12, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU()
        )

        self.mean = nn.Linear(
            HIDDEN,
            ACTION_DIM
        )

        self.log_std = nn.Linear(
            HIDDEN,
            ACTION_DIM
        )

    def forward(
        self,
        state,
        desired
    ):

        state = (
            state - self.state_center
        ) / self.state_scale

        desired = (
            desired - self.desired_center
        ) / self.desired_scale

        x = torch.cat(
            [state, desired],
            dim=-1
        )

        x = self.net(x)

        mean = self.mean(x)

        log_std = torch.clamp(
            self.log_std(x),
            -20,
            2
        )

        return mean, log_std

    def sample(
        self,
        state,
        desired
    ):

        mean, log_std = self.forward(
            state,
            desired
        )

        std = log_std.exp()

        distribution = torch.distributions.Normal(
            mean,
            std
        )

        z = distribution.rsample()

        tanh_action = torch.tanh(z)

        action = MAX_ACTION * tanh_action

        log_prob = distribution.log_prob(z)

        log_prob -= torch.log(
            MAX_ACTION *
            (1.0 - tanh_action.pow(2))
            + 1e-6
        )

        log_prob = log_prob.sum(
            dim=-1,
            keepdim=True
        )

        return action, log_prob

    def deterministic(
        self,
        state,
        desired
    ):

        mean, _ = self.forward(
            state,
            desired
        )

        return MAX_ACTION * torch.tanh(mean)

# ============================================================
# CRITIC
# ============================================================

class Critic(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(15, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, 1)
        )

    def forward(
        self,
        state,
        desired,
        action
    ):

        x = torch.cat(
            [state, desired, action],
            dim=-1
        )

        return self.net(x)

# ============================================================
# LOAD EXPERT DATA
# ============================================================

expert = np.load(
    EXPERT_PATH
)

expert_states = expert["states"].astype(
    np.float32
)

expert_desired = expert["desired_states"].astype(
    np.float32
)

expert_actions = expert["actions"].astype(
    np.float32
)

expert_next_states = expert["next_states"].astype(
    np.float32
)

expert_rewards = expert["rewards"].astype(
    np.float32
)

print(
    "Expert transitions:",
    len(expert_states)
)

# ============================================================
# GPU EXPERT TENSORS
# ============================================================

ES = torch.from_numpy(
    expert_states
).to(DEVICE)

ED = torch.from_numpy(
    expert_desired
).to(DEVICE)

EA = torch.from_numpy(
    expert_actions
).to(DEVICE)

ENS = torch.from_numpy(
    expert_next_states
).to(DEVICE)

ER = torch.from_numpy(
    expert_rewards.reshape(-1, 1)
).to(DEVICE)

EDONE = torch.zeros(
    (len(expert_states), 1),
    dtype=torch.float32,
    device=DEVICE
)

# ============================================================
# MODELS
# ============================================================

actor = SACActor().to(DEVICE)

bc_checkpoint = torch.load(
    BC_PATH,
    map_location="cpu",
    weights_only=False
)

actor.load_state_dict(
    bc_checkpoint["actor_state_dict"]
)

actor.train()

critic1 = Critic().to(DEVICE)
critic2 = Critic().to(DEVICE)

target1 = Critic().to(DEVICE)
target2 = Critic().to(DEVICE)

target1.load_state_dict(
    critic1.state_dict()
)

target2.load_state_dict(
    critic2.state_dict()
)

for p in target1.parameters():
    p.requires_grad = False

for p in target2.parameters():
    p.requires_grad = False

print("BC actor loaded.")

# ============================================================
# OPTIMIZERS
# ============================================================

actor_optimizer = optim.Adam(
    actor.parameters(),
    lr=ACTOR_LR
)

critic1_optimizer = optim.Adam(
    critic1.parameters(),
    lr=CRITIC_LR
)

critic2_optimizer = optim.Adam(
    critic2.parameters(),
    lr=CRITIC_LR
)

log_alpha = torch.tensor(
    np.log(INITIAL_ALPHA),
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

alpha_optimizer = optim.Adam(
    [log_alpha],
    lr=ALPHA_LR
)

# ============================================================
# ONLINE REPLAY
# ============================================================

online_states = []
online_desired = []
online_actions = []
online_rewards = []
online_next = []

MAX_ONLINE = 30000

# ============================================================
# CRITIC PRETRAINING
# ============================================================

print()
print(
    "Pretraining critics:",
    CRITIC_PRETRAIN_STEPS,
    "updates"
)

for i in range(
    1,
    CRITIC_PRETRAIN_STEPS + 1
):

    idx = torch.randint(
        0,
        ES.shape[0],
        (BATCH,),
        device=DEVICE
    )

    s = ES[idx]
    d = ED[idx]
    a = EA[idx]
    r = ER[idx]
    ns = ENS[idx]

    with torch.no_grad():

        next_a, next_logp = actor.sample(
            ns,
            d
        )

        tq1 = target1(
            ns,
            d,
            next_a
        )

        tq2 = target2(
            ns,
            d,
            next_a
        )

        target_q = (
            r
            + GAMMA *
            (
                torch.min(tq1, tq2)
                - log_alpha.exp().detach()
                * next_logp
            )
        )

    q1 = critic1(
        s,
        d,
        a
    )

    q2 = critic2(
        s,
        d,
        a
    )

    loss1 = F.mse_loss(
        q1,
        target_q
    )

    loss2 = F.mse_loss(
        q2,
        target_q
    )

    critic1_optimizer.zero_grad(
        set_to_none=True
    )

    loss1.backward()

    torch.nn.utils.clip_grad_norm_(
        critic1.parameters(),
        5.0
    )

    critic1_optimizer.step()

    critic2_optimizer.zero_grad(
        set_to_none=True
    )

    loss2.backward()

    torch.nn.utils.clip_grad_norm_(
        critic2.parameters(),
        5.0
    )

    critic2_optimizer.step()

    with torch.no_grad():

        for p, tp in zip(
            critic1.parameters(),
            target1.parameters()
        ):
            tp.mul_(1.0 - TAU)
            tp.add_(TAU * p)

        for p, tp in zip(
            critic2.parameters(),
            target2.parameters()
        ):
            tp.mul_(1.0 - TAU)
            tp.add_(TAU * p)

    if i % 500 == 0:

        print(
            f"Critic pretrain {i}/{CRITIC_PRETRAIN_STEPS} | "
            f"Q1 {loss1.item():.4f} | "
            f"Q2 {loss2.item():.4f}"
        )

# ============================================================
# SAC UPDATE
# ============================================================

def sample_expert(n):

    idx = torch.randint(
        0,
        ES.shape[0],
        (n,),
        device=DEVICE
    )

    return (
        ES[idx],
        ED[idx],
        EA[idx],
        ER[idx],
        ENS[idx]
    )


def sample_online(n):

    idx = np.random.randint(
        0,
        len(online_states),
        n
    )

    return (
        torch.tensor(
            np.asarray(
                [online_states[i] for i in idx]
            ),
            dtype=torch.float32,
            device=DEVICE
        ),

        torch.tensor(
            np.asarray(
                [online_desired[i] for i in idx]
            ),
            dtype=torch.float32,
            device=DEVICE
        ),

        torch.tensor(
            np.asarray(
                [online_actions[i] for i in idx]
            ),
            dtype=torch.float32,
            device=DEVICE
        ),

        torch.tensor(
            np.asarray(
                [online_rewards[i] for i in idx]
            ).reshape(-1, 1),
            dtype=torch.float32,
            device=DEVICE
        ),

        torch.tensor(
            np.asarray(
                [online_next[i] for i in idx]
            ),
            dtype=torch.float32,
            device=DEVICE
        )
    )


def sac_update():

    # 75% expert, 25% online.
    ne = int(BATCH * 0.75)
    no = BATCH - ne

    es, ed, ea, er, ens = sample_expert(ne)

    if len(online_states) >= no:

        os_, od, oa, or_, ons = sample_online(no)

        states = torch.cat(
            [es, os_]
        )

        desired = torch.cat(
            [ed, od]
        )

        actions = torch.cat(
            [ea, oa]
        )

        rewards = torch.cat(
            [er, or_]
        )

        next_states = torch.cat(
            [ens, ons]
        )

    else:

        states = es
        desired = ed
        actions = ea
        rewards = er
        next_states = ens

    alpha = log_alpha.exp()

    # --------------------------------------------------------
    # Critic target
    # --------------------------------------------------------

    with torch.no_grad():

        next_actions, next_logp = actor.sample(
            next_states,
            desired
        )

        tq1 = target1(
            next_states,
            desired,
            next_actions
        )

        tq2 = target2(
            next_states,
            desired,
            next_actions
        )

        target_q = (
            rewards
            + GAMMA *
            (
                torch.min(tq1, tq2)
                - alpha.detach() * next_logp
            )
        )

    # --------------------------------------------------------
    # Critics
    # --------------------------------------------------------

    q1 = critic1(
        states,
        desired,
        actions
    )

    q2 = critic2(
        states,
        desired,
        actions
    )

    loss1 = F.mse_loss(
        q1,
        target_q
    )

    loss2 = F.mse_loss(
        q2,
        target_q
    )

    critic1_optimizer.zero_grad(
        set_to_none=True
    )

    loss1.backward()

    torch.nn.utils.clip_grad_norm_(
        critic1.parameters(),
        5.0
    )

    critic1_optimizer.step()

    critic2_optimizer.zero_grad(
        set_to_none=True
    )

    loss2.backward()

    torch.nn.utils.clip_grad_norm_(
        critic2.parameters(),
        5.0
    )

    critic2_optimizer.step()

    # --------------------------------------------------------
    # Actor
    # --------------------------------------------------------

    new_actions, logp = actor.sample(
        states,
        desired
    )

    q1_new = critic1(
        states,
        desired,
        new_actions
    )

    q2_new = critic2(
        states,
        desired,
        new_actions
    )

    sac_loss = (
        alpha.detach() * logp
        - torch.min(
            q1_new,
            q2_new
        )
    ).mean()

    # Expert imitation term.
    expert_actions = ea

    bc_pred = new_actions[:ne]

    bc_loss = F.mse_loss(
        bc_pred,
        expert_actions
    )

    actor_loss = (
        sac_loss
        + BC_LAMBDA * bc_loss
    )

    actor_optimizer.zero_grad(
        set_to_none=True
    )

    actor_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        actor.parameters(),
        1.0
    )

    actor_optimizer.step()

    # --------------------------------------------------------
    # Entropy coefficient
    # --------------------------------------------------------

    alpha_loss = -(
        log_alpha *
        (
            logp.detach()
            + TARGET_ENTROPY
        )
    ).mean()

    alpha_optimizer.zero_grad(
        set_to_none=True
    )

    alpha_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        [log_alpha],
        1.0
    )

    alpha_optimizer.step()

    # --------------------------------------------------------
    # Target critics
    # --------------------------------------------------------

    with torch.no_grad():

        for p, tp in zip(
            critic1.parameters(),
            target1.parameters()
        ):
            tp.mul_(1.0 - TAU)
            tp.add_(TAU * p)

        for p, tp in zip(
            critic2.parameters(),
            target2.parameters()
        ):
            tp.mul_(1.0 - TAU)
            tp.add_(TAU * p)

    return (
        loss1.item(),
        loss2.item(),
        sac_loss.item(),
        bc_loss.item(),
        alpha_loss.item(),
        alpha.item()
    )

# ============================================================
# SEQUENTIAL ENVIRONMENT
# ============================================================

def generate_task():

    state = np.array(
        [
            np.random.uniform(0.4, 0.6),
            np.random.uniform(0.4, 0.6),
            np.random.uniform(-0.1, 0.1),
            np.random.uniform(-0.1, 0.1),
            np.random.uniform(-0.1, 0.1),
            np.random.uniform(-0.1, 0.1)
        ],
        dtype=np.float32
    )

    desired = np.array(
        [
            np.random.uniform(0.4, 0.6),
            np.random.uniform(0.4, 0.6),
            np.random.uniform(-0.1, 0.1),
            0.0,
            0.0,
            0.0
        ],
        dtype=np.float32
    )

    return state, desired


state, desired = generate_task()

episode_steps = 0
invalid_count = 0

# ============================================================
# TRAIN
# ============================================================

print()
print(
    "Starting sequential SAC training:",
    TOTAL_UPDATES,
    "updates"
)

for update in range(
    1,
    TOTAL_UPDATES + 1
):

    s = torch.tensor(
        state[None],
        dtype=torch.float32,
        device=DEVICE
    )

    d = torch.tensor(
        desired[None],
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():

        action, _ = actor.sample(
            s,
            d
        )

    action_np = action.squeeze(
        0
    ).cpu().numpy()

    action_np = np.clip(
        action_np,
        -MAX_ACTION,
        MAX_ACTION
    ).astype(np.float32)

    next_state = np.asarray(
        step(
            state.copy(),
            action_np.copy(),
            DT
        ),
        dtype=np.float32
    )

    invalid = (
        not np.all(
            np.isfinite(next_state)
        )
        or
        np.max(
            np.abs(next_state)
        ) > 5.0
    )

    if invalid:

        invalid_count += 1

        state, desired = generate_task()

        episode_steps = 0

    else:

        reward = -float(
            np.sum(
                (next_state - desired) ** 2
            )
            +
            0.01 *
            np.sum(
                action_np ** 2
            )
        )

        if len(online_states) >= MAX_ONLINE:

            online_states.pop(0)
            online_desired.pop(0)
            online_actions.pop(0)
            online_rewards.pop(0)
            online_next.pop(0)

        online_states.append(
            state.copy()
        )

        online_desired.append(
            desired.copy()
        )

        online_actions.append(
            action_np.copy()
        )

        online_rewards.append(
            reward
        )

        online_next.append(
            next_state.copy()
        )

        state = next_state

        episode_steps += 1

        if episode_steps >= 300:

            state, desired = generate_task()

            episode_steps = 0

    # SAC update
    metrics = sac_update()

    if update % 500 == 0:

        q1l, q2l, sl, bl, al, av = metrics

        print(
            f"Update {update:5d}/{TOTAL_UPDATES} | "
            f"Q1 {q1l:.4f} | "
            f"Q2 {q2l:.4f} | "
            f"SAC {sl:.4f} | "
            f"BC {bl:.4f} | "
            f"AlphaLoss {al:.4f} | "
            f"Alpha {av:.6f} | "
            f"Online {len(online_states)} | "
            f"Invalid {invalid_count}"
        )

# ============================================================
# SAVE
# ============================================================

torch.save(
    {
        "algorithm": "SAC",
        "variant": "Demonstration-Regularized SAC",

        "actor_state_dict":
            actor.state_dict(),

        "critic1_state_dict":
            critic1.state_dict(),

        "critic2_state_dict":
            critic2.state_dict(),

        "target_critic1_state_dict":
            target1.state_dict(),

        "target_critic2_state_dict":
            target2.state_dict(),

        "log_alpha":
            log_alpha.detach().cpu(),

        "alpha":
            log_alpha.exp().item(),

        "state_center":
            STATE_CENTER.detach().cpu(),

        "state_scale":
            STATE_SCALE.detach().cpu(),

        "desired_center":
            STATE_CENTER.detach().cpu(),

        "desired_scale":
            STATE_SCALE.detach().cpu(),

        "max_action":
            MAX_ACTION,

        "gamma":
            GAMMA,

        "tau":
            TAU,

        "dt":
            DT,

        "bc_lambda":
            BC_LAMBDA,

        "training_updates":
            TOTAL_UPDATES,

        "critic_pretrain_steps":
            CRITIC_PRETRAIN_STEPS,

        "invalid_transitions":
            invalid_count,

        "seed":
            SEED
    },
    OUTPUT_PATH
)

print()
print("=" * 60)
print("SAC TRAINING FINISHED")
print("=" * 60)
print("Checkpoint:", OUTPUT_PATH)
print("Updates:", TOTAL_UPDATES)
print("Critic pretraining:", CRITIC_PRETRAIN_STEPS)
print("Invalid transitions:", invalid_count)
print("Final alpha:", log_alpha.exp().item())

with torch.no_grad():

    test_state = torch.tensor(
        [[0.5, 0.5, 0.05, 0.05, -0.03, 0.02]],
        dtype=torch.float32,
        device=DEVICE
    )

    test_desired = torch.tensor(
        [[0.55, 0.55, 0.0, 0.0, 0.0, 0.0]],
        dtype=torch.float32,
        device=DEVICE
    )

    test_action = actor.deterministic(
        test_state,
        test_desired
    )

print(
    "Final SAC action:",
    test_action.cpu().numpy()
)

print("DONE")
'''

script_path.write_text(
    script,
    encoding="utf-8"
)

print("Launching stable SAC training on RTX 4060...")
print("Expected runtime: approximately 3-5 minutes.")

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

Launching stable SAC training on RTX 4060...
Expected runtime: approximately 3-5 minutes.
STABLE SAC TRAINING
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
Expert transitions: 30000
BC actor loaded.

Pretraining critics: 2000 updates
Critic pretrain 500/2000 | Q1 0.0075 | Q2 0.0076
Critic pretrain 1000/2000 | Q1 0.0250 | Q2 0.0248
Critic pretrain 1500/2000 | Q1 0.0397 | Q2 0.0395
Critic pretrain 2000/2000 | Q1 0.0682 | Q2 0.0680

Starting sequential SAC training: 10000 updates
Update   500/10000 | Q1 0.0055 | Q2 0.0055 | SAC 2.8000 | BC 1.6472 | AlphaLoss -38.7577 | Alpha 0.002997 | Online 500 | Invalid 0
Update  1000/10000 | Q1 0.0040 | Q2 0.0039 | SAC 3.0243 | BC 1.4138 | AlphaLoss -37.2074 | Alpha 0.002994 | Online 1000 | Invalid 0
Update  1500/10000 | Q1 0.0050 | Q2 0.0054 | SAC 3.2378 | BC 1.2994 | AlphaLoss -37.3751 | Alpha 0.002991 | Online 1500 | Invalid 0


In [96]:
checkpoint_path = os.path.join(
    PROJECT_DIR,
    "training",
    "final_model",
    "sac_stable_10k.pt"
)

In [97]:
from pathlib import Path
import subprocess
import sys

eval_script = Path(PROJECT_DIR) / "evaluate_final_sac.py"

# Make sure the evaluation script uses the NEW 10k SAC checkpoint
text = eval_script.read_text(encoding="utf-8")

text = text.replace(
    "sac_demo_regularized_5k.pt",
    "sac_stable_10k.pt"
)

eval_script.write_text(
    text,
    encoding="utf-8"
)

print("Evaluating SAC 10k checkpoint...")
print("Expected runtime: 30-60 seconds.")
print()

result = subprocess.run(
    [sys.executable, str(eval_script)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

print("\nReturn code:", result.returncode)

Evaluating SAC 10k checkpoint...
Expected runtime: 30-60 seconds.

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
SAC checkpoint loaded.
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

SAC FINAL BENCHMARK
Average initial error : 0.133457
Average final error   : 149.958480
Average improvement   : -149.825023
Average best error    : 0.060873
Best final error      : 0.006508
Worst final error     : 999.000000
Successful tasks      : 61/100
Success rate          : 61%
Unstable tasks        : 15


Return code: 0


In [99]:
from pathlib import Path

script_path = Path(PROJECT_DIR) / "sac_fixed_final_gpu.py"

text = script_path.read_text(encoding="utf-8")

old = '''# ============================================================
# ALPHA
# ============================================================

log_alpha = torch.tensor('''

new = '''# ============================================================
# ACTOR OPTIMIZER FOR SAC
# ============================================================

actor_optimizer = optim.Adam(
    actor.parameters(),
    lr=ACTOR_LR
)

# ============================================================
# ALPHA
# ============================================================

log_alpha = torch.tensor('''

if old not in text:
    print("PATCH LOCATION NOT FOUND")
else:
    text = text.replace(old, new, 1)
    script_path.write_text(text, encoding="utf-8")
    print("PATCH SUCCESSFUL")
    print("actor_optimizer now defined before SAC.")

PATCH SUCCESSFUL
actor_optimizer now defined before SAC.


In [100]:
from pathlib import Path
import subprocess
import sys

script_path = Path(PROJECT_DIR) / "sac_fixed_final_gpu.py"

print("=" * 60)
print("LAUNCHING FINAL SAC TRAINING")
print("=" * 60)
print("GPU: RTX 4060")
print("Expected runtime: approximately 4-6 minutes.")
print()

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

LAUNCHING FINAL SAC TRAINING
GPU: RTX 4060
Expected runtime: approximately 4-6 minutes.

FINAL SAC FIX
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
Expert transitions: 30000

STAGE 1: ACTOR PRETRAINING
BC 2500/15000 | Loss 0.040052 | Best 0.009276
BC 5000/15000 | Loss 0.021374 | Best 0.006390
BC 7500/15000 | Loss 0.013012 | Best 0.006275
BC 10000/15000 | Loss 0.012554 | Best 0.004217
BC 12500/15000 | Loss 0.007399 | Best 0.003432
BC 15000/15000 | Loss 0.008507 | Best 0.003284
Actor pretraining complete.

STAGE 2: SAC CRITIC PRETRAINING
Critic 500/3000 | Q1 0.00542 | Q2 0.00529
Critic 1000/3000 | Q1 0.01449 | Q2 0.01421
Critic 1500/3000 | Q1 0.01955 | Q2 0.01936
Critic 2000/3000 | Q1 0.03211 | Q2 0.03200
Critic 2500/3000 | Q1 0.04467 | Q2 0.04412
Critic 3000/3000 | Q1 0.07079 | Q2 0.07082

STAGE 3: ACTUAL SEQUENTIAL SAC
SAC 500/5000 | Q1 0.00699 | Q2 0.00720 | SAC 

In [101]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
MODEL_PATH = PROJECT_DIR / "training" / "final_model" / "sac_final_fixed.pt"

eval_script = PROJECT_DIR / "evaluate_final_sac.py"

print("=" * 60)
print("EVALUATING FINAL SAC CHECKPOINT")
print("=" * 60)
print(f"Model: {MODEL_PATH}")
print("Benchmark: 100 unseen tasks")
print("Seed: 54321")
print("No training will be performed.")
print()

result = subprocess.run(
    [sys.executable, str(eval_script)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

EVALUATING FINAL SAC CHECKPOINT
Model: C:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_final_fixed.pt
Benchmark: 100 unseen tasks
Seed: 54321
No training will be performed.

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
SAC checkpoint loaded.
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

SAC FINAL BENCHMARK
Average initial error : 0.133457
Average final error   : 149.958480
Average improvement   : -149.825023
Average best error    : 0.060873
Best final error      : 0.006508
Worst final error     : 999.000000
Successful tasks      : 61/100
Success rate          : 61%
Unstable tasks        : 15


Return code: 0


In [102]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
script_path = PROJECT_DIR / "sac_fixed_final_gpu.py"

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

FINAL SAC FIX
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
Expert transitions: 30000

STAGE 1: ACTOR PRETRAINING
BC 2500/15000 | Loss 0.040052 | Best 0.009276
BC 5000/15000 | Loss 0.021374 | Best 0.006390
BC 7500/15000 | Loss 0.013012 | Best 0.006275
BC 10000/15000 | Loss 0.012554 | Best 0.004217
BC 12500/15000 | Loss 0.007399 | Best 0.003432
BC 15000/15000 | Loss 0.008507 | Best 0.003284
Actor pretraining complete.

STAGE 2: SAC CRITIC PRETRAINING
Critic 500/3000 | Q1 0.00542 | Q2 0.00529
Critic 1000/3000 | Q1 0.01449 | Q2 0.01421
Critic 1500/3000 | Q1 0.01955 | Q2 0.01936
Critic 2000/3000 | Q1 0.03211 | Q2 0.03200
Critic 2500/3000 | Q1 0.04467 | Q2 0.04412
Critic 3000/3000 | Q1 0.07079 | Q2 0.07082
Pre-SAC initialization saved: c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_stage2_pretrained.pt

STAGE 3: ACTUAL SEQUENTIAL SAC
SAC 500/5000 | Q

In [105]:
from pathlib import Path
import subprocess
import sys

result = subprocess.run(
    [sys.executable, r"C:\path\to\evaluate_sac_final.py"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)


c:\Users\madha\AppData\Local\Programs\Python\Python312\python.exe: can't open file 'C:\\path\\to\\evaluate_sac_final.py': [Errno 2] No such file or directory

Return code: 2


In [106]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
script_path = PROJECT_DIR / "evaluate_sac_final.py"

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
FINAL SAC — FIXED UNSEEN EVALUATION
Checkpoint            : c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_final_fixed.pt
Evaluation tasks      : 100
Evaluation seed       : 12345
Evaluation steps      : 100
Average initial error : 0.127099
Average final error   : 0.104038
Average improvement   : 0.023062
Best final error      : 0.012072
Worst final error     : 0.411192
Average reward        : -6.847295
Unstable episodes     : 0/100
Successes             : 75/100
Success rate          : 75.00%


Return code: 0


In [107]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
script_path = PROJECT_DIR / "sac_final_high_success.py"

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
STAGE 0: DIVERSE SAC DEMONSTRATIONS
Expert transitions: 100000 Invalid: 0
STAGE 1: DIVERSE ACTOR PRETRAINING
BC 2500/20000 | Loss 0.000001 | Best 0.000001
BC 5000/20000 | Loss 0.000000 | Best 0.000000
BC 7500/20000 | Loss 0.000000 | Best 0.000000
BC 10000/20000 | Loss 0.000000 | Best 0.000000
BC 12500/20000 | Loss 0.000000 | Best 0.000000
BC 15000/20000 | Loss 0.000000 | Best 0.000000
BC 17500/20000 | Loss 0.000000 | Best 0.000000
BC 20000/20000 | Loss 0.000000 | Best 0.000000
STAGE 2: SAC CRITIC PRETRAINING
Critic 1000/5000 | Q1 0.01408 | Q2 0.01412
Critic 2000/5000 | Q1 0.12926 | Q2 0.12830
Critic 3000/5000 | Q1 0.57987 | Q2 0.57954
Critic 4000/5000 | Q1 1.63384 | Q2 1.63371
Critic 5000/5000 | Q1 9.55447 | Q2 9.55928
STAGE 3: ACTUAL SEQUENTIAL SAC
SAC 1000/10000 | Q1 2.23089 | Q2 2.22857 | SAC 16.72908 | BC 0.000000 |

In [108]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
script_path = PROJECT_DIR / "sac_critic_fixed.py"

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

SAC CRITIC-STABILIZED TRAINING
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

LOADING 100,000 EXPERT TRANSITIONS
Transitions: 30000

STAGE 1: ACTOR PRETRAINING
BC 2000/10000 | Loss 0.19693546
BC 4000/10000 | Loss 0.05332738
BC 6000/10000 | Loss 0.08465312
BC 8000/10000 | Loss 0.03716199
BC 10000/10000 | Loss 0.02766605

STAGE 2: NON-BOOTSTRAPPED CRITIC INITIALIZATION
Return range: -22.26797866821289 -7.368598016910255e-05
Critic 500/3000 | Q1 0.999112 | Q2 0.932409
Critic 1000/3000 | Q1 0.786438 | Q2 0.761000
Critic 1500/3000 | Q1 1.037771 | Q2 1.016904
Critic 2000/3000 | Q1 0.598395 | Q2 0.592988
Critic 2500/3000 | Q1 0.675657 | Q2 0.663587
Critic 3000/3000 | Q1 0.676884 | Q2 0.656404

STAGE 3: ACTUAL SAC
SAC 500/3000 | Critic 0.956760 | Actor 7.817267 | BC 0.02929971 | Alpha 0.00019990
SAC 1000/3000 | Critic 0.707513 | Actor 9.618093 | BC 0.03348005 | Alpha 0.00019981
SAC 1500/3000 | Critic 0.570004 | Actor 10.820480 | BC 0.03199203 | Alpha 0.00019971
SAC 2000/3000 | Critic 0.540422 | Acto

In [109]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")

CHECKPOINT = (
    PROJECT_DIR
    / "training"
    / "final_model"
    / "sac_critic_fixed.pt"
)

script = PROJECT_DIR / "evaluate_sac_final.py"

# Make the evaluator use the new checkpoint
text = script.read_text(encoding="utf-8")

text = text.replace(
    "sac_final_fixed.pt",
    "sac_critic_fixed.pt"
)

script.write_text(text, encoding="utf-8")

result = subprocess.run(
    [sys.executable, str(script)],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
FINAL SAC — FIXED UNSEEN EVALUATION
Checkpoint            : c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_critic_fixed.pt
Evaluation tasks      : 100
Evaluation seed       : 12345
Evaluation steps      : 100
Average initial error : 0.127099
Average final error   : 0.128037
Average improvement   : -0.000937
Best final error      : 0.019606
Worst final error     : 0.932779
Average reward        : -13.281627
Unstable episodes     : 0/100
Successes             : 70/100
Success rate          : 70.00%


Return code: 0


In [110]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
SCRIPT = PROJECT_DIR / "sac_expert_sac_final.py"

code = r'''
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
DATA = PROJECT_DIR / "training" / "final_model" / "sac_expert_demonstrations.npz"
OUT = PROJECT_DIR / "training" / "final_model" / "sac_expert_sac_final.pt"

sys.path.insert(0, str(PROJECT_DIR))

from dynamics.dynamics import step

torch.set_float32_matmul_precision("high")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 70)
print("SAC EXPERT-PRETRAINED FINAL")
print("=" * 70)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# ------------------------------------------------------------
# SAC constants
# ------------------------------------------------------------
STATE_LOW  = np.array([0.0,0.0,-0.5,-0.2,-0.2,-0.2], dtype=np.float32)
STATE_HIGH = np.array([1.0,1.0, 0.5, 0.2, 0.2, 0.2], dtype=np.float32)

MAX_ACTION = 2.0
GAMMA = 0.99
TAU = 0.005

def normalize(x):
    lo = torch.tensor(STATE_LOW, device=x.device)
    hi = torch.tensor(STATE_HIGH, device=x.device)
    return 2.0 * (x - lo) / (hi - lo) - 1.0

# ------------------------------------------------------------
# Actor
# ------------------------------------------------------------
class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(12,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU()
        )
        self.mean = nn.Linear(256,3)
        self.log_std = nn.Linear(256,3)

    def forward(self, state, desired):
        x = torch.cat([normalize(state), normalize(desired)], dim=1)
        h = self.net(x)

        mean = self.mean(h)
        log_std = torch.clamp(self.log_std(h), -5.0, -1.0)
        std = log_std.exp()

        dist = torch.distributions.Normal(mean, std)
        z = dist.rsample()
        y = torch.tanh(z)

        action = MAX_ACTION * y

        log_prob = dist.log_prob(z)
        log_prob -= torch.log(1 - y.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=1, keepdim=True)

        return action, log_prob

    def deterministic(self, state, desired):
        x = torch.cat([normalize(state), normalize(desired)], dim=1)
        h = self.net(x)
        return MAX_ACTION * torch.tanh(self.mean(h))

# ------------------------------------------------------------
# Critic
# ------------------------------------------------------------
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.q = nn.Sequential(
            nn.Linear(15,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,1)
        )

    def forward(self, state, desired, action):
        x = torch.cat(
            [normalize(state), normalize(desired), action / MAX_ACTION],
            dim=1
        )
        return self.q(x)

# ------------------------------------------------------------
# Load demonstrations
# ------------------------------------------------------------
data = np.load(DATA)

states = torch.tensor(data["states"], dtype=torch.float32, device=DEVICE)
desired = torch.tensor(data["desired_states"], dtype=torch.float32, device=DEVICE)
actions = torch.tensor(data["actions"], dtype=torch.float32, device=DEVICE)
next_states = torch.tensor(data["next_states"], dtype=torch.float32, device=DEVICE)
rewards = torch.tensor(data["rewards"], dtype=torch.float32, device=DEVICE).unsqueeze(1)

N = len(states)

print("Expert transitions:", N)

# ------------------------------------------------------------
# Models
# ------------------------------------------------------------
actor = Actor().to(DEVICE)
critic1 = Critic().to(DEVICE)
critic2 = Critic().to(DEVICE)

target1 = Critic().to(DEVICE)
target2 = Critic().to(DEVICE)

target1.load_state_dict(critic1.state_dict())
target2.load_state_dict(critic2.state_dict())

# ------------------------------------------------------------
# STAGE 1 — Strong actor pretraining
# ------------------------------------------------------------
print("\nSTAGE 1 — SAC ACTOR EXPERT PRETRAINING")

actor_opt = torch.optim.Adam(actor.parameters(), lr=3e-4)

batch_size = 512
steps = 30000

best_loss = float("inf")

for i in range(steps):
    idx = torch.randint(0, N, (batch_size,), device=DEVICE)

    pred = actor.deterministic(states[idx], desired[idx])

    loss = F.mse_loss(pred, actions[idx])

    actor_opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
    actor_opt.step()

    if (i + 1) % 3000 == 0:
        print(f"BC {i+1:5d}/{steps}  loss={loss.item():.8f}")

    if loss.item() < best_loss:
        best_loss = loss.item()

print("Best BC loss:", best_loss)

# ------------------------------------------------------------
# STAGE 2 — Critic supervised initialization
# ------------------------------------------------------------
print("\nSTAGE 2 — CRITIC INITIALIZATION")

critic_opt1 = torch.optim.Adam(critic1.parameters(), lr=2e-4)
critic_opt2 = torch.optim.Adam(critic2.parameters(), lr=2e-4)

# Monte Carlo approximation from stored trajectories.
# Each episode contains 300 transitions.
returns = torch.zeros_like(rewards)

with torch.no_grad():
    running = torch.zeros(1,1,device=DEVICE)

    for i in range(N-1, -1, -1):
        if i % 300 == 299:
            running.zero_()

        running = rewards[i:i+1] + GAMMA * running
        returns[i:i+1] = running

print(
    "Return range:",
    returns.min().item(),
    "to",
    returns.max().item()
)

critic_steps = 8000

for i in range(critic_steps):
    idx = torch.randint(0, N, (batch_size,), device=DEVICE)

    q1 = critic1(states[idx], desired[idx], actions[idx])
    q2 = critic2(states[idx], desired[idx], actions[idx])

    loss1 = F.smooth_l1_loss(q1, returns[idx])
    loss2 = F.smooth_l1_loss(q2, returns[idx])

    critic_opt1.zero_grad(set_to_none=True)
    loss1.backward()
    torch.nn.utils.clip_grad_norm_(critic1.parameters(), 1.0)
    critic_opt1.step()

    critic_opt2.zero_grad(set_to_none=True)
    loss2.backward()
    torch.nn.utils.clip_grad_norm_(critic2.parameters(), 1.0)
    critic_opt2.step()

    if (i + 1) % 1000 == 0:
        print(
            f"Critic {i+1:5d}/{critic_steps} "
            f"Q1={loss1.item():.5f} Q2={loss2.item():.5f}"
        )

# ------------------------------------------------------------
# STAGE 3 — ACTUAL SAC
# ------------------------------------------------------------
print("\nSTAGE 3 — ACTUAL SAC")

# Very small actor learning rate so SAC improves rather than destroys
# the expert policy.
actor_opt = torch.optim.Adam(actor.parameters(), lr=1e-7)
critic_opt1 = torch.optim.Adam(critic1.parameters(), lr=5e-5)
critic_opt2 = torch.optim.Adam(critic2.parameters(), lr=5e-5)

log_alpha = torch.tensor(
    np.log(0.0001),
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

alpha_opt = torch.optim.Adam([log_alpha], lr=1e-7)

TARGET_ENTROPY = -3.0
BC_LAMBDA = 100.0

sac_steps = 5000

for i in range(sac_steps):

    idx = torch.randint(0, N, (batch_size,), device=DEVICE)

    s = states[idx]
    d = desired[idx]
    a = actions[idx]
    ns = next_states[idx]
    r = rewards[idx]

    alpha = log_alpha.exp().detach()

    # --------------------------------------------------------
    # SAC target
    # --------------------------------------------------------
    with torch.no_grad():

        na, nlogp = actor(ns, d)

        tq1 = target1(ns, d, na)
        tq2 = target2(ns, d, na)

        tq = torch.min(tq1, tq2) - alpha * nlogp

        target = r + GAMMA * tq

        target = torch.clamp(target, -100.0, 10.0)

    q1 = critic1(s,d,a)
    q2 = critic2(s,d,a)

    critic_loss = (
        F.smooth_l1_loss(q1,target)
        + F.smooth_l1_loss(q2,target)
    )

    critic_opt1.zero_grad(set_to_none=True)
    critic_opt2.zero_grad(set_to_none=True)

    critic_loss.backward()

    torch.nn.utils.clip_grad_norm_(critic1.parameters(),1.0)
    torch.nn.utils.clip_grad_norm_(critic2.parameters(),1.0)

    critic_opt1.step()
    critic_opt2.step()

    # --------------------------------------------------------
    # SAC actor
    # --------------------------------------------------------
    new_a, logp = actor(s,d)

    q1_new = critic1(s,d,new_a)
    q2_new = critic2(s,d,new_a)

    q_new = torch.min(q1_new,q2_new)

    sac_actor_loss = (alpha * logp - q_new).mean()

    # Keep actor close to expert behavior.
    expert_action = actor.deterministic(s,d)
    bc_loss = F.mse_loss(expert_action,a)

    actor_loss = sac_actor_loss + BC_LAMBDA * bc_loss

    actor_opt.zero_grad(set_to_none=True)
    actor_loss.backward()

    torch.nn.utils.clip_grad_norm_(actor.parameters(),0.25)
    actor_opt.step()

    # --------------------------------------------------------
    # Alpha
    # --------------------------------------------------------
    alpha_loss = -(log_alpha * (logp + TARGET_ENTROPY).detach()).mean()

    alpha_opt.zero_grad(set_to_none=True)
    alpha_loss.backward()
    alpha_opt.step()

    log_alpha.data.clamp_(np.log(1e-5), np.log(0.01))

    # --------------------------------------------------------
    # Target networks
    # --------------------------------------------------------
    with torch.no_grad():
        for p,t in zip(critic1.parameters(),target1.parameters()):
            t.mul_(1-TAU)
            t.add_(TAU*p)

        for p,t in zip(critic2.parameters(),target2.parameters()):
            t.mul_(1-TAU)
            t.add_(TAU*p)

    if (i+1) % 500 == 0:
        print(
            f"SAC {i+1:5d}/{sac_steps} "
            f"critic={critic_loss.item():.6f} "
            f"actor={actor_loss.item():.6f} "
            f"BC={bc_loss.item():.8f} "
            f"alpha={log_alpha.exp().item():.7f}"
        )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
torch.save({
    "actor": actor.state_dict(),
    "critic1": critic1.state_dict(),
    "critic2": critic2.state_dict(),
    "target1": target1.state_dict(),
    "target2": target2.state_dict(),
    "log_alpha": log_alpha.detach(),
    "algorithm": "SAC",
    "state_dim": 6,
    "desired_dim": 6,
    "action_dim": 3,
    "max_action": 2.0,
    "gamma": GAMMA,
    "tau": TAU
}, OUT)

print("\n" + "="*70)
print("SAVED:", OUT)
print("="*70)
'''

SCRIPT.write_text(code, encoding="utf-8")

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)


Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
SAC EXPERT-PRETRAINED FINAL
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Expert transitions: 30000

STAGE 1 — SAC ACTOR EXPERT PRETRAINING
BC  3000/30000  loss=0.02652295
BC  6000/30000  loss=0.01108635
BC  9000/30000  loss=0.00784572
BC 12000/30000  loss=0.00347150
BC 15000/30000  loss=0.01045770
BC 18000/30000  loss=0.00610661
BC 21000/30000  loss=0.00168050
BC 24000/30000  loss=0.00291770
BC 27000/30000  loss=0.00420094
BC 30000/30000  loss=0.00191945
Best BC loss: 0.00042983394814655185

STAGE 2 — CRITIC INITIALIZATION
Return range: -22.26797866821289 to -7.368598016910255e-05
Critic  1000/8000 Q1=0.10945 Q2=0.12845
Critic  2000/8000 Q1=0.05075 Q2=0.05380
Critic  3000/8000 Q1=0.01729 Q2=0.01564
Critic  4000/8000 Q1=0.01143 Q2=0.02032
Critic  5000/8000 Q1=0.00238 Q2=0.00301
Critic  6000/8000 Q1=0.00373 Q2=0.00265
Critic  7000/8000 Q1=0.00165 Q2=0.00210
Criti

In [112]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
EVAL = PROJECT_DIR / "evaluate_sac_expert_final.py"

code = r'''
import sys
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
CKPT = PROJECT_DIR / "training" / "final_model" / "sac_expert_sac_final.pt"

sys.path.insert(0, str(PROJECT_DIR))
from dynamics.dynamics import step

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

STATE_LOW = np.array(
    [0.0,0.0,-0.5,-0.2,-0.2,-0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0,1.0,0.5,0.2,0.2,0.2],
    dtype=np.float32
)

MAX_ACTION = 2.0


def normalize(x):
    lo = torch.tensor(STATE_LOW, device=x.device)
    hi = torch.tensor(STATE_HIGH, device=x.device)
    return 2.0 * (x - lo) / (hi - lo) - 1.0


class Actor(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(12,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU(),
            nn.Linear(256,256),
            nn.ReLU()
        )

        self.mean = nn.Linear(256,3)
        self.log_std = nn.Linear(256,3)

    def deterministic(self,state,desired):

        x = torch.cat(
            [normalize(state), normalize(desired)],
            dim=1
        )

        h = self.net(x)

        return MAX_ACTION * torch.tanh(self.mean(h))


# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

print("=" * 70)
print("SAC EXPERT-PRETRAINED — FIXED UNSEEN EVALUATION")
print("=" * 70)

print("Checkpoint:", CKPT)
print("Device:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")

ckpt = torch.load(
    CKPT,
    map_location=DEVICE,
    weights_only=False
)

actor = Actor().to(DEVICE)
actor.load_state_dict(ckpt["actor"])
actor.eval()

print("Checkpoint loaded successfully.")
print("Algorithm:", ckpt.get("algorithm","SAC"))

# ------------------------------------------------------------
# FIXED EVALUATION
# ------------------------------------------------------------

EVAL_SEED = 12345
EVAL_TASKS = 100
EVAL_STEPS = 100

rng = np.random.default_rng(EVAL_SEED)

initial_errors = []
final_errors = []
improvements = []
best_errors = []
rewards_all = []

successes = 0
unstable = 0

for task in range(EVAL_TASKS):

    state = np.array([
        rng.uniform(0.4,0.6),
        rng.uniform(0.4,0.6),
        rng.uniform(-0.1,0.1),
        rng.uniform(-0.1,0.1),
        rng.uniform(-0.1,0.1),
        rng.uniform(-0.1,0.1)
    ], dtype=np.float64)

    desired = np.array([
        rng.uniform(0.4,0.6),
        rng.uniform(0.4,0.6),
        rng.uniform(-0.1,0.1),
        0.0,
        0.0,
        0.0
    ], dtype=np.float64)

    initial_error = np.linalg.norm(
        state[:3] - desired[:3]
    )

    best_error = initial_error
    total_reward = 0.0
    failed = False

    for t in range(EVAL_STEPS):

        s_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        d_tensor = torch.tensor(
            desired,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        with torch.no_grad():
            action = actor.deterministic(
                s_tensor,
                d_tensor
            ).cpu().numpy()[0]

        action = np.clip(
            action,
            -MAX_ACTION,
            MAX_ACTION
        )

        try:
            next_state = np.asarray(
                step(state, action, dt=0.01),
                dtype=np.float64
            )
        except Exception:
            failed = True
            unstable += 1
            break

        if (
            not np.all(np.isfinite(next_state))
            or np.any(np.abs(next_state) > 5.0)
        ):
            failed = True
            unstable += 1
            break

        reward = (
            -np.sum((next_state - desired) ** 2)
            -0.01*np.sum(action**2)
        )

        total_reward += reward

        state = next_state

        error = np.linalg.norm(
            state[:3] - desired[:3]
        )

        best_error = min(best_error,error)

    if failed:
        final_error = 999.0
        improvement = -999.0
    else:
        final_error = np.linalg.norm(
            state[:3] - desired[:3]
        )

        improvement = initial_error - final_error

        if final_error < initial_error:
            successes += 1

    initial_errors.append(initial_error)
    final_errors.append(final_error)
    improvements.append(improvement)
    best_errors.append(best_error)
    rewards_all.append(total_reward)


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(f"Average initial error : {np.mean(initial_errors):.6f}")
print(f"Average final error   : {np.mean(final_errors):.6f}")
print(f"Average improvement   : {np.mean(improvements):.6f}")
print(f"Best final error      : {np.min(final_errors):.6f}")
print(f"Worst final error     : {np.max(final_errors):.6f}")
print(f"Average reward        : {np.mean(rewards_all):.6f}")
print(f"Unstable episodes     : {unstable}/100")
print(f"Successes             : {successes}/100")
print(f"Success rate          : {successes:.2f}%")
print("=" * 70)
'''

EVAL.write_text(code, encoding="utf-8")

result = subprocess.run(
    [sys.executable, str(EVAL)],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
SAC EXPERT-PRETRAINED — FIXED UNSEEN EVALUATION
Checkpoint: C:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_expert_sac_final.pt
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Checkpoint loaded successfully.
Algorithm: SAC

FINAL RESULTS
Average initial error : 0.127099
Average final error   : 0.084608
Average improvement   : 0.042491
Best final error      : 0.017716
Worst final error     : 0.356507
Average reward        : -4.913526
Unstable episodes     : 0/100
Successes             : 84/100
Success rate          : 84.00%


Return code: 0


In [113]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
SCRIPT = PROJECT_DIR / "sac_finetune_84_to_90.py"

code = r'''
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
CKPT = PROJECT_DIR / "training" / "final_model" / "sac_expert_sac_final.pt"
OUT = PROJECT_DIR / "training" / "final_model" / "sac_expert_sac_finetuned.pt"

sys.path.insert(0, str(PROJECT_DIR))
from dynamics.dynamics import step

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

STATE_LOW = np.array([0,0,-.5,-.2,-.2,-.2],dtype=np.float32)
STATE_HIGH = np.array([1,1,.5,.2,.2,.2],dtype=np.float32)
MAX_ACTION = 2.0
GAMMA = .99
TAU = .005

def norm(x):
    lo=torch.tensor(STATE_LOW,device=x.device)
    hi=torch.tensor(STATE_HIGH,device=x.device)
    return 2*(x-lo)/(hi-lo)-1

class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(12,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU()
        )
        self.mean=nn.Linear(256,3)
        self.log_std=nn.Linear(256,3)

    def forward(self,s,d):
        h=self.net(torch.cat([norm(s),norm(d)],1))
        mean=self.mean(h)
        ls=torch.clamp(self.log_std(h),-5,-1)
        std=ls.exp()
        dist=torch.distributions.Normal(mean,std)
        z=dist.rsample()
        y=torch.tanh(z)
        a=MAX_ACTION*y
        lp=dist.log_prob(z)-torch.log(1-y.pow(2)+1e-6)
        return a,lp.sum(1,keepdim=True)

    def deterministic(self,s,d):
        h=self.net(torch.cat([norm(s),norm(d)],1))
        return MAX_ACTION*torch.tanh(self.mean(h))

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.q=nn.Sequential(
            nn.Linear(15,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),
            nn.Linear(256,1)
        )

    def forward(self,s,d,a):
        return self.q(torch.cat([norm(s),norm(d),a/MAX_ACTION],1))

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

print("="*70)
print("SAC FINE-TUNING FROM 84% CHECKPOINT")
print("="*70)
print("Device:",torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")

ckpt=torch.load(CKPT,map_location=DEVICE,weights_only=False)

actor=Actor().to(DEVICE)
critic1=Critic().to(DEVICE)
critic2=Critic().to(DEVICE)
target1=Critic().to(DEVICE)
target2=Critic().to(DEVICE)

actor.load_state_dict(ckpt["actor"])
critic1.load_state_dict(ckpt["critic1"])
critic2.load_state_dict(ckpt["critic2"])
target1.load_state_dict(ckpt["target1"])
target2.load_state_dict(ckpt["target2"])

print("Checkpoint loaded.")

# ------------------------------------------------------------
# EXPERT DATA
# ------------------------------------------------------------

data=np.load(
    PROJECT_DIR/"training"/"final_model"/
    "sac_expert_demonstrations.npz"
)

states=torch.tensor(data["states"],dtype=torch.float32,device=DEVICE)
desired=torch.tensor(data["desired_states"],dtype=torch.float32,device=DEVICE)
actions=torch.tensor(data["actions"],dtype=torch.float32,device=DEVICE)
next_states=torch.tensor(data["next_states"],dtype=torch.float32,device=DEVICE)
rewards=torch.tensor(data["rewards"],dtype=torch.float32,device=DEVICE).unsqueeze(1)

N=len(states)
B=512

# ------------------------------------------------------------
# SAC OPTIMIZERS
# ------------------------------------------------------------

actor_opt=torch.optim.Adam(actor.parameters(),lr=3e-7)
critic_opt1=torch.optim.Adam(critic1.parameters(),lr=2e-5)
critic_opt2=torch.optim.Adam(critic2.parameters(),lr=2e-5)

log_alpha=torch.tensor(
    np.log(.0001),
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

alpha_opt=torch.optim.Adam([log_alpha],lr=3e-8)

# ------------------------------------------------------------
# FINE-TUNE
# ------------------------------------------------------------

STEPS=3000
BC_LAMBDA=100.0
TARGET_ENTROPY=-3.0

for i in range(STEPS):

    idx=torch.randint(0,N,(B,),device=DEVICE)

    s=states[idx]
    d=desired[idx]
    a=actions[idx]
    ns=next_states[idx]
    r=rewards[idx]

    alpha=log_alpha.exp().detach()

    # SAC target
    with torch.no_grad():

        na,nlp=actor(ns,d)

        tq1=target1(ns,d,na)
        tq2=target2(ns,d,na)

        target=r+GAMMA*(torch.min(tq1,tq2)-alpha*nlp)

        target=torch.clamp(target,-100,10)

    q1=critic1(s,d,a)
    q2=critic2(s,d,a)

    closs=(
        F.smooth_l1_loss(q1,target)+
        F.smooth_l1_loss(q2,target)
    )

    critic_opt1.zero_grad(set_to_none=True)
    critic_opt2.zero_grad(set_to_none=True)

    closs.backward()

    torch.nn.utils.clip_grad_norm_(critic1.parameters(),1)
    torch.nn.utils.clip_grad_norm_(critic2.parameters(),1)

    critic_opt1.step()
    critic_opt2.step()

    # Actor
    new_a,lp=actor(s,d)

    qnew=torch.min(
        critic1(s,d,new_a),
        critic2(s,d,new_a)
    )

    sac_loss=(alpha*lp-qnew).mean()

    expert_pred=actor.deterministic(s,d)
    bc_loss=F.mse_loss(expert_pred,a)

    actor_loss=sac_loss+BC_LAMBDA*bc_loss

    actor_opt.zero_grad(set_to_none=True)
    actor_loss.backward()

    torch.nn.utils.clip_grad_norm_(actor.parameters(),.1)
    actor_opt.step()

    # Alpha
    aloss=-(log_alpha*(lp+TARGET_ENTROPY).detach()).mean()

    alpha_opt.zero_grad(set_to_none=True)
    aloss.backward()
    alpha_opt.step()

    log_alpha.data.clamp_(np.log(1e-5),np.log(.01))

    # Target networks
    with torch.no_grad():
        for p,t in zip(critic1.parameters(),target1.parameters()):
            t.mul_(1-TAU)
            t.add_(TAU*p)

        for p,t in zip(critic2.parameters(),target2.parameters()):
            t.mul_(1-TAU)
            t.add_(TAU*p)

    if (i+1)%500==0:
        print(
            f"SAC {i+1}/{STEPS} "
            f"critic={closs.item():.6f} "
            f"actor={actor_loss.item():.6f} "
            f"BC={bc_loss.item():.8f} "
            f"alpha={log_alpha.exp().item():.7f}"
        )

torch.save({
    "actor":actor.state_dict(),
    "critic1":critic1.state_dict(),
    "critic2":critic2.state_dict(),
    "target1":target1.state_dict(),
    "target2":target2.state_dict(),
    "log_alpha":log_alpha.detach(),
    "algorithm":"SAC",
    "state_dim":6,
    "desired_dim":6,
    "action_dim":3,
    "max_action":2.0,
    "gamma":GAMMA,
    "tau":TAU
},OUT)

print("\nSAVED:",OUT)
'''

SCRIPT.write_text(code,encoding="utf-8")

result=subprocess.run(
    [sys.executable,str(SCRIPT)],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:",result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
SAC FINE-TUNING FROM 84% CHECKPOINT
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Checkpoint loaded.
SAC 500/3000 critic=0.325256 actor=7.996812 BC=0.00040114 alpha=0.0001000
SAC 1000/3000 critic=0.347930 actor=8.877541 BC=0.00092373 alpha=0.0001000
SAC 1500/3000 critic=0.334434 actor=9.521782 BC=0.00071909 alpha=0.0001000
SAC 2000/3000 critic=0.375754 actor=10.125516 BC=0.00106643 alpha=0.0001000
SAC 2500/3000 critic=0.428839 actor=10.788326 BC=0.00187282 alpha=0.0001000
SAC 3000/3000 critic=0.394002 actor=11.313920 BC=0.00049626 alpha=0.0001000

SAVED: C:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_expert_sac_finetuned.pt


Return code: 0


In [114]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
SCRIPT = PROJECT_DIR / "evaluate_sac_finetuned.py"

# Reuse the compatible evaluator from the previous step
old = PROJECT_DIR / "evaluate_sac_expert_final.py"
text = old.read_text(encoding="utf-8")

text = text.replace(
    "sac_expert_sac_final.pt",
    "sac_expert_sac_finetuned.pt"
)

SCRIPT.write_text(text, encoding="utf-8")

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
SAC EXPERT-PRETRAINED — FIXED UNSEEN EVALUATION
Checkpoint: C:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_expert_sac_finetuned.pt
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Checkpoint loaded successfully.
Algorithm: SAC

FINAL RESULTS
Average initial error : 0.127099
Average final error   : 0.084621
Average improvement   : 0.042479
Best final error      : 0.017563
Worst final error     : 0.356385
Average reward        : -4.910447
Unstable episodes     : 0/100
Successes             : 84/100
Success rate          : 84.00%


Return code: 0


In [115]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
SCRIPT = PROJECT_DIR / "sac_online_improvement.py"

code = r'''
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
CKPT = PROJECT_DIR/"training/final_model/sac_expert_sac_final.pt"
OUT  = PROJECT_DIR/"training/final_model/sac_online_improved.pt"

sys.path.insert(0,str(PROJECT_DIR))
from dynamics.dynamics import step

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")

LOW=np.array([0,0,-.5,-.2,-.2,-.2],np.float32)
HIGH=np.array([1,1,.5,.2,.2,.2],np.float32)

MAX_A=2.
GAMMA=.99
TAU=.005

def norm(x):
    lo=torch.tensor(LOW,device=x.device)
    hi=torch.tensor(HIGH,device=x.device)
    return 2*(x-lo)/(hi-lo)-1

class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(12,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU()
        )
        self.mean=nn.Linear(256,3)
        self.log_std=nn.Linear(256,3)

    def forward(self,s,d):
        h=self.net(torch.cat([norm(s),norm(d)],1))
        mean=self.mean(h)
        ls=torch.clamp(self.log_std(h),-5,-1)
        std=ls.exp()
        dist=torch.distributions.Normal(mean,std)
        z=dist.rsample()
        y=torch.tanh(z)
        a=MAX_A*y
        lp=dist.log_prob(z)-torch.log(1-y.pow(2)+1e-6)
        return a,lp.sum(1,keepdim=True)

    def deterministic(self,s,d):
        h=self.net(torch.cat([norm(s),norm(d)],1))
        return MAX_A*torch.tanh(self.mean(h))

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.q=nn.Sequential(
            nn.Linear(15,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),
            nn.Linear(256,1)
        )

    def forward(self,s,d,a):
        return self.q(torch.cat([norm(s),norm(d),a/MAX_A],1))

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

print("="*70)
print("SAC ONLINE IMPROVEMENT")
print("="*70)
print("Device:",torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")

ckpt=torch.load(CKPT,map_location=DEVICE,weights_only=False)

actor=Actor().to(DEVICE)
critic1=Critic().to(DEVICE)
critic2=Critic().to(DEVICE)
target1=Critic().to(DEVICE)
target2=Critic().to(DEVICE)

actor.load_state_dict(ckpt["actor"])
critic1.load_state_dict(ckpt["critic1"])
critic2.load_state_dict(ckpt["critic2"])
target1.load_state_dict(ckpt["target1"])
target2.load_state_dict(ckpt["target2"])

# ------------------------------------------------------------
# EXPERT DATA
# ------------------------------------------------------------

data=np.load(
    PROJECT_DIR/"training/final_model/sac_expert_demonstrations.npz"
)

ES=torch.tensor(data["states"],dtype=torch.float32,device=DEVICE)
ED=torch.tensor(data["desired_states"],dtype=torch.float32,device=DEVICE)
EA=torch.tensor(data["actions"],dtype=torch.float32,device=DEVICE)
ENS=torch.tensor(data["next_states"],dtype=torch.float32,device=DEVICE)
ER=torch.tensor(data["rewards"],dtype=torch.float32,device=DEVICE).unsqueeze(1)

N=len(ES)

print("Expert transitions:",N)

# ------------------------------------------------------------
# ONLINE REPLAY
# ------------------------------------------------------------

online=[]

rng=np.random.default_rng(20260914)

print("\nCOLLECTING ONLINE SAC TRANSITIONS")

# Use deterministic policy for data collection.
# This preserves the strong 84% policy while generating
# transitions from the actual dynamics.

for ep in range(150):

    s=np.array([
        rng.uniform(.4,.6),
        rng.uniform(.4,.6),
        rng.uniform(-.1,.1),
        rng.uniform(-.1,.1),
        rng.uniform(-.1,.1),
        rng.uniform(-.1,.1)
    ],dtype=np.float64)

    d=np.array([
        rng.uniform(.4,.6),
        rng.uniform(.4,.6),
        rng.uniform(-.1,.1),
        0,0,0
    ],dtype=np.float64)

    for t in range(100):

        st=torch.tensor(s,dtype=torch.float32,device=DEVICE).unsqueeze(0)
        dt=torch.tensor(d,dtype=torch.float32,device=DEVICE).unsqueeze(0)

        with torch.no_grad():
            a=actor.deterministic(st,dt).cpu().numpy()[0]

        a=np.clip(a,-2,2)

        try:
            ns=np.asarray(step(s,a,dt=.01),dtype=np.float64)
        except:
            break

        if not np.all(np.isfinite(ns)) or np.any(np.abs(ns)>5):
            break

        r=(
            -np.sum((ns-d)**2)
            -.01*np.sum(a**2)
        )

        done=float(t==99)

        online.append(
            (s.copy(),d.copy(),a.copy(),ns.copy(),r,done)
        )

        s=ns

print("Online transitions:",len(online))

# ------------------------------------------------------------
# SAC TRAINING
# ------------------------------------------------------------

aopt=torch.optim.Adam(actor.parameters(),lr=2e-7)
q1opt=torch.optim.Adam(critic1.parameters(),lr=3e-5)
q2opt=torch.optim.Adam(critic2.parameters(),lr=3e-5)

log_alpha=torch.tensor(
    np.log(.0001),
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

alphaopt=torch.optim.Adam([log_alpha],lr=1e-7)

B=512
STEPS=5000

# 70% expert / 30% online
for i in range(STEPS):

    ne=int(B*.7)
    no=B-ne

    ei=torch.randint(0,N,(ne,),device=DEVICE)

    if len(online)>0:
        oi=np.random.randint(0,len(online),no)

        os=np.array([online[j][0] for j in oi],np.float32)
        od=np.array([online[j][1] for j in oi],np.float32)
        oa=np.array([online[j][2] for j in oi],np.float32)
        ons=np.array([online[j][3] for j in oi],np.float32)
        orr=np.array([online[j][4] for j in oi],np.float32)[:,None]
        odone=np.array([online[j][5] for j in oi],np.float32)[:,None]

        s=torch.cat([
            ES[ei],
            torch.tensor(os,device=DEVICE)
        ])

        d=torch.cat([
            ED[ei],
            torch.tensor(od,device=DEVICE)
        ])

        act=torch.cat([
            EA[ei],
            torch.tensor(oa,device=DEVICE)
        ])

        ns=torch.cat([
            ENS[ei],
            torch.tensor(ons,device=DEVICE)
        ])

        r=torch.cat([
            ER[ei],
            torch.tensor(orr,device=DEVICE)
        ])

        done=torch.cat([
            torch.zeros((ne,1),device=DEVICE),
            torch.tensor(odone,device=DEVICE)
        ])
    else:
        continue

    alpha=log_alpha.exp().detach()

    with torch.no_grad():

        na,nlp=actor(ns,d)

        tq=torch.min(
            target1(ns,d,na),
            target2(ns,d,na)
        )

        y=r+GAMMA*(1-done)*(tq-alpha*nlp)
        y=torch.clamp(y,-100,10)

    q1=critic1(s,d,act)
    q2=critic2(s,d,act)

    qloss=(
        F.smooth_l1_loss(q1,y)+
        F.smooth_l1_loss(q2,y)
    )

    q1opt.zero_grad(set_to_none=True)
    q2opt.zero_grad(set_to_none=True)

    qloss.backward()

    torch.nn.utils.clip_grad_norm_(critic1.parameters(),1)
    torch.nn.utils.clip_grad_norm_(critic2.parameters(),1)

    q1opt.step()
    q2opt.step()

    # Actor
    newa,lp=actor(s,d)

    qnew=torch.min(
        critic1(s,d,newa),
        critic2(s,d,newa)
    )

    sac_loss=(alpha*lp-qnew).mean()

    # Expert BC only on expert portion
    expert_pred=actor.deterministic(ES[ei],ED[ei])
    bc=F.mse_loss(expert_pred,EA[ei])

    actor_loss=sac_loss+50*bc

    aopt.zero_grad(set_to_none=True)
    actor_loss.backward()

    torch.nn.utils.clip_grad_norm_(actor.parameters(),.1)

    aopt.step()

    # Alpha
    aloss=-(log_alpha*(lp+(-3)).detach()).mean()

    alphaopt.zero_grad(set_to_none=True)
    aloss.backward()
    alphaopt.step()

    log_alpha.data.clamp_(np.log(1e-5),np.log(.01))

    # Targets
    with torch.no_grad():

        for p,t in zip(critic1.parameters(),target1.parameters()):
            t.mul_(1-TAU)
            t.add_(TAU*p)

        for p,t in zip(critic2.parameters(),target2.parameters()):
            t.mul_(1-TAU)
            t.add_(TAU*p)

    if (i+1)%500==0:
        print(
            f"SAC {i+1}/{STEPS} "
            f"critic={qloss.item():.5f} "
            f"actor={actor_loss.item():.5f} "
            f"BC={bc.item():.7f} "
            f"alpha={log_alpha.exp().item():.7f}"
        )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

torch.save({
    "actor":actor.state_dict(),
    "critic1":critic1.state_dict(),
    "critic2":critic2.state_dict(),
    "target1":target1.state_dict(),
    "target2":target2.state_dict(),
    "log_alpha":log_alpha.detach(),
    "algorithm":"SAC",
    "state_dim":6,
    "desired_dim":6,
    "action_dim":3,
    "max_action":2.,
    "gamma":GAMMA,
    "tau":TAU
},OUT)

print("\nSAVED:",OUT)
'''

SCRIPT.write_text(code,encoding="utf-8")

result=subprocess.run(
    [sys.executable,str(SCRIPT)],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:",result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
SAC ONLINE IMPROVEMENT
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Expert transitions: 30000

COLLECTING ONLINE SAC TRANSITIONS
Online transitions: 15000
SAC 500/5000 critic=0.38924 actor=8.25393 BC=0.0005475 alpha=0.0001000
SAC 1000/5000 critic=0.38390 actor=8.82472 BC=0.0012347 alpha=0.0001000
SAC 1500/5000 critic=0.33080 actor=9.38247 BC=0.0017216 alpha=0.0001000
SAC 2000/5000 critic=0.34609 actor=9.90285 BC=0.0003812 alpha=0.0001000
SAC 2500/5000 critic=0.36527 actor=10.50356 BC=0.0004457 alpha=0.0001000
SAC 3000/5000 critic=0.38847 actor=10.96566 BC=0.0008723 alpha=0.0001000
SAC 3500/5000 critic=0.38268 actor=11.55947 BC=0.0005329 alpha=0.0001000
SAC 4000/5000 critic=0.35757 actor=12.12219 BC=0.0003905 alpha=0.0001000
SAC 4500/5000 critic=0.39361 actor=12.72028 BC=0.0007856 alpha=0.0001000
SAC 5000/5000 critic=0.37577 actor=13.11555 BC=0.0005379 alpha=0.0

In [116]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
OLD = PROJECT_DIR / "evaluate_sac_expert_final.py"
NEW = PROJECT_DIR / "evaluate_sac_online.py"

text = OLD.read_text(encoding="utf-8")

text = text.replace(
    "sac_expert_sac_final.pt",
    "sac_online_improved.pt"
)

NEW.write_text(text, encoding="utf-8")

result = subprocess.run(
    [sys.executable, str(NEW)],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
SAC EXPERT-PRETRAINED — FIXED UNSEEN EVALUATION
Checkpoint: C:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_online_improved.pt
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Checkpoint loaded successfully.
Algorithm: SAC

FINAL RESULTS
Average initial error : 0.127099
Average final error   : 0.085136
Average improvement   : 0.041963
Best final error      : 0.016245
Worst final error     : 0.359391
Average reward        : -4.904094
Unstable episodes     : 0/100
Successes             : 84/100
Success rate          : 84.00%


Return code: 0


In [117]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
SCRIPT = PROJECT_DIR / "sac_final_expert_guided.py"

code = r'''
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
CKPT = PROJECT_DIR/"training/final_model/sac_expert_sac_final.pt"
DATA = PROJECT_DIR/"training/final_model/sac_expert_demonstrations.npz"
OUT = PROJECT_DIR/"training/final_model/sac_final_expert_guided.pt"

sys.path.insert(0,str(PROJECT_DIR))
from dynamics.dynamics import step

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")

LOW=np.array([0,0,-.5,-.2,-.2,-.2],np.float32)
HIGH=np.array([1,1,.5,.2,.2,.2],np.float32)

MAX_A=2.
GAMMA=.99
TAU=.005

def norm(x):
    lo=torch.tensor(LOW,device=x.device)
    hi=torch.tensor(HIGH,device=x.device)
    return 2*(x-lo)/(hi-lo)-1

class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(12,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU()
        )
        self.mean=nn.Linear(256,3)
        self.log_std=nn.Linear(256,3)

    def forward(self,s,d):
        h=self.net(torch.cat([norm(s),norm(d)],1))
        mean=self.mean(h)
        ls=torch.clamp(self.log_std(h),-5,-1)
        std=ls.exp()
        dist=torch.distributions.Normal(mean,std)
        z=dist.rsample()
        y=torch.tanh(z)
        a=MAX_A*y
        lp=dist.log_prob(z)-torch.log(1-y.pow(2)+1e-6)
        return a,lp.sum(1,keepdim=True)

    def deterministic(self,s,d):
        h=self.net(torch.cat([norm(s),norm(d)],1))
        return MAX_A*torch.tanh(self.mean(h))

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.q=nn.Sequential(
            nn.Linear(15,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),
            nn.Linear(256,1)
        )

    def forward(self,s,d,a):
        return self.q(torch.cat([norm(s),norm(d),a/MAX_A],1))

# ============================================================
# LOAD 84% SAC
# ============================================================

print("="*70)
print("FINAL EXPERT-GUIDED SAC")
print("="*70)
print("Device:",
      torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")

ckpt=torch.load(
    CKPT,
    map_location=DEVICE,
    weights_only=False
)

actor=Actor().to(DEVICE)
critic1=Critic().to(DEVICE)
critic2=Critic().to(DEVICE)
target1=Critic().to(DEVICE)
target2=Critic().to(DEVICE)

actor.load_state_dict(ckpt["actor"])
critic1.load_state_dict(ckpt["critic1"])
critic2.load_state_dict(ckpt["critic2"])
target1.load_state_dict(ckpt["target1"])
target2.load_state_dict(ckpt["target2"])

print("Loaded 84% SAC checkpoint.")

# ============================================================
# EXPERT DATA
# ============================================================

data=np.load(DATA)

S=torch.tensor(
    data["states"],
    dtype=torch.float32,
    device=DEVICE
)

D=torch.tensor(
    data["desired_states"],
    dtype=torch.float32,
    device=DEVICE
)

A=torch.tensor(
    data["actions"],
    dtype=torch.float32,
    device=DEVICE
)

NS=torch.tensor(
    data["next_states"],
    dtype=torch.float32,
    device=DEVICE
)

R=torch.tensor(
    data["rewards"],
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(1)

N=len(S)

print("Expert transitions:",N)

# ============================================================
# STAGE 1 — CRITIC ONLY
# ============================================================

print("\nSTAGE 1 — CRITIC LEARNING")

q1opt=torch.optim.Adam(
    critic1.parameters(),
    lr=1e-5
)

q2opt=torch.optim.Adam(
    critic2.parameters(),
    lr=1e-5
)

B=512

for i in range(5000):

    idx=torch.randint(
        0,N,(B,),
        device=DEVICE
    )

    s=S[idx]
    d=D[idx]
    a=A[idx]
    ns=NS[idx]
    r=R[idx]

    with torch.no_grad():

        na,_=actor(ns,d)

        target_q=torch.min(
            target1(ns,d,na),
            target2(ns,d,na)
        )

        y=r+GAMMA*target_q

        y=torch.clamp(y,-100,10)

    q1=critic1(s,d,a)
    q2=critic2(s,d,a)

    l1=F.smooth_l1_loss(q1,y)
    l2=F.smooth_l1_loss(q2,y)

    q1opt.zero_grad(set_to_none=True)
    l1.backward()
    torch.nn.utils.clip_grad_norm_(
        critic1.parameters(),.5
    )
    q1opt.step()

    q2opt.zero_grad(set_to_none=True)
    l2.backward()
    torch.nn.utils.clip_grad_norm_(
        critic2.parameters(),.5
    )
    q2opt.step()

    with torch.no_grad():

        for p,t in zip(
            critic1.parameters(),
            target1.parameters()
        ):
            t.mul_(1-TAU)
            t.add_(TAU*p)

        for p,t in zip(
            critic2.parameters(),
            target2.parameters()
        ):
            t.mul_(1-TAU)
            t.add_(TAU*p)

    if (i+1)%1000==0:
        print(
            f"Critic {i+1}/5000 "
            f"Q1={l1.item():.6f} "
            f"Q2={l2.item():.6f}"
        )

# ============================================================
# STAGE 2 — VERY SMALL ACTOR SAC UPDATE
# ============================================================

print("\nSTAGE 2 — SAC ACTOR UPDATE")

actoropt=torch.optim.Adam(
    actor.parameters(),
    lr=3e-8
)

alpha_log=torch.tensor(
    np.log(.00005),
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

alphaopt=torch.optim.Adam(
    [alpha_log],
    lr=1e-8
)

TARGET_ENTROPY=-3.
BC_WEIGHT=250.

for i in range(3000):

    idx=torch.randint(
        0,N,(B,),
        device=DEVICE
    )

    s=S[idx]
    d=D[idx]
    expert=A[idx]

    alpha=alpha_log.exp().detach()

    newa,logp=actor(s,d)

    q=torch.min(
        critic1(s,d,newa),
        critic2(s,d,newa)
    )

    sac_loss=(alpha*logp-q).mean()

    # Keep policy extremely close to demonstrated control.
    pred=actor.deterministic(s,d)

    bc_loss=F.mse_loss(pred,expert)

    actor_loss=sac_loss+BC_WEIGHT*bc_loss

    actoropt.zero_grad(set_to_none=True)

    actor_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        actor.parameters(),
        .05
    )

    actoropt.step()

    # Entropy temperature
    alpha_loss=-(
        alpha_log*
        (logp+TARGET_ENTROPY).detach()
    ).mean()

    alphaopt.zero_grad(set_to_none=True)
    alpha_loss.backward()
    alphaopt.step()

    alpha_log.data.clamp_(
        np.log(1e-5),
        np.log(.005)
    )

    if (i+1)%500==0:
        print(
            f"SAC {i+1}/3000 "
            f"actor={actor_loss.item():.6f} "
            f"SAC={sac_loss.item():.6f} "
            f"BC={bc_loss.item():.8f} "
            f"alpha={alpha_log.exp().item():.7f}"
        )

# ============================================================
# SAVE
# ============================================================

torch.save({
    "actor":actor.state_dict(),
    "critic1":critic1.state_dict(),
    "critic2":critic2.state_dict(),
    "target1":target1.state_dict(),
    "target2":target2.state_dict(),
    "log_alpha":alpha_log.detach(),
    "algorithm":"SAC",
    "state_dim":6,
    "desired_dim":6,
    "action_dim":3,
    "max_action":2.,
    "gamma":GAMMA,
    "tau":TAU
},OUT)

print("\n"+"="*70)
print("SAVED:",OUT)
print("="*70)
'''

SCRIPT.write_text(code,encoding="utf-8")

result=subprocess.run(
    [sys.executable,str(SCRIPT)],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:",result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
FINAL EXPERT-GUIDED SAC
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Loaded 84% SAC checkpoint.
Expert transitions: 30000

STAGE 1 — CRITIC LEARNING
Critic 1000/5000 Q1=0.163721 Q2=0.160469
Critic 2000/5000 Q1=0.172343 Q2=0.170391
Critic 3000/5000 Q1=0.236946 Q2=0.238958
Critic 4000/5000 Q1=0.209853 Q2=0.209913
Critic 5000/5000 Q1=0.214095 Q2=0.216306

STAGE 2 — SAC ACTOR UPDATE
SAC 500/3000 actor=14.414690 SAC=14.008684 BC=0.00162403 alpha=0.0000500
SAC 1000/3000 actor=14.189691 SAC=13.894262 BC=0.00118171 alpha=0.0000500
SAC 1500/3000 actor=14.214812 SAC=13.971345 BC=0.00097387 alpha=0.0000500
SAC 2000/3000 actor=14.024884 SAC=13.942362 BC=0.00033009 alpha=0.0000500
SAC 2500/3000 actor=14.549223 SAC=14.017199 BC=0.00212810 alpha=0.0000500
SAC 3000/3000 actor=14.192531 SAC=14.036277 BC=0.00062501 alpha=0.0000500

SAVED: C:\Users\madha\OneDrive\Desktop\project_

In [118]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
OLD = PROJECT_DIR / "evaluate_sac_expert_final.py"
NEW = PROJECT_DIR / "evaluate_sac_final_expert_guided.py"

text = OLD.read_text(encoding="utf-8")
text = text.replace(
    "sac_expert_sac_final.pt",
    "sac_final_expert_guided.pt"
)

NEW.write_text(text, encoding="utf-8")

result = subprocess.run(
    [sys.executable, str(NEW)],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nReturn code:", result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
SAC EXPERT-PRETRAINED — FIXED UNSEEN EVALUATION
Checkpoint: C:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_final_expert_guided.pt
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Checkpoint loaded successfully.
Algorithm: SAC

FINAL RESULTS
Average initial error : 0.127099
Average final error   : 0.084575
Average improvement   : 0.042524
Best final error      : 0.018228
Worst final error     : 0.355983
Average reward        : -4.912094
Unstable episodes     : 0/100
Successes             : 84/100
Success rate          : 84.00%


Return code: 0


In [119]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")
SCRIPT = PROJECT_DIR / "sac_3rpr_sac_inference.py"

code = r'''
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn


# ============================================================
# PROJECT PATH
# ============================================================

PROJECT_DIR = Path(r"C:\Users\madha\OneDrive\Desktop\project_1")

CHECKPOINT = (
    PROJECT_DIR
    / "training"
    / "final_model"
    / "sac_expert_sac_final.pt"
)


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================================
# 3-RPR STATE LIMITS
# ============================================================

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

MAX_ACTION = 2.0


# ============================================================
# NORMALIZATION
# ============================================================

def normalize(x):
    lo = torch.tensor(
        STATE_LOW,
        dtype=torch.float32,
        device=x.device
    )

    hi = torch.tensor(
        STATE_HIGH,
        dtype=torch.float32,
        device=x.device
    )

    return 2.0 * (x - lo) / (hi - lo) - 1.0


# ============================================================
# SAC ACTOR
# ============================================================

class Actor(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(12, 256),
            nn.ReLU(),

            nn.Linear(256, 256),
            nn.ReLU(),

            nn.Linear(256, 256),
            nn.ReLU()
        )

        self.mean = nn.Linear(256, 3)

        self.log_std = nn.Linear(256, 3)


    def deterministic(self, state, desired_state):

        x = torch.cat(
            [
                normalize(state),
                normalize(desired_state)
            ],
            dim=1
        )

        h = self.net(x)

        action = MAX_ACTION * torch.tanh(
            self.mean(h)
        )

        return action


# ============================================================
# LOAD SAC MODEL
# ============================================================

print("=" * 70)
print("3-RPR SAC FORCE INFERENCE")
print("=" * 70)

print("Checkpoint:")
print(CHECKPOINT)

print("\nDevice:")

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CPU")


if not CHECKPOINT.exists():

    raise FileNotFoundError(
        f"\nCheckpoint not found:\n{CHECKPOINT}"
    )


checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)


actor = Actor().to(DEVICE)

actor.load_state_dict(
    checkpoint["actor"]
)

actor.eval()

print("\nSAC actor loaded successfully.")


# ============================================================
# USER INPUT
# ============================================================

print("\n" + "=" * 70)
print("ENTER CURRENT STATE")
print("=" * 70)

print("""
Current state format:

[x, y, theta, x_dot, y_dot, theta_dot]

Example:
[0.45, 0.55, 0.05, 0.02, -0.01, 0.03]
""")

current_input = input(
    "Current state: "
).strip()


print("\n" + "=" * 70)
print("ENTER DESIRED STATE")
print("=" * 70)

print("""
Desired state format:

[xd, yd, theta_d, 0, 0, 0]

Example:
[0.50, 0.50, 0.00, 0, 0, 0]
""")

desired_input = input(
    "Desired state: "
).strip()


# ============================================================
# PARSE INPUT
# ============================================================

try:

    current_state = np.array(
        [
            float(x.strip())
            for x in current_input.strip("[]").split(",")
        ],
        dtype=np.float32
    )

    desired_state = np.array(
        [
            float(x.strip())
            for x in desired_input.strip("[]").split(",")
        ],
        dtype=np.float32
    )

except ValueError:

    raise ValueError(
        "\nInvalid input. "
        "Enter six comma-separated numbers."
    )


# ============================================================
# VALIDATE
# ============================================================

if len(current_state) != 6:

    raise ValueError(
        f"Current state must contain 6 values. "
        f"Got {len(current_state)}."
    )


if len(desired_state) != 6:

    raise ValueError(
        f"Desired state must contain 6 values. "
        f"Got {len(desired_state)}."
    )


# ============================================================
# CONVERT TO TENSOR
# ============================================================

state_tensor = torch.tensor(
    current_state,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(0)


desired_tensor = torch.tensor(
    desired_state,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(0)


# ============================================================
# SAC INFERENCE
# ============================================================

with torch.no_grad():

    forces = actor.deterministic(
        state_tensor,
        desired_tensor
    )


forces = forces.squeeze(0).cpu().numpy()

forces = np.clip(
    forces,
    -MAX_ACTION,
    MAX_ACTION
)


# ============================================================
# OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("SAC OUTPUT")
print("=" * 70)

print("\nCurrent state:")
print(current_state)

print("\nDesired state:")
print(desired_state)

print("\nForces [F1, F2, F3]:")
print(forces)

print("\nIndividual forces:")

print(f"F1 = {forces[0]:.6f}")
print(f"F2 = {forces[1]:.6f}")
print(f"F3 = {forces[2]:.6f}")

print("\nNumPy array:")
print(
    np.array2string(
        forces,
        precision=6,
        separator=", "
    )
)

print("\n" + "=" * 70)
'''

SCRIPT.write_text(
    code,
    encoding="utf-8"
)

print("Created:")
print(SCRIPT)

Created:
C:\Users\madha\OneDrive\Desktop\project_1\sac_3rpr_sac_inference.py


In [120]:
import sys
import os
import numpy as np

# ============================================================
# FIXED EXPANDED RANGE — DO NOT CHANGE
# ============================================================
STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

DESIRED_LOW = np.array(
    [0.0, 0.0, -0.5],
    dtype=np.float32
)

DESIRED_HIGH = np.array(
    [1.0, 1.0, 0.5],
    dtype=np.float32
)

N = 10000
rng = np.random.default_rng(20260914)

# Generate CURRENT states from the FULL expanded range
states = rng.uniform(
    STATE_LOW,
    STATE_HIGH,
    size=(N, 6)
).astype(np.float32)

# Generate DESIRED states from the FULL expanded pose range
desired_states = np.zeros((N, 6), dtype=np.float32)

desired_states[:, :3] = rng.uniform(
    DESIRED_LOW,
    DESIRED_HIGH,
    size=(N, 3)
).astype(np.float32)

print("=" * 70)
print("EXPANDED RANGE VERIFICATION")
print("=" * 70)

names = [
    "x",
    "y",
    "theta",
    "x_dot",
    "y_dot",
    "theta_dot"
]

print("\nCURRENT STATE:")
for i, name in enumerate(names):
    print(
        f"{name:8s}: "
        f"min={states[:, i].min(): .6f}   "
        f"max={states[:, i].max(): .6f}   "
        f"required=[{STATE_LOW[i]}, {STATE_HIGH[i]}]"
    )

print("\nDESIRED STATE:")
for i, name in enumerate(names):
    print(
        f"{name:8s}: "
        f"min={desired_states[:, i].min(): .6f}   "
        f"max={desired_states[:, i].max(): .6f}   "
        f"required=[{STATE_LOW[i]}, {STATE_HIGH[i]}]"
    )

# Verify every generated sample is actually inside the required range
state_ok = (
    np.all(states >= STATE_LOW - 1e-6)
    and
    np.all(states <= STATE_HIGH + 1e-6)
)

desired_ok = (
    np.all(desired_states >= STATE_LOW - 1e-6)
    and
    np.all(desired_states <= STATE_HIGH + 1e-6)
)

print("\n" + "=" * 70)
print(f"CURRENT STATES INSIDE EXPANDED RANGE : {state_ok}")
print(f"DESIRED STATES INSIDE EXPANDED RANGE : {desired_ok}")
print("=" * 70)

if state_ok and desired_ok:
    print("\n✅ RANGE CHECK PASSED")
    print("The data generator is using the required expanded range.")
else:
    print("\n❌ RANGE CHECK FAILED")
    print("DO NOT TRAIN.")

EXPANDED RANGE VERIFICATION

CURRENT STATE:
x       : min= 0.000000   max= 0.999885   required=[0.0, 1.0]
y       : min= 0.000043   max= 0.999955   required=[0.0, 1.0]
theta   : min=-0.499998   max= 0.499946   required=[-0.5, 0.5]
x_dot   : min=-0.199977   max= 0.199944   required=[-0.20000000298023224, 0.20000000298023224]
y_dot   : min=-0.199947   max= 0.199884   required=[-0.20000000298023224, 0.20000000298023224]
theta_dot: min=-0.199843   max= 0.199931   required=[-0.20000000298023224, 0.20000000298023224]

DESIRED STATE:
x       : min= 0.000055   max= 0.999901   required=[0.0, 1.0]
y       : min= 0.000128   max= 0.999995   required=[0.0, 1.0]
theta   : min=-0.499995   max= 0.499665   required=[-0.5, 0.5]
x_dot   : min= 0.000000   max= 0.000000   required=[-0.20000000298023224, 0.20000000298023224]
y_dot   : min= 0.000000   max= 0.000000   required=[-0.20000000298023224, 0.20000000298023224]
theta_dot: min= 0.000000   max= 0.000000   required=[-0.20000000298023224, 0.2000000029802

In [124]:
import numpy as np

# Use NumPy arrays, NOT Python lists
test_state = np.array([
    0.5, 0.5, 0.0,
    0.0, 0.0, 0.0
], dtype=np.float64)

test_force = np.array([
    0.0, 0.0, 0.0
], dtype=np.float64)

test_next = step(
    test_state,
    test_force,
    dt=0.01
)

print("=" * 60)
print("3-RPR DYNAMICS SANITY CHECK")
print("=" * 60)

print("State:")
print(test_state)

print("\nForce:")
print(test_force)

print("\nNext state:")
print(np.asarray(test_next))

print("\nFinite:", np.all(np.isfinite(np.asarray(test_next))))
print("=" * 60)

print("✅ Dynamics step works correctly.")

3-RPR DYNAMICS SANITY CHECK
State:
[0.5 0.5 0.  0.  0.  0. ]

Force:
[0. 0. 0.]

Next state:
[0.5 0.5 0.  0.  0.  0. ]

Finite: True
✅ Dynamics step works correctly.


In [125]:
import numpy as np

# ============================================================
# FIXED PROJECT RANGES — DO NOT CHANGE
# ============================================================
STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float64
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float64
)

DESIRED_LOW = np.array(
    [0.0, 0.0, -0.5],
    dtype=np.float64
)

DESIRED_HIGH = np.array(
    [1.0, 1.0, 0.5],
    dtype=np.float64
)

ACTION_LOW = -2.0
ACTION_HIGH = 2.0
DT = 0.01

KP = np.array([2.0, 2.0, 2.0], dtype=np.float64)
KD = np.array([1.5, 1.5, 1.5], dtype=np.float64)
RIDGE = 1e-4

N_TASKS = 100

rng = np.random.default_rng(12345)


# ============================================================
# NUMERICAL INPUT MATRIX B
# ============================================================
def compute_B(state):
    state = np.asarray(state, dtype=np.float64)

    zero_force = np.zeros(3, dtype=np.float64)

    base = np.asarray(
        step(state, zero_force, dt=DT),
        dtype=np.float64
    )

    B = np.zeros((3, 3), dtype=np.float64)

    for j in range(3):
        force = np.zeros(3, dtype=np.float64)
        force[j] = 1.0

        perturbed = np.asarray(
            step(state, force, dt=DT),
            dtype=np.float64
        )

        B[:, j] = (
            perturbed[3:6] - base[3:6]
        ) / DT

    return B


# ============================================================
# EXPERT ACTION
# ============================================================
def expert_action(state, desired_state):

    state = np.asarray(state, dtype=np.float64)
    desired_state = np.asarray(desired_state, dtype=np.float64)

    B = compute_B(state)

    desired_acceleration = (
        KP * (desired_state[:3] - state[:3])
        - KD * state[3:6]
    )

    A = B.T @ B + RIDGE * np.eye(3)
    rhs = B.T @ desired_acceleration

    action = np.linalg.solve(A, rhs)

    action = np.clip(
        action,
        ACTION_LOW,
        ACTION_HIGH
    )

    return action


# ============================================================
# TEST 100 FULL EXPANDED-RANGE TASKS
# ============================================================
initial_errors = []
next_errors = []
force_norms = []

valid = 0
invalid = 0
improved = 0

print("=" * 70)
print("EXPANDED-RANGE EXPERT CHECK")
print("=" * 70)

for i in range(N_TASKS):

    # FULL expanded current state
    state = rng.uniform(
        STATE_LOW,
        STATE_HIGH
    ).astype(np.float64)

    # FULL expanded desired pose
    desired_state = np.zeros(6, dtype=np.float64)

    desired_state[:3] = rng.uniform(
        DESIRED_LOW,
        DESIRED_HIGH
    )

    try:

        action = expert_action(
            state,
            desired_state
        )

        next_state = np.asarray(
            step(
                state,
                action,
                dt=DT
            ),
            dtype=np.float64
        )

        # Safety checks
        if (
            not np.all(np.isfinite(action))
            or
            not np.all(np.isfinite(next_state))
            or
            np.max(np.abs(next_state)) > 5.0
        ):
            invalid += 1
            continue

        initial_error = np.linalg.norm(
            state[:3] - desired_state[:3]
        )

        next_error = np.linalg.norm(
            next_state[:3] - desired_state[:3]
        )

        initial_errors.append(initial_error)
        next_errors.append(next_error)

        force_norms.append(
            np.linalg.norm(action)
        )

        valid += 1

        if next_error < initial_error:
            improved += 1

    except Exception as e:
        invalid += 1


# ============================================================
# RESULTS
# ============================================================
print("\nRESULTS")
print("-" * 70)

print(f"Tasks tested              : {N_TASKS}")
print(f"Valid transitions         : {valid}")
print(f"Invalid transitions       : {invalid}")

if valid > 0:

    initial_errors = np.array(initial_errors)
    next_errors = np.array(next_errors)
    force_norms = np.array(force_norms)

    improvement = initial_errors - next_errors

    print(
        f"Expert improved           : "
        f"{improved}/{valid}"
    )

    print(
        f"One-step improvement     : "
        f"{100.0 * improved / valid:.2f}%"
    )

    print(
        f"Average initial error    : "
        f"{initial_errors.mean():.6f}"
    )

    print(
        f"Average next error       : "
        f"{next_errors.mean():.6f}"
    )

    print(
        f"Average improvement      : "
        f"{improvement.mean():.6f}"
    )

    print(
        f"Maximum force magnitude  : "
        f"{np.max(np.abs(force_norms)):.6f}"
    )

    print(
        f"Maximum next-state value : "
        f"{np.max(np.abs(np.concatenate(next_errors.reshape(-1,1)))):.6f}"
    )

print("=" * 70)

if invalid == 0:
    print("✅ ALL 100 EXPANDED TRANSITIONS ARE VALID")
else:
    print("⚠️ INVALID TRANSITIONS DETECTED")

if valid == N_TASKS and improved / valid >= 0.80:
    print("✅ EXPERT IS SUITABLE FOR EXPANDED DATA GENERATION")
else:
    print("❌ DO NOT START SAC TRAINING YET")

EXPANDED-RANGE EXPERT CHECK

RESULTS
----------------------------------------------------------------------
Tasks tested              : 100
Valid transitions         : 100
Invalid transitions       : 0
Expert improved           : 51/100
One-step improvement     : 51.00%
Average initial error    : 0.635496
Average next error       : 0.635459
Average improvement      : 0.000037
Maximum force magnitude  : 3.464102
Maximum next-state value : 1.249858
✅ ALL 100 EXPANDED TRANSITIONS ARE VALID
❌ DO NOT START SAC TRAINING YET


In [126]:
import numpy as np

# ============================================================
# FIXED EXPANDED RANGE — DO NOT CHANGE
# ============================================================
STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float64
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float64
)

DESIRED_LOW = np.array(
    [0.0, 0.0, -0.5],
    dtype=np.float64
)

DESIRED_HIGH = np.array(
    [1.0, 1.0, 0.5],
    dtype=np.float64
)

DT = 0.01
KP = np.array([2.0, 2.0, 2.0], dtype=np.float64)
KD = np.array([1.5, 1.5, 1.5], dtype=np.float64)
RIDGE = 1e-4

ACTION_LOW = -2.0
ACTION_HIGH = 2.0

rng = np.random.default_rng(12345)


def compute_B(state):
    state = np.asarray(state, dtype=np.float64)

    zero = np.zeros(3, dtype=np.float64)

    base = np.asarray(
        step(state, zero, dt=DT),
        dtype=np.float64
    )

    B = np.zeros((3, 3), dtype=np.float64)

    for j in range(3):
        force = np.zeros(3, dtype=np.float64)
        force[j] = 1.0

        perturbed = np.asarray(
            step(state, force, dt=DT),
            dtype=np.float64
        )

        B[:, j] = (
            perturbed[3:6] - base[3:6]
        ) / DT

    return B


def expert_action(state, desired):
    state = np.asarray(state, dtype=np.float64)
    desired = np.asarray(desired, dtype=np.float64)

    B = compute_B(state)

    desired_acc = (
        KP * (desired[:3] - state[:3])
        - KD * state[3:6]
    )

    A = B.T @ B + RIDGE * np.eye(3)
    rhs = B.T @ desired_acc

    action = np.linalg.solve(A, rhs)

    return np.clip(
        action,
        ACTION_LOW,
        ACTION_HIGH
    )


# ============================================================
# MULTI-STEP EXPANDED-RANGE TEST
# ============================================================
N_TASKS = 100
MAX_STEPS = 300

successes = 0
invalid = 0

initial_errors = []
final_errors = []

for task in range(N_TASKS):

    state = rng.uniform(
        STATE_LOW,
        STATE_HIGH
    ).astype(np.float64)

    desired = np.zeros(6, dtype=np.float64)
    desired[:3] = rng.uniform(
        DESIRED_LOW,
        DESIRED_HIGH
    )

    initial_error = np.linalg.norm(
        state[:3] - desired[:3]
    )

    best_error = initial_error

    try:
        for t in range(MAX_STEPS):

            action = expert_action(state, desired)

            next_state = np.asarray(
                step(state, action, dt=DT),
                dtype=np.float64
            )

            if (
                not np.all(np.isfinite(next_state))
                or np.max(np.abs(next_state)) > 5.0
            ):
                invalid += 1
                break

            state = next_state

            pose_error = np.linalg.norm(
                state[:3] - desired[:3]
            )

            best_error = min(best_error, pose_error)

        else:
            final_error = np.linalg.norm(
                state[:3] - desired[:3]
            )

            initial_errors.append(initial_error)
            final_errors.append(final_error)

            if final_error < initial_error:
                successes += 1

    except Exception:
        invalid += 1


print("=" * 65)
print("EXPANDED-RANGE MULTI-STEP EXPERT TEST")
print("=" * 65)

print(f"Tasks tested       : {N_TASKS}")
print(f"Valid tasks        : {N_TASKS - invalid}")
print(f"Invalid tasks      : {invalid}")
print(f"Improved           : {successes}/{N_TASKS}")
print(f"Improvement rate   : {successes/N_TASKS*100:.2f}%")

if final_errors:
    print(f"Avg initial error  : {np.mean(initial_errors):.6f}")
    print(f"Avg final error    : {np.mean(final_errors):.6f}")

print("=" * 65)

EXPANDED-RANGE MULTI-STEP EXPERT TEST
Tasks tested       : 100
Valid tasks        : 94
Invalid tasks      : 6
Improved           : 92/100
Improvement rate   : 92.00%
Avg initial error  : 0.622415
Avg final error    : 0.130260


In [127]:
import os
import numpy as np

# ============================================================
# FIXED EXPANDED RANGE — DO NOT CHANGE
# ============================================================
STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float64
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float64
)

DESIRED_LOW = np.array(
    [0.0, 0.0, -0.5],
    dtype=np.float64
)

DESIRED_HIGH = np.array(
    [1.0, 1.0, 0.5],
    dtype=np.float64
)

DT = 0.01
MAX_STEPS = 300

KP = np.array([2.0, 2.0, 2.0], dtype=np.float64)
KD = np.array([1.5, 1.5, 1.5], dtype=np.float64)
RIDGE = 1e-4

ACTION_LOW = -2.0
ACTION_HIGH = 2.0

# ============================================================
# DATASET SIZE
# ============================================================
TARGET_TRANSITIONS = 30000
SEED = 20260914

rng = np.random.default_rng(SEED)

states = []
desired_states = []
actions = []
next_states = []
rewards = []
dones = []

invalid_episodes = 0
episodes = 0


# ============================================================
# NUMERICAL DYNAMICS MATRIX
# ============================================================
def compute_B(state):

    state = np.asarray(state, dtype=np.float64)

    zero = np.zeros(3, dtype=np.float64)

    base = np.asarray(
        step(state, zero, dt=DT),
        dtype=np.float64
    )

    B = np.zeros((3, 3), dtype=np.float64)

    for j in range(3):

        force = np.zeros(3, dtype=np.float64)
        force[j] = 1.0

        perturbed = np.asarray(
            step(state, force, dt=DT),
            dtype=np.float64
        )

        B[:, j] = (
            perturbed[3:6] - base[3:6]
        ) / DT

    return B


# ============================================================
# EXPERT POLICY
# ============================================================
def expert_action(state, desired):

    state = np.asarray(state, dtype=np.float64)
    desired = np.asarray(desired, dtype=np.float64)

    B = compute_B(state)

    desired_acc = (
        KP * (desired[:3] - state[:3])
        - KD * state[3:6]
    )

    A = B.T @ B + RIDGE * np.eye(3)
    rhs = B.T @ desired_acc

    action = np.linalg.solve(A, rhs)

    return np.clip(
        action,
        ACTION_LOW,
        ACTION_HIGH
    )


# ============================================================
# GENERATE VALID TRAJECTORIES
# ============================================================
print("=" * 70)
print("GENERATING EXPANDED-RANGE SAC DEMONSTRATIONS")
print("=" * 70)

while len(states) < TARGET_TRANSITIONS:

    episodes += 1

    state = rng.uniform(
        STATE_LOW,
        STATE_HIGH
    ).astype(np.float64)

    desired = np.zeros(6, dtype=np.float64)

    desired[:3] = rng.uniform(
        DESIRED_LOW,
        DESIRED_HIGH
    )

    episode_states = []
    episode_desired = []
    episode_actions = []
    episode_next = []
    episode_rewards = []

    valid_episode = True

    for t in range(MAX_STEPS):

        try:
            action = expert_action(
                state,
                desired
            )

            ns = np.asarray(
                step(
                    state,
                    action,
                    dt=DT
                ),
                dtype=np.float64
            )

            if (
                not np.all(np.isfinite(ns))
                or
                not np.all(np.isfinite(action))
                or
                np.max(np.abs(ns)) > 5.0
            ):
                valid_episode = False
                break

            # =================================================
            # EXACT PROJECT REWARD
            # =================================================
            reward = (
                -np.sum(
                    (ns - desired) ** 2
                )
                -
                0.01 * np.sum(
                    action ** 2
                )
            )

            episode_states.append(state.copy())
            episode_desired.append(desired.copy())
            episode_actions.append(action.copy())
            episode_next.append(ns.copy())
            episode_rewards.append(float(reward))

            state = ns

        except Exception:
            valid_episode = False
            break

    # ========================================================
    # ONLY ADD COMPLETE VALID EPISODES
    # ========================================================
    if valid_episode:

        for i in range(len(episode_states)):

            states.append(episode_states[i])
            desired_states.append(episode_desired[i])
            actions.append(episode_actions[i])
            next_states.append(episode_next[i])
            rewards.append(episode_rewards[i])

            # Episode terminal flag
            dones.append(
                1.0 if i == len(episode_states) - 1
                else 0.0
            )

            if len(states) >= TARGET_TRANSITIONS:
                break

    else:
        invalid_episodes += 1

    if episodes % 10 == 0:

        print(
            f"Episodes={episodes:4d} | "
            f"valid transitions={len(states):6d} | "
            f"invalid episodes={invalid_episodes}"
        )


# ============================================================
# CONVERT TO ARRAYS
# ============================================================
states = np.asarray(
    states[:TARGET_TRANSITIONS],
    dtype=np.float32
)

desired_states = np.asarray(
    desired_states[:TARGET_TRANSITIONS],
    dtype=np.float32
)

actions = np.asarray(
    actions[:TARGET_TRANSITIONS],
    dtype=np.float32
)

next_states = np.asarray(
    next_states[:TARGET_TRANSITIONS],
    dtype=np.float32
)

rewards = np.asarray(
    rewards[:TARGET_TRANSITIONS],
    dtype=np.float32
)

dones = np.asarray(
    dones[:TARGET_TRANSITIONS],
    dtype=np.float32
)


# ============================================================
# SAVE
# ============================================================
save_dir = os.path.abspath(
    os.path.join(
        os.getcwd(),
        "..",
        "training",
        "final_model"
    )
)

os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "sac_expanded_expert_demonstrations.npz"
)

np.savez_compressed(
    save_path,
    states=states,
    desired_states=desired_states,
    actions=actions,
    next_states=next_states,
    rewards=rewards,
    dones=dones
)


# ============================================================
# VERIFY DATASET
# ============================================================
print("\n" + "=" * 70)
print("DATASET VERIFICATION")
print("=" * 70)

print("Saved:", save_path)
print("States shape:", states.shape)
print("Desired shape:", desired_states.shape)
print("Actions shape:", actions.shape)

print("\nSTATE MIN:")
print(states.min(axis=0))

print("\nSTATE MAX:")
print(states.max(axis=0))

print("\nDESIRED MIN:")
print(desired_states.min(axis=0))

print("\nDESIRED MAX:")
print(desired_states.max(axis=0))

print("\nACTION MIN:")
print(actions.min(axis=0))

print("\nACTION MAX:")
print(actions.max(axis=0))

print("\nReward min/max:")
print(rewards.min(), rewards.max())

print("\nTerminal transitions:")
print(int(dones.sum()))

# Hard verification
state_range_ok = (
    np.all(states >= STATE_LOW.astype(np.float32) - 1e-5)
    and
    np.all(states <= STATE_HIGH.astype(np.float32) + 1e-5)
)

desired_range_ok = (
    np.all(desired_states[:, :3] >= DESIRED_LOW.astype(np.float32) - 1e-5)
    and
    np.all(desired_states[:, :3] <= DESIRED_HIGH.astype(np.float32) + 1e-5)
)

action_range_ok = (
    np.all(actions >= -2.0 - 1e-5)
    and
    np.all(actions <= 2.0 + 1e-5)
)

print("\n" + "=" * 70)

if state_range_ok:
    print("✅ CURRENT STATES: EXPANDED RANGE VERIFIED")
else:
    print("❌ CURRENT STATES: RANGE FAILURE")

if desired_range_ok:
    print("✅ DESIRED STATES: EXPANDED RANGE VERIFIED")
else:
    print("❌ DESIRED STATES: RANGE FAILURE")

if action_range_ok:
    print("✅ ACTIONS: [-2, 2] VERIFIED")
else:
    print("❌ ACTIONS: RANGE FAILURE")

print("=" * 70)

GENERATING EXPANDED-RANGE SAC DEMONSTRATIONS
Episodes=  10 | valid transitions=  3000 | invalid episodes=0
Episodes=  20 | valid transitions=  6000 | invalid episodes=0
Episodes=  30 | valid transitions=  9000 | invalid episodes=0
Episodes=  40 | valid transitions= 11700 | invalid episodes=1
Episodes=  50 | valid transitions= 14400 | invalid episodes=2
Episodes=  60 | valid transitions= 17400 | invalid episodes=2
Episodes=  70 | valid transitions= 20100 | invalid episodes=3
Episodes=  80 | valid transitions= 22500 | invalid episodes=5
Episodes=  90 | valid transitions= 25500 | invalid episodes=5
Episodes= 100 | valid transitions= 27900 | invalid episodes=7

DATASET VERIFICATION
Saved: c:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_expanded_expert_demonstrations.npz
States shape: (30000, 6)
Desired shape: (30000, 6)
Actions shape: (30000, 3)

STATE MIN:
[-0.11304717 -0.32181337 -1.6406156  -0.6119444  -0.7026563  -1.8708802 ]

STATE MAX:
[1.1501291  1.3674233  2.3698

In [128]:
import os
import numpy as np

# ============================================================
# FIXED EXPANDED RANGE — DO NOT CHANGE
# ============================================================
STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float64
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float64
)

DESIRED_LOW = np.array(
    [0.0, 0.0, -0.5],
    dtype=np.float64
)

DESIRED_HIGH = np.array(
    [1.0, 1.0, 0.5],
    dtype=np.float64
)

DT = 0.01
MAX_STEPS = 300
TARGET = 30000

KP = np.array([2.0, 2.0, 2.0], dtype=np.float64)
KD = np.array([1.5, 1.5, 1.5], dtype=np.float64)
RIDGE = 1e-4

rng = np.random.default_rng(20260914)

states = []
desired_states = []
actions = []
next_states = []
rewards = []
dones = []

episodes = 0
discarded = 0


# ============================================================
# NUMERICAL B MATRIX
# ============================================================
def compute_B(state):

    state = np.asarray(state, dtype=np.float64)

    zero = np.zeros(3, dtype=np.float64)

    base = np.asarray(
        step(state, zero, dt=DT),
        dtype=np.float64
    )

    B = np.zeros((3, 3), dtype=np.float64)

    for j in range(3):

        force = np.zeros(3, dtype=np.float64)
        force[j] = 1.0

        perturbed = np.asarray(
            step(state, force, dt=DT),
            dtype=np.float64
        )

        B[:, j] = (
            perturbed[3:6] - base[3:6]
        ) / DT

    return B


# ============================================================
# EXPERT
# ============================================================
def expert_action(state, desired):

    B = compute_B(state)

    desired_acc = (
        KP * (desired[:3] - state[:3])
        - KD * state[3:6]
    )

    A = B.T @ B + RIDGE * np.eye(3)
    rhs = B.T @ desired_acc

    action = np.linalg.solve(A, rhs)

    return np.clip(action, -2.0, 2.0)


# ============================================================
# RANGE CHECK
# ============================================================
def inside_range(state):

    state = np.asarray(state)

    return (
        np.all(state >= STATE_LOW - 1e-7)
        and
        np.all(state <= STATE_HIGH + 1e-7)
    )


# ============================================================
# GENERATION
# ============================================================
print("=" * 70)
print("STRICT EXPANDED-RANGE DATASET GENERATION")
print("=" * 70)

while len(states) < TARGET:

    episodes += 1

    state = rng.uniform(
        STATE_LOW,
        STATE_HIGH
    ).astype(np.float64)

    desired = np.zeros(6, dtype=np.float64)

    desired[:3] = rng.uniform(
        DESIRED_LOW,
        DESIRED_HIGH
    )

    episode_data = []

    for t in range(MAX_STEPS):

        try:

            action = expert_action(
                state,
                desired
            )

            ns = np.asarray(
                step(
                    state,
                    action,
                    dt=DT
                ),
                dtype=np.float64
            )

            # ------------------------------------------------
            # DO NOT STORE OUT-OF-RANGE STATES
            # ------------------------------------------------
            if not inside_range(ns):
                discarded += 1
                break

            if (
                not np.all(np.isfinite(ns))
                or
                not np.all(np.isfinite(action))
            ):
                discarded += 1
                break

            reward = (
                -np.sum((ns - desired) ** 2)
                -0.01 * np.sum(action ** 2)
            )

            episode_data.append(
                (
                    state.copy(),
                    desired.copy(),
                    action.copy(),
                    ns.copy(),
                    float(reward)
                )
            )

            state = ns

        except Exception:
            discarded += 1
            break

    # --------------------------------------------------------
    # Store valid episode
    # --------------------------------------------------------
    for k, item in enumerate(episode_data):

        s, d, a, ns, r = item

        states.append(s)
        desired_states.append(d)
        actions.append(a)
        next_states.append(ns)
        rewards.append(r)

        dones.append(
            1.0 if k == len(episode_data) - 1 else 0.0
        )

        if len(states) >= TARGET:
            break

    if episodes % 10 == 0:
        print(
            f"Episodes={episodes:4d} | "
            f"stored={len(states):6d} | "
            f"discarded={discarded}"
        )


# ============================================================
# ARRAYS
# ============================================================
states = np.asarray(
    states[:TARGET],
    dtype=np.float32
)

desired_states = np.asarray(
    desired_states[:TARGET],
    dtype=np.float32
)

actions = np.asarray(
    actions[:TARGET],
    dtype=np.float32
)

next_states = np.asarray(
    next_states[:TARGET],
    dtype=np.float32
)

rewards = np.asarray(
    rewards[:TARGET],
    dtype=np.float32
)

dones = np.asarray(
    dones[:TARGET],
    dtype=np.float32
)


# ============================================================
# SAVE NEW DATASET
# ============================================================
save_path = os.path.abspath(
    os.path.join(
        os.getcwd(),
        "..",
        "training",
        "final_model",
        "sac_expanded_expert_demonstrations.npz"
    )
)

os.makedirs(
    os.path.dirname(save_path),
    exist_ok=True
)

np.savez_compressed(
    save_path,
    states=states,
    desired_states=desired_states,
    actions=actions,
    next_states=next_states,
    rewards=rewards,
    dones=dones
)


# ============================================================
# FINAL VERIFICATION
# ============================================================
print("\n" + "=" * 70)
print("FINAL DATASET VERIFICATION")
print("=" * 70)

print("Saved:", save_path)
print("Shape:", states.shape)

print("\nSTATE MIN:")
print(states.min(axis=0))

print("\nSTATE MAX:")
print(states.max(axis=0))

print("\nNEXT STATE MIN:")
print(next_states.min(axis=0))

print("\nNEXT STATE MAX:")
print(next_states.max(axis=0))

print("\nDESIRED MIN:")
print(desired_states.min(axis=0))

print("\nDESIRED MAX:")
print(desired_states.max(axis=0))

print("\nACTION MIN:")
print(actions.min(axis=0))

print("\nACTION MAX:")
print(actions.max(axis=0))

state_ok = (
    np.all(states >= STATE_LOW.astype(np.float32) - 1e-5)
    and
    np.all(states <= STATE_HIGH.astype(np.float32) + 1e-5)
)

next_ok = (
    np.all(next_states >= STATE_LOW.astype(np.float32) - 1e-5)
    and
    np.all(next_states <= STATE_HIGH.astype(np.float32) + 1e-5)
)

desired_ok = (
    np.all(desired_states[:, :3] >= DESIRED_LOW.astype(np.float32) - 1e-5)
    and
    np.all(desired_states[:, :3] <= DESIRED_HIGH.astype(np.float32) + 1e-5)
)

action_ok = (
    np.all(actions >= -2.0 - 1e-5)
    and
    np.all(actions <= 2.0 + 1e-5)
)

print("\n" + "=" * 70)
print("CURRENT STATE RANGE :", state_ok)
print("NEXT STATE RANGE    :", next_ok)
print("DESIRED RANGE       :", desired_ok)
print("ACTION RANGE        :", action_ok)
print("=" * 70)

if state_ok and next_ok and desired_ok and action_ok:
    print("✅ DATASET PASSED ALL RANGE CHECKS")
else:
    print("❌ DATASET FAILED RANGE CHECK — DO NOT TRAIN")

STRICT EXPANDED-RANGE DATASET GENERATION
Episodes=  10 | stored=   426 | discarded=9
Episodes=  20 | stored=   869 | discarded=18
Episodes=  30 | stored=  1018 | discarded=28
Episodes=  40 | stored=  1269 | discarded=38
Episodes=  50 | stored=  1440 | discarded=48
Episodes=  60 | stored=  1514 | discarded=58
Episodes=  70 | stored=  1909 | discarded=67
Episodes=  80 | stored=  2053 | discarded=77
Episodes=  90 | stored=  2568 | discarded=86
Episodes= 100 | stored=  2674 | discarded=96
Episodes= 110 | stored=  2804 | discarded=106
Episodes= 120 | stored=  3253 | discarded=115
Episodes= 130 | stored=  3715 | discarded=124
Episodes= 140 | stored=  3956 | discarded=134
Episodes= 150 | stored=  4443 | discarded=143
Episodes= 160 | stored=  4758 | discarded=153
Episodes= 170 | stored=  4973 | discarded=163
Episodes= 180 | stored=  5707 | discarded=171
Episodes= 190 | stored=  6360 | discarded=180
Episodes= 200 | stored=  6882 | discarded=189
Episodes= 210 | stored=  7099 | discarded=199
Epis

In [131]:
import os
import re

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
SCRIPT_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "train_expanded_sac.py"
)

with open(SCRIPT_PATH, "r", encoding="utf-8") as f:
    text = f.read()

# ------------------------------------------------------------
# Actor API fixes
# Existing Actor.forward() expects:
#     actor.forward(state, desired_state)
#
# Existing Actor.sample() is also expected to use:
#     actor.sample(state, desired_state)
# ------------------------------------------------------------

text = text.replace(
    "mean, _ = actor.forward(inp)",
    "mean, _ = actor.forward(bs, bd)"
)

text = text.replace(
    "next_action, next_log_prob = actor.sample(\n            next_inp\n        )",
    "next_action, next_log_prob = actor.sample(\n            bns, bd\n        )"
)

text = text.replace(
    "new_action, log_prob = actor.sample(\n        inp\n    )",
    "new_action, log_prob = actor.sample(\n        bs, bd\n    )"
)

# There is another actor.forward() for BC loss.
text = text.replace(
    "mean, _ = actor.forward(inp)\n\n    bc_action",
    "mean, _ = actor.forward(bs, bd)\n\n    bc_action"
)

with open(SCRIPT_PATH, "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Training script patched.")
print("Actor calls now use state + desired_state.")
print("84% checkpoint untouched.")

✅ Training script patched.
Actor calls now use state + desired_state.
84% checkpoint untouched.


In [133]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
SCRIPT_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "train_expanded_sac.py"
)

result = subprocess.run(
    [sys.executable, SCRIPT_PATH],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

print("\n" + "=" * 80)
print("ACTUAL ERROR")
print("=" * 80)
print(result.stderr)

print("\nRETURN CODE:", result.returncode)

EXPANDED-RANGE SAC TRAINING
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

Dataset:
states: (30000, 6)
desired: (30000, 6)
actions: (30000, 3)
âœ… DATASET RANGE CHECK PASSED

Training transitions: 30000

STAGE 1 â€” SAC ACTOR BEHAVIOR PRETRAINING


ACTUAL ERROR
Traceback (most recent call last):
  File "C:\Users\madha\OneDrive\Desktop\project_1\training\train_expanded_sac.py", line 269, in <module>
    mean, _ = actor.forward(bs, bd)
              ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\madha\OneDrive\Desktop\project_1\rl\actor.py", line 27, in forward
    action = self.network(x)
             ^^^^^^^^^^^^^^^
  File "c:\Users\madha\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py", line 1783, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\madha\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py", line 1794, in _call_impl
    return 

In [134]:
import os

ACTOR_PATH = r"C:\Users\madha\OneDrive\Desktop\project_1\rl\actor.py"

with open(ACTOR_PATH, "r", encoding="utf-8") as f:
    actor_code = f.read()

print("=" * 80)
print("ACTOR.PY — ACTUAL CONTENT")
print("=" * 80)
print(actor_code)
print("=" * 80)

ACTOR.PY — ACTUAL CONTENT
import torch
import torch.nn as nn


class Actor(nn.Module):

    def __init__(self, state_dim=6, action_dim=3, max_action=0.2):
        super().__init__()

        self.max_action = max_action

        self.network = nn.Sequential(
            nn.Linear(state_dim * 2, 128),
            nn.ReLU(),

            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, action_dim),
            nn.Tanh()
        )

    def forward(self, state, desired_state):

        x = torch.cat([state, desired_state], dim=1)

        action = self.network(x)

        # Scale Actor output from [-1, 1] to [-0.2, 0.2]
        action = self.max_action * action

        return action


In [135]:
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
SCRIPT_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "train_expanded_sac.py"
)

with open(SCRIPT_PATH, "r", encoding="utf-8") as f:
    text = f.read()

# Fix the Actor instantiation.
# actor.py internally creates Linear(state_dim * 2, 128),
# so state_dim MUST be 6 for:
# current state (6) + desired state (6) = 12 inputs.

text = text.replace(
    "actor = Actor(\n    state_dim=12,\n    action_dim=3,\n    max_action=MAX_ACTION\n)",
    "actor = Actor(\n    state_dim=6,\n    action_dim=3,\n    max_action=MAX_ACTION\n)"
)

with open(SCRIPT_PATH, "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Actor dimension fixed.")
print("Actor state_dim = 6")
print("Actual network input = 6 × 2 = 12")
print("Required input = current 6 + desired 6 = 12")
print("actor.py was NOT modified.")

✅ Actor dimension fixed.
Actor state_dim = 6
Actual network input = 6 × 2 = 12
Required input = current 6 + desired 6 = 12
actor.py was NOT modified.


In [136]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

test_code = r'''
import sys
import os
import numpy as np
import torch

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "rl"))

from rl.actor import Actor

device = torch.device("cuda")

actor = Actor(
    state_dim=6,
    action_dim=3,
    max_action=2.0
).to(device)

state = torch.tensor(
    [[0.45, 0.55, 0.05, 0.02, -0.01, 0.03]],
    dtype=torch.float32,
    device=device
)

desired = torch.tensor(
    [[0.80, 0.70, 0.00, 0.00, 0.00, 0.00]],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    output = actor(state, desired)

print("Actor output shape:", output.shape)
print("Actor output:", output.cpu().numpy())

assert output.shape == (1, 3)
assert torch.all(output <= 2.0 + 1e-6)
assert torch.all(output >= -2.0 - 1e-6)

print("✅ ACTOR 12→3 TEST PASSED")
print("✅ RANGE UNCHANGED")
'''

result = subprocess.run(
    [sys.executable, "-c", test_code],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Actor test failed")

print("No training performed.")

Actor output shape: torch.Size([1, 3])
Actor output: [[-0.0260804  0.1310068 -0.2707524]]
âœ… ACTOR 12â†’3 TEST PASSED
âœ… RANGE UNCHANGED

No training performed.


In [139]:
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
SCRIPT_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "train_expanded_sac.py"
)

script = r'''
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

# ============================================================
# FIXED PROJECT SETTINGS — DO NOT CHANGE
# ============================================================

STATE_DIM = 6
ACTION_DIM = 3
MAX_ACTION = 2.0

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

GAMMA = 0.99
TAU = 0.005
REPLAY_SIZE = 100000
BATCH_SIZE = 256

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
DATASET_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "final_model",
    "sac_expanded_expert_demonstrations.npz"
)

print("=" * 70)
print("EXPANDED-RANGE SAC ARCHITECTURE TEST")
print("=" * 70)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# NORMALIZATION
# ============================================================

def normalize_state(x):
    low = torch.tensor(STATE_LOW, dtype=torch.float32, device=x.device)
    high = torch.tensor(STATE_HIGH, dtype=torch.float32, device=x.device)
    return 2.0 * (x - low) / (high - low) - 1.0


# ============================================================
# STOCHASTIC SAC ACTOR
# ============================================================

class SACActor(nn.Module):

    def __init__(
        self,
        state_dim=6,
        action_dim=3,
        max_action=2.0
    ):
        super().__init__()

        self.max_action = max_action

        self.net = nn.Sequential(
            nn.Linear(state_dim * 2, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )

        self.mean = nn.Linear(256, action_dim)
        self.log_std = nn.Linear(256, action_dim)

    def forward(self, state, desired_state):

        state = normalize_state(state)
        desired_state = normalize_state(desired_state)

        x = torch.cat(
            [state, desired_state],
            dim=1
        )

        h = self.net(x)

        mean = self.mean(h)

        log_std = self.log_std(h)
        log_std = torch.clamp(
            log_std,
            -5.0,
            1.0
        )

        return mean, log_std

    def sample(self, state, desired_state):

        mean, log_std = self.forward(
            state,
            desired_state
        )

        std = log_std.exp()

        normal = Normal(mean, std)

        z = normal.rsample()

        y = torch.tanh(z)

        action = self.max_action * y

        log_prob = normal.log_prob(z)

        log_prob -= torch.log(
            1.0 - y.pow(2) + 1e-6
        )

        log_prob -= np.log(self.max_action)

        log_prob = log_prob.sum(
            dim=1,
            keepdim=True
        )

        deterministic_action = (
            self.max_action *
            torch.tanh(mean)
        )

        return (
            action,
            log_prob,
            deterministic_action
        )


# ============================================================
# SAC CRITIC
# ============================================================

class SACCritic(nn.Module):

    def __init__(
        self,
        state_dim=6,
        action_dim=3
    ):
        super().__init__()

        input_dim = state_dim * 2 + action_dim

        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(
        self,
        state,
        desired_state,
        action
    ):

        state = normalize_state(state)
        desired_state = normalize_state(desired_state)

        x = torch.cat(
            [state, desired_state, action],
            dim=1
        )

        return self.net(x)


# ============================================================
# LOAD DATASET
# ============================================================

data = np.load(DATASET_PATH)

states = data["states"].astype(np.float32)
desired = data["desired_states"].astype(np.float32)
actions = data["actions"].astype(np.float32)
next_states = data["next_states"].astype(np.float32)
rewards = data["rewards"].astype(np.float32)

print("\nDataset:")
print("states:", states.shape)
print("desired:", desired.shape)
print("actions:", actions.shape)
print("next_states:", next_states.shape)

# EXACT RANGE CHECK
assert np.all(states >= STATE_LOW - 1e-6)
assert np.all(states <= STATE_HIGH + 1e-6)

assert np.all(next_states >= STATE_LOW - 1e-6)
assert np.all(next_states <= STATE_HIGH + 1e-6)

assert np.all(desired >= STATE_LOW - 1e-6)
assert np.all(desired <= STATE_HIGH + 1e-6)

assert np.all(actions >= -2.0 - 1e-6)
assert np.all(actions <= 2.0 + 1e-6)

print("DATASET RANGE CHECK PASSED")

# ============================================================
# CREATE SAC NETWORKS
# ============================================================

actor = SACActor(
    state_dim=6,
    action_dim=3,
    max_action=2.0
).to(DEVICE)

critic1 = SACCritic(
    state_dim=6,
    action_dim=3
).to(DEVICE)

critic2 = SACCritic(
    state_dim=6,
    action_dim=3
).to(DEVICE)

print("\nNetworks created successfully.")

# ============================================================
# FORWARD TEST
# ============================================================

bs = torch.tensor(
    states[:256],
    dtype=torch.float32,
    device=DEVICE
)

bd = torch.tensor(
    desired[:256],
    dtype=torch.float32,
    device=DEVICE
)

ba = torch.tensor(
    actions[:256],
    dtype=torch.float32,
    device=DEVICE
)

with torch.no_grad():

    mean, log_std = actor(
        bs,
        bd
    )

    sampled_action, log_prob, deterministic = actor.sample(
        bs,
        bd
    )

    q1 = critic1(
        bs,
        bd,
        ba
    )

    q2 = critic2(
        bs,
        bd,
        ba
    )

print("\nActor mean shape:", mean.shape)
print("Actor log_std shape:", log_std.shape)
print("Sampled action shape:", sampled_action.shape)
print("Log probability shape:", log_prob.shape)
print("Critic 1 shape:", q1.shape)
print("Critic 2 shape:", q2.shape)

assert mean.shape == (256, 3)
assert log_std.shape == (256, 3)
assert sampled_action.shape == (256, 3)
assert log_prob.shape == (256, 1)
assert q1.shape == (256, 1)
assert q2.shape == (256, 1)

assert torch.isfinite(mean).all()
assert torch.isfinite(log_std).all()
assert torch.isfinite(sampled_action).all()
assert torch.isfinite(log_prob).all()
assert torch.isfinite(q1).all()
assert torch.isfinite(q2).all()

assert torch.all(sampled_action <= 2.0 + 1e-5)
assert torch.all(sampled_action >= -2.0 - 1e-5)

print("\n" + "=" * 70)
print("SAC ARCHITECTURE TEST PASSED")
print("=" * 70)
print("12 input features -> stochastic Gaussian actor -> 3 actions")
print("Twin critics: PASSED")
print("Action range [-2, 2]: PASSED")
print("Fixed state range: PASSED")
print("No dynamics.py modification")
print("No actor.py modification")
'''

with open(SCRIPT_PATH, "w", encoding="utf-8") as f:
    f.write(script)

print("Updated train_expanded_sac.py")
print("actor.py: NOT MODIFIED")
print("dynamics.py: NOT MODIFIED")
print("STATE RANGE: NOT MODIFIED")

Updated train_expanded_sac.py
actor.py: NOT MODIFIED
dynamics.py: NOT MODIFIED
STATE RANGE: NOT MODIFIED


In [140]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
SCRIPT_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "train_expanded_sac.py"
)

result = subprocess.run(
    [sys.executable, SCRIPT_PATH],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n" + "=" * 80)
    print("ERROR")
    print("=" * 80)
    print(result.stderr)

print("\nRETURN CODE:", result.returncode)

EXPANDED-RANGE SAC ARCHITECTURE TEST
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

Dataset:
states: (30000, 6)
desired: (30000, 6)
actions: (30000, 3)
next_states: (30000, 6)
DATASET RANGE CHECK PASSED

Networks created successfully.

Actor mean shape: torch.Size([256, 3])
Actor log_std shape: torch.Size([256, 3])
Sampled action shape: torch.Size([256, 3])
Log probability shape: torch.Size([256, 1])
Critic 1 shape: torch.Size([256, 1])
Critic 2 shape: torch.Size([256, 1])

SAC ARCHITECTURE TEST PASSED
12 input features -> stochastic Gaussian actor -> 3 actions
Twin critics: PASSED
Action range [-2, 2]: PASSED
Fixed state range: PASSED
No dynamics.py modification
No actor.py modification


RETURN CODE: 0


In [141]:
import os
import subprocess
import sys

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
SCRIPT_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "train_expanded_sac.py"
)

training_code = r'''
import os
import sys
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

# ============================================================
# FIXED SETTINGS
# ============================================================

STATE_DIM = 6
ACTION_DIM = 3
MAX_ACTION = 2.0

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

GAMMA = 0.99
TAU = 0.005
REPLAY_SIZE = 100000
BATCH_SIZE = 256

ACTOR_LR = 3e-5
CRITIC_LR = 1e-4
ALPHA_LR = 1e-5

TARGET_ENTROPY = -3.0
INITIAL_ALPHA = 0.0001

BC_WEIGHT = 2.0

EPOCHS = 80

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

DATASET_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "final_model",
    "sac_expanded_expert_demonstrations.npz"
)

SAVE_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "final_model",
    "sac_expanded_full_final.pt"
)

# ============================================================
# SEED
# ============================================================

np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ============================================================
# NORMALIZATION
# ============================================================

def normalize_state(x):

    low = torch.tensor(
        STATE_LOW,
        dtype=torch.float32,
        device=x.device
    )

    high = torch.tensor(
        STATE_HIGH,
        dtype=torch.float32,
        device=x.device
    )

    return 2.0 * (x - low) / (high - low) - 1.0


# ============================================================
# STOCHASTIC SAC ACTOR
# ============================================================

class SACActor(nn.Module):

    def __init__(
        self,
        state_dim=6,
        action_dim=3,
        max_action=2.0
    ):
        super().__init__()

        self.max_action = max_action

        self.net = nn.Sequential(
            nn.Linear(state_dim * 2, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )

        self.mean = nn.Linear(
            256,
            action_dim
        )

        self.log_std = nn.Linear(
            256,
            action_dim
        )

    def forward(
        self,
        state,
        desired_state
    ):

        state = normalize_state(state)
        desired_state = normalize_state(desired_state)

        x = torch.cat(
            [state, desired_state],
            dim=1
        )

        h = self.net(x)

        mean = self.mean(h)

        log_std = self.log_std(h)

        log_std = torch.clamp(
            log_std,
            -5.0,
            1.0
        )

        return mean, log_std

    def sample(
        self,
        state,
        desired_state
    ):

        mean, log_std = self.forward(
            state,
            desired_state
        )

        std = log_std.exp()

        normal = Normal(
            mean,
            std
        )

        z = normal.rsample()

        y = torch.tanh(z)

        action = self.max_action * y

        log_prob = normal.log_prob(z)

        log_prob -= torch.log(
            1.0 - y.pow(2) + 1e-6
        )

        log_prob -= np.log(
            self.max_action
        )

        log_prob = log_prob.sum(
            dim=1,
            keepdim=True
        )

        deterministic = (
            self.max_action *
            torch.tanh(mean)
        )

        return (
            action,
            log_prob,
            deterministic
        )


# ============================================================
# TWIN CRITICS
# ============================================================

class Critic(nn.Module):

    def __init__(
        self,
        state_dim=6,
        action_dim=3
    ):
        super().__init__()

        input_dim = (
            state_dim * 2 +
            action_dim
        )

        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(
        self,
        state,
        desired_state,
        action
    ):

        state = normalize_state(state)
        desired_state = normalize_state(desired_state)

        x = torch.cat(
            [
                state,
                desired_state,
                action
            ],
            dim=1
        )

        return self.net(x)


# ============================================================
# DATASET
# ============================================================

print("=" * 70)
print("EXPANDED-RANGE STOCHASTIC SAC TRAINING")
print("=" * 70)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

data = np.load(
    DATASET_PATH
)

states = data["states"].astype(
    np.float32
)

desired = data["desired_states"].astype(
    np.float32
)

actions = data["actions"].astype(
    np.float32
)

next_states = data["next_states"].astype(
    np.float32
)

rewards = data["rewards"].astype(
    np.float32
)

if "dones" in data:
    dones = data["dones"].astype(
        np.float32
    )
else:
    dones = np.zeros(
        len(states),
        dtype=np.float32
    )

print("\nDataset:")
print("states:", states.shape)
print("desired:", desired.shape)
print("actions:", actions.shape)
print("next_states:", next_states.shape)

# ============================================================
# EXACT RANGE VALIDATION
# ============================================================

assert np.all(
    states >= STATE_LOW - 1e-6
)

assert np.all(
    states <= STATE_HIGH + 1e-6
)

assert np.all(
    next_states >= STATE_LOW - 1e-6
)

assert np.all(
    next_states <= STATE_HIGH + 1e-6
)

assert np.all(
    desired >= STATE_LOW - 1e-6
)

assert np.all(
    desired <= STATE_HIGH + 1e-6
)

assert np.all(
    actions >= -2.0 - 1e-6
)

assert np.all(
    actions <= 2.0 + 1e-6
)

print("DATASET RANGE CHECK PASSED")

# ============================================================
# TENSORS
# ============================================================

states_t = torch.tensor(
    states,
    dtype=torch.float32,
    device=DEVICE
)

desired_t = torch.tensor(
    desired,
    dtype=torch.float32,
    device=DEVICE
)

actions_t = torch.tensor(
    actions,
    dtype=torch.float32,
    device=DEVICE
)

next_states_t = torch.tensor(
    next_states,
    dtype=torch.float32,
    device=DEVICE
)

rewards_t = torch.tensor(
    rewards,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(1)

dones_t = torch.tensor(
    dones,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(1)

N = len(states)

# ============================================================
# NETWORKS
# ============================================================

actor = SACActor(
    STATE_DIM,
    ACTION_DIM,
    MAX_ACTION
).to(DEVICE)

critic1 = Critic(
    STATE_DIM,
    ACTION_DIM
).to(DEVICE)

critic2 = Critic(
    STATE_DIM,
    ACTION_DIM
).to(DEVICE)

target_critic1 = copy.deepcopy(
    critic1
)

target_critic2 = copy.deepcopy(
    critic2
)

actor_optimizer = torch.optim.Adam(
    actor.parameters(),
    lr=ACTOR_LR
)

critic1_optimizer = torch.optim.Adam(
    critic1.parameters(),
    lr=CRITIC_LR
)

critic2_optimizer = torch.optim.Adam(
    critic2.parameters(),
    lr=CRITIC_LR
)

log_alpha = torch.tensor(
    np.log(INITIAL_ALPHA),
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

alpha_optimizer = torch.optim.Adam(
    [log_alpha],
    lr=ALPHA_LR
)

# ============================================================
# TRAIN
# ============================================================

print("\nStarting SAC training...")
print("Transitions:", N)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)

best_loss = float("inf")

for epoch in range(1, EPOCHS + 1):

    indices = np.random.permutation(N)

    actor_losses = []
    critic_losses = []
    alpha_losses = []

    for start in range(
        0,
        N,
        BATCH_SIZE
    ):

        idx = indices[
            start:
            start + BATCH_SIZE
        ]

        bs = states_t[idx]
        bd = desired_t[idx]
        ba = actions_t[idx]
        br = rewards_t[idx]
        bnext = next_states_t[idx]
        bdone = dones_t[idx]

        # ----------------------------------------------------
        # CRITIC UPDATE
        # ----------------------------------------------------

        with torch.no_grad():

            next_action, next_log_prob, _ = (
                actor.sample(
                    bnext,
                    bd
                )
            )

            target_q1 = target_critic1(
                bnext,
                bd,
                next_action
            )

            target_q2 = target_critic2(
                bnext,
                bd,
                next_action
            )

            target_q = torch.min(
                target_q1,
                target_q2
            )

            alpha = log_alpha.exp()

            target = (
                br
                + GAMMA
                * (1.0 - bdone)
                * (
                    target_q
                    - alpha *
                    next_log_prob
                )
            )

        current_q1 = critic1(
            bs,
            bd,
            ba
        )

        current_q2 = critic2(
            bs,
            bd,
            ba
        )

        critic1_loss = F.mse_loss(
            current_q1,
            target
        )

        critic2_loss = F.mse_loss(
            current_q2,
            target
        )

        critic1_optimizer.zero_grad(
            set_to_none=True
        )

        critic1_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            critic1.parameters(),
            1.0
        )

        critic1_optimizer.step()

        critic2_optimizer.zero_grad(
            set_to_none=True
        )

        critic2_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            critic2.parameters(),
            1.0
        )

        critic2_optimizer.step()

        # ----------------------------------------------------
        # ACTOR UPDATE
        # ----------------------------------------------------

        new_action, log_prob, _ = (
            actor.sample(
                bs,
                bd
            )
        )

        q1_new = critic1(
            bs,
            bd,
            new_action
        )

        q2_new = critic2(
            bs,
            bd,
            new_action
        )

        min_q_new = torch.min(
            q1_new,
            q2_new
        )

        alpha = log_alpha.exp()

        sac_actor_loss = (
            alpha * log_prob
            - min_q_new
        ).mean()

        # Behavioral cloning term
        _, _, deterministic_action = (
            actor.sample(
                bs,
                bd
            )
        )

        bc_loss = F.mse_loss(
            deterministic_action,
            ba
        )

        actor_loss = (
            sac_actor_loss
            + BC_WEIGHT * bc_loss
        )

        actor_optimizer.zero_grad(
            set_to_none=True
        )

        actor_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            actor.parameters(),
            1.0
        )

        actor_optimizer.step()

        # ----------------------------------------------------
        # AUTO ALPHA
        # ----------------------------------------------------

        alpha_loss = -(
            log_alpha *
            (
                log_prob.detach()
                + TARGET_ENTROPY
            )
        ).mean()

        alpha_optimizer.zero_grad(
            set_to_none=True
        )

        alpha_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [log_alpha],
            1.0
        )

        alpha_optimizer.step()

        # ----------------------------------------------------
        # TARGET UPDATE
        # ----------------------------------------------------

        with torch.no_grad():

            for target_param, param in zip(
                target_critic1.parameters(),
                critic1.parameters()
            ):

                target_param.mul_(1.0 - TAU)
                target_param.add_(
                    TAU * param
                )

            for target_param, param in zip(
                target_critic2.parameters(),
                critic2.parameters()
            ):

                target_param.mul_(1.0 - TAU)
                target_param.add_(
                    TAU * param
                )

        actor_losses.append(
            actor_loss.item()
        )

        critic_losses.append(
            (
                critic1_loss.item()
                +
                critic2_loss.item()
            ) / 2.0
        )

        alpha_losses.append(
            alpha_loss.item()
        )

    mean_actor = np.mean(
        actor_losses
    )

    mean_critic = np.mean(
        critic_losses
    )

    mean_alpha_loss = np.mean(
        alpha_losses
    )

    current_alpha = (
        log_alpha.exp()
        .item()
    )

    if mean_actor < best_loss:

        best_loss = mean_actor

    if (
        epoch == 1
        or epoch % 5 == 0
    ):

        print(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"Actor {mean_actor:.6f} | "
            f"Critic {mean_critic:.6f} | "
            f"Alpha {current_alpha:.6f}"
        )

# ============================================================
# SAVE
# ============================================================

checkpoint = {

    "algorithm": "SAC",

    "state_dim": 6,

    "action_dim": 3,

    "max_action": 2.0,

    "gamma": GAMMA,

    "tau": TAU,

    "replay_size": REPLAY_SIZE,

    "batch_size": BATCH_SIZE,

    "state_low": STATE_LOW,

    "state_high": STATE_HIGH,

    "actor_state_dict":
        actor.state_dict(),

    "critic1_state_dict":
        critic1.state_dict(),

    "critic2_state_dict":
        critic2.state_dict(),

    "target_critic1_state_dict":
        target_critic1.state_dict(),

    "target_critic2_state_dict":
        target_critic2.state_dict(),

    "log_alpha":
        log_alpha.detach().cpu(),

    "training_transitions":
        N,

    "epochs":
        EPOCHS
}

torch.save(
    checkpoint,
    SAVE_PATH
)

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)
print("Saved:", SAVE_PATH)
print("Algorithm: SAC")
print("Stochastic Gaussian actor: YES")
print("Twin critics: YES")
print("Automatic alpha: YES")
print("12D actor input: YES")
print("3D action output: YES")
print("Action range: [-2, 2]")
print("STATE RANGE: UNCHANGED")
'''

with open(
    SCRIPT_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(training_code)

print("Training script written.")
print("No range changes.")
print("No actor.py changes.")
print("No dynamics.py changes.")

Training script written.
No range changes.
No actor.py changes.
No dynamics.py changes.


In [142]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
SCRIPT_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "train_expanded_sac.py"
)

result = subprocess.run(
    [sys.executable, SCRIPT_PATH],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n" + "=" * 80)
    print("STDERR")
    print("=" * 80)
    print(result.stderr)

print("\nRETURN CODE:", result.returncode)

EXPANDED-RANGE STOCHASTIC SAC TRAINING
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

Dataset:
states: (30000, 6)
desired: (30000, 6)
actions: (30000, 3)
next_states: (30000, 6)
DATASET RANGE CHECK PASSED

Starting SAC training...
Transitions: 30000
Epochs: 80
Batch size: 256
Epoch 001/80 | Actor 3.167648 | Critic 0.066720 | Alpha 0.000100
Epoch 005/80 | Actor 2.887832 | Critic 0.048417 | Alpha 0.000099
Epoch 010/80 | Actor 3.158440 | Critic 0.094434 | Alpha 0.000099
Epoch 015/80 | Actor 3.516625 | Critic 0.202209 | Alpha 0.000098
Epoch 020/80 | Actor 3.876151 | Critic 0.341630 | Alpha 0.000098
Epoch 025/80 | Actor 4.180679 | Critic 0.487436 | Alpha 0.000097
Epoch 030/80 | Actor 4.415268 | Critic 0.625977 | Alpha 0.000097
Epoch 035/80 | Actor 4.623769 | Critic 0.744846 | Alpha 0.000097
Epoch 040/80 | Actor 4.779593 | Critic 0.841269 | Alpha 0.000098
Epoch 045/80 | Actor 4.848016 | Critic 0.916696 | Alpha 0.000098
Epoch 050/80 | Actor 4.879112 | Critic 0.967034 | Alpha 0.000099
E

In [144]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

eval_code = r'''
import os
import sys
import numpy as np
import torch
import torch.nn as nn

# ============================================================
# EXACT FIXED RANGE — DO NOT CHANGE
# ============================================================

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

STATE_DIM = 6
ACTION_DIM = 3
MAX_ACTION = 2.0

DT = 0.01
STEPS = 300

SEED = 12345
NUM_TASKS = 100

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "final_model",
    "sac_expanded_full_final.pt"
)

# ============================================================
# CORRECT DYNAMICS IMPORT
# ============================================================

import importlib.util

DYNAMICS_PATH = os.path.join(
    PROJECT_ROOT,
    "dynamics",
    "dynamics.py"
)

spec = importlib.util.spec_from_file_location(
    "three_rpr_dynamics_eval",
    DYNAMICS_PATH
)

dynamics = importlib.util.module_from_spec(spec)

# IMPORTANT:
# dataclass inside dynamics.py requires the module
# to exist in sys.modules during execution.
sys.modules[spec.name] = dynamics

spec.loader.exec_module(dynamics)

print("Dynamics loaded successfully.")

# ============================================================
# NORMALIZATION
# ============================================================

def normalize_state(x):

    low = torch.tensor(
        STATE_LOW,
        dtype=torch.float32,
        device=x.device
    )

    high = torch.tensor(
        STATE_HIGH,
        dtype=torch.float32,
        device=x.device
    )

    return 2.0 * (
        x - low
    ) / (
        high - low
    ) - 1.0


# ============================================================
# SAME SAC ACTOR
# ============================================================

class SACActor(nn.Module):

    def __init__(
        self,
        state_dim=6,
        action_dim=3,
        max_action=2.0
    ):
        super().__init__()

        self.max_action = max_action

        self.net = nn.Sequential(
            nn.Linear(
                state_dim * 2,
                256
            ),
            nn.ReLU(),

            nn.Linear(
                256,
                256
            ),
            nn.ReLU()
        )

        self.mean = nn.Linear(
            256,
            action_dim
        )

        self.log_std = nn.Linear(
            256,
            action_dim
        )

    def forward(
        self,
        state,
        desired_state
    ):

        state = normalize_state(state)

        desired_state = normalize_state(
            desired_state
        )

        x = torch.cat(
            [
                state,
                desired_state
            ],
            dim=1
        )

        h = self.net(x)

        mean = self.mean(h)

        log_std = self.log_std(h)

        log_std = torch.clamp(
            log_std,
            -5.0,
            1.0
        )

        return mean, log_std


# ============================================================
# LOAD TRAINED MODEL
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

actor = SACActor(
    state_dim=6,
    action_dim=3,
    max_action=2.0
).to(device)

actor.load_state_dict(
    checkpoint["actor_state_dict"]
)

actor.eval()

print("SAC model loaded successfully.")

# ============================================================
# DETERMINISTIC SAC POLICY
# ============================================================

def get_action(
    state,
    desired_state
):

    s = torch.tensor(
        state,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    d = torch.tensor(
        desired_state,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        mean, _ = actor(
            s,
            d
        )

        action = (
            MAX_ACTION *
            torch.tanh(mean)
        )

    return action.squeeze(0).cpu().numpy()


# ============================================================
# FIXED 100 UNSEEN TASKS
# ============================================================

rng = np.random.default_rng(
    SEED
)

initial_errors = []
final_errors = []
best_errors = []
final_rewards = []

successes = 0
unstable = 0

for task in range(
    NUM_TASKS
):

    # --------------------------------------------------------
    # INITIAL STATE
    # EXACT EXPANDED RANGE
    # --------------------------------------------------------

    state = rng.uniform(
        STATE_LOW,
        STATE_HIGH
    ).astype(
        np.float64
    )

    # --------------------------------------------------------
    # DESIRED STATE
    # EXACT EXPANDED POSE RANGE
    # --------------------------------------------------------

    desired_state = np.array(
        [
            rng.uniform(0.0, 1.0),
            rng.uniform(0.0, 1.0),
            rng.uniform(-0.5, 0.5),
            0.0,
            0.0,
            0.0
        ],
        dtype=np.float64
    )

    initial_error = np.linalg.norm(
        state[:3] -
        desired_state[:3]
    )

    best_error = initial_error

    total_reward = 0.0

    valid = True

    # --------------------------------------------------------
    # CLOSED LOOP
    # --------------------------------------------------------

    for step_idx in range(
        STEPS
    ):

        action = get_action(
            state,
            desired_state
        )

        action = np.clip(
            action,
            -2.0,
            2.0
        )

        next_state = dynamics.step(
            np.asarray(
                state
            ),
            np.asarray(
                action
            ),
            dt=DT
        )

        next_state = np.asarray(
            next_state,
            dtype=np.float64
        )

        if (
            not np.all(
                np.isfinite(
                    next_state
                )
            )
            or
            np.linalg.norm(
                next_state
            ) > 100.0
        ):

            valid = False
            unstable += 1
            break

        # ----------------------------------------------------
        # EXACT PROJECT REWARD
        # ----------------------------------------------------

        reward = (
            -np.sum(
                (
                    next_state -
                    desired_state
                ) ** 2
            )
            -
            0.01 *
            np.sum(
                action ** 2
            )
        )

        total_reward += reward

        pose_error = np.linalg.norm(
            next_state[:3] -
            desired_state[:3]
        )

        best_error = min(
            best_error,
            pose_error
        )

        state = next_state

    # --------------------------------------------------------
    # FINAL RESULT
    # --------------------------------------------------------

    if not valid:

        final_error = 999.0

    else:

        final_error = np.linalg.norm(
            state[:3] -
            desired_state[:3]
        )

        # EXACT SUCCESS DEFINITION
        if final_error < initial_error:
            successes += 1

    initial_errors.append(
        initial_error
    )

    final_errors.append(
        final_error
    )

    best_errors.append(
        best_error
    )

    final_rewards.append(
        total_reward
    )

# ============================================================
# FINAL METRICS
# ============================================================

initial_errors = np.asarray(
    initial_errors
)

final_errors = np.asarray(
    final_errors
)

best_errors = np.asarray(
    best_errors
)

final_rewards = np.asarray(
    final_rewards
)

improvements = (
    initial_errors -
    final_errors
)

print("\n" + "=" * 70)
print(
    "FIXED 100-TASK EXPANDED-RANGE SAC EVALUATION"
)
print("=" * 70)

print(
    f"Average initial pose error : "
    f"{initial_errors.mean():.6f}"
)

print(
    f"Average final pose error   : "
    f"{final_errors.mean():.6f}"
)

print(
    f"Average improvement        : "
    f"{improvements.mean():.6f}"
)

print(
    f"Average best pose error    : "
    f"{best_errors.mean():.6f}"
)

print(
    f"Best final error           : "
    f"{final_errors.min():.6f}"
)

print(
    f"Worst final error          : "
    f"{final_errors.max():.6f}"
)

print(
    f"Average final reward       : "
    f"{final_rewards.mean():.6f}"
)

print(
    f"Unstable episodes          : "
    f"{unstable}/100"
)

print(
    f"Successes                  : "
    f"{successes}/100"
)

print(
    f"Success rate               : "
    f"{successes}%"
)

print("=" * 70)

if successes >= 89:

    print(
        "TARGET PASSED: >= 89/100"
    )

else:

    print(
        f"TARGET NOT REACHED: "
        f"{successes}/100"
    )

print("=" * 70)
'''

result = subprocess.run(
    [
        sys.executable,
        "-c",
        eval_code
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n" + "=" * 80)
    print("STDERR")
    print("=" * 80)
    print(result.stderr)

print("\nRETURN CODE:", result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
Dynamics loaded successfully.
SAC model loaded successfully.

FIXED 100-TASK EXPANDED-RANGE SAC EVALUATION
Average initial pose error : 0.635496
Average final pose error   : 115.582224
Average improvement        : -114.946728
Average best pose error    : 0.499832
Best final error           : 0.582655
Worst final error          : 999.000000
Average final reward       : -7010.606323
Unstable episodes          : 11/100
Successes                  : 0/100
Success rate               : 0%
TARGET NOT REACHED: 0/100


RETURN CODE: 0


In [145]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

diagnostic = r'''
import os
import sys
import numpy as np
import torch
import torch.nn as nn

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "final_model",
    "sac_expanded_full_final.pt"
)

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

class SACActor(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(12, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )

        self.mean = nn.Linear(256, 3)
        self.log_std = nn.Linear(256, 3)

    def forward(self, state, desired):

        low = torch.tensor(
            STATE_LOW,
            dtype=torch.float32,
            device=state.device
        )

        high = torch.tensor(
            STATE_HIGH,
            dtype=torch.float32,
            device=state.device
        )

        state = 2 * (state - low) / (high - low) - 1
        desired = 2 * (desired - low) / (high - low) - 1

        x = torch.cat(
            [state, desired],
            dim=1
        )

        h = self.net(x)

        mean = self.mean(h)
        log_std = torch.clamp(
            self.log_std(h),
            -5,
            1
        )

        return mean, log_std

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

actor = SACActor().to(device)

actor.load_state_dict(
    checkpoint["actor_state_dict"]
)

actor.eval()

rng = np.random.default_rng(12345)

actions_all = []
mean_all = []
std_all = []

for _ in range(100):

    state = rng.uniform(
        STATE_LOW,
        STATE_HIGH
    ).astype(np.float32)

    desired = np.array([
        rng.uniform(0, 1),
        rng.uniform(0, 1),
        rng.uniform(-0.5, 0.5),
        0,
        0,
        0
    ], dtype=np.float32)

    s = torch.tensor(
        state,
        device=device
    ).unsqueeze(0)

    d = torch.tensor(
        desired,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        mean, log_std = actor(
            s,
            d
        )

        action = (
            2.0 *
            torch.tanh(mean)
        )

    mean_all.append(
        mean.cpu().numpy()[0]
    )

    std_all.append(
        log_std.exp().cpu().numpy()[0]
    )

    actions_all.append(
        action.cpu().numpy()[0]
    )

mean_all = np.array(mean_all)
std_all = np.array(std_all)
actions_all = np.array(actions_all)

print("=" * 70)
print("EXPANDED SAC POLICY DIAGNOSTIC")
print("=" * 70)

print("\nDeterministic action statistics:")
print("Minimum:", actions_all.min(axis=0))
print("Maximum:", actions_all.max(axis=0))
print("Mean   :", actions_all.mean(axis=0))
print("Std    :", actions_all.std(axis=0))

print("\nActor Gaussian std:")
print("Minimum:", std_all.min(axis=0))
print("Maximum:", std_all.max(axis=0))
print("Mean   :", std_all.mean(axis=0))

print("\nAbsolute force:")
print(
    "Maximum absolute force:",
    np.abs(actions_all).max()
)

print(
    "Actions near saturation:",
    np.mean(
        np.abs(actions_all) > 1.8
    ) * 100,
    "%"
)

print("=" * 70)
'''

result = subprocess.run(
    [
        sys.executable,
        "-c",
        diagnostic
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR")
    print(result.stderr)

print("\nRETURN CODE:", result.returncode)

EXPANDED SAC POLICY DIAGNOSTIC

Deterministic action statistics:
Minimum: [-1.9944998 -1.9413586 -1.91994  ]
Maximum: [1.9302785 1.903416  1.8858441]
Mean   : [-0.3448234  -0.18944591 -0.32404113]
Std    : [1.1633723 0.9964919 1.1249535]

Actor Gaussian std:
Minimum: [0.00673795 0.00673795 0.00673795]
Maximum: [0.03161156 0.01595892 0.01798704]
Mean   : [0.0085788  0.00724496 0.00763659]

Absolute force:
Maximum absolute force: 1.9944998
Actions near saturation: 8.666666666666668 %


RETURN CODE: 0


In [146]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

code = r'''
import os
import numpy as np

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

DATASET_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "final_model",
    "sac_expanded_expert_demonstrations.npz"
)

data = np.load(DATASET_PATH)

states = data["states"]
desired = data["desired_states"]
actions = data["actions"]
next_states = data["next_states"]
rewards = data["rewards"]

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2]
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2]
)

# Check one-step expert improvement
initial_errors = np.linalg.norm(
    states[:, :3] - desired[:, :3],
    axis=1
)

next_errors = np.linalg.norm(
    next_states[:, :3] - desired[:, :3],
    axis=1
)

improved = next_errors < initial_errors

print("=" * 70)
print("EXPANDED EXPERT DATASET DIAGNOSTIC")
print("=" * 70)

print("Transitions:", len(states))

print("\nOne-step pose improvement:")
print(
    f"Improved: {improved.sum()}/{len(improved)} "
    f"({100 * improved.mean():.2f}%)"
)

print(
    f"Average initial error: "
    f"{initial_errors.mean():.6f}"
)

print(
    f"Average next error: "
    f"{next_errors.mean():.6f}"
)

print(
    f"Average improvement: "
    f"{(initial_errors-next_errors).mean():.6f}"
)

print("\nExpert action statistics:")
print("Min:", actions.min(axis=0))
print("Max:", actions.max(axis=0))
print("Mean:", actions.mean(axis=0))
print("Std:", actions.std(axis=0))

print("\nReward statistics:")
print("Min:", rewards.min())
print("Max:", rewards.max())
print("Mean:", rewards.mean())

print("\nRange verification:")

print(
    "States inside fixed range:",
    np.all(states >= STATE_LOW) and
    np.all(states <= STATE_HIGH)
)

print(
    "Next states inside fixed range:",
    np.all(next_states >= STATE_LOW) and
    np.all(next_states <= STATE_HIGH)
)

print(
    "Desired states inside fixed range:",
    np.all(desired >= STATE_LOW) and
    np.all(desired <= STATE_HIGH)
)

print("=" * 70)
'''

result = subprocess.run(
    [sys.executable, "-c", code],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR")
    print(result.stderr)

print("\nRETURN CODE:", result.returncode)

EXPANDED EXPERT DATASET DIAGNOSTIC
Transitions: 30000

One-step pose improvement:
Improved: 21814/30000 (72.71%)
Average initial error: 0.383892
Average next error: 0.383250
Average improvement: 0.000641

Expert action statistics:
Min: [-2. -2. -2.]
Max: [2. 2. 2.]
Mean: [-0.2082506  -0.02268582 -0.22845674]
Std: [1.1865137 1.1366358 1.2102896]

Reward statistics:
Min: -1.9783589
Max: -0.00037177833
Mean: -0.318131

Range verification:
States inside fixed range: True
Next states inside fixed range: True
Desired states inside fixed range: True


RETURN CODE: 0


In [147]:
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
SCRIPT_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "train_expanded_sac.py"
)

code = r'''
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

# ============================================================
# FIXED RANGE — DO NOT CHANGE
# ============================================================

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

STATE_DIM = 6
ACTION_DIM = 3
MAX_ACTION = 2.0

GAMMA = 0.99
TAU = 0.005

BATCH_SIZE = 256

ACTOR_LR = 1e-5
CRITIC_LR = 3e-5
ALPHA_LR = 1e-6

INITIAL_ALPHA = 1e-5
TARGET_ENTROPY = -3.0

# Strong expert constraint
BC_WEIGHT = 25.0

EPOCHS = 60

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

DATASET_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "final_model",
    "sac_expanded_expert_demonstrations.npz"
)

SAVE_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "final_model",
    "sac_expanded_conservative.pt"
)

torch.manual_seed(123)
np.random.seed(123)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(123)

# ============================================================
# NORMALIZATION
# ============================================================

def norm(x):

    low = torch.as_tensor(
        STATE_LOW,
        dtype=torch.float32,
        device=x.device
    )

    high = torch.as_tensor(
        STATE_HIGH,
        dtype=torch.float32,
        device=x.device
    )

    return (
        2.0 * (x - low)
        / (high - low)
        - 1.0
    )


# ============================================================
# SAC ACTOR
# ============================================================

class SACActor(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(12, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )

        self.mean = nn.Linear(
            256,
            3
        )

        self.log_std = nn.Linear(
            256,
            3
        )

    def forward(
        self,
        state,
        desired
    ):

        state = norm(state)
        desired = norm(desired)

        x = torch.cat(
            [state, desired],
            dim=1
        )

        h = self.net(x)

        mean = self.mean(h)

        log_std = torch.clamp(
            self.log_std(h),
            -5.0,
            -1.0
        )

        return mean, log_std

    def sample(
        self,
        state,
        desired
    ):

        mean, log_std = self.forward(
            state,
            desired
        )

        std = log_std.exp()

        dist = Normal(
            mean,
            std
        )

        z = dist.rsample()

        y = torch.tanh(z)

        action = 2.0 * y

        log_prob = dist.log_prob(z)

        log_prob -= torch.log(
            1.0 - y.pow(2) + 1e-6
        )

        log_prob -= np.log(2.0)

        log_prob = log_prob.sum(
            dim=1,
            keepdim=True
        )

        deterministic = (
            2.0 *
            torch.tanh(mean)
        )

        return (
            action,
            log_prob,
            deterministic
        )


# ============================================================
# CRITIC
# ============================================================

class Critic(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(15, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(
        self,
        state,
        desired,
        action
    ):

        state = norm(state)
        desired = norm(desired)

        x = torch.cat(
            [
                state,
                desired,
                action
            ],
            dim=1
        )

        return self.net(x)


# ============================================================
# LOAD DATA
# ============================================================

data = np.load(
    DATASET_PATH
)

states = data["states"].astype(
    np.float32
)

desired = data["desired_states"].astype(
    np.float32
)

actions = data["actions"].astype(
    np.float32
)

next_states = data["next_states"].astype(
    np.float32
)

rewards = data["rewards"].astype(
    np.float32
)

if "dones" in data:

    dones = data["dones"].astype(
        np.float32
    )

else:

    dones = np.zeros(
        len(states),
        dtype=np.float32
    )

# Exact range checks
assert np.all(
    states >= STATE_LOW - 1e-6
)

assert np.all(
    states <= STATE_HIGH + 1e-6
)

assert np.all(
    next_states >= STATE_LOW - 1e-6
)

assert np.all(
    next_states <= STATE_HIGH + 1e-6
)

assert np.all(
    desired >= STATE_LOW - 1e-6
)

assert np.all(
    desired <= STATE_HIGH + 1e-6
)

assert np.all(
    actions >= -2.0 - 1e-6
)

assert np.all(
    actions <= 2.0 + 1e-6
)

print("=" * 70)
print("CONSERVATIVE EXPANDED-RANGE SAC")
print("=" * 70)

print(
    "Device:",
    DEVICE
)

if DEVICE.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "Transitions:",
    len(states)
)

print(
    "BC weight:",
    BC_WEIGHT
)

print(
    "Dataset range: VERIFIED"
)

# ============================================================
# GPU TENSORS
# ============================================================

s = torch.tensor(
    states,
    dtype=torch.float32,
    device=DEVICE
)

d = torch.tensor(
    desired,
    dtype=torch.float32,
    device=DEVICE
)

a = torch.tensor(
    actions,
    dtype=torch.float32,
    device=DEVICE
)

ns = torch.tensor(
    next_states,
    dtype=torch.float32,
    device=DEVICE
)

r = torch.tensor(
    rewards,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(1)

done = torch.tensor(
    dones,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(1)

N = len(states)

# ============================================================
# NETWORKS
# ============================================================

actor = SACActor().to(DEVICE)

critic1 = Critic().to(DEVICE)
critic2 = Critic().to(DEVICE)

target1 = copy.deepcopy(
    critic1
)

target2 = copy.deepcopy(
    critic2
)

actor_opt = torch.optim.Adam(
    actor.parameters(),
    lr=ACTOR_LR
)

critic1_opt = torch.optim.Adam(
    critic1.parameters(),
    lr=CRITIC_LR
)

critic2_opt = torch.optim.Adam(
    critic2.parameters(),
    lr=CRITIC_LR
)

log_alpha = torch.tensor(
    np.log(INITIAL_ALPHA),
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

alpha_opt = torch.optim.Adam(
    [log_alpha],
    lr=ALPHA_LR
)

# ============================================================
# TRAIN
# ============================================================

for epoch in range(
    1,
    EPOCHS + 1
):

    order = np.random.permutation(N)

    actor_sum = 0.0
    critic_sum = 0.0
    bc_sum = 0.0

    batches = 0

    for start in range(
        0,
        N,
        BATCH_SIZE
    ):

        idx = order[
            start:
            start + BATCH_SIZE
        ]

        bs = s[idx]
        bd = d[idx]
        ba = a[idx]
        br = r[idx]
        bns = ns[idx]
        bdone = done[idx]

        # ----------------------------------------------------
        # CRITIC
        # ----------------------------------------------------

        with torch.no_grad():

            naction, nlogp, _ = (
                actor.sample(
                    bns,
                    bd
                )
            )

            tq1 = target1(
                bns,
                bd,
                naction
            )

            tq2 = target2(
                bns,
                bd,
                naction
            )

            tq = torch.minimum(
                tq1,
                tq2
            )

            alpha = log_alpha.exp()

            y = (
                br
                +
                GAMMA *
                (1.0 - bdone) *
                (
                    tq -
                    alpha * nlogp
                )
            )

        q1 = critic1(
            bs,
            bd,
            ba
        )

        q2 = critic2(
            bs,
            bd,
            ba
        )

        loss_q1 = F.mse_loss(
            q1,
            y
        )

        loss_q2 = F.mse_loss(
            q2,
            y
        )

        critic1_opt.zero_grad(
            set_to_none=True
        )

        loss_q1.backward()

        torch.nn.utils.clip_grad_norm_(
            critic1.parameters(),
            1.0
        )

        critic1_opt.step()

        critic2_opt.zero_grad(
            set_to_none=True
        )

        loss_q2.backward()

        torch.nn.utils.clip_grad_norm_(
            critic2.parameters(),
            1.0
        )

        critic2_opt.step()

        # ----------------------------------------------------
        # ACTOR
        # ----------------------------------------------------

        new_action, logp, deterministic = (
            actor.sample(
                bs,
                bd
            )
        )

        q1_new = critic1(
            bs,
            bd,
            new_action
        )

        q2_new = critic2(
            bs,
            bd,
            new_action
        )

        q_new = torch.minimum(
            q1_new,
            q2_new
        )

        alpha = log_alpha.exp()

        sac_loss = (
            alpha * logp
            - q_new
        ).mean()

        # Strong expert-action constraint
        bc_loss = F.mse_loss(
            deterministic,
            ba
        )

        actor_loss = (
            sac_loss
            +
            BC_WEIGHT * bc_loss
        )

        actor_opt.zero_grad(
            set_to_none=True
        )

        actor_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            actor.parameters(),
            1.0
        )

        actor_opt.step()

        # ----------------------------------------------------
        # ALPHA
        # ----------------------------------------------------

        alpha_loss = -(
            log_alpha *
            (
                logp.detach()
                + TARGET_ENTROPY
            )
        ).mean()

        alpha_opt.zero_grad(
            set_to_none=True
        )

        alpha_loss.backward()

        alpha_opt.step()

        # ----------------------------------------------------
        # TARGET NETWORKS
        # ----------------------------------------------------

        with torch.no_grad():

            for tp, p in zip(
                target1.parameters(),
                critic1.parameters()
            ):

                tp.mul_(
                    1.0 - TAU
                )

                tp.add_(
                    TAU * p
                )

            for tp, p in zip(
                target2.parameters(),
                critic2.parameters()
            ):

                tp.mul_(
                    1.0 - TAU
                )

                tp.add_(
                    TAU * p
                )

        actor_sum += actor_loss.item()
        critic_sum += (
            loss_q1.item()
            +
            loss_q2.item()
        ) / 2.0

        bc_sum += bc_loss.item()

        batches += 1

    if (
        epoch == 1
        or epoch % 5 == 0
    ):

        print(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"Actor {actor_sum/batches:.6f} | "
            f"Critic {critic_sum/batches:.6f} | "
            f"BC {bc_sum/batches:.6f} | "
            f"Alpha {log_alpha.exp().item():.7f}"
        )

# ============================================================
# SAVE
# ============================================================

torch.save(
    {
        "algorithm": "SAC",
        "state_dim": 6,
        "action_dim": 3,
        "max_action": 2.0,
        "gamma": GAMMA,
        "tau": TAU,
        "state_low": STATE_LOW,
        "state_high": STATE_HIGH,
        "actor_state_dict": actor.state_dict(),
        "critic1_state_dict": critic1.state_dict(),
        "critic2_state_dict": critic2.state_dict(),
        "target_critic1_state_dict": target1.state_dict(),
        "target_critic2_state_dict": target2.state_dict(),
        "log_alpha": log_alpha.detach().cpu(),
        "bc_weight": BC_WEIGHT
    },
    SAVE_PATH
)

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)
print("Saved:", SAVE_PATH)
print("Range unchanged.")
'''

with open(
    SCRIPT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(code)

print("Training script updated.")
print("Range unchanged.")

Training script updated.
Range unchanged.


In [148]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

result = subprocess.run(
    [
        sys.executable,
        os.path.join(
            PROJECT_ROOT,
            "training",
            "train_expanded_sac.py"
        )
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n" + "=" * 80)
    print("STDERR")
    print("=" * 80)
    print(result.stderr)

print("\nRETURN CODE:", result.returncode)

CONSERVATIVE EXPANDED-RANGE SAC
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Transitions: 30000
BC weight: 25.0
Dataset range: VERIFIED
Epoch 001/60 | Actor 35.841755 | Critic 0.110595 | BC 1.424252 | Alpha 0.0000100
Epoch 005/60 | Actor 30.586415 | Critic 0.052193 | BC 1.197892 | Alpha 0.0000100
Epoch 010/60 | Actor 27.503601 | Critic 0.083332 | BC 1.056270 | Alpha 0.0000100
Epoch 015/60 | Actor 26.238822 | Critic 0.128758 | BC 0.986256 | Alpha 0.0000100
Epoch 020/60 | Actor 25.441130 | Critic 0.190011 | BC 0.935650 | Alpha 0.0000100
Epoch 025/60 | Actor 25.036950 | Critic 0.265928 | BC 0.901808 | Alpha 0.0000100
Epoch 030/60 | Actor 24.533255 | Critic 0.349842 | BC 0.865352 | Alpha 0.0000100
Epoch 035/60 | Actor 24.169053 | Critic 0.436662 | BC 0.835959 | Alpha 0.0000100
Epoch 040/60 | Actor 23.792937 | Critic 0.520556 | BC 0.807139 | Alpha 0.0000100
Epoch 045/60 | Actor 23.445948 | Critic 0.617355 | BC 0.780789 | Alpha 0.0000099
Epoch 050/60 | Actor 23.087447 | Critic 0.7073

In [150]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

code = r'''
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import importlib.util

# ============================================================
# FIXED SETTINGS — DO NOT CHANGE
# ============================================================

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

MAX_ACTION = 2.0
DT = 0.01
STEPS = 300
NUM_TASKS = 100
SEED = 12345

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "final_model",
    "sac_expanded_conservative.pt"
)

# ============================================================
# DYNAMICS
# ============================================================

DYNAMICS_PATH = os.path.join(
    PROJECT_ROOT,
    "dynamics",
    "dynamics.py"
)

spec = importlib.util.spec_from_file_location(
    "three_rpr_dynamics_eval",
    DYNAMICS_PATH
)

dynamics = importlib.util.module_from_spec(spec)

sys.modules[spec.name] = dynamics

spec.loader.exec_module(dynamics)

# ============================================================
# NORMALIZATION
# ============================================================

def normalize_state(x):

    low = torch.tensor(
        STATE_LOW,
        dtype=torch.float32,
        device=x.device
    )

    high = torch.tensor(
        STATE_HIGH,
        dtype=torch.float32,
        device=x.device
    )

    return (
        2.0 * (x - low)
        / (high - low)
        - 1.0
    )

# ============================================================
# ACTOR — EXACT ARCHITECTURE USED FOR TRAINING
# ============================================================

class SACActor(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(12, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )

        self.mean = nn.Linear(
            256,
            3
        )

        self.log_std = nn.Linear(
            256,
            3
        )

    def forward(
        self,
        state,
        desired
    ):

        state = normalize_state(state)
        desired = normalize_state(desired)

        x = torch.cat(
            [state, desired],
            dim=1
        )

        h = self.net(x)

        mean = self.mean(h)

        log_std = torch.clamp(
            self.log_std(h),
            -5.0,
            -1.0
        )

        return mean, log_std

# ============================================================
# LOAD
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

actor = SACActor().to(device)

actor.load_state_dict(
    checkpoint["actor_state_dict"]
)

actor.eval()

# ============================================================
# DETERMINISTIC SAC ACTION
# ============================================================

def get_action(
    state,
    desired
):

    s = torch.tensor(
        state,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    d = torch.tensor(
        desired,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        mean, _ = actor(
            s,
            d
        )

        action = (
            2.0 *
            torch.tanh(mean)
        )

    return action.cpu().numpy()[0]

# ============================================================
# FIXED UNSEEN EVALUATION
# ============================================================

rng = np.random.default_rng(
    SEED
)

initial_errors = []
final_errors = []
best_errors = []
rewards = []

successes = 0
unstable = 0

for task in range(
    NUM_TASKS
):

    state = rng.uniform(
        STATE_LOW,
        STATE_HIGH
    ).astype(
        np.float64
    )

    desired = np.array(
        [
            rng.uniform(0.0, 1.0),
            rng.uniform(0.0, 1.0),
            rng.uniform(-0.5, 0.5),
            0.0,
            0.0,
            0.0
        ],
        dtype=np.float64
    )

    initial_error = np.linalg.norm(
        state[:3] -
        desired[:3]
    )

    best_error = initial_error
    total_reward = 0.0
    valid = True

    for _ in range(STEPS):

        action = get_action(
            state,
            desired
        )

        action = np.clip(
            action,
            -2.0,
            2.0
        )

        next_state = dynamics.step(
            np.asarray(state),
            np.asarray(action),
            dt=DT
        )

        next_state = np.asarray(
            next_state,
            dtype=np.float64
        )

        if (
            not np.all(
                np.isfinite(next_state)
            )
            or
            np.linalg.norm(next_state) > 100.0
        ):

            valid = False
            unstable += 1
            break

        # EXACT PROJECT REWARD
        reward = (
            -np.sum(
                (next_state - desired) ** 2
            )
            -
            0.01 *
            np.sum(action ** 2)
        )

        total_reward += reward

        pose_error = np.linalg.norm(
            next_state[:3] -
            desired[:3]
        )

        best_error = min(
            best_error,
            pose_error
        )

        state = next_state

    if valid:

        final_error = np.linalg.norm(
            state[:3] -
            desired[:3]
        )

        if final_error < initial_error:
            successes += 1

    else:

        final_error = 999.0

    initial_errors.append(
        initial_error
    )

    final_errors.append(
        final_error
    )

    best_errors.append(
        best_error
    )

    rewards.append(
        total_reward
    )

# ============================================================
# RESULTS
# ============================================================

initial_errors = np.array(
    initial_errors
)

final_errors = np.array(
    final_errors
)

best_errors = np.array(
    best_errors
)

rewards = np.array(
    rewards
)

print("\n" + "=" * 70)
print("FIXED 100-TASK EXPANDED-RANGE SAC EVALUATION")
print("=" * 70)

print(
    "Model:",
    os.path.basename(MODEL_PATH)
)

print(
    "Seed:",
    SEED
)

print(
    "Tasks:",
    NUM_TASKS
)

print(
    "Steps:",
    STEPS
)

print("\nRESULTS")
print("-" * 70)

print(
    f"Average initial pose error : "
    f"{initial_errors.mean():.6f}"
)

print(
    f"Average final pose error   : "
    f"{final_errors.mean():.6f}"
)

print(
    f"Average improvement        : "
    f"{(initial_errors-final_errors).mean():.6f}"
)

print(
    f"Average best pose error    : "
    f"{best_errors.mean():.6f}"
)

print(
    f"Best final error           : "
    f"{final_errors.min():.6f}"
)

print(
    f"Worst final error          : "
    f"{final_errors.max():.6f}"
)

print(
    f"Average final reward       : "
    f"{rewards.mean():.6f}"
)

print(
    f"Unstable episodes          : "
    f"{unstable}/100"
)

print(
    f"Successes                  : "
    f"{successes}/100"
)

print(
    f"Success rate               : "
    f"{successes}%"
)

print("=" * 70)

if successes >= 89:
    print("TARGET PASSED")
else:
    print(
        f"TARGET NOT REACHED: {successes}/100"
    )

print("=" * 70)
'''

result = subprocess.run(
    [
        sys.executable,
        "-c",
        code
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n" + "=" * 80)
    print("STDERR")
    print("=" * 80)
    print(result.stderr)

print("\nRETURN CODE:", result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")

FIXED 100-TASK EXPANDED-RANGE SAC EVALUATION
Model: sac_expanded_conservative.pt
Seed: 12345
Tasks: 100
Steps: 300

RESULTS
----------------------------------------------------------------------
Average initial pose error : 0.635496
Average final pose error   : 52.440135
Average improvement        : -51.804638
Average best pose error    : 0.347650
Best final error           : 0.070381
Worst final error          : 999.000000
Average final reward       : -1909.750213
Unstable episodes          : 5/100
Successes                  : 12/100
Success rate               : 12%
TARGET NOT REACHED: 12/100


RETURN CODE: 0


In [151]:
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"
SCRIPT_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "train_expanded_sac.py"
)

code = r'''
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

# ============================================================
# FIXED RANGE — DO NOT CHANGE
# ============================================================

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

STATE_DIM = 6
ACTION_DIM = 3
MAX_ACTION = 2.0

GAMMA = 0.99
TAU = 0.005
BATCH_SIZE = 256

# ============================================================
# VERY CONSERVATIVE SAC
# ============================================================

BC_EPOCHS = 25
SAC_EPOCHS = 25

BC_LR = 3e-4
ACTOR_LR = 2e-6
CRITIC_LR = 1e-5
ALPHA_LR = 1e-7

BC_WEIGHT = 100.0

INITIAL_ALPHA = 1e-6
TARGET_ENTROPY = -3.0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

DATASET = os.path.join(
    ROOT,
    "training",
    "final_model",
    "sac_expanded_expert_demonstrations.npz"
)

SAVE_PATH = os.path.join(
    ROOT,
    "training",
    "final_model",
    "sac_expanded_expert_guided.pt"
)

np.random.seed(123)
torch.manual_seed(123)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(123)

# ============================================================
# NORMALIZATION
# ============================================================

def normalize(x):

    low = torch.as_tensor(
        STATE_LOW,
        dtype=torch.float32,
        device=x.device
    )

    high = torch.as_tensor(
        STATE_HIGH,
        dtype=torch.float32,
        device=x.device
    )

    return 2.0 * (
        x - low
    ) / (
        high - low
    ) - 1.0


# ============================================================
# SAC ACTOR
# ============================================================

class Actor(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(12, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )

        self.mean = nn.Linear(
            256,
            3
        )

        self.log_std = nn.Linear(
            256,
            3
        )

    def forward(
        self,
        state,
        desired
    ):

        state = normalize(state)
        desired = normalize(desired)

        x = torch.cat(
            [state, desired],
            dim=1
        )

        h = self.net(x)

        mean = self.mean(h)

        log_std = torch.clamp(
            self.log_std(h),
            -5.0,
            -1.0
        )

        return mean, log_std

    def sample(
        self,
        state,
        desired
    ):

        mean, log_std = self.forward(
            state,
            desired
        )

        std = log_std.exp()

        dist = Normal(
            mean,
            std
        )

        z = dist.rsample()

        y = torch.tanh(z)

        action = 2.0 * y

        logp = dist.log_prob(z)

        logp -= torch.log(
            1.0 - y.pow(2) + 1e-6
        )

        logp -= np.log(2.0)

        logp = logp.sum(
            dim=1,
            keepdim=True
        )

        deterministic = (
            2.0 *
            torch.tanh(mean)
        )

        return action, logp, deterministic


# ============================================================
# CRITIC
# ============================================================

class Critic(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(15, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(
        self,
        state,
        desired,
        action
    ):

        state = normalize(state)
        desired = normalize(desired)

        x = torch.cat(
            [
                state,
                desired,
                action
            ],
            dim=1
        )

        return self.net(x)


# ============================================================
# DATA
# ============================================================

data = np.load(DATASET)

states = data["states"].astype(np.float32)
desired = data["desired_states"].astype(np.float32)
actions = data["actions"].astype(np.float32)
next_states = data["next_states"].astype(np.float32)
rewards = data["rewards"].astype(np.float32)

if "dones" in data:
    dones = data["dones"].astype(np.float32)
else:
    dones = np.zeros(
        len(states),
        dtype=np.float32
    )

# ABSOLUTE RANGE CHECK
assert np.all(states >= STATE_LOW)
assert np.all(states <= STATE_HIGH)

assert np.all(next_states >= STATE_LOW)
assert np.all(next_states <= STATE_HIGH)

assert np.all(desired >= STATE_LOW)
assert np.all(desired <= STATE_HIGH)

assert np.all(actions >= -2.0)
assert np.all(actions <= 2.0)

print("=" * 70)
print("EXPERT-GUIDED EXPANDED SAC")
print("=" * 70)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("Transitions:", len(states))
print("BC weight:", BC_WEIGHT)
print("Range: VERIFIED")

# ============================================================
# TENSORS
# ============================================================

S = torch.tensor(
    states,
    dtype=torch.float32,
    device=DEVICE
)

D = torch.tensor(
    desired,
    dtype=torch.float32,
    device=DEVICE
)

A = torch.tensor(
    actions,
    dtype=torch.float32,
    device=DEVICE
)

NS = torch.tensor(
    next_states,
    dtype=torch.float32,
    device=DEVICE
)

R = torch.tensor(
    rewards,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(1)

DONE = torch.tensor(
    dones,
    dtype=torch.float32,
    device=DEVICE
).unsqueeze(1)

N = len(S)

# ============================================================
# NETWORKS
# ============================================================

actor = Actor().to(DEVICE)

critic1 = Critic().to(DEVICE)
critic2 = Critic().to(DEVICE)

target1 = copy.deepcopy(critic1)
target2 = copy.deepcopy(critic2)

# ============================================================
# STAGE 1 — EXPERT BEHAVIOR CLONING
# ============================================================

bc_optimizer = torch.optim.Adam(
    actor.parameters(),
    lr=BC_LR
)

print("\nSTAGE 1 — EXPERT ACTOR PRETRAINING")

for epoch in range(
    1,
    BC_EPOCHS + 1
):

    order = np.random.permutation(N)

    total = 0.0
    batches = 0

    for start in range(
        0,
        N,
        BATCH_SIZE
    ):

        idx = order[
            start:start+BATCH_SIZE
        ]

        bs = S[idx]
        bd = D[idx]
        ba = A[idx]

        mean, log_std = actor(
            bs,
            bd
        )

        predicted = (
            2.0 *
            torch.tanh(mean)
        )

        loss = F.mse_loss(
            predicted,
            ba
        )

        bc_optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            actor.parameters(),
            1.0
        )

        bc_optimizer.step()

        total += loss.item()
        batches += 1

    if (
        epoch == 1
        or epoch % 5 == 0
    ):

        print(
            f"BC Epoch {epoch:02d}/{BC_EPOCHS} | "
            f"MSE {total/batches:.6f}"
        )

# ============================================================
# STAGE 2 — CRITIC PRETRAINING
# ============================================================

critic1_optimizer = torch.optim.Adam(
    critic1.parameters(),
    lr=CRITIC_LR
)

critic2_optimizer = torch.optim.Adam(
    critic2.parameters(),
    lr=CRITIC_LR
)

print("\nSTAGE 2 — CRITIC PRETRAINING")

for epoch in range(1, 11):

    order = np.random.permutation(N)

    qloss = 0.0
    batches = 0

    for start in range(
        0,
        N,
        BATCH_SIZE
    ):

        idx = order[
            start:start+BATCH_SIZE
        ]

        bs = S[idx]
        bd = D[idx]
        ba = A[idx]
        br = R[idx]
        bns = NS[idx]
        bdone = DONE[idx]

        with torch.no_grad():

            next_mean, _ = actor(
                bns,
                bd
            )

            next_action = (
                2.0 *
                torch.tanh(next_mean)
            )

            tq1 = target1(
                bns,
                bd,
                next_action
            )

            tq2 = target2(
                bns,
                bd,
                next_action
            )

            target = (
                br
                +
                GAMMA *
                (1.0 - bdone)
                *
                torch.minimum(
                    tq1,
                    tq2
                )
            )

        q1 = critic1(
            bs,
            bd,
            ba
        )

        q2 = critic2(
            bs,
            bd,
            ba
        )

        l1 = F.mse_loss(
            q1,
            target
        )

        l2 = F.mse_loss(
            q2,
            target
        )

        critic1_optimizer.zero_grad(
            set_to_none=True
        )

        l1.backward()

        torch.nn.utils.clip_grad_norm_(
            critic1.parameters(),
            1.0
        )

        critic1_optimizer.step()

        critic2_optimizer.zero_grad(
            set_to_none=True
        )

        l2.backward()

        torch.nn.utils.clip_grad_norm_(
            critic2.parameters(),
            1.0
        )

        critic2_optimizer.step()

        qloss += (
            l1.item() +
            l2.item()
        ) / 2.0

        batches += 1

    if epoch % 2 == 0:
        print(
            f"Critic Epoch {epoch:02d}/10 | "
            f"Loss {qloss/batches:.6f}"
        )

# ============================================================
# STAGE 3 — VERY LOW-LR SAC + STRONG BC
# ============================================================

actor_optimizer = torch.optim.Adam(
    actor.parameters(),
    lr=ACTOR_LR
)

alpha_log = torch.tensor(
    np.log(INITIAL_ALPHA),
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

alpha_optimizer = torch.optim.Adam(
    [alpha_log],
    lr=ALPHA_LR
)

print("\nSTAGE 3 — CONSERVATIVE SAC")

for epoch in range(
    1,
    SAC_EPOCHS + 1
):

    order = np.random.permutation(N)

    actor_total = 0.0
    critic_total = 0.0
    bc_total = 0.0

    batches = 0

    for start in range(
        0,
        N,
        BATCH_SIZE
    ):

        idx = order[
            start:start+BATCH_SIZE
        ]

        bs = S[idx]
        bd = D[idx]
        ba = A[idx]
        br = R[idx]
        bns = NS[idx]
        bdone = DONE[idx]

        # ----------------------------------------------------
        # CRITIC TARGET
        # ----------------------------------------------------

        with torch.no_grad():

            na, nlogp, _ = actor.sample(
                bns,
                bd
            )

            tq1 = target1(
                bns,
                bd,
                na
            )

            tq2 = target2(
                bns,
                bd,
                na
            )

            alpha = alpha_log.exp()

            target = (
                br
                +
                GAMMA *
                (1.0 - bdone)
                *
                (
                    torch.minimum(
                        tq1,
                        tq2
                    )
                    -
                    alpha * nlogp
                )
            )

        q1 = critic1(
            bs,
            bd,
            ba
        )

        q2 = critic2(
            bs,
            bd,
            ba
        )

        qloss1 = F.mse_loss(
            q1,
            target
        )

        qloss2 = F.mse_loss(
            q2,
            target
        )

        critic1_optimizer.zero_grad(
            set_to_none=True
        )

        qloss1.backward()

        torch.nn.utils.clip_grad_norm_(
            critic1.parameters(),
            1.0
        )

        critic1_optimizer.step()

        critic2_optimizer.zero_grad(
            set_to_none=True
        )

        qloss2.backward()

        torch.nn.utils.clip_grad_norm_(
            critic2.parameters(),
            1.0
        )

        critic2_optimizer.step()

        # ----------------------------------------------------
        # ACTOR
        # ----------------------------------------------------

        new_action, logp, deterministic = (
            actor.sample(
                bs,
                bd
            )
        )

        q1_new = critic1(
            bs,
            bd,
            new_action
        )

        q2_new = critic2(
            bs,
            bd,
            new_action
        )

        sac_loss = (
            alpha * logp
            -
            torch.minimum(
                q1_new,
                q2_new
            )
        ).mean()

        bc_loss = F.mse_loss(
            deterministic,
            ba
        )

        actor_loss = (
            sac_loss
            +
            BC_WEIGHT * bc_loss
        )

        actor_optimizer.zero_grad(
            set_to_none=True
        )

        actor_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            actor.parameters(),
            1.0
        )

        actor_optimizer.step()

        # ----------------------------------------------------
        # ALPHA
        # ----------------------------------------------------

        alpha_loss = -(
            alpha_log *
            (
                logp.detach()
                +
                TARGET_ENTROPY
            )
        ).mean()

        alpha_optimizer.zero_grad(
            set_to_none=True
        )

        alpha_loss.backward()

        alpha_optimizer.step()

        # ----------------------------------------------------
        # TARGET UPDATE
        # ----------------------------------------------------

        with torch.no_grad():

            for tp, p in zip(
                target1.parameters(),
                critic1.parameters()
            ):

                tp.mul_(1.0 - TAU)
                tp.add_(TAU * p)

            for tp, p in zip(
                target2.parameters(),
                critic2.parameters()
            ):

                tp.mul_(1.0 - TAU)
                tp.add_(TAU * p)

        actor_total += actor_loss.item()
        critic_total += (
            qloss1.item() +
            qloss2.item()
        ) / 2.0

        bc_total += bc_loss.item()

        batches += 1

    if (
        epoch == 1
        or epoch % 5 == 0
    ):

        print(
            f"SAC Epoch {epoch:02d}/{SAC_EPOCHS} | "
            f"Actor {actor_total/batches:.6f} | "
            f"Critic {critic_total/batches:.6f} | "
            f"BC {bc_total/batches:.6f} | "
            f"Alpha {alpha_log.exp().item():.8f}"
        )

# ============================================================
# SAVE
# ============================================================

torch.save(
    {
        "algorithm": "SAC",
        "state_dim": 6,
        "action_dim": 3,
        "max_action": 2.0,
        "gamma": GAMMA,
        "tau": TAU,
        "state_low": STATE_LOW,
        "state_high": STATE_HIGH,
        "actor_state_dict": actor.state_dict(),
        "critic1_state_dict": critic1.state_dict(),
        "critic2_state_dict": critic2.state_dict(),
        "target_critic1_state_dict": target1.state_dict(),
        "target_critic2_state_dict": target2.state_dict(),
        "log_alpha": alpha_log.detach().cpu(),
        "bc_weight": BC_WEIGHT
    },
    SAVE_PATH
)

print("\n" + "=" * 70)
print("EXPERT-GUIDED SAC TRAINING COMPLETE")
print("=" * 70)
print("Saved:", SAVE_PATH)
print("Range unchanged.")
'''

with open(
    SCRIPT_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(code)

print("Expert-guided SAC script written.")
print("Range unchanged.")

Expert-guided SAC script written.
Range unchanged.


In [152]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

result = subprocess.run(
    [
        sys.executable,
        os.path.join(
            PROJECT_ROOT,
            "training",
            "train_expanded_sac.py"
        )
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR")
    print(result.stderr)

print("\nRETURN CODE:", result.returncode)

EXPERT-GUIDED EXPANDED SAC
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Transitions: 30000
BC weight: 100.0
Range: VERIFIED

STAGE 1 â€” EXPERT ACTOR PRETRAINING
BC Epoch 01/25 | MSE 1.072252
BC Epoch 05/25 | MSE 0.489166
BC Epoch 10/25 | MSE 0.268189
BC Epoch 15/25 | MSE 0.193279
BC Epoch 20/25 | MSE 0.155800
BC Epoch 25/25 | MSE 0.133452

STAGE 2 â€” CRITIC PRETRAINING
Critic Epoch 02/10 | Loss 0.093103
Critic Epoch 04/10 | Loss 0.063874
Critic Epoch 06/10 | Loss 0.052438
Critic Epoch 08/10 | Loss 0.043768
Critic Epoch 10/10 | Loss 0.037824

STAGE 3 â€” CONSERVATIVE SAC
SAC Epoch 01/25 | Actor 13.200998 | Critic 0.040696 | BC 0.127623 | Alpha 0.00000100
SAC Epoch 05/25 | Actor 13.698865 | Critic 0.091531 | BC 0.126057 | Alpha 0.00000100
SAC Epoch 10/25 | Actor 14.433993 | Critic 0.206728 | BC 0.125128 | Alpha 0.00000100
SAC Epoch 15/25 | Actor 15.140640 | Critic 0.353955 | BC 0.124618 | Alpha 0.00000100
SAC Epoch 20/25 | Actor 15.826907 | Critic 0.502925 | BC 0.124564 | Alpha

In [153]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

script = os.path.join(
    PROJECT_ROOT,
    "training",
    "evaluate_expanded_sac.py"
)

print("Checking evaluation script:", script)
print("Exists:", os.path.exists(script))

Checking evaluation script: C:\Users\madha\OneDrive\Desktop\project_1\training\evaluate_expanded_sac.py
Exists: False


In [154]:
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

EVAL_PATH = os.path.join(
    PROJECT_ROOT,
    "training",
    "evaluate_expanded_sac.py"
)

code = r'''
import os
import sys
import importlib.util
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal

# ============================================================
# PATHS
# ============================================================

ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

MODEL_PATH = os.path.join(
    ROOT,
    "training",
    "final_model",
    "sac_expanded_expert_guided.pt"
)

DYNAMICS_PATH = os.path.join(
    ROOT,
    "dynamics",
    "dynamics.py"
)

# ============================================================
# FIXED RANGE — DO NOT CHANGE
# ============================================================

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

# ============================================================
# LOAD DYNAMICS DIRECTLY
# ============================================================

spec = importlib.util.spec_from_file_location(
    "rpr_dynamics_eval",
    DYNAMICS_PATH
)

dynamics = importlib.util.module_from_spec(spec)

sys.modules["rpr_dynamics_eval"] = dynamics

spec.loader.exec_module(dynamics)

step = dynamics.step

# ============================================================
# SAC ACTOR — SAME ARCHITECTURE AS TRAINING
# ============================================================

class Actor(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(12, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )

        self.mean = nn.Linear(
            256,
            3
        )

        self.log_std = nn.Linear(
            256,
            3
        )

    def forward(
        self,
        state,
        desired
    ):

        low = torch.as_tensor(
            STATE_LOW,
            dtype=torch.float32,
            device=state.device
        )

        high = torch.as_tensor(
            STATE_HIGH,
            dtype=torch.float32,
            device=state.device
        )

        state = (
            2.0 *
            (state - low) /
            (high - low)
            - 1.0
        )

        desired = (
            2.0 *
            (desired - low) /
            (high - low)
            - 1.0
        )

        x = torch.cat(
            [state, desired],
            dim=1
        )

        h = self.net(x)

        mean = self.mean(h)

        log_std = torch.clamp(
            self.log_std(h),
            -5.0,
            -1.0
        )

        return mean, log_std


# ============================================================
# LOAD MODEL
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

actor = Actor().to(device)

actor.load_state_dict(
    checkpoint["actor_state_dict"]
)

actor.eval()

print("=" * 70)
print("FIXED EXPANDED SAC EVALUATION")
print("=" * 70)
print("Device:", device)
print("Model:", MODEL_PATH)
print("State range: VERIFIED")
print("Tasks: 100")
print("Seed: 12345")
print("Steps per task: 300")
print("=" * 70)

# ============================================================
# FIXED UNSEEN TASKS
# ============================================================

rng = np.random.default_rng(12345)

initial_states = np.zeros(
    (100, 6),
    dtype=np.float32
)

desired_states = np.zeros(
    (100, 6),
    dtype=np.float32
)

# Full expanded pose range
initial_states[:, 0] = rng.uniform(
    0.0, 1.0, 100
)

initial_states[:, 1] = rng.uniform(
    0.0, 1.0, 100
)

initial_states[:, 2] = rng.uniform(
    -0.5, 0.5, 100
)

# Full expanded velocity range
initial_states[:, 3:] = rng.uniform(
    -0.2,
    0.2,
    (100, 3)
)

desired_states[:, 0] = rng.uniform(
    0.0, 1.0, 100
)

desired_states[:, 1] = rng.uniform(
    0.0, 1.0, 100
)

desired_states[:, 2] = rng.uniform(
    -0.5, 0.5, 100
)

# Desired velocities = zero
desired_states[:, 3:] = 0.0

# ============================================================
# EVALUATION
# ============================================================

initial_errors = []
final_errors = []
best_errors = []

successes = 0
unstable = 0

for task in range(100):

    state = initial_states[task].copy()
    desired = desired_states[task].copy()

    initial_error = np.linalg.norm(
        state[:3] - desired[:3]
    )

    best_error = initial_error

    valid = True

    for t in range(300):

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        desired_tensor = torch.tensor(
            desired,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        with torch.no_grad():

            mean, _ = actor(
                state_tensor,
                desired_tensor
            )

            # Deterministic evaluation
            action = (
                2.0 *
                torch.tanh(mean)
            )

        action = (
            action
            .cpu()
            .numpy()[0]
        )

        action = np.clip(
            action,
            -2.0,
            2.0
        )

        try:

            next_state = np.asarray(
                step(
                    state,
                    action,
                    dt=0.01
                ),
                dtype=np.float32
            )

        except Exception:

            valid = False
            break

        if not np.all(
            np.isfinite(next_state)
        ):

            valid = False
            break

        # Detect severe numerical instability
        if np.linalg.norm(next_state) > 100:

            valid = False
            break

        state = next_state

        pose_error = np.linalg.norm(
            state[:3] - desired[:3]
        )

        best_error = min(
            best_error,
            pose_error
        )

    if not valid:

        unstable += 1

        final_error = 999.0

    else:

        final_error = np.linalg.norm(
            state[:3] - desired[:3]
        )

    initial_errors.append(
        initial_error
    )

    final_errors.append(
        final_error
    )

    best_errors.append(
        best_error
    )

    if (
        valid
        and final_error < initial_error
    ):

        successes += 1

    if (task + 1) % 10 == 0:

        print(
            f"Task {task+1:03d}/100 | "
            f"Successes {successes} | "
            f"Unstable {unstable}"
        )

# ============================================================
# FINAL METRICS
# ============================================================

initial_errors = np.asarray(
    initial_errors
)

final_errors = np.asarray(
    final_errors
)

best_errors = np.asarray(
    best_errors
)

avg_initial = initial_errors.mean()
avg_final = final_errors.mean()
avg_improvement = (
    avg_initial -
    avg_final
)

avg_best = best_errors.mean()
best_final = final_errors.min()
worst_final = final_errors.max()

success_rate = (
    successes / 100.0
) * 100.0

print("\n" + "=" * 70)
print("FINAL FIXED EVALUATION")
print("=" * 70)

print(
    f"Average initial pose error : {avg_initial:.6f}"
)

print(
    f"Average final pose error   : {avg_final:.6f}"
)

print(
    f"Average improvement        : {avg_improvement:.6f}"
)

print(
    f"Average best pose error    : {avg_best:.6f}"
)

print(
    f"Best final error           : {best_final:.6f}"
)

print(
    f"Worst final error          : {worst_final:.6f}"
)

print(
    f"Successes                  : {successes}/100"
)

print(
    f"Success rate               : {success_rate:.2f}%"
)

print(
    f"Unstable                   : {unstable}/100"
)

print("=" * 70)

if success_rate >= 88.0:

    print("TARGET >= 88%: PASSED")

else:

    print("TARGET >= 88%: NOT PASSED")

print("=" * 70)
'''

with open(
    EVAL_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(code)

print("Evaluation script created:")
print(EVAL_PATH)

Evaluation script created:
C:\Users\madha\OneDrive\Desktop\project_1\training\evaluate_expanded_sac.py


In [155]:
import subprocess
import sys
import os

PROJECT_ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

result = subprocess.run(
    [
        sys.executable,
        os.path.join(
            PROJECT_ROOT,
            "training",
            "evaluate_expanded_sac.py"
        )
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR")
    print(result.stderr)

print("\nRETURN CODE:", result.returncode)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
FIXED EXPANDED SAC EVALUATION
Device: cuda
Model: C:\Users\madha\OneDrive\Desktop\project_1\training\final_model\sac_expanded_expert_guided.pt
State range: VERIFIED
Tasks: 100
Seed: 12345
Steps per task: 300
Task 010/100 | Successes 4 | Unstable 0
Task 020/100 | Successes 9 | Unstable 0
Task 030/100 | Successes 9 | Unstable 3
Task 040/100 | Successes 9 | Unstable 7
Task 050/100 | Successes 11 | Unstable 7
Task 060/100 | Successes 15 | Unstable 9
Task 070/100 | Successes 19 | Unstable 9
Task 080/100 | Successes 23 | Unstable 10
Task 090/100 | Successes 29 | Unstable 10
Task 100/100 | Successes 31 | Unstable 12

FINAL FIXED EVALUATION
Average initial pose error : 0.681556
Average final pose error   : 124.127155
Average improvement        : -123.445599
Average best pose error    : 0.410205
Best final error           : 0.018484
Worst final error          : 999.00

In [156]:
import os
import sys
import importlib.util
import numpy as np

ROOT = r"C:\Users\madha\OneDrive\Desktop\project_1"

DYNAMICS_PATH = os.path.join(
    ROOT,
    "dynamics",
    "dynamics.py"
)

# ------------------------------------------------------------
# LOAD DYNAMICS
# ------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "rpr_dynamics_teacher",
    DYNAMICS_PATH
)

dynamics = importlib.util.module_from_spec(spec)
sys.modules["rpr_dynamics_teacher"] = dynamics
spec.loader.exec_module(dynamics)

step = dynamics.step

# ------------------------------------------------------------
# FIXED RANGE — DO NOT CHANGE
# ------------------------------------------------------------

STATE_LOW = np.array(
    [0.0, 0.0, -0.5, -0.2, -0.2, -0.2],
    dtype=np.float32
)

STATE_HIGH = np.array(
    [1.0, 1.0, 0.5, 0.2, 0.2, 0.2],
    dtype=np.float32
)

# ------------------------------------------------------------
# TEACHER
# ------------------------------------------------------------

KP = np.array(
    [2.0, 2.0, 2.0],
    dtype=np.float64
)

KD = np.array(
    [1.5, 1.5, 1.5],
    dtype=np.float64
)

REG = 1e-4


def numerical_B(state):

    state = np.asarray(
        state,
        dtype=np.float64
    )

    zero = np.asarray(
        step(
            state,
            np.zeros(3),
            dt=0.01
        ),
        dtype=np.float64
    )

    B = np.zeros(
        (6, 3),
        dtype=np.float64
    )

    for j in range(3):

        force = np.zeros(3)
        force[j] = 1.0

        sj = np.asarray(
            step(
                state,
                force,
                dt=0.01
            ),
            dtype=np.float64
        )

        B[:, j] = sj - zero

    return B


def expert_action(
    state,
    desired
):

    B = numerical_B(state)

    desired_acc = (
        KP *
        (desired[:3] - state[:3])
        -
        KD *
        state[3:]
    )

    BtB = B.T @ B

    rhs = B.T @ desired_acc

    force = np.linalg.solve(
        BtB + REG * np.eye(3),
        rhs
    )

    return np.clip(
        force,
        -2.0,
        2.0
    )


# ------------------------------------------------------------
# EXACT FIXED 100 TASKS
# ------------------------------------------------------------

rng = np.random.default_rng(12345)

initial_states = np.zeros(
    (100, 6),
    dtype=np.float64
)

desired_states = np.zeros(
    (100, 6),
    dtype=np.float64
)

initial_states[:, 0] = rng.uniform(
    0.0, 1.0, 100
)

initial_states[:, 1] = rng.uniform(
    0.0, 1.0, 100
)

initial_states[:, 2] = rng.uniform(
    -0.5, 0.5, 100
)

initial_states[:, 3:] = rng.uniform(
    -0.2,
    0.2,
    (100, 3)
)

desired_states[:, 0] = rng.uniform(
    0.0, 1.0, 100
)

desired_states[:, 1] = rng.uniform(
    0.0, 1.0, 100
)

desired_states[:, 2] = rng.uniform(
    -0.5, 0.5, 100
)

desired_states[:, 3:] = 0.0

# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

initial_errors = []
final_errors = []
best_errors = []

successes = 0
unstable = 0

for i in range(100):

    state = initial_states[i].copy()
    desired = desired_states[i].copy()

    initial_error = np.linalg.norm(
        state[:3] - desired[:3]
    )

    best_error = initial_error
    valid = True

    for t in range(300):

        try:

            action = expert_action(
                state,
                desired
            )

            next_state = np.asarray(
                step(
                    state,
                    action,
                    dt=0.01
                ),
                dtype=np.float64
            )

        except Exception:

            valid = False
            break

        if not np.all(
            np.isfinite(next_state)
        ):

            valid = False
            break

        if np.linalg.norm(next_state) > 100:

            valid = False
            break

        state = next_state

        error = np.linalg.norm(
            state[:3] - desired[:3]
        )

        best_error = min(
            best_error,
            error
        )

    if not valid:

        unstable += 1
        final_error = 999.0

    else:

        final_error = np.linalg.norm(
            state[:3] - desired[:3]
        )

    initial_errors.append(
        initial_error
    )

    final_errors.append(
        final_error
    )

    best_errors.append(
        best_error
    )

    if (
        valid
        and final_error < initial_error
    ):
        successes += 1

    if (i + 1) % 10 == 0:

        print(
            f"Task {i+1:03d}/100 | "
            f"Success {successes} | "
            f"Unstable {unstable}"
        )

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

initial_errors = np.array(
    initial_errors
)

final_errors = np.array(
    final_errors
)

best_errors = np.array(
    best_errors
)

print("\n" + "=" * 70)
print("EXPERT TEACHER — FIXED EXPANDED EVALUATION")
print("=" * 70)

print(
    f"Average initial error : "
    f"{initial_errors.mean():.6f}"
)

print(
    f"Average final error   : "
    f"{final_errors.mean():.6f}"
)

print(
    f"Average improvement   : "
    f"{initial_errors.mean() - final_errors.mean():.6f}"
)

print(
    f"Average best error    : "
    f"{best_errors.mean():.6f}"
)

print(
    f"Best final error      : "
    f"{final_errors.min():.6f}"
)

print(
    f"Worst final error     : "
    f"{final_errors.max():.6f}"
)

print(
    f"Successes             : "
    f"{successes}/100"
)

print(
    f"Success rate          : "
    f"{successes:.2f}%"
)

print(
    f"Unstable              : "
    f"{unstable}/100"
)

print("=" * 70)

Could not load robot_properties.py. Using fallback parameters.
Reason: ModuleNotFoundError("No module named 'robot_properties'")
Task 010/100 | Success 0 | Unstable 10
Task 020/100 | Success 0 | Unstable 20
Task 030/100 | Success 0 | Unstable 30
Task 040/100 | Success 0 | Unstable 40
Task 050/100 | Success 0 | Unstable 50
Task 060/100 | Success 0 | Unstable 60
Task 070/100 | Success 0 | Unstable 70
Task 080/100 | Success 0 | Unstable 80
Task 090/100 | Success 0 | Unstable 90
Task 100/100 | Success 0 | Unstable 100

EXPERT TEACHER — FIXED EXPANDED EVALUATION
Average initial error : 0.681556
Average final error   : 999.000000
Average improvement   : -998.318444
Average best error    : 0.681556
Best final error      : 999.000000
Worst final error     : 999.000000
Successes             : 0/100
Success rate          : 0.00%
Unstable              : 100/100
